# LoD2 Height Integration

This notebook matches LoD2 building data to the classified OSM buildings and extracts usable building-height information.

**Input**
- Classified OSM buildings.
- LoD2 building data.

**Output**
- Buildings enriched with matched LoD2 height information.

In [ ]:
# ============================================================
# Germany-wide official LoD2 height enrichment
# ============================================================

from pathlib import Path
import sys
import platform
import importlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "germany_lod2"

RAW_DIR = DATA_ROOT / "raw"
EXTRACTED_DIR = DATA_ROOT / "extracted"
MATCHED_DIR = DATA_ROOT / "matched_to_osm"
REPORT_DIR = DATA_ROOT / "quality_reports"
FINAL_DIR = DATA_ROOT / "final"

DIRECTORIES = [RAW_DIR, EXTRACTED_DIR, MATCHED_DIR, REPORT_DIR, FINAL_DIR]

for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

GERMAN_STATES = {
    "BW": "Baden-Württemberg",
    "BY": "Bavaria",
    "BE": "Berlin",
    "BB": "Brandenburg",
    "HB": "Bremen",
    "HH": "Hamburg",
    "HE": "Hesse",
    "MV": "Mecklenburg-Western Pomerania",
    "NI": "Lower Saxony",
    "NW": "North Rhine-Westphalia",
    "RP": "Rhineland-Palatinate",
    "SL": "Saarland",
    "SN": "Saxony",
    "ST": "Saxony-Anhalt",
    "SH": "Schleswig-Holstein",
    "TH": "Thuringia",
}

REQUIRED_PACKAGES = ["geopandas", "shapely", "pyproj", "pyarrow", "lxml", "requests", "tqdm"]

package_status = []

for package_name in REQUIRED_PACKAGES:
    try:
        module = importlib.import_module(package_name)
        version = getattr(module, "__version__", "installed")
        package_status.append({"package": package_name, "status": "available", "version": version})
    except ImportError:
        package_status.append({"package": package_name, "status": "MISSING", "version": None})

package_status_df = pd.DataFrame(package_status)

print("=" * 70)
print("GERMANY LoD2 HEIGHT ENRICHMENT — NOTEBOOK SETUP")
print("=" * 70)
print(f"\nPython version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")
print(f"Project root   : {PROJECT_ROOT}")
print(f"Data root      : {DATA_ROOT}")
print("\nCreated directories:")

for directory in DIRECTORIES:
    print(f"  - {directory}")

print(f"\nFederal states configured: {len(GERMAN_STATES)}")
print("\nPackage status:")
display(package_status_df)

GERMANY LoD2 HEIGHT ENRICHMENT — NOTEBOOK SETUP

Python version : 3.10.20
Platform       : Linux-4.18.0-477.15.1.el8_8.x86_64-x86_64-with-glibc2.28
Project root   : /fast/home/o-olajuyigbe
Data root      : /fast/home/o-olajuyigbe/data/germany_lod2

Created directories:
  - /fast/home/o-olajuyigbe/data/germany_lod2/raw
  - /fast/home/o-olajuyigbe/data/germany_lod2/extracted
  - /fast/home/o-olajuyigbe/data/germany_lod2/matched_to_osm
  - /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports
  - /fast/home/o-olajuyigbe/data/germany_lod2/final

Federal states configured: 16

Package status:


,package,status,version
0,geopandas,available,1.1.3
1,shapely,available,2.1.2
2,pyproj,available,3.7.1
3,pyarrow,available,23.0.1
4,lxml,available,6.1.1
5,requests,available,2.32.5
6,tqdm,available,4.67.3


In [2]:
# ============================================================
# 01 — OFFICIAL LoD2 SOURCE REGISTRY
# ============================================================

source_registry = pd.DataFrame([
    {"state_code": code, "state_name": name, "portal_url": None, "download_url": None, "download_method": None, "format": "CityGML", "citygml_version": None, "horizontal_crs": None, "vertical_crs": None, "release_date": None, "licence": None, "download_status": "pending", "notes": None}
    for code, name in GERMAN_STATES.items()
])

source_registry_path = DATA_ROOT / "lod2_source_registry.csv"
source_registry.to_csv(source_registry_path, index=False)

print("=" * 70)
print("LoD2 SOURCE REGISTRY")
print("=" * 70)
print(f"\nStates included : {len(source_registry):,}")
print(f"Saved to        : {source_registry_path}")

display(source_registry)

LoD2 SOURCE REGISTRY

States included : 16
Saved to        : /fast/home/o-olajuyigbe/data/germany_lod2/lod2_source_registry.csv


,state_code,state_name,portal_url,download_url,download_method,format,citygml_version,horizontal_crs,vertical_crs,release_date,licence,download_status,notes
0,BW,Baden-Württemberg,None,None,None,CityGML,None,None,None,None,None,pending,None
1,BY,Bavaria,None,None,None,CityGML,None,None,None,None,None,pending,None
2,BE,Berlin,None,None,None,CityGML,None,None,None,None,None,pending,None
3,BB,Brandenburg,None,None,None,CityGML,None,None,None,None,None,pending,None
4,HB,Bremen,None,None,None,CityGML,None,None,None,None,None,pending,None
5,HH,Hamburg,None,None,None,CityGML,None,None,None,None,None,pending,None
6,HE,Hesse,None,None,None,CityGML,None,None,None,None,None,pending,None
7,MV,Mecklenburg-Western Pomerania,None,None,None,CityGML,None,None,None,None,None,pending,None
8,NI,Lower Saxony,None,None,None,CityGML,None,None,None,None,None,pending,None
9,NW,North Rhine-Westphalia,None,None,None,CityGML,None,None,None,None,None,pending,None


In [3]:
import requests
from tqdm.auto import tqdm
from zipfile import ZipFile, BadZipFile
from lxml import etree
from collections import Counter

/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ============================================================
# 02 — REGISTER AND DOWNLOAD BREMEN LoD2 DATA
# ============================================================



BREMEN_DOWNLOADS = {
    "Bremen": "https://gdi2.geo.bremen.de/inspire/download/LoD/data/LOD2_CITYGML_HB.zip",
    "Bremerhaven": "https://gdi2.geo.bremen.de/inspire/download/LoD/data/LOD2_CITYGML_BHV.zip"
}

hb_index = source_registry["state_code"].eq("HB")
source_registry.loc[hb_index, ["portal_url", "download_method", "format", "horizontal_crs", "release_date", "licence", "download_status", "notes"]] = [
    "https://metaver.de/trefferanzeige?docuuid=226971C2-6677-4B79-95F3-C5311F1275C8",
    "direct_zip",
    "CityGML",
    "EPSG:25832",
    "2025-04-28",
    "CC BY 4.0",
    "ready",
    "Separate CityGML downloads for Bremen and Bremerhaven"
]

source_registry.to_csv(source_registry_path, index=False)

HB_RAW_DIR = RAW_DIR / "HB"
HB_RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_file(url, output_path, chunk_size=1024 * 1024):
    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"Already downloaded: {output_path.name} ({output_path.stat().st_size / 1024**2:,.1f} MB)")
        return

    with requests.get(url, stream=True, timeout=(30, 600)) as response:
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))

        with open(output_path, "wb") as file, tqdm(total=total_size, unit="B", unit_scale=True, desc=output_path.name) as progress:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    print(f"Saved: {output_path}")

for city, url in BREMEN_DOWNLOADS.items():
    output_path = HB_RAW_DIR / f"LoD2_{city}.zip"
    download_file(url, output_path)

print("\nDownloaded Bremen files:")

for file in sorted(HB_RAW_DIR.glob("*.zip")):
    print(f"  {file.name}: {file.stat().st_size / 1024**2:,.1f} MB")

display(source_registry[source_registry["state_code"] == "HB"])

Already downloaded: LoD2_Bremen.zip (384.7 MB)
Already downloaded: LoD2_Bremerhaven.zip (74.5 MB)

Downloaded Bremen files:
  LoD2_Bremen.zip: 384.7 MB
  LoD2_Bremerhaven.zip: 74.5 MB


,state_code,state_name,portal_url,download_url,download_method,format,citygml_version,horizontal_crs,vertical_crs,release_date,licence,download_status,notes
4,HB,Bremen,https://metaver.de/trefferanzeige?docuuid=2269...,None,direct_zip,CityGML,None,EPSG:25832,None,2025-04-28,CC BY 4.0,ready,Separate CityGML downloads for Bremen and Brem...


In [5]:
# ============================================================
# 03 — INSPECT AND EXTRACT BREMEN LoD2 ZIP FILES
# ============================================================



HB_EXTRACT_DIR = HB_RAW_DIR / "extracted"
HB_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

zip_summary = []

for zip_path in sorted(HB_RAW_DIR.glob("*.zip")):
    print("\n" + "=" * 70)
    print(f"Inspecting: {zip_path.name}")
    print("=" * 70)

    try:
        with ZipFile(zip_path, "r") as archive:
            members = archive.namelist()
            gml_files = [name for name in members if name.lower().endswith((".gml", ".xml"))]
            print(f"Files in archive : {len(members):,}")
            print(f"GML/XML files    : {len(gml_files):,}")

            for name in members[:20]:
                print(f"  {name}")

            if len(members) > 20:
                print(f"  ... and {len(members) - 20:,} more files")

            extract_folder = HB_EXTRACT_DIR / zip_path.stem
            extract_folder.mkdir(parents=True, exist_ok=True)
            archive.extractall(extract_folder)

            zip_summary.append({"zip_file": zip_path.name, "total_files": len(members), "gml_files": len(gml_files), "extract_folder": str(extract_folder), "status": "extracted"})

    except BadZipFile:
        print(f"ERROR: {zip_path.name} is not a valid ZIP file.")
        zip_summary.append({"zip_file": zip_path.name, "total_files": None, "gml_files": None, "extract_folder": None, "status": "invalid_zip"})

zip_summary_df = pd.DataFrame(zip_summary)

all_gml_files = sorted(HB_EXTRACT_DIR.rglob("*.gml")) + sorted(HB_EXTRACT_DIR.rglob("*.xml"))

print("\n" + "=" * 70)
print("EXTRACTION SUMMARY")
print("=" * 70)
display(zip_summary_df)

print(f"\nTotal extracted GML/XML files: {len(all_gml_files):,}")

for file in all_gml_files[:20]:
    print(f"  {file}")

if len(all_gml_files) > 20:
    print(f"  ... and {len(all_gml_files) - 20:,} more files")


Inspecting: LoD2_Bremen.zip
Files in archive : 1
GML/XML files    : 0
  LOD2_CITYGML_HB_2026_03.zip

Inspecting: LoD2_Bremerhaven.zip
Files in archive : 1
GML/XML files    : 0
  LOD2_CITYGML_BHV_2026_03.zip

EXTRACTION SUMMARY


,zip_file,total_files,gml_files,extract_folder,status
0,LoD2_Bremen.zip,1,0,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,extracted
1,LoD2_Bremerhaven.zip,1,0,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,extracted



Total extracted GML/XML files: 143
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_466_5894_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_466_5896_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_468_5892_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_468_5894_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_468_5896_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_470_5890_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Brem

In [6]:
# ============================================================
# 04 — EXTRACT NESTED ZIP FILES
# ============================================================

def extract_nested_zips(root_dir):
    processed = set()

    while True:
        zip_files = [path for path in root_dir.rglob("*.zip") if path not in processed]

        if not zip_files:
            break

        for zip_path in zip_files:
            extract_folder = zip_path.parent / zip_path.stem
            extract_folder.mkdir(parents=True, exist_ok=True)

            try:
                with ZipFile(zip_path, "r") as archive:
                    archive.extractall(extract_folder)

                processed.add(zip_path)
                print(f"Extracted: {zip_path.name} → {extract_folder}")

            except BadZipFile:
                processed.add(zip_path)
                print(f"Invalid ZIP skipped: {zip_path}")

extract_nested_zips(HB_EXTRACT_DIR)

all_extracted_files = [path for path in HB_EXTRACT_DIR.rglob("*") if path.is_file()]
all_gml_files = sorted([path for path in all_extracted_files if path.suffix.lower() in {".gml", ".xml"}])

print("\n" + "=" * 70)
print("NESTED EXTRACTION SUMMARY")
print("=" * 70)
print(f"All extracted files : {len(all_extracted_files):,}")
print(f"GML/XML files found : {len(all_gml_files):,}")

print("\nFirst GML/XML files:")
for path in all_gml_files[:20]:
    print(f"  {path}")

if not all_gml_files:
    print("\nNo GML files found. File types currently present:")
    display(pd.Series([path.suffix.lower() or "[no extension]" for path in all_extracted_files]).value_counts().rename_axis("extension").reset_index(name="count"))

Extracted: LOD2_CITYGML_HB_2026_03.zip → /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03
Extracted: LOD2_CITYGML_BHV_2026_03.zip → /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremerhaven/LOD2_CITYGML_BHV_2026_03

NESTED EXTRACTION SUMMARY
All extracted files : 145
GML/XML files found : 143

First GML/XML files:
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_466_5894_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_466_5896_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_468_5892_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_468_5894_2_HB.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/

In [7]:
# ============================================================
# 05 — INSPECT BREMEN CITYGML STRUCTURE
# ============================================================



sample_gml_path = None
building_examples = []

for gml_path in all_gml_files:
    context = etree.iterparse(gml_path, events=("end",), huge_tree=True)

    for event, element in context:
        local_name = etree.QName(element).localname

        if local_name in {"Building", "BuildingPart"}:
            sample_gml_path = gml_path
            building_examples.append(element)

            if len(building_examples) >= 3:
                break

    if len(building_examples) >= 3:
        break

if sample_gml_path is None:
    raise ValueError("No Building or BuildingPart elements were found in the Bremen CityGML files.")

print("=" * 70)
print("CITYGML STRUCTURE INSPECTION")
print("=" * 70)
print(f"\nSample file : {sample_gml_path}")
print(f"File size   : {sample_gml_path.stat().st_size / 1024**2:,.2f} MB")

start_context = etree.iterparse(sample_gml_path, events=("start",), huge_tree=True)
event, root = next(start_context)

print("\nNamespaces:")
for prefix, uri in root.nsmap.items():
    print(f"  {prefix or 'default'}: {uri}")

del start_context

srs_names = set()
tag_counts = Counter()

for event, element in etree.iterparse(sample_gml_path, events=("start",), huge_tree=True):
    local_name = etree.QName(element).localname
    tag_counts[local_name] += 1

    if element.get("srsName"):
        srs_names.add(element.get("srsName"))

print("\nDetected coordinate reference systems:")
for srs_name in sorted(srs_names):
    print(f"  {srs_name}")

print("\nMost frequent tags:")
for tag, count in tag_counts.most_common(30):
    print(f"  {tag:<35} {count:>10,}")

selected_attributes = {"measuredHeight", "function", "class", "usage", "roofType", "storeysAboveGround", "storeysBelowGround", "yearOfConstruction"}

for number, building in enumerate(building_examples, start=1):
    print("\n" + "=" * 70)
    print(f"BUILDING EXAMPLE {number}")
    print("=" * 70)

    building_id = building.get("{http://www.opengis.net/gml}id")
    print(f"gml:id: {building_id}")

    attributes = {}

    for descendant in building.iter():
        local_name = etree.QName(descendant).localname
        text = descendant.text.strip() if descendant.text and descendant.text.strip() else None

        if text and local_name in selected_attributes:
            attributes.setdefault(local_name, []).append(text)

    print("\nSelected attributes:")

    if attributes:
        for name, values in attributes.items():
            print(f"  {name:<25}: {sorted(set(values))}")
    else:
        print("  No selected attributes found.")

    geometry_counts = Counter(etree.QName(descendant).localname for descendant in building.iter())

    print("\nGeometry-related tags:")
    for tag in ["lod0FootPrint", "lod0RoofEdge", "lod1Solid", "lod2Solid", "boundedBy", "GroundSurface", "WallSurface", "RoofSurface", "Polygon", "LinearRing", "posList", "pos"]:
        if geometry_counts[tag]:
            print(f"  {tag:<25}: {geometry_counts[tag]:,}")

    print("\nDirect child tags:")
    for child in building:
        print(f"  {etree.QName(child).localname}")

CITYGML STRUCTURE INSPECTION

Sample file : /fast/home/o-olajuyigbe/data/germany_lod2/raw/HB/extracted/LoD2_Bremen/LOD2_CITYGML_HB_2026_03/LoD2_Bremen_Citygml/LoD2_32_466_5894_2_HB.gml
File size   : 19.70 MB

Namespaces:
  bldg: http://www.opengis.net/citygml/building/1.0
  xAL: urn:oasis:names:tc:ciq:xsdschema:xAL:2.0
  gml: http://www.opengis.net/gml
  app: http://www.opengis.net/citygml/appearance/1.0
  grp: http://www.opengis.net/citygml/cityobjectgroup/1.0
  core: http://www.opengis.net/citygml/1.0
  gen: http://www.opengis.net/citygml/generics/1.0
  xlink: http://www.w3.org/1999/xlink
  xsi: http://www.w3.org/2001/XMLSchema-instance

Detected coordinate reference systems:
  urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH

Most frequent tags:
  surfaceMember                           33,196
  exterior                                17,882
  LinearRing                              16,629
  posList                                 16,629
  Polygon                                 16,598
  bou

In [8]:
# ============================================================
# 06 — EXTRACT BUILDINGS FROM ONE CITYGML TILE
# ============================================================

import geopandas as gpd
from shapely.geometry import Polygon
from shapely.ops import unary_union

GML_ID = "{http://www.opengis.net/gml}id"

def get_first_text(element, tag_name):
    for child in element.iter():
        if etree.QName(child).localname == tag_name and child.text:
            return child.text.strip()
    return None

def parse_number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

def extract_ground_geometry(building):
    polygons = []

    for surface in building.iter():
        if etree.QName(surface).localname != "GroundSurface":
            continue

        for pos_list in surface.iter():
            if etree.QName(pos_list).localname != "posList" or not pos_list.text:
                continue

            values = np.fromstring(pos_list.text, sep=" ")
            dimension = int(pos_list.get("srsDimension", 3))

            if len(values) % dimension != 0:
                dimension = 3 if len(values) % 3 == 0 else 2

            coordinates = values.reshape(-1, dimension)[:, :2]

            if len(coordinates) < 3:
                continue

            if not np.array_equal(coordinates[0], coordinates[-1]):
                coordinates = np.vstack([coordinates, coordinates[0]])

            polygon = Polygon(coordinates)

            if not polygon.is_empty and polygon.area > 0:
                polygons.append(polygon)

    if not polygons:
        return None

    return unary_union(polygons)

def extract_citygml_tile(gml_path):
    records = []

    for event, building in etree.iterparse(gml_path, events=("end",), huge_tree=True):
        if etree.QName(building).localname != "Building":
            continue

        geometry = extract_ground_geometry(building)

        records.append({
            "lod2_id": building.get(GML_ID),
            "creation_date": get_first_text(building, "creationDate"),
            "function": get_first_text(building, "function"),
            "roof_type": get_first_text(building, "roofType"),
            "measured_height_m": parse_number(get_first_text(building, "measuredHeight")),
            "storeys_above_ground": parse_number(get_first_text(building, "storeysAboveGround")),
            "source_file": gml_path.name,
            "geometry": geometry
        })

        building.clear()

        while building.getprevious() is not None:
            del building.getparent()[0]

    return gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:25832")

sample_gdf = extract_citygml_tile(sample_gml_path)

print("=" * 70)
print("SAMPLE TILE EXTRACTION")
print("=" * 70)
print(f"\nSource file             : {sample_gml_path.name}")
print(f"Buildings extracted     : {len(sample_gdf):,}")
print(f"Valid geometries        : {sample_gdf.geometry.notna().sum():,}")
print(f"Measured heights        : {sample_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights         : {sample_gdf['measured_height_m'].isna().sum():,}")
print(f"CRS                     : {sample_gdf.crs}")

display(sample_gdf.head())
display(sample_gdf[["measured_height_m", "storeys_above_ground"]].describe())

SAMPLE TILE EXTRACTION

Source file             : LoD2_32_466_5894_2_HB.gml
Buildings extracted     : 1,284
Valid geometries        : 1,284
Measured heights        : 1,284
Missing heights         : 0
CRS                     : EPSG:25832


,lod2_id,creation_date,function,roof_type,measured_height_m,storeys_above_ground,source_file,geometry
0,DEHB01ALg0001fk2,2026-03-09,31001_2463,3100,4.376,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467456.09 5895994.972, 467458.609 58..."
1,DEHB01ALg0001gqP,2026-03-09,31001_2463,1000,2.869,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467193.743 5895923.153, 467193.84 58..."
2,DEHB01ALg0001eWT,2026-03-09,31001_2463,3100,4.702,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467368.543 5895969.863, 467369.876 5..."
3,DEHB01ALg0001f7b,2026-03-09,31001_2463,1000,2.919,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467408.469 5895811.288, 467412.879 5..."
4,DEHB01ALg0001goo,2026-03-09,31001_2463,3100,5.871,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467336.853 5895635.147, 467342.728 5..."


,measured_height_m,storeys_above_ground
count,1284.000000,730.000000
mean,6.850963,1.076712
std,8.695963,0.286179
min,1.000000,1.000000
25%,2.992750,1.000000
50%,5.044000,1.000000
75%,8.526750,1.000000
max,149.255000,3.000000


In [9]:
# ============================================================
# 07 — EXTRACT ALL BREMEN LoD2 FILES
# ============================================================

bremen_parts = []
failed_files = []

for gml_path in tqdm(all_gml_files, desc="Extracting Bremen LoD2 files"):
    try:
        tile_gdf = extract_citygml_tile(gml_path)
        tile_gdf["state_code"] = "HB"
        tile_gdf["source_city"] = "Bremerhaven" if "BHV" in gml_path.name else "Bremen"
        bremen_parts.append(tile_gdf)
    except Exception as error:
        failed_files.append({"source_file": str(gml_path), "error": str(error)})

if not bremen_parts:
    raise RuntimeError("No Bremen LoD2 files were successfully extracted.")

gdf_lod2_hb = gpd.GeoDataFrame(pd.concat(bremen_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
failed_files_df = pd.DataFrame(failed_files)

print("=" * 70)
print("BREMEN LoD2 EXTRACTION SUMMARY")
print("=" * 70)
print(f"\nFiles processed       : {len(all_gml_files):,}")
print(f"Files successful      : {len(bremen_parts):,}")
print(f"Files failed          : {len(failed_files):,}")
print(f"Buildings extracted   : {len(gdf_lod2_hb):,}")
print(f"Unique LoD2 IDs       : {gdf_lod2_hb['lod2_id'].nunique():,}")
print(f"Valid geometries      : {gdf_lod2_hb.geometry.notna().sum():,}")
print(f"Measured heights      : {gdf_lod2_hb['measured_height_m'].notna().sum():,}")
print(f"Missing heights       : {gdf_lod2_hb['measured_height_m'].isna().sum():,}")
print(f"Duplicate LoD2 IDs    : {gdf_lod2_hb['lod2_id'].duplicated().sum():,}")

display(gdf_lod2_hb.head())
display(gdf_lod2_hb.groupby("source_city").size().rename("building_count").reset_index())

if not failed_files_df.empty:
    display(failed_files_df)

Extracting Bremen LoD2 files: 100%|██████████████████████████████████████████████| 143/143 [03:19<00:00,  1.40s/it]


BREMEN LoD2 EXTRACTION SUMMARY

Files processed       : 143
Files successful      : 143
Files failed          : 0
Buildings extracted   : 331,804
Unique LoD2 IDs       : 331,804
Valid geometries      : 331,804
Measured heights      : 331,804
Missing heights       : 0
Duplicate LoD2 IDs    : 0


,lod2_id,creation_date,function,roof_type,measured_height_m,storeys_above_ground,source_file,geometry,state_code,source_city
0,DEHB01ALg0001fk2,2026-03-09,31001_2463,3100,4.376,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467456.09 5895994.972, 467458.609 58...",HB,Bremen
1,DEHB01ALg0001gqP,2026-03-09,31001_2463,1000,2.869,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467193.743 5895923.153, 467193.84 58...",HB,Bremen
2,DEHB01ALg0001eWT,2026-03-09,31001_2463,3100,4.702,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467368.543 5895969.863, 467369.876 5...",HB,Bremen
3,DEHB01ALg0001f7b,2026-03-09,31001_2463,1000,2.919,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467408.469 5895811.288, 467412.879 5...",HB,Bremen
4,DEHB01ALg0001goo,2026-03-09,31001_2463,3100,5.871,1.0,LoD2_32_466_5894_2_HB.gml,"POLYGON ((467336.853 5895635.147, 467342.728 5...",HB,Bremen


,source_city,building_count
0,Bremen,331804


In [10]:
# ============================================================
# 08 — CORRECT CITY LABELS AND SAVE BREMEN LoD2
# ============================================================

file_city_lookup = {path.name: ("Bremerhaven" if "Bremerhaven" in str(path) else "Bremen") for path in all_gml_files}
duplicate_filenames = pd.Series([path.name for path in all_gml_files]).duplicated().sum()

print(f"Duplicate GML filenames across folders: {duplicate_filenames:,}")

if duplicate_filenames > 0:
    raise ValueError("Some Bremen and Bremerhaven files have identical filenames. We need to use full source paths instead.")

gdf_lod2_hb["source_city"] = gdf_lod2_hb["source_file"].map(file_city_lookup)

print("\nBuilding counts after correction:")
display(gdf_lod2_hb.groupby("source_city", dropna=False).size().rename("building_count").reset_index())

print(f"\nUnassigned city labels: {gdf_lod2_hb['source_city'].isna().sum():,}")

HB_OUTPUT_DIR = EXTRACTED_DIR / "HB"
HB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

hb_output_path = HB_OUTPUT_DIR / "lod2_buildings_HB.parquet"
gdf_lod2_hb.to_parquet(hb_output_path, index=False)

print(f"\nSaved to: {hb_output_path}")
print(f"File size: {hb_output_path.stat().st_size / 1024**2:,.1f} MB")

Duplicate GML filenames across folders: 0

Building counts after correction:


,source_city,building_count
0,Bremen,271459
1,Bremerhaven,60345



Unassigned city labels: 0

Saved to: /fast/home/o-olajuyigbe/data/germany_lod2/extracted/HB/lod2_buildings_HB.parquet
File size: 35.7 MB


In [11]:
# ============================================================
# 09 — VALIDATE BREMEN LoD2 HEIGHTS
# ============================================================

gdf_lod2_hb["height_valid"] = gdf_lod2_hb["measured_height_m"].between(1, 300)
gdf_lod2_hb["geometry_valid"] = gdf_lod2_hb.geometry.is_valid
gdf_lod2_hb["footprint_area_m2"] = gdf_lod2_hb.geometry.area

validation_summary = pd.DataFrame({
    "check": [
        "Total buildings",
        "Missing heights",
        "Heights below 1 m",
        "Heights above 300 m",
        "Valid heights",
        "Invalid geometries",
        "Empty geometries",
        "Duplicate LoD2 IDs",
        "Footprints below 1 m²"
    ],
    "count": [
        len(gdf_lod2_hb),
        gdf_lod2_hb["measured_height_m"].isna().sum(),
        (gdf_lod2_hb["measured_height_m"] < 1).sum(),
        (gdf_lod2_hb["measured_height_m"] > 300).sum(),
        gdf_lod2_hb["height_valid"].sum(),
        (~gdf_lod2_hb["geometry_valid"]).sum(),
        gdf_lod2_hb.geometry.is_empty.sum(),
        gdf_lod2_hb["lod2_id"].duplicated().sum(),
        (gdf_lod2_hb["footprint_area_m2"] < 1).sum()
    ]
})

print("=" * 70)
print("BREMEN LoD2 VALIDATION SUMMARY")
print("=" * 70)
display(validation_summary)

print("\nHeight distribution:")
display(gdf_lod2_hb["measured_height_m"].describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).to_frame())

print("\nTallest 20 buildings:")
display(gdf_lod2_hb.nlargest(20, "measured_height_m")[["lod2_id", "source_city", "function", "measured_height_m", "storeys_above_ground", "footprint_area_m2"]])

print("\nLargest 20 footprints:")
display(gdf_lod2_hb.nlargest(20, "footprint_area_m2")[["lod2_id", "source_city", "function", "measured_height_m", "footprint_area_m2"]])

BREMEN LoD2 VALIDATION SUMMARY


,check,count
0,Total buildings,331804
1,Missing heights,0
2,Heights below 1 m,15
3,Heights above 300 m,0
4,Valid heights,331789
5,Invalid geometries,0
6,Empty geometries,0
7,Duplicate LoD2 IDs,0
8,Footprints below 1 m²,697



Height distribution:


,measured_height_m
count,331804.000000
mean,6.787625
std,5.071337
min,0.010000
1%,1.696000
5%,2.244000
25%,2.801000
50%,5.424000
75%,9.630000
95%,15.011850



Tallest 20 buildings:


,lod2_id,source_city,function,measured_height_m,storeys_above_ground,footprint_area_m2
56737,DEHB01ALt00012fY,Bremen,51002_1290,249.020,NaN,237.883862
42829,DEHB01ALt0000xbU,Bremen,31001_2521,197.420,NaN,52.444453
132262,DEHB01ALU000055w,Bremen,51001_1008,176.160,NaN,131.271833
275856,DEHB01ALya00009y,Bremerhaven,51002_1220,152.490,NaN,137.095254
527,DEHB01ALg0001orV,Bremen,51009_1610,149.255,NaN,29.133088
202515,DEHB01ALk00017Pr,Bremen,31001_3024,144.932,2.0,2058.031005
1232,DEHB01ALoA0000O0,Bremen,51002_1250,133.640,NaN,363.441660
321290,DEHB01AL1yX0001u,Bremerhaven,51002_1220,132.030,NaN,114.844077
320008,DEHB01AL1uN0009t,Bremerhaven,51002_1220,130.610,NaN,99.464335
321291,DEHB01AL1yX0001v,Bremerhaven,51002_1220,130.230,NaN,114.844077



Largest 20 footprints:


,lod2_id,source_city,function,measured_height_m,footprint_area_m2
43545,DEHB01ALt0000xzQ,Bremen,31001_2100,27.398,125999.701694
43120,DEHB01ALt0000wVQ,Bremen,31001_2100,48.318,102255.331053
78281,DEHB01ALt0000xM0,Bremen,31001_2010,20.514,89176.639728
212939,DEHB01ALu0001Zcy,Bremen,31001_2100,32.780,85510.865434
55565,DEHB01AL4hQ0005R,Bremen,31001_2100,14.347,83580.424990
212971,DEHB01ALu0001cP0,Bremen,31001_2100,28.564,80769.795696
213235,DEHB01ALu0001bBF,Bremen,31001_2100,32.509,75426.239530
260520,DEHB01ALs0001Tb0,Bremen,31001_2010,16.016,72604.755168
75472,DEHB01ALS00003aL,Bremen,31001_2143,43.480,70766.231608
39857,DEHB01ALj0001mFj,Bremen,31001_2100,27.974,61179.878564


In [12]:
# ============================================================
# 10 — SAVE RAW STANDARDISED BREMEN LoD2 DATA
# ============================================================

HB_OUTPUT_DIR = EXTRACTED_DIR / "HB"
HB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gdf_lod2_hb["height_source"] = "lod2_measured"
gdf_lod2_hb["source_state"] = "Bremen"
gdf_lod2_hb["source_crs"] = "EPSG:25832"
gdf_lod2_hb["citygml_version"] = "1.0"

hb_output_path = HB_OUTPUT_DIR / "lod2_buildings_HB_raw.parquet"
gdf_lod2_hb.to_parquet(hb_output_path, index=False)

hb_summary = pd.DataFrame([{
    "state_code": "HB",
    "state_name": "Bremen",
    "gml_files": len(all_gml_files),
    "buildings": len(gdf_lod2_hb),
    "unique_lod2_ids": gdf_lod2_hb["lod2_id"].nunique(),
    "valid_geometries": gdf_lod2_hb.geometry.notna().sum(),
    "measured_heights": gdf_lod2_hb["measured_height_m"].notna().sum(),
    "missing_heights": gdf_lod2_hb["measured_height_m"].isna().sum(),
    "minimum_height_m": gdf_lod2_hb["measured_height_m"].min(),
    "maximum_height_m": gdf_lod2_hb["measured_height_m"].max(),
    "output_file": str(hb_output_path)
}])

hb_summary_path = REPORT_DIR / "lod2_extraction_summary_HB.csv"
hb_summary.to_csv(hb_summary_path, index=False)

print("=" * 70)
print("BREMEN RAW LoD2 DATA SAVED")
print("=" * 70)
print(f"\nBuildings saved : {len(gdf_lod2_hb):,}")
print(f"Output file     : {hb_output_path}")
print(f"File size       : {hb_output_path.stat().st_size / 1024**2:,.1f} MB")
print(f"Summary file    : {hb_summary_path}")

display(hb_summary)

BREMEN RAW LoD2 DATA SAVED

Buildings saved : 331,804
Output file     : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/HB/lod2_buildings_HB_raw.parquet
File size       : 38.5 MB
Summary file    : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_HB.csv


,state_code,state_name,gml_files,buildings,unique_lod2_ids,valid_geometries,measured_heights,missing_heights,minimum_height_m,maximum_height_m,output_file
0,HB,Bremen,143,331804,331804,331804,331804,0,0.01,249.02,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


In [13]:
# ============================================================
# 11 — REUSABLE STATE LoD2 PROCESSING FUNCTION
# ============================================================

def find_citygml_files(folder):
    return sorted([path for path in Path(folder).rglob("*") if path.is_file() and path.suffix.lower() in {".gml", ".xml"}])

def process_lod2_state(state_code, state_name, input_folder, source_crs, citygml_version=None):
    gml_files = find_citygml_files(input_folder)

    if not gml_files:
        raise FileNotFoundError(f"No GML/XML files found for {state_code} inside {input_folder}")

    state_parts = []
    failed_files = []

    print("=" * 70)
    print(f"PROCESSING {state_name.upper()} ({state_code})")
    print("=" * 70)
    print(f"\nGML/XML files found: {len(gml_files):,}")

    for gml_path in tqdm(gml_files, desc=f"Extracting {state_code}"):
        try:
            tile_gdf = extract_citygml_tile(gml_path)
            tile_gdf = tile_gdf.set_crs(source_crs, allow_override=True)
            tile_gdf["state_code"] = state_code
            tile_gdf["source_state"] = state_name
            tile_gdf["source_crs"] = source_crs
            tile_gdf["citygml_version"] = citygml_version
            tile_gdf["height_source"] = "lod2_measured"
            tile_gdf["source_path"] = str(gml_path)
            state_parts.append(tile_gdf)
        except Exception as error:
            failed_files.append({"state_code": state_code, "source_file": str(gml_path), "error": str(error)})

    if not state_parts:
        raise RuntimeError(f"No files were successfully processed for {state_code}")

    state_gdf = gpd.GeoDataFrame(pd.concat(state_parts, ignore_index=True), geometry="geometry", crs=source_crs)
    failed_df = pd.DataFrame(failed_files)

    output_dir = EXTRACTED_DIR / state_code
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / f"lod2_buildings_{state_code}_raw.parquet"
    state_gdf.to_parquet(output_path, index=False)

    summary = pd.DataFrame([{
        "state_code": state_code,
        "state_name": state_name,
        "gml_files": len(gml_files),
        "successful_files": len(state_parts),
        "failed_files": len(failed_files),
        "buildings": len(state_gdf),
        "unique_lod2_ids": state_gdf["lod2_id"].nunique(),
        "valid_geometries": state_gdf.geometry.notna().sum(),
        "measured_heights": state_gdf["measured_height_m"].notna().sum(),
        "missing_heights": state_gdf["measured_height_m"].isna().sum(),
        "minimum_height_m": state_gdf["measured_height_m"].min(),
        "maximum_height_m": state_gdf["measured_height_m"].max(),
        "output_file": str(output_path)
    }])

    summary.to_csv(REPORT_DIR / f"lod2_extraction_summary_{state_code}.csv", index=False)

    if not failed_df.empty:
        failed_df.to_csv(REPORT_DIR / f"lod2_failed_files_{state_code}.csv", index=False)

    print(f"\nBuildings extracted : {len(state_gdf):,}")
    print(f"Files failed        : {len(failed_files):,}")
    print(f"Missing heights     : {state_gdf['measured_height_m'].isna().sum():,}")
    print(f"Saved to            : {output_path}")

    return state_gdf, summary, failed_df

In [14]:
# ============================================================
# 12 — DOWNLOAD AND EXTRACT HAMBURG LoD2
# ============================================================

HH_RAW_DIR = RAW_DIR / "HH"
HH_RAW_DIR.mkdir(parents=True, exist_ok=True)

HH_DOWNLOAD_URL = "https://daten-hamburg.de/opendata/3d_stadtmodell_lod2/LoD2-DE_HH_2026-04-28.zip"
HH_ZIP_PATH = HH_RAW_DIR / "LoD2-DE_HH_2026-04-28.zip"

hh_index = source_registry["state_code"].eq("HH")
source_registry.loc[hh_index, ["portal_url", "download_url", "download_method", "format", "citygml_version", "release_date", "licence", "download_status", "notes"]] = [
    "https://suche.transparenz.hamburg.de/dataset/3d-gebaeudemodell-lod2-de-hamburg2",
    HH_DOWNLOAD_URL,
    "direct_zip",
    "CityGML",
    "1.0",
    "2026-04-28",
    "Datenlizenz Deutschland Namensnennung 2.0",
    "ready",
    "Complete Hamburg dataset including Neuwerk; CRS to be confirmed from CityGML"
]

source_registry.to_csv(source_registry_path, index=False)

download_file(HH_DOWNLOAD_URL, HH_ZIP_PATH)

HH_EXTRACT_DIR = HH_RAW_DIR / "extracted"
HH_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with ZipFile(HH_ZIP_PATH, "r") as archive:
    archive.extractall(HH_EXTRACT_DIR)

extract_nested_zips(HH_EXTRACT_DIR)

hh_gml_files = find_citygml_files(HH_EXTRACT_DIR)

print("=" * 70)
print("HAMBURG DOWNLOAD AND EXTRACTION")
print("=" * 70)
print(f"\nZIP file size       : {HH_ZIP_PATH.stat().st_size / 1024**2:,.1f} MB")
print(f"GML/XML files found : {len(hh_gml_files):,}")
print(f"Extraction folder   : {HH_EXTRACT_DIR}")

print("\nFirst files:")
for path in hh_gml_files[:20]:
    print(f"  {path}")

if not hh_gml_files:
    raise FileNotFoundError("No Hamburg GML/XML files were found after extraction.")

Already downloaded: LoD2-DE_HH_2026-04-28.zip (629.0 MB)
HAMBURG DOWNLOAD AND EXTRACTION

ZIP file size       : 629.0 MB
GML/XML files found : 783
Extraction folder   : /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted

First files:
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_466_5974_1_HH.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_467_5974_1_HH.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_467_5975_1_HH.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_548_5935_1_HH.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_548_5936_1_HH.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_548_5937_1_HH.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_549_5935_1_HH.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_549_5936_1_HH.gml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/L

In [15]:
# ============================================================
# 13 — INSPECT ONE HAMBURG CITYGML FILE
# ============================================================

hh_sample_path = hh_gml_files[0]

print("=" * 70)
print("HAMBURG CITYGML INSPECTION")
print("=" * 70)
print(f"\nSample file : {hh_sample_path}")
print(f"File size   : {hh_sample_path.stat().st_size / 1024**2:,.2f} MB")

context = etree.iterparse(hh_sample_path, events=("start",), huge_tree=True)
event, root = next(context)

print("\nNamespaces:")
for prefix, uri in root.nsmap.items():
    print(f"  {prefix or 'default'}: {uri}")

del context

srs_names = set()
building_count = 0
measured_height_count = 0
function_count = 0
roof_type_count = 0
storeys_count = 0

for event, element in etree.iterparse(hh_sample_path, events=("end",), huge_tree=True):
    local_name = etree.QName(element).localname

    if element.get("srsName"):
        srs_names.add(element.get("srsName"))

    if local_name == "Building":
        building_count += 1

        if get_first_text(element, "measuredHeight") is not None:
            measured_height_count += 1

        if get_first_text(element, "function") is not None:
            function_count += 1

        if get_first_text(element, "roofType") is not None:
            roof_type_count += 1

        if get_first_text(element, "storeysAboveGround") is not None:
            storeys_count += 1

        element.clear()

        while element.getprevious() is not None:
            del element.getparent()[0]

print("\nDetected CRS:")
for srs_name in sorted(srs_names):
    print(f"  {srs_name}")

print("\nAttribute availability:")
print(f"  Buildings              : {building_count:,}")
print(f"  Measured height        : {measured_height_count:,}")
print(f"  Function               : {function_count:,}")
print(f"  Roof type              : {roof_type_count:,}")
print(f"  Storeys above ground   : {storeys_count:,}")

HAMBURG CITYGML INSPECTION

Sample file : /fast/home/o-olajuyigbe/data/germany_lod2/raw/HH/extracted/LoD2_32_466_5974_1_HH.gml
File size   : 0.86 MB

Namespaces:
  bldg: http://www.opengis.net/citygml/building/1.0
  xAL: urn:oasis:names:tc:ciq:xsdschema:xAL:2.0
  gml: http://www.opengis.net/gml
  app: http://www.opengis.net/citygml/appearance/1.0
  grp: http://www.opengis.net/citygml/cityobjectgroup/1.0
  core: http://www.opengis.net/citygml/1.0
  gen: http://www.opengis.net/citygml/generics/1.0
  xlink: http://www.w3.org/1999/xlink
  xsi: http://www.w3.org/2001/XMLSchema-instance

Detected CRS:
  urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH

Attribute availability:
  Buildings              : 56
  Measured height        : 56
  Function               : 56
  Roof type              : 56
  Storeys above ground   : 56


In [16]:
# ============================================================
# 14 — PROCESS ALL HAMBURG LoD2 FILES
# ============================================================

gdf_lod2_hh, hh_summary, hh_failed = process_lod2_state("HH", "Hamburg", HH_EXTRACT_DIR, "EPSG:25832", "1.0")

display(hh_summary)

if not hh_failed.empty:
    display(hh_failed.head(20))

PROCESSING HAMBURG (HH)

GML/XML files found: 783


Extracting HH:   0%|                                                                       | 0/783 [00:00<?, ?it/s]

Extracting HH: 100%|█████████████████████████████████████████████████████████████| 783/783 [03:34<00:00,  3.64it/s]



Buildings extracted : 388,729
Files failed        : 0
Missing heights     : 0
Saved to            : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/HH/lod2_buildings_HH_raw.parquet


,state_code,state_name,gml_files,successful_files,failed_files,buildings,unique_lod2_ids,valid_geometries,measured_heights,missing_heights,minimum_height_m,maximum_height_m,output_file
0,HH,Hamburg,783,783,0,388729,388729,388729,388729,0,0.737,158.47,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


In [17]:
from urllib.parse import urljoin


In [18]:
# ============================================================
# 15 — DISCOVER BERLIN LoD2 DOWNLOAD FILES
# ============================================================

from urllib.parse import urljoin

BE_RAW_DIR = RAW_DIR / "BE"
BE_RAW_DIR.mkdir(parents=True, exist_ok=True)

BE_PORTAL_URL = "https://daten.berlin.de/datensaetze/3d-gebaudemodelle-im-level-of-detail-2-lod-2-3c7c49af"
BE_ATOM_URL = "https://gdi.berlin.de/data/a_lod2/atom/0.atom"

be_index = source_registry["state_code"].eq("BE")
source_registry.loc[be_index, ["portal_url", "download_url", "download_method", "format", "licence", "download_status", "notes"]] = [BE_PORTAL_URL, BE_ATOM_URL, "ATOM", "CityGML", "Datenlizenz Deutschland Zero 2.0", "ready", "Statewide Berlin LoD2; download links discovered recursively from ATOM feeds"]
source_registry.to_csv(source_registry_path, index=False)

def discover_atom_downloads(start_url):
    pending = [start_url]
    visited = set()
    downloads = []

    while pending:
        feed_url = pending.pop()

        if feed_url in visited:
            continue

        visited.add(feed_url)
        response = requests.get(feed_url, timeout=(30, 120))
        response.raise_for_status()
        root = etree.fromstring(response.content)

        for link in root.xpath("//*[local-name()='link']"):
            href = link.get("href")

            if not href:
                continue

            link_url = urljoin(feed_url, href)
            link_type = (link.get("type") or "").lower()
            clean_url = link_url.lower().split("?")[0]

            if clean_url.endswith(".atom") or "application/atom+xml" in link_type:
                if link_url not in visited:
                    pending.append(link_url)

            elif clean_url.endswith((".zip", ".gml", ".xml")):
                downloads.append({"download_url": link_url, "filename": Path(clean_url).name, "source_feed": feed_url})

    return pd.DataFrame(downloads).drop_duplicates(subset="download_url").reset_index(drop=True)

be_downloads = discover_atom_downloads(BE_ATOM_URL)

print("=" * 70)
print("BERLIN LoD2 DOWNLOAD DISCOVERY")
print("=" * 70)
print(f"\nATOM feeds visited : {be_downloads['source_feed'].nunique() if not be_downloads.empty else 0:,}")
print(f"Downloads found    : {len(be_downloads):,}")

if be_downloads.empty:
    raise RuntimeError("No Berlin LoD2 download files were found in the ATOM service.")

display(be_downloads.head(20))

be_download_list_path = BE_RAW_DIR / "berlin_download_list.csv"
be_downloads.to_csv(be_download_list_path, index=False)

print(f"\nDownload list saved to: {be_download_list_path}")

BERLIN LoD2 DOWNLOAD DISCOVERY

ATOM feeds visited : 1
Downloads found    : 925


,download_url,filename,source_feed
0,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_371_5809.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
1,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_371_5812.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
2,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_371_5813.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
3,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_371_5814.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
4,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_372_5805.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
5,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_372_5806.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
6,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_372_5807.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
7,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_372_5808.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
8,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_372_5809.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom
9,https://gdi.berlin.de/data/a_lod2/atom/LoD2_37...,lod2_372_5810.zip,https://gdi.berlin.de/data/a_lod2/atom/0.atom



Download list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/berlin_download_list.csv


In [19]:
# ============================================================
# 16 — DOWNLOAD AND EXTRACT BERLIN LoD2 TILES
# ============================================================

BE_ZIP_DIR = BE_RAW_DIR / "zips"
BE_EXTRACT_DIR = BE_RAW_DIR / "extracted"
BE_ZIP_DIR.mkdir(parents=True, exist_ok=True)
BE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Berlin tile downloads found: {len(be_downloads):,}")

download_errors = []

for row in tqdm(be_downloads.itertuples(index=False), total=len(be_downloads), desc="Downloading Berlin tiles"):
    zip_path = BE_ZIP_DIR / row.filename

    try:
        if not zip_path.exists() or zip_path.stat().st_size == 0:
            response = requests.get(row.download_url, stream=True, timeout=(30, 600))
            response.raise_for_status()

            with open(zip_path, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)

    except Exception as error:
        download_errors.append({"filename": row.filename, "download_url": row.download_url, "error": str(error)})

download_errors_df = pd.DataFrame(download_errors)

print(f"\nTiles downloaded successfully: {len(be_downloads) - len(download_errors):,}")
print(f"Download failures            : {len(download_errors):,}")

if not download_errors_df.empty:
    display(download_errors_df.head(20))
    download_errors_df.to_csv(REPORT_DIR / "lod2_download_errors_BE.csv", index=False)

extract_errors = []

for zip_path in tqdm(sorted(BE_ZIP_DIR.glob("*.zip")), desc="Extracting Berlin tiles"):
    extract_folder = BE_EXTRACT_DIR / zip_path.stem
    extract_folder.mkdir(parents=True, exist_ok=True)

    try:
        if not any(extract_folder.iterdir()):
            with ZipFile(zip_path, "r") as archive:
                archive.extractall(extract_folder)
    except Exception as error:
        extract_errors.append({"zip_file": str(zip_path), "error": str(error)})

extract_nested_zips(BE_EXTRACT_DIR)

be_gml_files = find_citygml_files(BE_EXTRACT_DIR)
extract_errors_df = pd.DataFrame(extract_errors)

print("\n" + "=" * 70)
print("BERLIN DOWNLOAD AND EXTRACTION SUMMARY")
print("=" * 70)
print(f"ZIP files present    : {len(list(BE_ZIP_DIR.glob('*.zip'))):,}")
print(f"GML/XML files found  : {len(be_gml_files):,}")
print(f"Extraction failures  : {len(extract_errors):,}")

if not extract_errors_df.empty:
    display(extract_errors_df.head(20))
    extract_errors_df.to_csv(REPORT_DIR / "lod2_extraction_errors_BE.csv", index=False)

print("\nFirst Berlin CityGML files:")
for path in be_gml_files[:10]:
    print(f"  {path}")

Berlin tile downloads found: 925



Tiles downloaded successfully: 925
Download failures            : 0


Extracting Berlin tiles: 100%|█████████████████████████████████████████████████| 925/925 [00:00<00:00, 1512.52it/s]



BERLIN DOWNLOAD AND EXTRACTION SUMMARY
ZIP files present    : 925
GML/XML files found  : 925
Extraction failures  : 0

First Berlin CityGML files:
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/extracted/lod2_371_5809/LoD2_33_371_5809_1_BE.xml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/extracted/lod2_371_5812/LoD2_33_371_5812_1_BE.xml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/extracted/lod2_371_5813/LoD2_33_371_5813_1_BE.xml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/extracted/lod2_371_5814/LoD2_33_371_5814_1_BE.xml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/extracted/lod2_372_5805/LoD2_33_372_5805_1_BE.xml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/extracted/lod2_372_5806/LoD2_33_372_5806_1_BE.xml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/extracted/lod2_372_5807/LoD2_33_372_5807_1_BE.xml
  /fast/home/o-olajuyigbe/data/germany_lod2/raw/BE/extracted/lod2_372_5808/LoD2_33_372_5808_1_BE.xml
  /fast/home/o-olajuyigbe/data/germany_lod2/

In [20]:
import time


In [21]:
# ============================================================
# 17 — RETRY FAILED BERLIN TILE AND INSPECT CRS
# ============================================================


if not download_errors_df.empty:
    for row in download_errors_df.itertuples(index=False):
        zip_path = BE_ZIP_DIR / row.filename

        for attempt in range(1, 4):
            try:
                response = requests.get(row.download_url, stream=True, timeout=(30, 600))
                response.raise_for_status()

                with open(zip_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            file.write(chunk)

                print(f"Downloaded successfully on attempt {attempt}: {row.filename}")

                extract_folder = BE_EXTRACT_DIR / zip_path.stem
                extract_folder.mkdir(parents=True, exist_ok=True)

                with ZipFile(zip_path, "r") as archive:
                    archive.extractall(extract_folder)

                break

            except Exception as error:
                print(f"Attempt {attempt} failed for {row.filename}: {error}")
                time.sleep(5)

be_gml_files = find_citygml_files(BE_EXTRACT_DIR)

print(f"\nBerlin ZIP files present : {len(list(BE_ZIP_DIR.glob('*.zip'))):,}")
print(f"Berlin GML/XML files     : {len(be_gml_files):,}")

be_sample_path = be_gml_files[0]
srs_names = set()
building_count = 0
measured_height_count = 0

for event, element in etree.iterparse(be_sample_path, events=("end",), huge_tree=True):
    if element.get("srsName"):
        srs_names.add(element.get("srsName"))

    if etree.QName(element).localname == "Building":
        building_count += 1

        if get_first_text(element, "measuredHeight") is not None:
            measured_height_count += 1

        element.clear()

        while element.getprevious() is not None:
            del element.getparent()[0]

print(f"\nSample file       : {be_sample_path.name}")
print(f"Buildings         : {building_count:,}")
print(f"Measured heights  : {measured_height_count:,}")
print("Detected CRS:")

for srs_name in sorted(srs_names):
    print(f"  {srs_name}")


Berlin ZIP files present : 925
Berlin GML/XML files     : 925

Sample file       : LoD2_33_371_5809_1_BE.xml
Buildings         : 22
Measured heights  : 22
Detected CRS:
  urn:adv:crs:ETRS89_UTM33*DE_DHHN2016_NH


In [22]:
# ============================================================
# 18 — PROCESS ALL BERLIN LoD2 FILES
# ============================================================

gdf_lod2_be, be_summary, be_failed = process_lod2_state("BE", "Berlin", BE_EXTRACT_DIR, "EPSG:25833", "1.0")

display(be_summary)

if not be_failed.empty:
    display(be_failed.head(20))

PROCESSING BERLIN (BE)

GML/XML files found: 925


Extracting BE: 100%|█████████████████████████████████████████████████████████████| 925/925 [10:29<00:00,  1.47it/s]



Buildings extracted : 639,658
Files failed        : 3
Missing heights     : 0
Saved to            : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BE/lod2_buildings_BE_raw.parquet


,state_code,state_name,gml_files,successful_files,failed_files,buildings,unique_lod2_ids,valid_geometries,measured_heights,missing_heights,minimum_height_m,maximum_height_m,output_file
0,BE,Berlin,925,922,3,639658,639658,636938,639658,0,0.0,230.0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


,state_code,source_file,error
0,BE,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,Unknown column geometry
1,BE,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,TopologyException: side location conflict at 3...
2,BE,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,TopologyException: side location conflict at 3...


In [23]:
from shapely import make_valid, union_all



In [24]:
# ============================================================
# 19 — FIX PARSER AND RETRY FAILED BERLIN FILES
# ============================================================


OUTPUT_COLUMNS = ["lod2_id", "creation_date", "function", "roof_type", "measured_height_m", "storeys_above_ground", "source_file", "geometry"]

def extract_ground_geometry(building):
    polygons = []

    for surface in building.iter():
        if etree.QName(surface).localname != "GroundSurface":
            continue

        for pos_list in surface.iter():
            if etree.QName(pos_list).localname != "posList" or not pos_list.text:
                continue

            values = np.fromstring(pos_list.text, sep=" ")
            dimension = int(pos_list.get("srsDimension", 3))

            if len(values) % dimension != 0:
                dimension = 3 if len(values) % 3 == 0 else 2

            coordinates = values.reshape(-1, dimension)[:, :2]

            if len(coordinates) < 3:
                continue

            if not np.array_equal(coordinates[0], coordinates[-1]):
                coordinates = np.vstack([coordinates, coordinates[0]])

            polygon = make_valid(Polygon(coordinates))

            if not polygon.is_empty and polygon.area > 0:
                polygons.append(polygon)

    if not polygons:
        return None

    if len(polygons) == 1:
        return polygons[0]

    try:
        return make_valid(union_all(polygons, grid_size=0.01))
    except Exception:
        return max(polygons, key=lambda geometry: geometry.area)

def extract_citygml_tile(gml_path):
    records = []

    for event, building in etree.iterparse(gml_path, events=("end",), huge_tree=True):
        if etree.QName(building).localname != "Building":
            continue

        records.append({
            "lod2_id": building.get(GML_ID),
            "creation_date": get_first_text(building, "creationDate"),
            "function": get_first_text(building, "function"),
            "roof_type": get_first_text(building, "roofType"),
            "measured_height_m": parse_number(get_first_text(building, "measuredHeight")),
            "storeys_above_ground": parse_number(get_first_text(building, "storeysAboveGround")),
            "source_file": gml_path.name,
            "geometry": extract_ground_geometry(building)
        })

        building.clear()

        while building.getprevious() is not None:
            del building.getparent()[0]

    if not records:
        return gpd.GeoDataFrame(columns=OUTPUT_COLUMNS, geometry="geometry", crs="EPSG:25833")

    return gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:25833")

retry_parts = []
retry_results = []

for source_file in be_failed["source_file"]:
    gml_path = Path(source_file)

    try:
        retry_gdf = extract_citygml_tile(gml_path)

        if retry_gdf.empty:
            retry_results.append({"source_file": str(gml_path), "status": "empty_tile", "buildings": 0, "missing_geometry": 0})
            continue

        retry_gdf["state_code"] = "BE"
        retry_gdf["source_state"] = "Berlin"
        retry_gdf["source_crs"] = "EPSG:25833"
        retry_gdf["citygml_version"] = "1.0"
        retry_gdf["height_source"] = "lod2_measured"
        retry_gdf["source_path"] = str(gml_path)
        retry_parts.append(retry_gdf)
        retry_results.append({"source_file": str(gml_path), "status": "recovered", "buildings": len(retry_gdf), "missing_geometry": retry_gdf.geometry.isna().sum()})

    except Exception as error:
        retry_results.append({"source_file": str(gml_path), "status": "failed_again", "buildings": 0, "missing_geometry": None, "error": str(error)})

retry_results_df = pd.DataFrame(retry_results)

if retry_parts:
    recovered_gdf = gpd.GeoDataFrame(pd.concat(retry_parts, ignore_index=True), geometry="geometry", crs="EPSG:25833")
    gdf_lod2_be = gpd.GeoDataFrame(pd.concat([gdf_lod2_be, recovered_gdf], ignore_index=True), geometry="geometry", crs="EPSG:25833")
    gdf_lod2_be = gdf_lod2_be.drop_duplicates(subset="lod2_id", keep="last").reset_index(drop=True)

be_output_path = EXTRACTED_DIR / "BE" / "lod2_buildings_BE_raw.parquet"
gdf_lod2_be.to_parquet(be_output_path, index=False)

print("=" * 70)
print("BERLIN FAILED-FILE RETRY")
print("=" * 70)
display(retry_results_df)

print(f"\nBuildings after retry       : {len(gdf_lod2_be):,}")
print(f"Unique LoD2 IDs             : {gdf_lod2_be['lod2_id'].nunique():,}")
print(f"Buildings with geometry     : {gdf_lod2_be.geometry.notna().sum():,}")
print(f"Buildings without geometry  : {gdf_lod2_be.geometry.isna().sum():,}")
print(f"Missing measured heights    : {gdf_lod2_be['measured_height_m'].isna().sum():,}")
print(f"Updated file                : {be_output_path}")

BERLIN FAILED-FILE RETRY


,source_file,status,buildings,missing_geometry
0,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,empty_tile,0,0
1,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,recovered,1013,0
2,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,recovered,1142,0



Buildings after retry       : 641,813
Unique LoD2 IDs             : 641,813
Buildings with geometry     : 639,093
Buildings without geometry  : 2,720
Missing measured heights    : 0
Updated file                : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BE/lod2_buildings_BE_raw.parquet


In [25]:
# ============================================================
# 20 — INSPECT BERLIN BUILDINGS WITHOUT GEOMETRY
# ============================================================

missing_geometry_be = gdf_lod2_be[gdf_lod2_be.geometry.isna()].copy()

print("=" * 70)
print("BERLIN BUILDINGS WITHOUT EXTRACTED GEOMETRY")
print("=" * 70)
print(f"\nBuildings without geometry: {len(missing_geometry_be):,}")
print(f"Source files affected     : {missing_geometry_be['source_file'].nunique():,}")

print("\nMost affected source files:")
display(missing_geometry_be.groupby(["source_file", "source_path"]).size().rename("missing_geometry_count").sort_values(ascending=False).head(20).reset_index())

GML_XLINK_HREF = "{http://www.w3.org/1999/xlink}href"

def inspect_building_structure(gml_path, target_id):
    for event, element in etree.iterparse(gml_path, events=("end",), huge_tree=True):
        if etree.QName(element).localname != "Building" or element.get(GML_ID) != target_id:
            continue

        tag_counts = Counter(etree.QName(descendant).localname for descendant in element.iter())
        xlinks = [descendant.get(GML_XLINK_HREF) for descendant in element.iter() if descendant.get(GML_XLINK_HREF)]

        result = {
            "lod2_id": target_id,
            "source_file": Path(gml_path).name,
            "GroundSurface": tag_counts["GroundSurface"],
            "RoofSurface": tag_counts["RoofSurface"],
            "WallSurface": tag_counts["WallSurface"],
            "BuildingPart": tag_counts["BuildingPart"],
            "consistsOfBuildingPart": tag_counts["consistsOfBuildingPart"],
            "lod0FootPrint": tag_counts["lod0FootPrint"],
            "lod0RoofEdge": tag_counts["lod0RoofEdge"],
            "lod1Solid": tag_counts["lod1Solid"],
            "lod2Solid": tag_counts["lod2Solid"],
            "Polygon": tag_counts["Polygon"],
            "posList": tag_counts["posList"],
            "xlink_references": len(xlinks),
            "example_xlink": xlinks[0] if xlinks else None
        }

        return result

    return {"lod2_id": target_id, "source_file": Path(gml_path).name, "error": "Building not found"}

inspection_records = []

for row in missing_geometry_be.head(20).itertuples(index=False):
    inspection_records.append(inspect_building_structure(row.source_path, row.lod2_id))

missing_geometry_structure = pd.DataFrame(inspection_records)

display(missing_geometry_structure)

BERLIN BUILDINGS WITHOUT EXTRACTED GEOMETRY

Buildings without geometry: 2,720
Source files affected     : 551

Most affected source files:


,source_file,source_path,missing_geometry_count
0,LoD2_33_406_5812_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,34
1,LoD2_33_394_5813_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,33
2,LoD2_33_389_5816_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,29
3,LoD2_33_384_5818_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,29
4,LoD2_33_386_5822_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,29
5,LoD2_33_398_5814_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,27
6,LoD2_33_389_5817_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,27
7,LoD2_33_389_5822_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,27
8,LoD2_33_395_5829_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,24
9,LoD2_33_405_5812_1_BE.xml,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,24


,lod2_id,source_file,GroundSurface,RoofSurface,WallSurface,BuildingPart,consistsOfBuildingPart,lod0FootPrint,lod0RoofEdge,lod1Solid,lod2Solid,Polygon,posList,xlink_references,example_xlink
0,DEBEATKBlj00003M,LoD2_33_371_5809_1_BE.xml,0,0,26,0,0,0,0,0,1,26,26,26,#ID_52089dee-e97d-4530-99c6-44f43d046290
1,DEBEATKBIb000007,LoD2_33_372_5805_1_BE.xml,0,0,12,0,0,0,0,0,1,12,12,12,#ID_a32ce554-cfc3-42e6-ab0a-bc8825cd8760
2,DEBEATKBR5000006,LoD2_33_372_5806_1_BE.xml,0,0,8,0,0,0,0,0,1,8,8,8,#ID_68c9cdc3-4689-4e84-a3f3-c67a7fe98683
3,DEBEATKBR5000005,LoD2_33_372_5806_1_BE.xml,0,0,17,0,0,0,0,0,1,17,17,17,#ID_ff2780b4-6a2f-4b8a-8de6-d6c149888944
4,DEBEATKBIb000006,LoD2_33_372_5806_1_BE.xml,0,0,6,0,0,0,0,0,1,6,6,6,#ID_cb45bb2f-d371-408c-88bf-b38e8dedc4a5
5,DEBEATKBR5000007,LoD2_33_372_5806_1_BE.xml,0,0,9,0,0,0,0,0,1,9,9,9,#ID_d401fa38-a4f1-40e2-a853-0a234d9e68af
6,DEBEATKB10000LFT,LoD2_33_372_5808_1_BE.xml,0,0,18,0,0,0,0,0,1,18,18,18,#ID_513e1423-5d4a-423b-8e6e-3585237bad8e
7,DEBEATKB4Y00000P,LoD2_33_372_5808_1_BE.xml,0,0,18,0,0,0,0,0,1,18,18,18,#ID_38210b7e-3d83-4d67-aa76-2c9f708bcf06
8,DEBEATKB4Y00000T,LoD2_33_372_5808_1_BE.xml,0,0,18,0,0,0,0,0,1,18,18,18,#ID_c4940bd5-8fbb-4b83-837f-7071faf5cc86
9,DEBEATKB1Gq0000B,LoD2_33_372_5808_1_BE.xml,0,0,18,0,0,0,0,0,1,18,18,18,#ID_26f7bdc5-91ca-4f2d-8b90-b4ed9d6cac3b


In [26]:
# ============================================================
# 21 — TEST WALL-BASED FOOTPRINT EXTRACTION
# ============================================================

from shapely.geometry import LineString, MultiPoint
from shapely.ops import polygonize

def parse_poslist_3d(pos_list):
    values = np.fromstring(pos_list.text, sep=" ")
    dimension = int(pos_list.get("srsDimension", 3))

    if len(values) % dimension != 0:
        dimension = 3 if len(values) % 3 == 0 else 2

    coordinates = values.reshape(-1, dimension)

    if dimension == 2:
        coordinates = np.column_stack([coordinates, np.zeros(len(coordinates))])

    return coordinates[:, :3]

def extract_wall_footprint(building, z_tolerance=0.10):
    bottom_lines = []
    bottom_points = []

    for surface in building.iter():
        if etree.QName(surface).localname != "WallSurface":
            continue

        for pos_list in surface.iter():
            if etree.QName(pos_list).localname != "posList" or not pos_list.text:
                continue

            coordinates = parse_poslist_3d(pos_list)
            minimum_z = coordinates[:, 2].min()
            lower_coordinates = coordinates[coordinates[:, 2] <= minimum_z + z_tolerance, :2]

            if len(lower_coordinates) < 2:
                continue

            unique_coordinates = []

            for coordinate in lower_coordinates:
                if not unique_coordinates or not np.allclose(coordinate, unique_coordinates[-1]):
                    unique_coordinates.append(coordinate)

            if len(unique_coordinates) >= 2:
                line = LineString(unique_coordinates)
                if not line.is_empty and line.length > 0:
                    bottom_lines.append(line)
                    bottom_points.extend(unique_coordinates)

    if not bottom_lines:
        return None, "missing"

    merged_lines = union_all(bottom_lines, grid_size=0.01)
    polygons = [polygon for polygon in polygonize(merged_lines) if polygon.area > 0]

    if polygons:
        return make_valid(union_all(polygons, grid_size=0.01)), "wall_bottom_polygonized"

    fallback = MultiPoint(bottom_points).convex_hull

    if fallback.geom_type in {"Polygon", "MultiPolygon"} and fallback.area > 0:
        return make_valid(fallback), "wall_bottom_convex_hull"

    return None, "missing"

# ============================================================
# 21 — RETEST WALL-BASED FOOTPRINT EXTRACTION
# ============================================================

def extract_wall_footprint_by_id(gml_path, target_id):
    for event, building in etree.iterparse(gml_path, events=("end",), tag="{http://www.opengis.net/citygml/building/1.0}Building", huge_tree=True):
        building_id = building.get(GML_ID)

        if building_id == target_id:
            geometry, method = extract_wall_footprint(building)
            building.clear()
            while building.getprevious() is not None:
                del building.getparent()[0]
            return geometry, method

        building.clear()

        while building.getprevious() is not None:
            del building.getparent()[0]

    return None, "building_not_found"

wall_test_records = []

for row in missing_geometry_be.head(20).itertuples(index=False):
    geometry, method = extract_wall_footprint_by_id(row.source_path, row.lod2_id)
    wall_test_records.append({
        "lod2_id": row.lod2_id,
        "source_file": row.source_file,
        "method": method,
        "geometry_type": geometry.geom_type if geometry is not None else None,
        "footprint_area_m2": geometry.area if geometry is not None else np.nan
    })

wall_test_df = pd.DataFrame(wall_test_records)

print("=" * 70)
print("CORRECTED WALL-BASED FOOTPRINT TEST")
print("=" * 70)
print(f"\nBuildings tested     : {len(wall_test_df):,}")
print(f"Footprints recovered : {wall_test_df['footprint_area_m2'].notna().sum():,}")

display(wall_test_df)
display(wall_test_df["method"].value_counts(dropna=False).rename_axis("method").reset_index(name="count"))

CORRECTED WALL-BASED FOOTPRINT TEST

Buildings tested     : 20
Footprints recovered : 20


,lod2_id,source_file,method,geometry_type,footprint_area_m2
0,DEBEATKBlj00003M,LoD2_33_371_5809_1_BE.xml,wall_bottom_convex_hull,Polygon,75.278207
1,DEBEATKBIb000007,LoD2_33_372_5805_1_BE.xml,wall_bottom_polygonized,Polygon,63.157750
2,DEBEATKBR5000006,LoD2_33_372_5806_1_BE.xml,wall_bottom_polygonized,Polygon,383.206250
3,DEBEATKBR5000005,LoD2_33_372_5806_1_BE.xml,wall_bottom_convex_hull,Polygon,1.361648
4,DEBEATKBIb000006,LoD2_33_372_5806_1_BE.xml,wall_bottom_polygonized,MultiPolygon,4.118800
5,DEBEATKBR5000007,LoD2_33_372_5806_1_BE.xml,wall_bottom_polygonized,Polygon,8.220650
6,DEBEATKB10000LFT,LoD2_33_372_5808_1_BE.xml,wall_bottom_convex_hull,Polygon,11.112550
7,DEBEATKB4Y00000P,LoD2_33_372_5808_1_BE.xml,wall_bottom_polygonized,Polygon,58.992200
8,DEBEATKB4Y00000T,LoD2_33_372_5808_1_BE.xml,wall_bottom_polygonized,Polygon,58.908800
9,DEBEATKB1Gq0000B,LoD2_33_372_5808_1_BE.xml,wall_bottom_polygonized,Polygon,58.908800


,method,count
0,wall_bottom_polygonized,13
1,wall_bottom_convex_hull,7


In [27]:
# ============================================================
# 22 — ADD WALL FALLBACK AND REPAIR BERLIN GEOMETRIES
# ============================================================

def extract_lod2_footprint(building):
    geometry = extract_ground_geometry(building)

    if geometry is not None and not geometry.is_empty:
        return geometry, "ground_surface"

    return extract_wall_footprint(building)

def extract_citygml_tile(gml_path, source_crs=None):
    records = []

    for event, building in etree.iterparse(gml_path, events=("end",), tag="{http://www.opengis.net/citygml/building/1.0}Building", huge_tree=True):
        geometry, footprint_method = extract_lod2_footprint(building)

        records.append({
            "lod2_id": building.get(GML_ID),
            "creation_date": get_first_text(building, "creationDate"),
            "function": get_first_text(building, "function"),
            "roof_type": get_first_text(building, "roofType"),
            "measured_height_m": parse_number(get_first_text(building, "measuredHeight")),
            "storeys_above_ground": parse_number(get_first_text(building, "storeysAboveGround")),
            "source_file": gml_path.name,
            "footprint_method": footprint_method,
            "geometry": geometry
        })

        building.clear()

        while building.getprevious() is not None:
            del building.getparent()[0]

    columns = ["lod2_id", "creation_date", "function", "roof_type", "measured_height_m", "storeys_above_ground", "source_file", "footprint_method", "geometry"]

    if not records:
        return gpd.GeoDataFrame(columns=columns, geometry="geometry", crs=source_crs)

    return gpd.GeoDataFrame(records, geometry="geometry", crs=source_crs)

gdf_lod2_be["footprint_method"] = np.where(gdf_lod2_be.geometry.notna(), "ground_surface", "missing")

missing_ids_by_file = missing_geometry_be.groupby("source_path")["lod2_id"].apply(set).to_dict()
recovered_records = []
recovery_errors = []

for source_path, target_ids in tqdm(missing_ids_by_file.items(), total=len(missing_ids_by_file), desc="Recovering Berlin footprints"):
    gml_path = Path(source_path)

    try:
        for event, building in etree.iterparse(gml_path, events=("end",), tag="{http://www.opengis.net/citygml/building/1.0}Building", huge_tree=True):
            lod2_id = building.get(GML_ID)

            if lod2_id in target_ids:
                geometry, method = extract_lod2_footprint(building)
                recovered_records.append({"lod2_id": lod2_id, "geometry": geometry, "footprint_method": method})

            building.clear()

            while building.getprevious() is not None:
                del building.getparent()[0]

    except Exception as error:
        recovery_errors.append({"source_path": str(gml_path), "error": str(error)})

recovered_gdf = gpd.GeoDataFrame(recovered_records, geometry="geometry", crs="EPSG:25833")
recovered_gdf = recovered_gdf.drop_duplicates(subset="lod2_id", keep="last")

geometry_lookup = recovered_gdf.set_index("lod2_id")["geometry"]
method_lookup = recovered_gdf.set_index("lod2_id")["footprint_method"]
missing_mask = gdf_lod2_be.geometry.isna()

gdf_lod2_be.loc[missing_mask, "geometry"] = gdf_lod2_be.loc[missing_mask, "lod2_id"].map(geometry_lookup)
gdf_lod2_be.loc[missing_mask, "footprint_method"] = gdf_lod2_be.loc[missing_mask, "lod2_id"].map(method_lookup).fillna("missing")

be_output_path = EXTRACTED_DIR / "BE" / "lod2_buildings_BE_raw.parquet"
gdf_lod2_be.to_parquet(be_output_path, index=False)

recovery_errors_df = pd.DataFrame(recovery_errors)

print("=" * 70)
print("BERLIN GEOMETRY RECOVERY SUMMARY")
print("=" * 70)
print(f"\nOriginally missing       : {len(missing_geometry_be):,}")
print(f"Recovery records created : {len(recovered_gdf):,}")
print(f"Recovered geometries     : {recovered_gdf.geometry.notna().sum():,}")
print(f"Still missing geometry   : {gdf_lod2_be.geometry.isna().sum():,}")
print(f"Files with errors        : {len(recovery_errors_df):,}")
print(f"Updated file             : {be_output_path}")

display(gdf_lod2_be["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

if not recovery_errors_df.empty:
    display(recovery_errors_df)

Recovering Berlin footprints: 100%|██████████████████████████████████████████████| 551/551 [01:43<00:00,  5.35it/s]


BERLIN GEOMETRY RECOVERY SUMMARY

Originally missing       : 2,720
Recovery records created : 2,720
Recovered geometries     : 2,473
Still missing geometry   : 247
Files with errors        : 0
Updated file             : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BE/lod2_buildings_BE_raw.parquet


,footprint_method,building_count
0,ground_surface,639093
1,wall_bottom_polygonized,1323
2,wall_bottom_convex_hull,1150
3,missing,247


In [28]:
# ============================================================
# 23 — DISCOVER BRANDENBURG LoD2 TILES
# ============================================================

from lxml import html
from urllib.parse import urljoin

BB_RAW_DIR = RAW_DIR / "BB"
BB_RAW_DIR.mkdir(parents=True, exist_ok=True)

BB_PORTAL_URL = "https://geoportal.brandenburg.de/detailansichtdienst/render?url=https%3A%2F%2Fgeoportal.brandenburg.de%2Fgs-json%2Fxml%3Ffileid%3D0414a37a-a749-4ee6-9f59-a41226919c58"
BB_DOWNLOAD_DIR_URL = "https://data.geobasis-bb.de/geobasis/daten/3d_gebaeude/lod2_gml/"

bb_index = source_registry["state_code"].eq("BB")
source_registry.loc[bb_index, ["portal_url", "download_url", "download_method", "format", "horizontal_crs", "licence", "download_status", "notes"]] = [BB_PORTAL_URL, BB_DOWNLOAD_DIR_URL, "directory_tiles", "CityGML", "EPSG:25833", "Datenlizenz Deutschland Namensnennung 2.0", "ready", "Official statewide LoD2, individual 1 x 1 km ZIP tiles"]
source_registry.to_csv(source_registry_path, index=False)

response = requests.get(BB_DOWNLOAD_DIR_URL, timeout=(30, 120))
response.raise_for_status()

document = html.fromstring(response.content)
tile_urls = sorted({urljoin(BB_DOWNLOAD_DIR_URL, href) for href in document.xpath("//a/@href") if href.lower().endswith(".zip")})

bb_downloads = pd.DataFrame({"download_url": tile_urls})
bb_downloads["filename"] = bb_downloads["download_url"].str.rsplit("/", n=1).str[-1]

bb_download_list_path = BB_RAW_DIR / "brandenburg_download_list.csv"
bb_downloads.to_csv(bb_download_list_path, index=False)

print("=" * 70)
print("BRANDENBURG LoD2 TILE DISCOVERY")
print("=" * 70)
print(f"\nTiles found        : {len(bb_downloads):,}")
print(f"Download directory : {BB_DOWNLOAD_DIR_URL}")
print(f"List saved to      : {bb_download_list_path}")

display(bb_downloads.head(20))
display(bb_downloads.tail(20))

if bb_downloads.empty:
    raise RuntimeError("No Brandenburg LoD2 ZIP files were discovered.")

BRANDENBURG LoD2 TILE DISCOVERY

Tiles found        : 19,298
Download directory : https://data.geobasis-bb.de/geobasis/daten/3d_gebaeude/lod2_gml/
List saved to      : /fast/home/o-olajuyigbe/data/germany_lod2/raw/BB/brandenburg_download_list.csv


,download_url,filename
0,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33250-5889.zip
1,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33250-5890.zip
2,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33250-5891.zip
3,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33250-5892.zip
4,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33251-5888.zip
5,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33251-5889.zip
6,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33251-5891.zip
7,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33252-5887.zip
8,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33252-5888.zip
9,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33252-5890.zip


,download_url,filename
19278,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33481-5763.zip
19279,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33481-5764.zip
19280,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33481-5765.zip
19281,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33481-5766.zip
19282,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33481-5767.zip
19283,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33481-5768.zip
19284,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33481-5769.zip
19285,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33482-5716.zip
19286,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33482-5720.zip
19287,https://data.geobasis-bb.de/geobasis/daten/3d_...,lod2_33482-5721.zip


In [29]:
# ============================================================
# 24 — DOWNLOAD AND TEST ONE BRANDENBURG LoD2 TILE
# ============================================================

BB_TEST_DIR = BB_RAW_DIR / "test_tile"
BB_TEST_DIR.mkdir(parents=True, exist_ok=True)

bb_test_row = bb_downloads.iloc[len(bb_downloads) // 2]
bb_test_zip = BB_TEST_DIR / bb_test_row["filename"]
bb_test_extract_dir = BB_TEST_DIR / bb_test_zip.stem

download_file(bb_test_row["download_url"], bb_test_zip)

bb_test_extract_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(bb_test_zip, "r") as archive:
    archive.extractall(bb_test_extract_dir)

bb_test_gml_files = find_citygml_files(bb_test_extract_dir)

if not bb_test_gml_files:
    raise FileNotFoundError(f"No CityGML file found inside {bb_test_zip}")

bb_sample_path = bb_test_gml_files[0]
bb_srs_names = set()

for event, element in etree.iterparse(bb_sample_path, events=("start",), huge_tree=True):
    if element.get("srsName"):
        bb_srs_names.add(element.get("srsName"))

bb_sample_gdf = extract_citygml_tile(bb_sample_path, source_crs="EPSG:25833")

print("=" * 70)
print("BRANDENBURG SAMPLE TILE TEST")
print("=" * 70)
print(f"\nZIP file               : {bb_test_zip.name}")
print(f"CityGML file           : {bb_sample_path.name}")
print(f"Buildings extracted    : {len(bb_sample_gdf):,}")
print(f"Measured heights       : {bb_sample_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights        : {bb_sample_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries      : {bb_sample_gdf.geometry.notna().sum():,}")
print(f"Missing geometries     : {bb_sample_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs        : {bb_sample_gdf['lod2_id'].nunique():,}")
print(f"CRS assigned           : {bb_sample_gdf.crs}")

print("\nCRS written in the CityGML:")
for srs_name in sorted(bb_srs_names):
    print(f"  {srs_name}")

print("\nFootprint methods:")
display(bb_sample_gdf["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

display(bb_sample_gdf.head())

Already downloaded: lod2_33399-5765.zip (0.2 MB)
BRANDENBURG SAMPLE TILE TEST

ZIP file               : lod2_33399-5765.zip
CityGML file           : lod2_33399-5765_geb.gml
Buildings extracted    : 110
Measured heights       : 110
Missing heights        : 0
Usable geometries      : 110
Missing geometries     : 0
Unique LoD2 IDs        : 110
CRS assigned           : EPSG:25833

CRS written in the CityGML:
  urn:adv:crs:ETRS89_UTM33*DE_DHHN2016_NH

Footprint methods:


,footprint_method,building_count
0,ground_surface,110


,lod2_id,creation_date,function,roof_type,measured_height_m,storeys_above_ground,source_file,footprint_method,geometry
0,DEBBAL720004z3Q1,2017-10-17,31001_2463,1000,2.894,NaN,lod2_33399-5765_geb.gml,ground_surface,"POLYGON ((399222.16 5765976.829, 399225.354 57..."
1,DEBBAL720004z3Qr,2017-10-17,31001_2463,3100,7.377,NaN,lod2_33399-5765_geb.gml,ground_surface,"POLYGON ((399266.07 5765919.77, 399266.35 5765..."
2,DEBBAL720004z3Rn,2017-10-17,31001_9998,3100,3.895,NaN,lod2_33399-5765_geb.gml,ground_surface,"POLYGON ((399525.748 5765960.754, 399521.71 57..."
3,DEBBAL720004z3PM,2017-10-17,31001_9998,3100,3.061,NaN,lod2_33399-5765_geb.gml,ground_surface,"POLYGON ((399200.23 5765973.68, 399194.98 5765..."
4,DEBBAL720004z3RZ,2017-10-17,31001_1010,3100,8.673,NaN,lod2_33399-5765_geb.gml,ground_surface,"POLYGON ((399371.975 5766001.225, 399367.14 57..."


In [30]:
# ============================================================
# 25 — DOWNLOAD ALL BRANDENBURG LoD2 TILES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import time


BB_ZIP_DIR = BB_RAW_DIR / "zips"
BB_ZIP_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 8
MAX_RETRIES = 3
CHUNK_SIZE = 1024 * 1024

def download_tile(url, filename):
    output_path = BB_ZIP_DIR / filename
    temp_path = output_path.with_suffix(output_path.suffix + ".part")

    if output_path.exists() and output_path.stat().st_size > 0:
        return {"filename": filename, "status": "already_downloaded", "size_bytes": output_path.stat().st_size, "error": None}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 600)) as response:
                response.raise_for_status()
                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            file.write(chunk)

            temp_path.replace(output_path)
            return {"filename": filename, "status": "downloaded", "size_bytes": output_path.stat().st_size, "error": None}

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == MAX_RETRIES:
                return {"filename": filename, "status": "failed", "size_bytes": 0, "error": str(error)}

            time.sleep(attempt * 3)

download_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_tile, row.download_url, row.filename): row.filename for row in bb_downloads.itertuples(index=False)}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading Brandenburg tiles"):
        download_results.append(future.result())

bb_download_results = pd.DataFrame(download_results)
bb_download_results.to_csv(REPORT_DIR / "lod2_download_results_BB.csv", index=False)

successful_statuses = ["downloaded", "already_downloaded"]

print("=" * 70)
print("BRANDENBURG DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nTiles requested        : {len(bb_downloads):,}")
print(f"Tiles available locally: {len(list(BB_ZIP_DIR.glob('*.zip'))):,}")
print(f"Downloaded now         : {(bb_download_results['status'] == 'downloaded').sum():,}")
print(f"Already downloaded     : {(bb_download_results['status'] == 'already_downloaded').sum():,}")
print(f"Failed                 : {(bb_download_results['status'] == 'failed').sum():,}")
print(f"Downloaded size        : {sum(path.stat().st_size for path in BB_ZIP_DIR.glob('*.zip')) / 1024**3:,.2f} GB")

display(bb_download_results["status"].value_counts().rename_axis("status").reset_index(name="count"))

if (bb_download_results["status"] == "failed").any():
    display(bb_download_results[bb_download_results["status"] == "failed"].head(30))

BRANDENBURG DOWNLOAD SUMMARY

Tiles requested        : 19,298
Tiles available locally: 19,298
Downloaded now         : 0
Already downloaded     : 19,298
Failed                 : 0
Downloaded size        : 4.60 GB


,status,count
0,already_downloaded,19298


In [31]:
# ============================================================
# 26 — PROCESS BRANDENBURG LoD2 IN BATCHES
# ============================================================

import gc
import pyarrow.parquet as pq

BB_OUTPUT_DIR = EXTRACTED_DIR / "BB"
BB_PART_DIR = BB_OUTPUT_DIR / "parts"
BB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BB_PART_DIR.mkdir(parents=True, exist_ok=True)

BUILDING_TAG_V1 = "{http://www.opengis.net/citygml/building/1.0}Building"
BB_BATCH_SIZE = 250

def extract_citygml_stream(stream, source_file, source_crs):
    records = []

    for event, building in etree.iterparse(stream, events=("end",), tag=BUILDING_TAG_V1, huge_tree=True):
        geometry, footprint_method = extract_lod2_footprint(building)

        records.append({
            "lod2_id": building.get(GML_ID),
            "creation_date": get_first_text(building, "creationDate"),
            "function": get_first_text(building, "function"),
            "roof_type": get_first_text(building, "roofType"),
            "measured_height_m": parse_number(get_first_text(building, "measuredHeight")),
            "storeys_above_ground": parse_number(get_first_text(building, "storeysAboveGround")),
            "source_file": source_file,
            "footprint_method": footprint_method,
            "geometry": geometry
        })

        building.clear()

        while building.getprevious() is not None:
            del building.getparent()[0]

    columns = ["lod2_id", "creation_date", "function", "roof_type", "measured_height_m", "storeys_above_ground", "source_file", "footprint_method", "geometry"]

    if not records:
        return gpd.GeoDataFrame(columns=columns, geometry="geometry", crs=source_crs)

    return gpd.GeoDataFrame(records, geometry="geometry", crs=source_crs)

bb_zip_files = sorted(BB_ZIP_DIR.glob("*.zip"))

if len(bb_zip_files) != len(bb_downloads):
    print(f"WARNING: Expected {len(bb_downloads):,} ZIP files but found {len(bb_zip_files):,}.")

bb_batch_summaries = []
bb_processing_errors = []

for batch_start in tqdm(range(0, len(bb_zip_files), BB_BATCH_SIZE), desc="Processing Brandenburg batches"):
    batch_number = batch_start // BB_BATCH_SIZE + 1
    batch_zip_files = bb_zip_files[batch_start:batch_start + BB_BATCH_SIZE]
    part_path = BB_PART_DIR / f"lod2_buildings_BB_part_{batch_number:04d}.parquet"

    if part_path.exists() and part_path.stat().st_size > 0:
        row_count = pq.ParquetFile(part_path).metadata.num_rows
        bb_batch_summaries.append({"batch": batch_number, "zip_files": len(batch_zip_files), "buildings": row_count, "status": "already_processed", "output_file": str(part_path)})
        continue

    batch_parts = []

    for zip_path in batch_zip_files:
        try:
            with ZipFile(zip_path, "r") as archive:
                gml_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

                if not gml_members:
                    bb_processing_errors.append({"zip_file": str(zip_path), "source_file": None, "error": "No GML/XML file found"})
                    continue

                for member in gml_members:
                    with archive.open(member) as stream:
                        tile_gdf = extract_citygml_stream(stream, Path(member).name, "EPSG:25833")

                    if tile_gdf.empty:
                        continue

                    tile_gdf["state_code"] = "BB"
                    tile_gdf["source_state"] = "Brandenburg"
                    tile_gdf["source_crs"] = "EPSG:25833"
                    tile_gdf["citygml_version"] = "1.0"
                    tile_gdf["height_source"] = "lod2_measured"
                    tile_gdf["source_path"] = str(zip_path)
                    batch_parts.append(tile_gdf)

        except Exception as error:
            bb_processing_errors.append({"zip_file": str(zip_path), "source_file": None, "error": str(error)})

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25833")
        batch_gdf.to_parquet(part_path, index=False)
        building_count = len(batch_gdf)
        missing_heights = batch_gdf["measured_height_m"].isna().sum()
        missing_geometry = batch_gdf.geometry.isna().sum()
    else:
        building_count = 0
        missing_heights = 0
        missing_geometry = 0

    bb_batch_summaries.append({
        "batch": batch_number,
        "zip_files": len(batch_zip_files),
        "buildings": building_count,
        "missing_heights": missing_heights,
        "missing_geometry": missing_geometry,
        "status": "processed",
        "output_file": str(part_path) if building_count > 0 else None
    })

    del batch_parts

    if building_count > 0:
        del batch_gdf

    gc.collect()

bb_batch_summary_df = pd.DataFrame(bb_batch_summaries)
bb_processing_errors_df = pd.DataFrame(bb_processing_errors)

bb_batch_summary_path = REPORT_DIR / "lod2_batch_summary_BB.csv"
bb_error_path = REPORT_DIR / "lod2_processing_errors_BB.csv"

bb_batch_summary_df.to_csv(bb_batch_summary_path, index=False)

if not bb_processing_errors_df.empty:
    bb_processing_errors_df.to_csv(bb_error_path, index=False)

bb_part_files = sorted(BB_PART_DIR.glob("*.parquet"))
bb_total_buildings = sum(pq.ParquetFile(path).metadata.num_rows for path in bb_part_files)

print("=" * 70)
print("BRANDENBURG LoD2 BATCH PROCESSING SUMMARY")
print("=" * 70)
print(f"\nZIP files available      : {len(bb_zip_files):,}")
print(f"Parquet parts created    : {len(bb_part_files):,}")
print(f"Buildings extracted      : {bb_total_buildings:,}")
print(f"Processing errors        : {len(bb_processing_errors_df):,}")
print(f"Part directory           : {BB_PART_DIR}")
print(f"Batch summary            : {bb_batch_summary_path}")

display(bb_batch_summary_df.tail(20))

if not bb_processing_errors_df.empty:
    display(bb_processing_errors_df.head(30))

Processing Brandenburg batches: 100%|█████████████████████████████████████████████| 78/78 [00:00<00:00, 341.14it/s]

BRANDENBURG LoD2 BATCH PROCESSING SUMMARY

ZIP files available      : 19,298
Parquet parts created    : 78
Buildings extracted      : 2,376,794
Processing errors        : 0
Part directory           : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts
Batch summary            : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_batch_summary_BB.csv


,batch,zip_files,buildings,status,output_file
58,59,250,22660,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
59,60,250,26696,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
60,61,250,27700,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
61,62,250,20922,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
62,63,250,23215,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
63,64,250,26527,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
64,65,250,19737,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
65,66,250,16249,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
66,67,250,25604,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
67,68,250,23417,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


In [32]:
# ============================================================
# 27 — SUMMARISE BRANDENBURG RAW LoD2 DATA
# ============================================================

bb_summary_records = []
bb_method_counts = {}

for part_path in tqdm(bb_part_files, desc="Summarising Brandenburg parts"):
    part = pd.read_parquet(part_path, columns=["lod2_id", "measured_height_m", "footprint_method", "geometry"])

    bb_summary_records.append({
        "buildings": len(part),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    for method, count in part["footprint_method"].value_counts(dropna=False).items():
        bb_method_counts[method] = bb_method_counts.get(method, 0) + count

bb_part_stats = pd.DataFrame(bb_summary_records)

bb_summary = pd.DataFrame([{
    "state_code": "BB",
    "state_name": "Brandenburg",
    "zip_files": len(bb_zip_files),
    "parquet_parts": len(bb_part_files),
    "buildings": bb_part_stats["buildings"].sum(),
    "measured_heights": bb_part_stats["measured_heights"].sum(),
    "missing_heights": bb_part_stats["missing_heights"].sum(),
    "usable_geometries": bb_part_stats["usable_geometries"].sum(),
    "missing_geometries": bb_part_stats["missing_geometries"].sum(),
    "minimum_height_m": bb_part_stats["minimum_height_m"].min(),
    "maximum_height_m": bb_part_stats["maximum_height_m"].max(),
    "output_directory": str(BB_PART_DIR)
}])

bb_footprint_summary = pd.DataFrame([{"footprint_method": method, "building_count": count} for method, count in bb_method_counts.items()]).sort_values("building_count", ascending=False)

bb_summary_path = REPORT_DIR / "lod2_extraction_summary_BB.csv"
bb_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_BB.csv"

bb_summary.to_csv(bb_summary_path, index=False)
bb_footprint_summary.to_csv(bb_footprint_summary_path, index=False)

print("=" * 70)
print("BRANDENBURG RAW LoD2 SUMMARY")
print("=" * 70)

display(bb_summary)
display(bb_footprint_summary)

print(f"\nState summary saved to     : {bb_summary_path}")
print(f"Footprint summary saved to : {bb_footprint_summary_path}")


Summarising Brandenburg parts:   0%|                                                        | 0/78 [00:00<?, ?it/s]

Summarising Brandenburg parts: 100%|███████████████████████████████████████████████| 78/78 [00:05<00:00, 13.24it/s]

BRANDENBURG RAW LoD2 SUMMARY


,state_code,state_name,zip_files,parquet_parts,buildings,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,BB,Brandenburg,19298,78,2376794,2376794,0,2331500,45294,0.5,170.683,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


,footprint_method,building_count
0,ground_surface,2331500
1,missing,45294



State summary saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_BB.csv
Footprint summary saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_BB.csv


In [33]:
# ============================================================
# 28 — DISCOVER MV LoD2 DOWNLOADS
# ============================================================

import re
from urllib.parse import urljoin, urlparse

MV_RAW_DIR = RAW_DIR / "MV"
MV_RAW_DIR.mkdir(parents=True, exist_ok=True)

MV_PORTAL_URL = "https://www.geoportal-mv.de/portal/Download_und_Shop/Produkte/Landschaftsdaten"
MV_ATOM_URL = "https://www.geodaten-mv.de/dienste/gebaeude_atom?"

mv_index = source_registry["state_code"].eq("MV")
source_registry.loc[mv_index, ["portal_url", "download_url", "download_method", "format", "citygml_version", "horizontal_crs", "download_status", "notes"]] = [MV_PORTAL_URL, MV_ATOM_URL, "ATOM", "CityGML", "1.0", "EPSG:25833", "ready", "Official statewide LoD2 ATOM service"]
source_registry.to_csv(source_registry_path, index=False)

def discover_atom_downloads_flexible(start_url):
    pending = [start_url]
    visited = set()
    downloads = []
    link_log = []
    headers = {"Accept": "application/atom+xml, application/xml, text/xml, */*", "User-Agent": "Mozilla/5.0"}

    while pending:
        feed_url = pending.pop(0)

        if feed_url in visited:
            continue

        visited.add(feed_url)
        response = requests.get(feed_url, headers=headers, timeout=(30, 180))
        response.raise_for_status()
        root = etree.fromstring(response.content)

        for element in root.iter():
            local_name = etree.QName(element).localname.lower()

            if local_name not in {"link", "content"}:
                continue

            href = element.get("href") or element.get("src")

            if not href:
                continue

            link_url = urljoin(feed_url, href)
            link_type = (element.get("type") or "").lower()
            link_rel = (element.get("rel") or "").lower()
            clean_url = link_url.lower().split("?")[0]
            extension = Path(urlparse(clean_url).path).suffix.lower()

            link_log.append({"source_feed": feed_url, "url": link_url, "type": link_type, "rel": link_rel, "extension": extension})

            is_atom_feed = "atom+xml" in link_type or link_rel == "subsection" or extension in {".atom", ".feed"}
            is_download = extension in {".zip", ".gml", ".xml"} or any(value in link_type for value in ["application/zip", "application/x-zip", "application/gml", "octet-stream", "gzip"])

            if is_atom_feed and link_url not in visited:
                pending.append(link_url)
            elif is_download:
                downloads.append({"download_url": link_url, "filename": Path(urlparse(clean_url).path).name, "source_feed": feed_url})

        text_urls = re.findall(r'https?://[^\s"<>]+(?:\.zip|\.gml|\.xml)(?:\?[^\s"<>]*)?', response.text, flags=re.IGNORECASE)

        for link_url in text_urls:
            clean_url = link_url.split("?")[0]
            downloads.append({"download_url": link_url, "filename": Path(urlparse(clean_url).path).name, "source_feed": feed_url})

    downloads_df = pd.DataFrame(downloads).drop_duplicates(subset="download_url").reset_index(drop=True) if downloads else pd.DataFrame(columns=["download_url", "filename", "source_feed"])
    link_log_df = pd.DataFrame(link_log)
    return downloads_df, link_log_df, visited

mv_downloads, mv_link_log, mv_feeds_visited = discover_atom_downloads_flexible(MV_ATOM_URL)

print("=" * 70)
print("MECKLENBURG-WESTERN POMERANIA DOWNLOAD DISCOVERY")
print("=" * 70)
print(f"\nFeeds visited   : {len(mv_feeds_visited):,}")
print(f"Links inspected : {len(mv_link_log):,}")
print(f"Downloads found : {len(mv_downloads):,}")

if not mv_downloads.empty:
    display(mv_downloads.head(20))
    display(mv_downloads.tail(20))
    mv_download_list_path = MV_RAW_DIR / "mv_download_list.csv"
    mv_downloads.to_csv(mv_download_list_path, index=False)
    print(f"\nDownload list saved to: {mv_download_list_path}")
else:
    print("\nLink types returned by the service:")
    display(mv_link_log.groupby(["type", "rel", "extension"], dropna=False).size().rename("count").reset_index().sort_values("count", ascending=False).head(30))

MECKLENBURG-WESTERN POMERANIA DOWNLOAD DISCOVERY

Feeds visited   : 3
Links inspected : 6,359
Downloads found : 12,688


,download_url,filename,source_feed
0,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
1,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
2,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
3,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
4,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
5,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
6,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
7,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
8,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
9,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...


,download_url,filename,source_feed
12668,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12669,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12670,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12671,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12672,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12673,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12674,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12675,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12676,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...
12677,https://www.geodaten-mv.de/dienste/gebaeude_do...,gebaeude_download,https://www.geodaten-mv.de/dienste/gebaeude_at...



Download list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/MV/mv_download_list.csv


In [34]:
# ============================================================
# 29 — INSPECT MV DOWNLOAD URL STRUCTURE
# ============================================================

from urllib.parse import urlparse, parse_qs
import hashlib

def create_mv_filename(url, index):
    parsed = urlparse(url)
    params = parse_qs(parsed.query)

    for key in ["file", "filename", "name", "tile", "id", "bbox", "blatt", "kachel"]:
        if key in params and params[key]:
            value = re.sub(r"[^A-Za-z0-9_-]+", "_", params[key][0]).strip("_")
            return f"lod2_mv_{value}.zip"

    url_hash = hashlib.md5(url.encode()).hexdigest()[:12]
    return f"lod2_mv_{index:05d}_{url_hash}.zip"

mv_downloads["filename"] = [create_mv_filename(url, index) for index, url in enumerate(mv_downloads["download_url"], start=1)]

print("=" * 70)
print("MV DOWNLOAD URL INSPECTION")
print("=" * 70)
print(f"\nDownloads found        : {len(mv_downloads):,}")
print(f"Unique URLs            : {mv_downloads['download_url'].nunique():,}")
print(f"Unique filenames       : {mv_downloads['filename'].nunique():,}")
print(f"Duplicate filenames    : {mv_downloads['filename'].duplicated().sum():,}")

print("\nFirst five complete URLs:")
for url in mv_downloads["download_url"].head():
    print(url)

display(mv_downloads.head(20))

mv_test_url = mv_downloads.iloc[0]["download_url"]
mv_test_response = requests.get(mv_test_url, stream=True, timeout=(30, 180))
mv_test_response.raise_for_status()

print("\nTest response headers:")
print(f"Content-Type        : {mv_test_response.headers.get('content-type')}")
print(f"Content-Length      : {mv_test_response.headers.get('content-length')}")
print(f"Content-Disposition : {mv_test_response.headers.get('content-disposition')}")

mv_test_response.close()

mv_download_list_path = MV_RAW_DIR / "mv_download_list.csv"
mv_downloads.to_csv(mv_download_list_path, index=False)

print(f"\nUpdated download list saved to: {mv_download_list_path}")

MV DOWNLOAD URL INSPECTION

Downloads found        : 12,688
Unique URLs            : 12,688
Unique filenames       : 12,688
Duplicate filenames    : 0

First five complete URLs:
https://www.geodaten-mv.de/dienste/gebaeude_download?index=0&dataset=8397b554-5cb9-4274-8be8-c20490d9a6e8&file=lod2_33_206_5920_2_gml.zip
https://www.geodaten-mv.de/dienste/gebaeude_download?index=0&dataset=8397b554-5cb9-4274-8be8-c20490d9a6e8&file=lod2_33_206_5922_2_gml.zip
https://www.geodaten-mv.de/dienste/gebaeude_download?index=0&dataset=8397b554-5cb9-4274-8be8-c20490d9a6e8&file=lod2_33_208_5922_2_gml.zip
https://www.geodaten-mv.de/dienste/gebaeude_download?index=0&dataset=8397b554-5cb9-4274-8be8-c20490d9a6e8&file=lod2_33_208_5924_2_gml.zip
https://www.geodaten-mv.de/dienste/gebaeude_download?index=0&dataset=8397b554-5cb9-4274-8be8-c20490d9a6e8&file=lod2_33_208_5926_2_gml.zip


,download_url,filename,source_feed
0,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_206_5920_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
1,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_206_5922_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
2,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_208_5922_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
3,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_208_5924_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
4,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_208_5926_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
5,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_208_5928_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
6,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_208_5930_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
7,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_210_5920_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
8,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_210_5922_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...
9,https://www.geodaten-mv.de/dienste/gebaeude_do...,lod2_mv_lod2_33_210_5924_2_gml_zip.zip,https://www.geodaten-mv.de/dienste/gebaeude_at...



Test response headers:
Content-Type        : application/zip
Content-Length      : None
Content-Disposition : attachment; filename="lod2_33_206_5920_2_gml.zip"

Updated download list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/MV/mv_download_list.csv


In [35]:
# ============================================================
# 30 — NORMALISE AND DOWNLOAD ALL MV LoD2 TILES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse, parse_qs
from html import unescape
import time

MV_ZIP_DIR = MV_RAW_DIR / "zips"
MV_ZIP_DIR.mkdir(parents=True, exist_ok=True)

mv_discovered_count = len(mv_downloads)

# Convert "&amp;" back to "&", then remove duplicated URLs
mv_downloads["download_url"] = mv_downloads["download_url"].astype(str).apply(unescape)
mv_downloads = mv_downloads.drop_duplicates(subset="download_url").reset_index(drop=True)

def get_mv_filename(url):
    params = parse_qs(urlparse(url).query)
    return params.get("file", [None])[0]

mv_downloads["filename"] = mv_downloads["download_url"].apply(get_mv_filename)

# Keep only genuine ZIP download links containing a file parameter
mv_invalid_downloads = mv_downloads[mv_downloads["filename"].isna()].copy()
mv_downloads = mv_downloads[mv_downloads["filename"].notna()].copy()
mv_downloads = mv_downloads[mv_downloads["filename"].str.lower().str.endswith(".zip")].copy()
mv_downloads = mv_downloads.drop_duplicates(subset="filename").reset_index(drop=True)

print("=" * 70)
print("MV DOWNLOAD-LIST CORRECTION")
print("=" * 70)
print(f"\nOriginally discovered : {mv_discovered_count:,}")
print(f"Valid unique ZIP URLs : {len(mv_downloads):,}")
print(f"Invalid links removed : {len(mv_invalid_downloads):,}")
print(f"Duplicate filenames   : {mv_downloads['filename'].duplicated().sum():,}")

if mv_downloads.empty:
    raise RuntimeError("No valid MV ZIP download links remain after normalisation.")

if mv_downloads["filename"].duplicated().any():
    raise ValueError("Duplicate MV filenames remain after normalisation.")

mv_download_list_path = MV_RAW_DIR / "mv_download_list.csv"
mv_downloads.to_csv(mv_download_list_path, index=False)

MV_MAX_WORKERS = 8
MV_MAX_RETRIES = 3

def download_mv_tile(url, filename):
    output_path = MV_ZIP_DIR / filename
    temp_path = MV_ZIP_DIR / f"{filename}.part"

    if output_path.exists() and output_path.stat().st_size > 0:
        try:
            with ZipFile(output_path, "r") as archive:
                bad_member = archive.testzip()

            if bad_member is None:
                return {"filename": filename, "status": "already_downloaded", "size_bytes": output_path.stat().st_size, "error": None}

            output_path.unlink()

        except Exception:
            output_path.unlink()

    for attempt in range(1, MV_MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 600)) as response:
                response.raise_for_status()

                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            file.write(chunk)

            with ZipFile(temp_path, "r") as archive:
                bad_member = archive.testzip()

                if bad_member is not None:
                    raise ValueError(f"Corrupt ZIP member: {bad_member}")

            temp_path.replace(output_path)
            return {"filename": filename, "status": "downloaded", "size_bytes": output_path.stat().st_size, "error": None}

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == MV_MAX_RETRIES:
                return {"filename": filename, "status": "failed", "size_bytes": 0, "error": str(error)}

            time.sleep(attempt * 3)

mv_download_results = []

with ThreadPoolExecutor(max_workers=MV_MAX_WORKERS) as executor:
    futures = {executor.submit(download_mv_tile, row.download_url, row.filename): row.filename for row in mv_downloads.itertuples(index=False)}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading MV tiles"):
        mv_download_results.append(future.result())

mv_download_results_df = pd.DataFrame(mv_download_results)
mv_download_results_df.to_csv(REPORT_DIR / "lod2_download_results_MV.csv", index=False)

mv_local_zips = sorted(MV_ZIP_DIR.glob("*.zip"))

print("=" * 70)
print("MV DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nTiles requested       : {len(mv_downloads):,}")
print(f"ZIP files present     : {len(mv_local_zips):,}")
print(f"Downloaded now        : {(mv_download_results_df['status'] == 'downloaded').sum():,}")
print(f"Already downloaded    : {(mv_download_results_df['status'] == 'already_downloaded').sum():,}")
print(f"Failed                : {(mv_download_results_df['status'] == 'failed').sum():,}")
print(f"Total downloaded size : {sum(path.stat().st_size for path in mv_local_zips) / 1024**3:,.2f} GB")

display(mv_download_results_df["status"].value_counts().rename_axis("status").reset_index(name="count"))

if (mv_download_results_df["status"] == "failed").any():
    display(mv_download_results_df[mv_download_results_df["status"] == "failed"].head(30))

MV DOWNLOAD-LIST CORRECTION

Originally discovered : 12,688
Valid unique ZIP URLs : 6,344
Invalid links removed : 0
Duplicate filenames   : 0


MV DOWNLOAD SUMMARY

Tiles requested       : 6,344
ZIP files present     : 6,344
Downloaded now        : 0
Already downloaded    : 6,344
Failed                : 0
Total downloaded size : 1.72 GB


,status,count
0,already_downloaded,6344


In [36]:
# ============================================================
# 31 — TEST ONE MV LoD2 TILE
# ============================================================

mv_test_zip = sorted(MV_ZIP_DIR.glob("*.zip"))[len(list(MV_ZIP_DIR.glob("*.zip"))) // 2]

with ZipFile(mv_test_zip, "r") as archive:
    mv_test_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

    if not mv_test_members:
        raise FileNotFoundError(f"No GML/XML file found inside {mv_test_zip.name}")

    mv_test_member = mv_test_members[0]

    with archive.open(mv_test_member) as stream:
        mv_test_content = stream.read()

mv_test_root = etree.fromstring(mv_test_content)
mv_bldg_namespace = mv_test_root.nsmap.get("bldg")
mv_building_tag = f"{{{mv_bldg_namespace}}}Building"

mv_srs_names = sorted({element.get("srsName") for element in mv_test_root.iter() if element.get("srsName")})

with ZipFile(mv_test_zip, "r") as archive:
    with archive.open(mv_test_member) as stream:
        mv_sample_gdf = extract_citygml_stream(stream, Path(mv_test_member).name, "EPSG:25833")

print("=" * 70)
print("MV SAMPLE TILE TEST")
print("=" * 70)
print(f"\nZIP file             : {mv_test_zip.name}")
print(f"CityGML file         : {mv_test_member}")
print(f"Building namespace   : {mv_bldg_namespace}")
print(f"Building tag         : {mv_building_tag}")
print(f"Buildings extracted  : {len(mv_sample_gdf):,}")
print(f"Measured heights     : {mv_sample_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights      : {mv_sample_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries    : {mv_sample_gdf.geometry.notna().sum():,}")
print(f"Missing geometries   : {mv_sample_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs      : {mv_sample_gdf['lod2_id'].nunique():,}")
print(f"Assigned CRS         : {mv_sample_gdf.crs}")

print("\nCRS values in CityGML:")
for srs_name in mv_srs_names:
    print(f"  {srs_name}")

print("\nFootprint methods:")
display(mv_sample_gdf["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

display(mv_sample_gdf.head())

MV SAMPLE TILE TEST

ZIP file             : lod2_33_344_5982_2_gml.zip
CityGML file         : LoD2_33_344_5982_2_MV.gml
Building namespace   : http://www.opengis.net/citygml/building/1.0
Building tag         : {http://www.opengis.net/citygml/building/1.0}Building
Buildings extracted  : 197
Measured heights     : 197
Missing heights      : 0
Usable geometries    : 197
Missing geometries   : 0
Unique LoD2 IDs      : 197
Assigned CRS         : EPSG:25833

CRS values in CityGML:
  urn:adv:crs:ETRS89_UTM33*DE_DHHN2016_NH

Footprint methods:


,footprint_method,building_count
0,ground_surface,197


,lod2_id,creation_date,function,roof_type,measured_height_m,storeys_above_ground,source_file,footprint_method,geometry
0,DEMVAL72000yDGCs,2022-05-19,31001_2723,1000,2.457,NaN,LoD2_33_344_5982_2_MV.gml,ground_surface,"POLYGON ((345664.022 5983580.516, 345665.495 5..."
1,DEMVAL72000yDHaA,2019-12-18,31001_2724,3100,6.720,NaN,LoD2_33_344_5982_2_MV.gml,ground_surface,"POLYGON ((345859.103 5983547.436, 345866.471 5..."
2,DEMVAL72000mooAr,2019-12-18,31001_2463,3100,3.540,NaN,LoD2_33_344_5982_2_MV.gml,ground_surface,"POLYGON ((345572.426 5983288.854, 345584.815 5..."
3,DEMVAL72000yDHcm,2019-12-18,31001_2724,3100,7.815,NaN,LoD2_33_344_5982_2_MV.gml,ground_surface,"POLYGON ((345977.933 5983510.616, 345969.192 5..."
4,DEMVAL72000yDHca,2022-05-18,31001_1010,3100,7.780,NaN,LoD2_33_344_5982_2_MV.gml,ground_surface,"POLYGON ((345969.192 5983511.72, 345954.863 59..."


In [37]:
# ============================================================
# 32 — PROCESS ALL MV LoD2 TILES IN BATCHES
# ============================================================

MV_OUTPUT_DIR = EXTRACTED_DIR / "MV"
MV_PART_DIR = MV_OUTPUT_DIR / "parts"
MV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MV_PART_DIR.mkdir(parents=True, exist_ok=True)

MV_BATCH_SIZE = 250
mv_zip_files = sorted(MV_ZIP_DIR.glob("*.zip"))
mv_batch_summaries = []
mv_processing_errors = []

for batch_start in tqdm(range(0, len(mv_zip_files), MV_BATCH_SIZE), desc="Processing MV batches"):
    batch_number = batch_start // MV_BATCH_SIZE + 1
    batch_zip_files = mv_zip_files[batch_start:batch_start + MV_BATCH_SIZE]
    part_path = MV_PART_DIR / f"lod2_buildings_MV_part_{batch_number:04d}.parquet"

    if part_path.exists() and part_path.stat().st_size > 0:
        row_count = pq.ParquetFile(part_path).metadata.num_rows
        mv_batch_summaries.append({
            "batch": batch_number,
            "zip_files": len(batch_zip_files),
            "buildings": row_count,
            "status": "already_processed",
            "output_file": str(part_path)
        })
        continue

    batch_parts = []

    for zip_path in batch_zip_files:
        try:
            with ZipFile(zip_path, "r") as archive:
                gml_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

                if not gml_members:
                    mv_processing_errors.append({"zip_file": str(zip_path), "source_file": None, "error": "No GML/XML file found"})
                    continue

                for member in gml_members:
                    with archive.open(member) as stream:
                        tile_gdf = extract_citygml_stream(stream, Path(member).name, "EPSG:25833")

                    if tile_gdf.empty:
                        continue

                    tile_gdf["state_code"] = "MV"
                    tile_gdf["source_state"] = "Mecklenburg-Western Pomerania"
                    tile_gdf["source_crs"] = "EPSG:25833"
                    tile_gdf["citygml_version"] = "1.0"
                    tile_gdf["height_source"] = "lod2_measured"
                    tile_gdf["source_path"] = str(zip_path)
                    batch_parts.append(tile_gdf)

        except Exception as error:
            mv_processing_errors.append({"zip_file": str(zip_path), "source_file": None, "error": str(error)})

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25833")
        batch_gdf.to_parquet(part_path, index=False)

        building_count = len(batch_gdf)
        missing_heights = batch_gdf["measured_height_m"].isna().sum()
        missing_geometry = batch_gdf.geometry.isna().sum()
    else:
        building_count = 0
        missing_heights = 0
        missing_geometry = 0

    mv_batch_summaries.append({
        "batch": batch_number,
        "zip_files": len(batch_zip_files),
        "buildings": building_count,
        "missing_heights": missing_heights,
        "missing_geometry": missing_geometry,
        "status": "processed",
        "output_file": str(part_path) if building_count > 0 else None
    })

    del batch_parts

    if building_count > 0:
        del batch_gdf

    gc.collect()

mv_batch_summary_df = pd.DataFrame(mv_batch_summaries)
mv_processing_errors_df = pd.DataFrame(mv_processing_errors)

mv_batch_summary_path = REPORT_DIR / "lod2_batch_summary_MV.csv"
mv_error_path = REPORT_DIR / "lod2_processing_errors_MV.csv"

mv_batch_summary_df.to_csv(mv_batch_summary_path, index=False)

if not mv_processing_errors_df.empty:
    mv_processing_errors_df.to_csv(mv_error_path, index=False)

mv_part_files = sorted(MV_PART_DIR.glob("*.parquet"))
mv_total_buildings = sum(pq.ParquetFile(path).metadata.num_rows for path in mv_part_files)

print("=" * 70)
print("MV LoD2 BATCH PROCESSING SUMMARY")
print("=" * 70)
print(f"\nZIP files available   : {len(mv_zip_files):,}")
print(f"Parquet parts created : {len(mv_part_files):,}")
print(f"Buildings extracted   : {mv_total_buildings:,}")
print(f"Processing errors     : {len(mv_processing_errors_df):,}")
print(f"Part directory        : {MV_PART_DIR}")
print(f"Batch summary         : {mv_batch_summary_path}")

display(mv_batch_summary_df.tail(20))

if not mv_processing_errors_df.empty:
    display(mv_processing_errors_df.head(30))

Processing MV batches: 100%|██████████████████████████████████████████████████████| 26/26 [00:00<00:00, 339.13it/s]

MV LoD2 BATCH PROCESSING SUMMARY

ZIP files available   : 6,344
Parquet parts created : 26
Buildings extracted   : 1,332,053
Processing errors     : 0
Part directory        : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/MV/parts
Batch summary         : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_batch_summary_MV.csv


,batch,zip_files,buildings,status,output_file
6,7,250,51904,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
7,8,250,61258,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
8,9,250,110227,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
9,10,250,63271,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
10,11,250,43121,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
11,12,250,42572,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
12,13,250,44656,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
13,14,250,52808,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
14,15,250,31219,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
15,16,250,30026,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


In [38]:
# ============================================================
# 33 — SUMMARISE MV RAW LoD2 DATA
# ============================================================

mv_summary_records = []
mv_method_counts = {}

for part_path in tqdm(mv_part_files, desc="Summarising MV parts"):
    part = pd.read_parquet(part_path, columns=["lod2_id", "measured_height_m", "footprint_method", "geometry"])

    mv_summary_records.append({
        "buildings": len(part),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    for method, count in part["footprint_method"].value_counts(dropna=False).items():
        mv_method_counts[method] = mv_method_counts.get(method, 0) + count

mv_part_stats = pd.DataFrame(mv_summary_records)

mv_summary = pd.DataFrame([{
    "state_code": "MV",
    "state_name": "Mecklenburg-Western Pomerania",
    "zip_files": len(mv_zip_files),
    "parquet_parts": len(mv_part_files),
    "buildings": mv_part_stats["buildings"].sum(),
    "measured_heights": mv_part_stats["measured_heights"].sum(),
    "missing_heights": mv_part_stats["missing_heights"].sum(),
    "usable_geometries": mv_part_stats["usable_geometries"].sum(),
    "missing_geometries": mv_part_stats["missing_geometries"].sum(),
    "minimum_height_m": mv_part_stats["minimum_height_m"].min(),
    "maximum_height_m": mv_part_stats["maximum_height_m"].max(),
    "output_directory": str(MV_PART_DIR)
}])

mv_footprint_summary = pd.DataFrame([{"footprint_method": method, "building_count": count} for method, count in mv_method_counts.items()]).sort_values("building_count", ascending=False)

mv_summary_path = REPORT_DIR / "lod2_extraction_summary_MV.csv"
mv_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_MV.csv"

mv_summary.to_csv(mv_summary_path, index=False)
mv_footprint_summary.to_csv(mv_footprint_summary_path, index=False)

print("=" * 70)
print("MV RAW LoD2 SUMMARY")
print("=" * 70)

display(mv_summary)
display(mv_footprint_summary)

print(f"\nState summary saved to     : {mv_summary_path}")
print(f"Footprint summary saved to : {mv_footprint_summary_path}")

Summarising MV parts:   0%|                                                                 | 0/26 [00:00<?, ?it/s]

Summarising MV parts: 100%|████████████████████████████████████████████████████████| 26/26 [00:02<00:00,  9.55it/s]

MV RAW LoD2 SUMMARY


,state_code,state_name,zip_files,parquet_parts,buildings,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,MV,Mecklenburg-Western Pomerania,6344,26,1332053,1332053,0,1332053,0,0.11,268.0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


,footprint_method,building_count
0,ground_surface,1332053



State summary saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_MV.csv
Footprint summary saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_MV.csv


In [39]:
# ============================================================
# 34 — DOWNLOAD SCHLESWIG-HOLSTEIN LoD2 TILE INDEX
# ============================================================

SH_RAW_DIR = RAW_DIR / "SH"
SH_RAW_DIR.mkdir(parents=True, exist_ok=True)

SH_PORTAL_URL = "https://geodaten.schleswig-holstein.de/gaialight-sh/_apps/dladownload/dl-lod2.html"
SH_INDEX_URL = "https://geodaten.schleswig-holstein.de/gaialight-sh/_apps/dladownload/single.php?file=LOD2_SH_Massendownload.geojson&id=4"
SH_INDEX_PATH = SH_RAW_DIR / "LOD2_SH_Massendownload.geojson"

sh_index = source_registry["state_code"].eq("SH")
source_registry.loc[sh_index, ["portal_url", "download_url", "download_method", "format", "horizontal_crs", "download_status", "notes"]] = [SH_PORTAL_URL, SH_INDEX_URL, "GeoJSON tile index", "CityGML", "EPSG:25832", "ready", "Official statewide LoD2 mass-download index"]
source_registry.to_csv(source_registry_path, index=False)

response = requests.get(SH_INDEX_URL, timeout=(30, 300))
response.raise_for_status()
SH_INDEX_PATH.write_bytes(response.content)

sh_tile_index = gpd.read_file(SH_INDEX_PATH)

print("=" * 70)
print("SCHLESWIG-HOLSTEIN LoD2 TILE INDEX")
print("=" * 70)
print(f"\nIndex file       : {SH_INDEX_PATH}")
print(f"File size        : {SH_INDEX_PATH.stat().st_size / 1024**2:,.2f} MB")
print(f"Features found   : {len(sh_tile_index):,}")
print(f"Index CRS        : {sh_tile_index.crs}")
print(f"Columns          : {list(sh_tile_index.columns)}")

display(sh_tile_index.drop(columns="geometry", errors="ignore").head(20))

print("\nExample values containing URLs or ZIP names:")

for column in sh_tile_index.select_dtypes(include="object").columns:
    matches = sh_tile_index[column].astype(str).str.contains(r"https?://|\.zip", case=False, regex=True, na=False)

    if matches.any():
        print(f"\nColumn: {column}")
        display(sh_tile_index.loc[matches, [column]].head(10))
        

SCHLESWIG-HOLSTEIN LoD2 TILE INDEX

Index file       : /fast/home/o-olajuyigbe/data/germany_lod2/raw/SH/LOD2_SH_Massendownload.geojson
File size        : 5.57 MB
Features found   : 13,805
Index CRS        : EPSG:25832
Columns          : ['id', 'datum', 'data_link', 'geometry']


,id,datum,data_link
0,324266004,2023-06-30,https://geodaten.schleswig-holstein.de/gaialig...
1,324276003,2023-06-30,https://geodaten.schleswig-holstein.de/gaialig...
2,324276004,2024-06-30,https://geodaten.schleswig-holstein.de/gaialig...
3,324286004,2023-06-30,https://geodaten.schleswig-holstein.de/gaialig...
4,324296004,2023-06-30,https://geodaten.schleswig-holstein.de/gaialig...
5,324556058,2023-06-30,https://geodaten.schleswig-holstein.de/gaialig...
6,324566055,2023-06-30,https://geodaten.schleswig-holstein.de/gaialig...
7,324566057,2023-06-30,https://geodaten.schleswig-holstein.de/gaialig...
8,324566058,2023-06-30,https://geodaten.schleswig-holstein.de/gaialig...
9,324566059,2024-06-30,https://geodaten.schleswig-holstein.de/gaialig...



Example values containing URLs or ZIP names:

Column: data_link


,data_link
0,https://geodaten.schleswig-holstein.de/gaialig...
1,https://geodaten.schleswig-holstein.de/gaialig...
2,https://geodaten.schleswig-holstein.de/gaialig...
3,https://geodaten.schleswig-holstein.de/gaialig...
4,https://geodaten.schleswig-holstein.de/gaialig...
5,https://geodaten.schleswig-holstein.de/gaialig...
6,https://geodaten.schleswig-holstein.de/gaialig...
7,https://geodaten.schleswig-holstein.de/gaialig...
8,https://geodaten.schleswig-holstein.de/gaialig...
9,https://geodaten.schleswig-holstein.de/gaialig...


In [40]:
# ============================================================
# 35 — PREPARE SH DOWNLOAD LIST AND TEST ONE TILE
# ============================================================

from urllib.parse import urlparse
from email.message import Message

SH_TEST_DIR = SH_RAW_DIR / "test_tile"
SH_TEST_DIR.mkdir(parents=True, exist_ok=True)

sh_downloads = sh_tile_index[["id", "datum", "data_link"]].copy()
sh_downloads = sh_downloads.dropna(subset=["data_link"]).drop_duplicates(subset=["data_link"]).reset_index(drop=True)
sh_downloads["download_url"] = sh_downloads["data_link"].astype(str)

def get_sh_filename(row):
    path_name = Path(urlparse(row["download_url"]).path).name

    if path_name.lower().endswith(".zip"):
        return path_name

    return f"lod2_sh_{row['id']}.zip"

sh_downloads["filename"] = sh_downloads.apply(get_sh_filename, axis=1)

print("=" * 70)
print("SH DOWNLOAD-LIST CHECK")
print("=" * 70)
print(f"\nIndex features       : {len(sh_tile_index):,}")
print(f"Unique download URLs : {sh_downloads['download_url'].nunique():,}")
print(f"Unique filenames     : {sh_downloads['filename'].nunique():,}")
print(f"Duplicate filenames  : {sh_downloads['filename'].duplicated().sum():,}")

if sh_downloads["filename"].duplicated().any():
    sh_downloads["filename"] = sh_downloads.apply(lambda row: f"lod2_sh_{row['id']}.zip", axis=1)

sh_download_list_path = SH_RAW_DIR / "sh_download_list.csv"
sh_downloads.to_csv(sh_download_list_path, index=False)

sh_test_row = sh_downloads.iloc[len(sh_downloads) // 2]
sh_test_response = requests.get(sh_test_row["download_url"], timeout=(30, 300))
sh_test_response.raise_for_status()

content_disposition = sh_test_response.headers.get("content-disposition")
sh_test_filename = sh_test_row["filename"]

if content_disposition:
    message = Message()
    message["content-disposition"] = content_disposition
    header_filename = message.get_filename()

    if header_filename:
        sh_test_filename = header_filename

sh_test_zip = SH_TEST_DIR / sh_test_filename
sh_test_zip.write_bytes(sh_test_response.content)

with ZipFile(sh_test_zip, "r") as archive:
    bad_member = archive.testzip()

    if bad_member is not None:
        raise ValueError(f"Corrupt ZIP member: {bad_member}")

    sh_test_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

    if not sh_test_members:
        raise FileNotFoundError(f"No GML/XML file found in {sh_test_zip.name}")

    sh_test_member = sh_test_members[0]

    with archive.open(sh_test_member) as stream:
        sh_test_content = stream.read()

sh_test_root = etree.fromstring(sh_test_content)
sh_bldg_namespace = sh_test_root.nsmap.get("bldg")
sh_building_tag = f"{{{sh_bldg_namespace}}}Building"
sh_srs_names = sorted({element.get("srsName") for element in sh_test_root.iter() if element.get("srsName")})

def extract_citygml_stream_dynamic(stream, source_file, source_crs, building_tag):
    records = []

    for event, building in etree.iterparse(stream, events=("end",), tag=building_tag, huge_tree=True):
        geometry, footprint_method = extract_lod2_footprint(building)

        records.append({
            "lod2_id": building.get(GML_ID),
            "creation_date": get_first_text(building, "creationDate"),
            "function": get_first_text(building, "function"),
            "roof_type": get_first_text(building, "roofType"),
            "measured_height_m": parse_number(get_first_text(building, "measuredHeight")),
            "storeys_above_ground": parse_number(get_first_text(building, "storeysAboveGround")),
            "source_file": source_file,
            "footprint_method": footprint_method,
            "geometry": geometry
        })

        building.clear()

        while building.getprevious() is not None:
            del building.getparent()[0]

    columns = ["lod2_id", "creation_date", "function", "roof_type", "measured_height_m", "storeys_above_ground", "source_file", "footprint_method", "geometry"]

    if not records:
        return gpd.GeoDataFrame(columns=columns, geometry="geometry", crs=source_crs)

    return gpd.GeoDataFrame(records, geometry="geometry", crs=source_crs)

with ZipFile(sh_test_zip, "r") as archive:
    with archive.open(sh_test_member) as stream:
        sh_sample_gdf = extract_citygml_stream_dynamic(stream, Path(sh_test_member).name, "EPSG:25832", sh_building_tag)

print("\n" + "=" * 70)
print("SH SAMPLE TILE TEST")
print("=" * 70)
print(f"\nZIP file             : {sh_test_zip.name}")
print(f"CityGML file         : {sh_test_member}")
print(f"Building namespace   : {sh_bldg_namespace}")
print(f"Buildings extracted  : {len(sh_sample_gdf):,}")
print(f"Measured heights     : {sh_sample_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights      : {sh_sample_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries    : {sh_sample_gdf.geometry.notna().sum():,}")
print(f"Missing geometries   : {sh_sample_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs      : {sh_sample_gdf['lod2_id'].nunique():,}")
print(f"Assigned CRS         : {sh_sample_gdf.crs}")
print(f"Content-Type         : {sh_test_response.headers.get('content-type')}")
print(f"Content-Disposition  : {content_disposition}")

print("\nCRS values in CityGML:")
for srs_name in sh_srs_names:
    print(f"  {srs_name}")

print("\nFootprint methods:")
display(sh_sample_gdf["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

display(sh_sample_gdf.head())

SH DOWNLOAD-LIST CHECK

Index features       : 13,805
Unique download URLs : 13,805
Unique filenames     : 13,805
Duplicate filenames  : 0


BadZipFile: File is not a zip file

In [41]:
# ============================================================
# 35A — INSPECT THE ACTUAL SH DOWNLOAD RESPONSE
# ============================================================

SH_TEST_DIR = SH_RAW_DIR / "test_tile"
SH_TEST_DIR.mkdir(parents=True, exist_ok=True)

sh_test_row = sh_downloads.iloc[len(sh_downloads) // 2]
sh_test_url = sh_test_row["download_url"]

sh_test_response = requests.get(sh_test_url, allow_redirects=True, timeout=(30, 300))
sh_test_response.raise_for_status()

sh_test_content = sh_test_response.content
sh_content_type = sh_test_response.headers.get("content-type", "")
sh_content_disposition = sh_test_response.headers.get("content-disposition", "")
sh_first_bytes = sh_test_content[:100]
sh_lower_start = sh_test_content[:1000].lstrip().lower()

if sh_test_content.startswith(b"PK\x03\x04"):
    sh_response_type = "ZIP"
elif sh_test_content.startswith(b"\x1f\x8b"):
    sh_response_type = "GZIP"
elif sh_lower_start.startswith(b"<?xml") or b"<core:citymodel" in sh_lower_start or b"<citymodel" in sh_lower_start:
    sh_response_type = "XML / CityGML"
elif b"<html" in sh_lower_start or b"<!doctype html" in sh_lower_start:
    sh_response_type = "HTML"
else:
    sh_response_type = "unknown binary/text"

sh_diagnostic_path = SH_TEST_DIR / "sh_test_response.bin"
sh_diagnostic_path.write_bytes(sh_test_content)

print("=" * 70)
print("SH DOWNLOAD RESPONSE DIAGNOSTIC")
print("=" * 70)
print(f"\nRequested URL        : {sh_test_url}")
print(f"Final URL            : {sh_test_response.url}")
print(f"HTTP status          : {sh_test_response.status_code}")
print(f"Redirects followed   : {len(sh_test_response.history)}")
print(f"Content-Type         : {sh_content_type}")
print(f"Content-Disposition  : {sh_content_disposition}")
print(f"Response size        : {len(sh_test_content) / 1024:,.2f} KB")
print(f"Detected response    : {sh_response_type}")
print(f"First bytes          : {sh_first_bytes!r}")
print(f"Saved diagnostic file: {sh_diagnostic_path}")

if sh_response_type in {"HTML", "XML / CityGML", "unknown binary/text"}:
    print("\nBeginning of response:")
    print(sh_test_content[:1000].decode("utf-8", errors="replace"))

SH DOWNLOAD RESPONSE DIAGNOSTIC

Requested URL        : https://geodaten.schleswig-holstein.de/gaialight-sh/_apps/dladownload/massen.php?file=LoD2_32_553_5938_1_SH.xml&id=4&live=2024&km=32550_5930
Final URL            : https://geodaten.schleswig-holstein.de/gaialight-sh/_apps/dladownload/massen.php?file=LoD2_32_553_5938_1_SH.xml&id=4&live=2024&km=32550_5930
HTTP status          : 200
Redirects followed   : 0
Content-Type         : application/xml
Content-Disposition  : attachment; filename="LoD2_32_553_5938_1_SH.xml"
Response size        : 616.65 KB
Detected response    : XML / CityGML
First bytes          : b'<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<core:CityModel xmlns:bldg="http://www.opengis.net/citygml/bu'
Saved diagnostic file: /fast/home/o-olajuyigbe/data/germany_lod2/raw/SH/test_tile/sh_test_response.bin

Beginning of response:
<?xml version='1.0' encoding='utf-8'?>
<core:CityModel xmlns:bldg="http://www.opengis.net/citygml/building/1.0" xmlns:core="http://www.opengis.net/

In [42]:
# ============================================================
# 35D — PARSE THE CLEANED SH SAMPLE
# ============================================================

from io import BytesIO
import re

def clean_sh_citygml(content):
    citymodel_end = list(re.finditer(br"</(?:[A-Za-z_][\w.-]*:)?CityModel\s*>", content))

    if len(citymodel_end) != 1:
        raise ValueError(f"Expected one closing CityModel tag, found {len(citymodel_end)}.")

    return content[:citymodel_end[0].end()]

sh_clean_content = clean_sh_citygml(sh_test_content)
sh_test_root = etree.fromstring(sh_clean_content)
sh_bldg_namespace = sh_test_root.nsmap.get("bldg")
sh_srs_names = sorted({element.get("srsName") for element in sh_test_root.iter() if element.get("srsName")})
sh_test_filename = parse_qs(urlparse(sh_test_url).query).get("file", [f"lod2_sh_{sh_test_row['id']}.xml"])[0]

sh_sample_gdf = extract_citygml_stream(BytesIO(sh_clean_content), sh_test_filename, "EPSG:25832")

print("=" * 70)
print("SH SAMPLE TILE TEST")
print("=" * 70)
print(f"\nCityGML file         : {sh_test_filename}")
print(f"Building namespace   : {sh_bldg_namespace}")
print(f"Buildings extracted  : {len(sh_sample_gdf):,}")
print(f"Measured heights     : {sh_sample_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights      : {sh_sample_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries    : {sh_sample_gdf.geometry.notna().sum():,}")
print(f"Missing geometries   : {sh_sample_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs      : {sh_sample_gdf['lod2_id'].nunique():,}")
print(f"Assigned CRS         : {sh_sample_gdf.crs}")
print(f"Trailing HTML removed: {len(sh_test_content) - len(sh_clean_content):,} bytes")

print("\nCRS values in CityGML:")
for srs_name in sh_srs_names:
    print(f"  {srs_name}")

print("\nFootprint methods:")
display(sh_sample_gdf["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

display(sh_sample_gdf.head())

SH SAMPLE TILE TEST

CityGML file         : LoD2_32_553_5938_1_SH.xml
Building namespace   : http://www.opengis.net/citygml/building/1.0
Buildings extracted  : 46
Measured heights     : 46
Missing heights      : 0
Usable geometries    : 46
Missing geometries   : 0
Unique LoD2 IDs      : 46
Assigned CRS         : EPSG:25832
Trailing HTML removed: 761 bytes

CRS values in CityGML:
  urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH

Footprint methods:


,footprint_method,building_count
0,ground_surface,46


,lod2_id,creation_date,function,roof_type,measured_height_m,storeys_above_ground,source_file,footprint_method,geometry
0,DESHPDHK0003YpQA,2023-08-15,31001_1010,3100,7.705,2.0,LoD2_32_553_5938_1_SH.xml,ground_surface,"POLYGON ((553674.404 5938810.684, 553674.673 5..."
1,DESHPDHK0003YpQD,2023-08-15,31001_9998,3100,7.377,2.0,LoD2_32_553_5938_1_SH.xml,ground_surface,"POLYGON ((553669.106 5938818.461, 553674.804 5..."
2,DESHPDHK0003YpQC,2023-08-15,31001_9998,2100,6.648,1.0,LoD2_32_553_5938_1_SH.xml,ground_surface,"POLYGON ((553662.538 5938823.649, 553668.276 5..."
3,DESHPDHK0003YpQX,2023-08-15,31001_9998,1000,2.484,NaN,LoD2_32_553_5938_1_SH.xml,ground_surface,"POLYGON ((553798.917 5938914.62, 553801.396 59..."
4,DESHPDHK0003YpQI,2023-08-15,31001_1010,2100,7.628,1.0,LoD2_32_553_5938_1_SH.xml,ground_surface,"POLYGON ((553715.638 5938837.182, 553717.447 5..."


In [43]:
# ============================================================
# 36 — DOWNLOAD ALL SH LoD2 XML TILES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse, parse_qs
import time

SH_XML_DIR = SH_RAW_DIR / "xml"
SH_XML_DIR.mkdir(parents=True, exist_ok=True)

def get_sh_xml_filename(row):
    filename = parse_qs(urlparse(row["download_url"]).query).get("file", [None])[0]
    return filename if filename else f"lod2_sh_{row['id']}.xml"

sh_downloads["filename"] = sh_downloads.apply(get_sh_xml_filename, axis=1)
sh_downloads["filename"] = sh_downloads["filename"].str.replace(r"[^A-Za-z0-9_.-]+", "_", regex=True)

if sh_downloads["filename"].duplicated().any():
    sh_downloads["filename"] = sh_downloads.apply(lambda row: f"{row['id']}_{row['filename']}", axis=1)

print("=" * 70)
print("SH DOWNLOAD-LIST CHECK")
print("=" * 70)
print(f"\nTiles requested      : {len(sh_downloads):,}")
print(f"Unique URLs          : {sh_downloads['download_url'].nunique():,}")
print(f"Unique filenames     : {sh_downloads['filename'].nunique():,}")
print(f"Duplicate filenames  : {sh_downloads['filename'].duplicated().sum():,}")

sh_downloads.to_csv(SH_RAW_DIR / "sh_download_list.csv", index=False)

SH_MAX_WORKERS = 8
SH_MAX_RETRIES = 3

def validate_sh_response(content):
    clean_content = clean_sh_citygml(content)
    root = etree.fromstring(clean_content)
    return etree.QName(root).localname == "CityModel"

def download_sh_tile(url, filename):
    output_path = SH_XML_DIR / filename
    temp_path = SH_XML_DIR / f"{filename}.part"

    if output_path.exists() and output_path.stat().st_size > 0:
        try:
            if validate_sh_response(output_path.read_bytes()):
                return {"filename": filename, "status": "already_downloaded", "size_bytes": output_path.stat().st_size, "error": None}
        except Exception:
            output_path.unlink()

    for attempt in range(1, SH_MAX_RETRIES + 1):
        try:
            response = requests.get(url, timeout=(30, 600))
            response.raise_for_status()
            content = response.content

            if not validate_sh_response(content):
                raise ValueError("Response does not contain a valid CityModel document.")

            temp_path.write_bytes(content)
            temp_path.replace(output_path)

            return {"filename": filename, "status": "downloaded", "size_bytes": output_path.stat().st_size, "error": None}

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == SH_MAX_RETRIES:
                return {"filename": filename, "status": "failed", "size_bytes": 0, "error": str(error)}

            time.sleep(attempt * 3)

sh_download_results = []

with ThreadPoolExecutor(max_workers=SH_MAX_WORKERS) as executor:
    futures = {executor.submit(download_sh_tile, row.download_url, row.filename): row.filename for row in sh_downloads.itertuples(index=False)}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SH XML tiles"):
        sh_download_results.append(future.result())

sh_download_results_df = pd.DataFrame(sh_download_results)
sh_download_results_df.to_csv(REPORT_DIR / "lod2_download_results_SH.csv", index=False)

sh_local_xml = sorted(SH_XML_DIR.glob("*.xml"))

print("=" * 70)
print("SH DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nTiles requested       : {len(sh_downloads):,}")
print(f"XML files present     : {len(sh_local_xml):,}")
print(f"Downloaded now        : {(sh_download_results_df['status'] == 'downloaded').sum():,}")
print(f"Already downloaded    : {(sh_download_results_df['status'] == 'already_downloaded').sum():,}")
print(f"Failed                : {(sh_download_results_df['status'] == 'failed').sum():,}")
print(f"Total downloaded size : {sum(path.stat().st_size for path in sh_local_xml) / 1024**3:,.2f} GB")

display(sh_download_results_df["status"].value_counts().rename_axis("status").reset_index(name="count"))

if (sh_download_results_df["status"] == "failed").any():
    display(sh_download_results_df[sh_download_results_df["status"] == "failed"].head(30))

SH DOWNLOAD-LIST CHECK

Tiles requested      : 13,805
Unique URLs          : 13,805
Unique filenames     : 13,805
Duplicate filenames  : 0


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 37 — PROCESS ALL SH LoD2 XML TILES IN BATCHES
# ============================================================

SH_OUTPUT_DIR = EXTRACTED_DIR / "SH"
SH_PART_DIR = SH_OUTPUT_DIR / "parts"
SH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SH_PART_DIR.mkdir(parents=True, exist_ok=True)

SH_BATCH_SIZE = 5000
sh_xml_files = sorted(SH_XML_DIR.glob("*.xml"))
sh_batch_summaries = []
sh_processing_errors = []

for batch_start in tqdm(range(0, len(sh_xml_files), SH_BATCH_SIZE), desc="Processing SH batches"):
    batch_number = batch_start // SH_BATCH_SIZE + 1
    batch_xml_files = sh_xml_files[batch_start:batch_start + SH_BATCH_SIZE]
    part_path = SH_PART_DIR / f"lod2_buildings_SH_part_{batch_number:04d}.parquet"

    if part_path.exists() and part_path.stat().st_size > 0:
        row_count = pq.ParquetFile(part_path).metadata.num_rows
        sh_batch_summaries.append({"batch": batch_number, "xml_files": len(batch_xml_files), "buildings": row_count, "status": "already_processed", "output_file": str(part_path)})
        continue

    batch_parts = []

    for xml_path in batch_xml_files:
        try:
            content = xml_path.read_bytes()
            clean_content = clean_sh_citygml(content)
            tile_gdf = extract_citygml_stream(BytesIO(clean_content), xml_path.name, "EPSG:25832")

            if tile_gdf.empty:
                continue

            tile_gdf["state_code"] = "SH"
            tile_gdf["source_state"] = "Schleswig-Holstein"
            tile_gdf["source_crs"] = "EPSG:25832"
            tile_gdf["citygml_version"] = "1.0"
            tile_gdf["height_source"] = "lod2_measured"
            tile_gdf["source_path"] = str(xml_path)
            batch_parts.append(tile_gdf)

        except Exception as error:
            sh_processing_errors.append({"source_file": str(xml_path), "error": str(error)})

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)

        building_count = len(batch_gdf)
        missing_heights = batch_gdf["measured_height_m"].isna().sum()
        missing_geometry = batch_gdf.geometry.isna().sum()
    else:
        building_count = 0
        missing_heights = 0
        missing_geometry = 0

    sh_batch_summaries.append({
        "batch": batch_number,
        "xml_files": len(batch_xml_files),
        "buildings": building_count,
        "missing_heights": missing_heights,
        "missing_geometry": missing_geometry,
        "status": "processed",
        "output_file": str(part_path) if building_count > 0 else None
    })

    del batch_parts

    if building_count > 0:
        del batch_gdf

    gc.collect()

sh_batch_summary_df = pd.DataFrame(sh_batch_summaries)
sh_processing_errors_df = pd.DataFrame(sh_processing_errors)

sh_batch_summary_path = REPORT_DIR / "lod2_batch_summary_SH.csv"
sh_error_path = REPORT_DIR / "lod2_processing_errors_SH.csv"

sh_batch_summary_df.to_csv(sh_batch_summary_path, index=False)

if not sh_processing_errors_df.empty:
    sh_processing_errors_df.to_csv(sh_error_path, index=False)

sh_part_files = sorted(SH_PART_DIR.glob("*.parquet"))
sh_total_buildings = sum(pq.ParquetFile(path).metadata.num_rows for path in sh_part_files)

print("=" * 70)
print("SH LoD2 BATCH PROCESSING SUMMARY")
print("=" * 70)
print(f"\nXML files available   : {len(sh_xml_files):,}")
print(f"Parquet parts created : {len(sh_part_files):,}")
print(f"Buildings extracted   : {sh_total_buildings:,}")
print(f"Processing errors     : {len(sh_processing_errors_df):,}")
print(f"Part directory        : {SH_PART_DIR}")
print(f"Batch summary         : {sh_batch_summary_path}")

display(sh_batch_summary_df.tail(20))

if not sh_processing_errors_df.empty:
    display(sh_processing_errors_df.head(30))

Processing SH batches: 100%|█████████████████████████████████████████████████████████████| 3/3 [18:15<00:00, 365.04s/it]

SH LoD2 BATCH PROCESSING SUMMARY

XML files available   : 13,805
Parquet parts created : 3
Buildings extracted   : 2,319,725
Processing errors     : 0
Part directory        : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/SH/parts
Batch summary         : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_batch_summary_SH.csv


,batch,xml_files,buildings,missing_heights,missing_geometry,status,output_file
0,1,5000,608643,0,0,processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
1,2,5000,1030249,0,0,processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
2,3,3805,680833,0,0,processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


In [ ]:
# ============================================================
# 38 — VALIDATE AND SUMMARISE SH RAW LoD2 DATA
# ============================================================

sh_part_files = sorted(SH_PART_DIR.glob("*.parquet"))
sh_expected_files = {path.name for path in sh_xml_files}
sh_processed_files = set()
sh_summary_records = []
sh_method_counts = {}

for part_path in tqdm(sh_part_files, desc="Validating SH parts"):
    part = pd.read_parquet(part_path, columns=["lod2_id", "measured_height_m", "footprint_method", "source_path", "geometry"])
    sh_processed_files.update(Path(path).name for path in part["source_path"].dropna().unique())

    sh_summary_records.append({
        "buildings": len(part),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    for method, count in part["footprint_method"].value_counts(dropna=False).items():
        sh_method_counts[method] = sh_method_counts.get(method, 0) + count

sh_missing_source_files = sorted(sh_expected_files - sh_processed_files)
sh_unexpected_source_files = sorted(sh_processed_files - sh_expected_files)
sh_part_stats = pd.DataFrame(sh_summary_records)

sh_summary = pd.DataFrame([{
    "state_code": "SH",
    "state_name": "Schleswig-Holstein",
    "xml_files_expected": len(sh_expected_files),
    "xml_files_processed": len(sh_processed_files),
    "missing_source_files": len(sh_missing_source_files),
    "unexpected_source_files": len(sh_unexpected_source_files),
    "parquet_parts": len(sh_part_files),
    "buildings": sh_part_stats["buildings"].sum(),
    "measured_heights": sh_part_stats["measured_heights"].sum(),
    "missing_heights": sh_part_stats["missing_heights"].sum(),
    "usable_geometries": sh_part_stats["usable_geometries"].sum(),
    "missing_geometries": sh_part_stats["missing_geometries"].sum(),
    "minimum_height_m": sh_part_stats["minimum_height_m"].min(),
    "maximum_height_m": sh_part_stats["maximum_height_m"].max(),
    "output_directory": str(SH_PART_DIR)
}])

sh_footprint_summary = pd.DataFrame([{"footprint_method": method, "building_count": count} for method, count in sh_method_counts.items()]).sort_values("building_count", ascending=False)

sh_summary_path = REPORT_DIR / "lod2_extraction_summary_SH.csv"
sh_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_SH.csv"
sh_missing_files_path = REPORT_DIR / "lod2_missing_source_files_SH.csv"

sh_summary.to_csv(sh_summary_path, index=False)
sh_footprint_summary.to_csv(sh_footprint_summary_path, index=False)

if sh_missing_source_files:
    pd.DataFrame({"missing_source_file": sh_missing_source_files}).to_csv(sh_missing_files_path, index=False)

print("=" * 70)
print("SH RAW LoD2 VALIDATION SUMMARY")
print("=" * 70)

display(sh_summary)
display(sh_footprint_summary)

print(f"\nState summary saved to     : {sh_summary_path}")
print(f"Footprint summary saved to : {sh_footprint_summary_path}")

if sh_missing_source_files:
    print(f"Missing-file list saved to : {sh_missing_files_path}")
    display(pd.DataFrame({"missing_source_file": sh_missing_source_files}).head(30))

Validating SH parts: 100%|████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.21it/s]

SH RAW LoD2 VALIDATION SUMMARY


,state_code,state_name,xml_files_expected,xml_files_processed,missing_source_files,unexpected_source_files,parquet_parts,buildings,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,SH,Schleswig-Holstein,13805,13805,0,0,3,2319725,2319725,0,2319725,0,1.0,230.176,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


,footprint_method,building_count
0,ground_surface,2319725



State summary saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_SH.csv
Footprint summary saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_SH.csv


In [46]:
# ============================================================
# 39 — DISCOVER LOWER SAXONY LoD2 THROUGH STAC
# ============================================================
import json
NI_RAW_DIR = RAW_DIR / "NI"
NI_RAW_DIR.mkdir(parents=True, exist_ok=True)

NI_PORTAL_URL = "https://lgln-geodaten.niedersachsen.de/startseite/luftbilder_und_3d_produkte/3d_produkte/3d_gebaudemodelle/3d-gebaudemodelle-lod1-und-lod2-142891.html"
NI_STAC_URL = "https://lod.stac.lgln.niedersachsen.de"
NI_COLLECTION_ID = "lod2"

ni_index = source_registry["state_code"].eq("NI")
source_registry.loc[ni_index, ["portal_url", "download_url", "download_method", "format", "citygml_version", "horizontal_crs", "licence", "download_status", "notes"]] = [NI_PORTAL_URL, NI_STAC_URL, "STAC API", "CityGML", "1.0", "EPSG:25832", "CC BY 4.0", "ready", "Official Lower Saxony LoD2 STAC collection"]
source_registry.to_csv(source_registry_path, index=False)

ni_root_response = requests.get(NI_STAC_URL, timeout=(30, 180))
ni_root_response.raise_for_status()
ni_root = ni_root_response.json()

ni_collection_response = requests.get(f"{NI_STAC_URL}/collections/{NI_COLLECTION_ID}", timeout=(30, 180))
ni_collection_response.raise_for_status()
ni_collection = ni_collection_response.json()

ni_items_response = requests.get(f"{NI_STAC_URL}/collections/{NI_COLLECTION_ID}/items", params={"limit": 10}, timeout=(30, 180))
ni_items_response.raise_for_status()
ni_items_sample = ni_items_response.json()

ni_asset_records = []

for item in ni_items_sample.get("features", []):
    for asset_key, asset in item.get("assets", {}).items():
        ni_asset_records.append({
            "item_id": item.get("id"),
            "asset_key": asset_key,
            "href": asset.get("href"),
            "type": asset.get("type"),
            "title": asset.get("title"),
            "roles": ", ".join(asset.get("roles", []))
        })

ni_sample_assets = pd.DataFrame(ni_asset_records)

print("=" * 70)
print("LOWER SAXONY LoD2 STAC DISCOVERY")
print("=" * 70)
print(f"\nCatalog title       : {ni_root.get('title')}")
print(f"STAC version        : {ni_root.get('stac_version')}")
print(f"Collection ID       : {ni_collection.get('id')}")
print(f"Collection title    : {ni_collection.get('title')}")
print(f"Items returned      : {len(ni_items_sample.get('features', [])):,}")
print(f"Total items matched : {ni_items_sample.get('numberMatched', 'not reported')}")
print(f"Asset records       : {len(ni_sample_assets):,}")

print("\nCollection extent:")
display(ni_collection.get("extent", {}))

print("\nSample assets:")
display(ni_sample_assets)

ni_collection_path = NI_RAW_DIR / "ni_lod2_collection.json"
ni_collection_path.write_text(json.dumps(ni_collection, indent=2, ensure_ascii=False), encoding="utf-8")

ni_sample_assets_path = NI_RAW_DIR / "ni_lod2_sample_assets.csv"
ni_sample_assets.to_csv(ni_sample_assets_path, index=False)

print(f"\nCollection metadata saved to: {ni_collection_path}")
print(f"Sample assets saved to      : {ni_sample_assets_path}")

LOWER SAXONY LoD2 STAC DISCOVERY

Catalog title       : 3D-Gebäudemodelle (LoD1 und LoD2)
STAC version        : 1.0.0
Collection ID       : lod2
Collection title    : Buildings 3D - Level of Detail 2 (LoD2)
Items returned      : 10
Total items matched : not reported
Asset records       : 20

Collection extent:


{'spatial': {'bbox': [[6.643266678258008,
    51.30511504732237,
    11.580558426752328,
    53.89605760371077]]},
 'temporal': {'interval': [['2024-05-31T00:00:00Z', '2024-08-30T00:00:00Z']]}}


Sample assets:


,item_id,asset_key,href,type,title,roles
0,LoD2_32_384_5872_1_ni,lod2-gml,https://lod2.s3.eu-de.cloud-object-storage.app...,text/xml,LoD2 cityGML,data
1,LoD2_32_384_5872_1_ni,lod2-shp,https://lod2.s3.eu-de.cloud-object-storage.app...,application/x-zip,LoD2 ESRI 3D-Shape,data
2,LoD2_32_551_5785_1_ni,lod2-gml,https://lod2.s3.eu-de.cloud-object-storage.app...,text/xml,LoD2 cityGML,data
3,LoD2_32_551_5785_1_ni,lod2-shp,https://lod2.s3.eu-de.cloud-object-storage.app...,application/x-zip,LoD2 ESRI 3D-Shape,data
4,LoD2_32_551_5783_1_ni,lod2-gml,https://lod2.s3.eu-de.cloud-object-storage.app...,text/xml,LoD2 cityGML,data
5,LoD2_32_551_5783_1_ni,lod2-shp,https://lod2.s3.eu-de.cloud-object-storage.app...,application/x-zip,LoD2 ESRI 3D-Shape,data
6,LoD2_32_551_5782_1_ni,lod2-gml,https://lod2.s3.eu-de.cloud-object-storage.app...,text/xml,LoD2 cityGML,data
7,LoD2_32_551_5782_1_ni,lod2-shp,https://lod2.s3.eu-de.cloud-object-storage.app...,application/x-zip,LoD2 ESRI 3D-Shape,data
8,LoD2_32_551_5781_1_ni,lod2-gml,https://lod2.s3.eu-de.cloud-object-storage.app...,text/xml,LoD2 cityGML,data
9,LoD2_32_551_5781_1_ni,lod2-shp,https://lod2.s3.eu-de.cloud-object-storage.app...,application/x-zip,LoD2 ESRI 3D-Shape,data



Collection metadata saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/NI/ni_lod2_collection.json
Sample assets saved to      : /fast/home/o-olajuyigbe/data/germany_lod2/raw/NI/ni_lod2_sample_assets.csv


In [47]:
# ============================================================
# 40 — RETRIEVE ALL LOWER SAXONY LoD2 GML ASSETS
# ============================================================

from urllib.parse import urljoin, urlparse

NI_ITEMS_URL = f"{NI_STAC_URL}/collections/{NI_COLLECTION_ID}/items"
ni_next_url = f"{NI_ITEMS_URL}?limit=1000"
ni_download_records = []
ni_pages_read = 0
ni_seen_pages = set()

while ni_next_url:
    if ni_next_url in ni_seen_pages:
        raise RuntimeError("Repeated STAC page detected.")

    ni_seen_pages.add(ni_next_url)
    response = requests.get(ni_next_url, timeout=(30, 300))
    response.raise_for_status()
    page = response.json()
    ni_pages_read += 1

    for item in page.get("features", []):
        asset = item.get("assets", {}).get("lod2-gml")

        if not asset or not asset.get("href"):
            continue

        href = asset["href"]
        filename = Path(urlparse(href).path).name

        if not filename:
            filename = f"{item.get('id')}.gml"

        ni_download_records.append({
            "item_id": item.get("id"),
            "datetime": item.get("properties", {}).get("datetime"),
            "download_url": href,
            "filename": filename,
            "content_type": asset.get("type"),
            "asset_title": asset.get("title")
        })

    next_links = [link.get("href") for link in page.get("links", []) if link.get("rel") == "next" and link.get("href")]
    ni_next_url = urljoin(ni_next_url, next_links[0]) if next_links else None

ni_downloads = pd.DataFrame(ni_download_records)
ni_downloads = ni_downloads.drop_duplicates(subset="download_url").reset_index(drop=True)

print("=" * 70)
print("LOWER SAXONY LoD2 DOWNLOAD DISCOVERY")
print("=" * 70)
print(f"\nSTAC pages read       : {ni_pages_read:,}")
print(f"GML assets found      : {len(ni_downloads):,}")
print(f"Unique item IDs       : {ni_downloads['item_id'].nunique():,}")
print(f"Unique URLs           : {ni_downloads['download_url'].nunique():,}")
print(f"Unique filenames      : {ni_downloads['filename'].nunique():,}")
print(f"Duplicate filenames   : {ni_downloads['filename'].duplicated().sum():,}")
print(f"Missing URLs          : {ni_downloads['download_url'].isna().sum():,}")

if ni_downloads.empty:
    raise RuntimeError("No Lower Saxony LoD2 GML assets were found.")

if ni_downloads["filename"].duplicated().any():
    ni_downloads["filename"] = ni_downloads.apply(lambda row: f"{row['item_id']}.gml", axis=1)

ni_download_list_path = NI_RAW_DIR / "ni_download_list.csv"
ni_downloads.to_csv(ni_download_list_path, index=False)

display(ni_downloads.head(20))
display(ni_downloads.tail(20))

print(f"\nDownload list saved to: {ni_download_list_path}")

LOWER SAXONY LoD2 DOWNLOAD DISCOVERY

STAC pages read       : 38
GML assets found      : 37,928
Unique item IDs       : 37,928
Unique URLs           : 37,928
Unique filenames      : 37,928
Duplicate filenames   : 0
Missing URLs          : 0


,item_id,datetime,download_url,filename,content_type,asset_title
0,LoD2_32_384_5872_1_ni,2024-08-30T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_384_5872_1_ni.gml,text/xml,LoD2 cityGML
1,LoD2_32_551_5785_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5785_1_ni.gml,text/xml,LoD2 cityGML
2,LoD2_32_551_5783_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5783_1_ni.gml,text/xml,LoD2 cityGML
3,LoD2_32_551_5782_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5782_1_ni.gml,text/xml,LoD2 cityGML
4,LoD2_32_551_5781_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5781_1_ni.gml,text/xml,LoD2 cityGML
5,LoD2_32_551_5780_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5780_1_ni.gml,text/xml,LoD2 cityGML
6,LoD2_32_551_5779_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5779_1_ni.gml,text/xml,LoD2 cityGML
7,LoD2_32_551_5778_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5778_1_ni.gml,text/xml,LoD2 cityGML
8,LoD2_32_551_5777_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5777_1_ni.gml,text/xml,LoD2 cityGML
9,LoD2_32_551_5776_1_ni,2024-08-04T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_551_5776_1_ni.gml,text/xml,LoD2 cityGML


,item_id,datetime,download_url,filename,content_type,asset_title
37908,LoD2_32_385_5908_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5908_1_ni.gml,text/xml,LoD2 cityGML
37909,LoD2_32_385_5907_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5907_1_ni.gml,text/xml,LoD2 cityGML
37910,LoD2_32_385_5906_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5906_1_ni.gml,text/xml,LoD2 cityGML
37911,LoD2_32_385_5904_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5904_1_ni.gml,text/xml,LoD2 cityGML
37912,LoD2_32_385_5902_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5902_1_ni.gml,text/xml,LoD2 cityGML
37913,LoD2_32_385_5901_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5901_1_ni.gml,text/xml,LoD2 cityGML
37914,LoD2_32_385_5900_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5900_1_ni.gml,text/xml,LoD2 cityGML
37915,LoD2_32_385_5899_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5899_1_ni.gml,text/xml,LoD2 cityGML
37916,LoD2_32_385_5898_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5898_1_ni.gml,text/xml,LoD2 cityGML
37917,LoD2_32_385_5816_1_ni,2024-05-31T00:00:00Z,https://lod2.s3.eu-de.cloud-object-storage.app...,LoD2_32_385_5816_1_ni.gml,text/xml,LoD2 cityGML



Download list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/NI/ni_download_list.csv


In [49]:
# ============================================================
# 41 — DOWNLOAD AND TEST ONE LOWER SAXONY LoD2 TILE
# ============================================================

from io import BytesIO

NI_TEST_DIR = NI_RAW_DIR / "test_tile"
NI_TEST_DIR.mkdir(parents=True, exist_ok=True)

ni_test_row = ni_downloads.iloc[len(ni_downloads) // 2]
ni_test_url = ni_test_row["download_url"]
ni_test_filename = ni_test_row["filename"]
ni_test_path = NI_TEST_DIR / ni_test_filename

ni_test_response = requests.get(ni_test_url, timeout=(30, 300))
ni_test_response.raise_for_status()
ni_test_content = ni_test_response.content
ni_test_path.write_bytes(ni_test_content)

ni_first_bytes = ni_test_content[:100]
ni_lower_start = ni_test_content[:1000].lstrip().lower()

if ni_lower_start.startswith(b"<?xml") or b"<core:citymodel" in ni_lower_start or b"<citymodel" in ni_lower_start:
    ni_response_type = "XML / CityGML"
elif ni_test_content.startswith(b"PK\x03\x04"):
    ni_response_type = "ZIP"
elif b"<html" in ni_lower_start or b"<!doctype html" in ni_lower_start:
    ni_response_type = "HTML"
else:
    ni_response_type = "unknown"

if ni_response_type != "XML / CityGML":
    raise ValueError(f"Expected CityGML XML but received: {ni_response_type}")

ni_test_root = etree.fromstring(ni_test_content)
ni_bldg_namespace = ni_test_root.nsmap.get("bldg")
ni_building_tag = f"{{{ni_bldg_namespace}}}Building"
ni_srs_names = sorted({element.get("srsName") for element in ni_test_root.iter() if element.get("srsName")})

ni_sample_gdf = extract_citygml_stream(BytesIO(ni_test_content), ni_test_filename, "EPSG:25832")

print("=" * 70)
print("LOWER SAXONY SAMPLE TILE TEST")
print("=" * 70)
print(f"\nCityGML file         : {ni_test_filename}")
print(f"Response type        : {ni_response_type}")
print(f"Content-Type         : {ni_test_response.headers.get('content-type')}")
print(f"Response size        : {len(ni_test_content) / 1024:,.2f} KB")
print(f"Building namespace   : {ni_bldg_namespace}")
print(f"Buildings extracted  : {len(ni_sample_gdf):,}")
print(f"Measured heights     : {ni_sample_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights      : {ni_sample_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries    : {ni_sample_gdf.geometry.notna().sum():,}")
print(f"Missing geometries   : {ni_sample_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs      : {ni_sample_gdf['lod2_id'].nunique():,}")
print(f"Assigned CRS         : {ni_sample_gdf.crs}")
print(f"Saved sample         : {ni_test_path}")

print("\nCRS values in CityGML:")
for srs_name in ni_srs_names:
    print(f"  {srs_name}")

print("\nFootprint methods:")
display(ni_sample_gdf["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

display(ni_sample_gdf.head())

LOWER SAXONY SAMPLE TILE TEST

CityGML file         : LoD2_32_510_5925_1_ni.gml
Response type        : XML / CityGML
Content-Type         : application/octet-stream
Response size        : 16,507.16 KB
Building namespace   : http://www.opengis.net/citygml/building/1.0
Buildings extracted  : 955
Measured heights     : 955
Missing heights      : 0
Usable geometries    : 955
Missing geometries   : 0
Unique LoD2 IDs      : 955
Assigned CRS         : EPSG:25832
Saved sample         : /fast/home/o-olajuyigbe/data/germany_lod2/raw/NI/test_tile/LoD2_32_510_5925_1_ni.gml

CRS values in CityGML:
  urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH

Footprint methods:


,footprint_method,building_count
0,ground_surface,955


,lod2_id,creation_date,function,roof_type,measured_height_m,storeys_above_ground,source_file,footprint_method,geometry
0,DENILD31000070FD,2020-12-25,31001_2000,1000,2.868,NaN,LoD2_32_510_5925_1_ni.gml,ground_surface,"POLYGON ((510670.183 5925506.94, 510675.581 59..."
1,DENILD310000eXlV,2020-12-25,51009_1610,9999,6.583,NaN,LoD2_32_510_5925_1_ni.gml,ground_surface,"POLYGON ((510613.129 5925740.038, 510613.628 5..."
2,DENILD31000072hG,2020-12-25,31001_2000,1000,2.326,NaN,LoD2_32_510_5925_1_ni.gml,ground_surface,"POLYGON ((510584.79 5925782.921, 510584.89 592..."
3,DENILD31000070VR,2020-12-25,31001_2000,1000,2.317,NaN,LoD2_32_510_5925_1_ni.gml,ground_surface,"POLYGON ((510608.677 5925501.163, 510608.741 5..."
4,DENILD31000072p2,2020-12-25,31001_2000,2100,3.039,NaN,LoD2_32_510_5925_1_ni.gml,ground_surface,"POLYGON ((510646.446 5925778.022, 510646.786 5..."


In [50]:
# ============================================================
# 42 — DOWNLOAD ALL LOWER SAXONY LoD2 GML TILES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import time

NI_GML_DIR = NI_RAW_DIR / "gml"
NI_GML_DIR.mkdir(parents=True, exist_ok=True)

NI_MAX_WORKERS = 8
NI_MAX_RETRIES = 3
NI_CHUNK_SIZE = 1024 * 1024

def validate_ni_gml_file(path):
    if not path.exists() or path.stat().st_size < 100:
        return False

    with open(path, "rb") as file:
        beginning = file.read(8192).lower()
        file.seek(max(0, path.stat().st_size - 8192))
        ending = file.read().lower()

    has_start = b"citymodel" in beginning
    has_end = b"</core:citymodel" in ending or re.search(br"</(?:[a-z_][\w.-]*:)?citymodel\s*>", ending) is not None

    return has_start and has_end

def download_ni_tile(url, filename):
    output_path = NI_GML_DIR / filename
    temp_path = NI_GML_DIR / f"{filename}.part"

    if validate_ni_gml_file(output_path):
        return {
            "filename": filename,
            "status": "already_downloaded",
            "size_bytes": output_path.stat().st_size,
            "error": None
        }

    if output_path.exists():
        output_path.unlink()

    for attempt in range(1, NI_MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 900)) as response:
                response.raise_for_status()
                expected_size = int(response.headers.get("content-length", 0))

                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=NI_CHUNK_SIZE):
                        if chunk:
                            file.write(chunk)

            actual_size = temp_path.stat().st_size

            if expected_size and actual_size != expected_size:
                raise ValueError(f"Incomplete download: expected {expected_size:,} bytes, received {actual_size:,}")

            if not validate_ni_gml_file(temp_path):
                raise ValueError("Downloaded file does not appear to contain a complete CityModel document.")

            temp_path.replace(output_path)

            return {
                "filename": filename,
                "status": "downloaded",
                "size_bytes": output_path.stat().st_size,
                "error": None
            }

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == NI_MAX_RETRIES:
                return {
                    "filename": filename,
                    "status": "failed",
                    "size_bytes": 0,
                    "error": str(error)
                }

            time.sleep(attempt * 3)

ni_download_results = []

with ThreadPoolExecutor(max_workers=NI_MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_ni_tile, row.download_url, row.filename): row.filename
        for row in ni_downloads.itertuples(index=False)
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading Lower Saxony GML tiles"):
        ni_download_results.append(future.result())

ni_download_results_df = pd.DataFrame(ni_download_results)
ni_download_results_path = REPORT_DIR / "lod2_download_results_NI.csv"
ni_download_results_df.to_csv(ni_download_results_path, index=False)

ni_local_gml = sorted(NI_GML_DIR.glob("*.gml"))

print("=" * 70)
print("LOWER SAXONY DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nTiles requested       : {len(ni_downloads):,}")
print(f"GML files present     : {len(ni_local_gml):,}")
print(f"Downloaded now        : {(ni_download_results_df['status'] == 'downloaded').sum():,}")
print(f"Already downloaded    : {(ni_download_results_df['status'] == 'already_downloaded').sum():,}")
print(f"Failed                : {(ni_download_results_df['status'] == 'failed').sum():,}")
print(f"Total downloaded size : {sum(path.stat().st_size for path in ni_local_gml) / 1024**3:,.2f} GB")
print(f"Download report       : {ni_download_results_path}")

display(ni_download_results_df["status"].value_counts().rename_axis("status").reset_index(name="count"))

if (ni_download_results_df["status"] == "failed").any():
    display(ni_download_results_df[ni_download_results_df["status"] == "failed"].head(30))

LOWER SAXONY DOWNLOAD SUMMARY

Tiles requested       : 37,928
GML files present     : 37,928
Downloaded now        : 0
Already downloaded    : 37,928
Failed                : 0
Total downloaded size : 136.66 GB
Download report       : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_download_results_NI.csv


,status,count
0,already_downloaded,37928


In [51]:
# ============================================================
# 43 — PROCESS ALL LOWER SAXONY LoD2 TILES IN BATCHES
# ============================================================

NI_OUTPUT_DIR = EXTRACTED_DIR / "NI"
NI_PART_DIR = NI_OUTPUT_DIR / "parts"
NI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NI_PART_DIR.mkdir(parents=True, exist_ok=True)

NI_BATCH_SIZE = 7500
ni_gml_files = sorted(NI_GML_DIR.glob("*.gml"))
ni_batch_starts = list(range(0, len(ni_gml_files), NI_BATCH_SIZE))

print("=" * 70)
print("LOWER SAXONY PROCESSING SETUP")
print("=" * 70)
print(f"\nGML files found       : {len(ni_gml_files):,}")
print(f"Batch size            : {NI_BATCH_SIZE:,}")
print(f"Expected Parquet parts: {len(ni_batch_starts):,}")

if len(ni_gml_files) != len(ni_downloads):
    print(f"\nWARNING: Expected {len(ni_downloads):,} GML files but found {len(ni_gml_files):,}.")

ni_batch_summaries = []
ni_processing_errors = []

for batch_start in tqdm(ni_batch_starts, total=len(ni_batch_starts), desc="Processing Lower Saxony batches"):
    batch_number = batch_start // NI_BATCH_SIZE + 1
    batch_gml_files = ni_gml_files[batch_start:batch_start + NI_BATCH_SIZE]
    part_path = NI_PART_DIR / f"lod2_buildings_NI_part_{batch_number:04d}.parquet"

    if part_path.exists() and part_path.stat().st_size > 0:
        row_count = pq.ParquetFile(part_path).metadata.num_rows
        ni_batch_summaries.append({
            "batch": batch_number,
            "gml_files": len(batch_gml_files),
            "buildings": row_count,
            "status": "already_processed",
            "output_file": str(part_path)
        })
        continue

    batch_parts = []

    for gml_path in batch_gml_files:
        try:
            with open(gml_path, "rb") as stream:
                tile_gdf = extract_citygml_stream(stream, gml_path.name, "EPSG:25832")

            if tile_gdf.empty:
                continue

            tile_gdf["state_code"] = "NI"
            tile_gdf["source_state"] = "Lower Saxony"
            tile_gdf["source_crs"] = "EPSG:25832"
            tile_gdf["citygml_version"] = "1.0"
            tile_gdf["height_source"] = "lod2_measured"
            tile_gdf["source_path"] = str(gml_path)
            batch_parts.append(tile_gdf)

        except Exception as error:
            ni_processing_errors.append({"source_file": str(gml_path), "error": str(error)})

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)

        building_count = len(batch_gdf)
        missing_heights = batch_gdf["measured_height_m"].isna().sum()
        missing_geometry = batch_gdf.geometry.isna().sum()
    else:
        building_count = 0
        missing_heights = 0
        missing_geometry = 0

    ni_batch_summaries.append({
        "batch": batch_number,
        "gml_files": len(batch_gml_files),
        "buildings": building_count,
        "missing_heights": missing_heights,
        "missing_geometry": missing_geometry,
        "status": "processed",
        "output_file": str(part_path) if building_count > 0 else None
    })

    del batch_parts

    if building_count > 0:
        del batch_gdf

    gc.collect()

ni_batch_summary_df = pd.DataFrame(ni_batch_summaries)
ni_processing_errors_df = pd.DataFrame(ni_processing_errors)

ni_batch_summary_path = REPORT_DIR / "lod2_batch_summary_NI.csv"
ni_error_path = REPORT_DIR / "lod2_processing_errors_NI.csv"

ni_batch_summary_df.to_csv(ni_batch_summary_path, index=False)

if not ni_processing_errors_df.empty:
    ni_processing_errors_df.to_csv(ni_error_path, index=False)

ni_part_files = sorted(NI_PART_DIR.glob("*.parquet"))
ni_total_buildings = sum(pq.ParquetFile(path).metadata.num_rows for path in ni_part_files)

print("=" * 70)
print("LOWER SAXONY LoD2 BATCH PROCESSING SUMMARY")
print("=" * 70)
print(f"\nGML files available   : {len(ni_gml_files):,}")
print(f"Expected parts        : {len(ni_batch_starts):,}")
print(f"Parquet parts created : {len(ni_part_files):,}")
print(f"Buildings extracted   : {ni_total_buildings:,}")
print(f"Processing errors     : {len(ni_processing_errors_df):,}")
print(f"Part directory        : {NI_PART_DIR}")
print(f"Batch summary         : {ni_batch_summary_path}")

display(ni_batch_summary_df.tail(20))

if not ni_processing_errors_df.empty:
    display(ni_processing_errors_df.head(30))

LOWER SAXONY PROCESSING SETUP

GML files found       : 37,928
Batch size            : 7,500
Expected Parquet parts: 6


Processing Lower Saxony batches: 100%|██████████████████████████████████████████████| 6/6 [00:00<00:00, 397.23it/s]

LOWER SAXONY LoD2 BATCH PROCESSING SUMMARY

GML files available   : 37,928
Expected parts        : 6
Parquet parts created : 6
Buildings extracted   : 4,268,233
Processing errors     : 0
Part directory        : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/NI/parts
Batch summary         : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_batch_summary_NI.csv


,batch,gml_files,buildings,status,output_file
0,1,7500,9001,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
1,2,7500,4961,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
2,3,7500,1134862,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
3,4,7500,1572537,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
4,5,7500,1496001,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
5,6,428,50871,already_processed,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


In [52]:
# ============================================================
# 44 — VALIDATE LOWER SAXONY SOURCE-FILE COVERAGE
# ============================================================

from collections import Counter

ni_gml_files = sorted(NI_GML_DIR.glob("*.gml"))
ni_part_files = sorted(NI_PART_DIR.glob("*.parquet"))
ni_expected_files = {path.name for path in ni_gml_files}

ni_effective_batch_size = int(ni_batch_summary_df["gml_files"].max())
ni_processed_files = set()
ni_file_part_counts = Counter()
ni_part_validation_records = []

for part_path in tqdm(ni_part_files, desc="Validating Lower Saxony parts"):
    part_number = int(part_path.stem.rsplit("_", 1)[-1])
    start = (part_number - 1) * ni_effective_batch_size
    expected_part_files = {path.name for path in ni_gml_files[start:start + ni_effective_batch_size]}

    part = pd.read_parquet(part_path, columns=["source_path", "measured_height_m", "geometry"])
    processed_part_files = {Path(path).name for path in part["source_path"].dropna().unique()}

    ni_processed_files.update(processed_part_files)

    for filename in processed_part_files:
        ni_file_part_counts[filename] += 1

    ni_part_validation_records.append({
        "part": part_number,
        "expected_source_files": len(expected_part_files),
        "processed_source_files": len(processed_part_files),
        "missing_source_files": len(expected_part_files - processed_part_files),
        "unexpected_source_files": len(processed_part_files - expected_part_files),
        "buildings": len(part),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "part_path": str(part_path)
    })

    del part
    gc.collect()

ni_part_validation = pd.DataFrame(ni_part_validation_records).sort_values("part").reset_index(drop=True)
ni_missing_source_files = sorted(ni_expected_files - ni_processed_files)
ni_unexpected_source_files = sorted(ni_processed_files - ni_expected_files)
ni_source_files_in_multiple_parts = sorted([filename for filename, count in ni_file_part_counts.items() if count > 1])

ni_validation_summary = pd.DataFrame([{
    "gml_files_expected": len(ni_expected_files),
    "gml_files_processed": len(ni_processed_files),
    "missing_source_files": len(ni_missing_source_files),
    "unexpected_source_files": len(ni_unexpected_source_files),
    "source_files_in_multiple_parts": len(ni_source_files_in_multiple_parts),
    "parquet_parts": len(ni_part_files),
    "buildings": ni_part_validation["buildings"].sum(),
    "missing_heights": ni_part_validation["missing_heights"].sum(),
    "missing_geometries": ni_part_validation["missing_geometries"].sum()
}])

ni_part_validation_path = REPORT_DIR / "lod2_part_validation_NI.csv"
ni_missing_files_path = REPORT_DIR / "lod2_missing_source_files_NI.csv"

ni_part_validation.to_csv(ni_part_validation_path, index=False)

if ni_missing_source_files:
    pd.DataFrame({"missing_source_file": ni_missing_source_files}).to_csv(ni_missing_files_path, index=False)

print("=" * 70)
print("LOWER SAXONY RAW LoD2 VALIDATION")
print("=" * 70)

display(ni_validation_summary)
display(ni_part_validation)

print(f"\nPart validation saved to : {ni_part_validation_path}")

if ni_missing_source_files:
    print(f"Missing-file list saved to: {ni_missing_files_path}")
    display(pd.DataFrame({"missing_source_file": ni_missing_source_files}).head(30))

Validating Lower Saxony parts: 100%|█████████████████████████████████████████████████| 6/6 [00:09<00:00,  1.62s/it]

LOWER SAXONY RAW LoD2 VALIDATION


,gml_files_expected,gml_files_processed,missing_source_files,unexpected_source_files,source_files_in_multiple_parts,parquet_parts,buildings,missing_heights,missing_geometries
0,37928,23127,14801,0,0,6,4268233,0,25


,part,expected_source_files,processed_source_files,missing_source_files,unexpected_source_files,buildings,missing_heights,missing_geometries,part_path
0,1,7500,100,7400,0,9001,0,0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
1,2,7500,100,7500,100,4961,0,0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
2,3,7500,7500,0,0,1134862,0,11,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
3,4,7500,7500,0,0,1572537,0,7,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
4,5,7500,7499,1,0,1496001,0,7,/fast/home/o-olajuyigbe/data/germany_lod2/extr...
5,6,428,428,0,0,50871,0,0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...



Part validation saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_part_validation_NI.csv
Missing-file list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_missing_source_files_NI.csv


,missing_source_file
0,LoD2_32_353_5819_1_ni.gml
1,LoD2_32_353_5821_1_ni.gml
2,LoD2_32_353_5822_1_ni.gml
3,LoD2_32_353_5823_1_ni.gml
4,LoD2_32_353_5824_1_ni.gml
5,LoD2_32_353_5825_1_ni.gml
6,LoD2_32_353_5826_1_ni.gml
7,LoD2_32_353_5827_1_ni.gml
8,LoD2_32_353_5828_1_ni.gml
9,LoD2_32_353_5829_1_ni.gml


In [53]:
# ============================================================
# 45 — REPAIR MISSING LOWER SAXONY LoD2 TILES
# ============================================================

NI_REPAIR_DIR = NI_PART_DIR / "repair"
NI_REPAIR_DIR.mkdir(parents=True, exist_ok=True)

NI_REPAIR_BATCH_SIZE = 500

if "ni_missing_source_files" not in globals():
    ni_missing_source_files = pd.read_csv(REPORT_DIR / "lod2_missing_source_files_NI.csv")["missing_source_file"].tolist()

ni_gml_lookup = {path.name: path for path in NI_GML_DIR.glob("*.gml")}
ni_repair_files = [ni_gml_lookup[name] for name in ni_missing_source_files if name in ni_gml_lookup]
ni_missing_on_disk = sorted(set(ni_missing_source_files) - set(ni_gml_lookup))

if ni_missing_on_disk:
    raise FileNotFoundError(f"{len(ni_missing_on_disk):,} missing source files are not present in NI_GML_DIR.")

ni_repair_batch_starts = list(range(0, len(ni_repair_files), NI_REPAIR_BATCH_SIZE))
ni_repair_summaries = []

print("=" * 70)
print("LOWER SAXONY REPAIR SETUP")
print("=" * 70)
print(f"\nMissing source files : {len(ni_missing_source_files):,}")
print(f"Files available      : {len(ni_repair_files):,}")
print(f"Repair batch size    : {NI_REPAIR_BATCH_SIZE:,}")
print(f"Expected repair parts: {len(ni_repair_batch_starts):,}")

for batch_start in tqdm(ni_repair_batch_starts, total=len(ni_repair_batch_starts), desc="Repairing Lower Saxony tiles"):
    batch_number = batch_start // NI_REPAIR_BATCH_SIZE + 1
    batch_files = ni_repair_files[batch_start:batch_start + NI_REPAIR_BATCH_SIZE]

    part_path = NI_REPAIR_DIR / f"lod2_buildings_NI_repair_{batch_number:04d}.parquet"
    manifest_path = NI_REPAIR_DIR / f"lod2_manifest_NI_repair_{batch_number:04d}.csv"

    if manifest_path.exists():
        existing_manifest = pd.read_csv(manifest_path)
        complete_manifest = len(existing_manifest) == len(batch_files) and not existing_manifest["status"].eq("failed").any()
        output_complete = existing_manifest["buildings"].sum() == 0 or part_path.exists()

        if complete_manifest and output_complete:
            ni_repair_summaries.append({
                "batch": batch_number,
                "source_files": len(existing_manifest),
                "processed_files": existing_manifest["status"].eq("processed").sum(),
                "empty_files": existing_manifest["status"].eq("empty").sum(),
                "failed_files": existing_manifest["status"].eq("failed").sum(),
                "buildings": existing_manifest["buildings"].sum(),
                "status": "already_processed"
            })
            continue

    if part_path.exists():
        part_path.unlink()

    batch_parts = []
    batch_manifest = []

    for gml_path in batch_files:
        try:
            with open(gml_path, "rb") as stream:
                tile_gdf = extract_citygml_stream(stream, gml_path.name, "EPSG:25832")

            if tile_gdf.empty:
                batch_manifest.append({"source_file": gml_path.name, "source_path": str(gml_path), "status": "empty", "buildings": 0, "error": None})
                continue

            tile_gdf["state_code"] = "NI"
            tile_gdf["source_state"] = "Lower Saxony"
            tile_gdf["source_crs"] = "EPSG:25832"
            tile_gdf["citygml_version"] = "1.0"
            tile_gdf["height_source"] = "lod2_measured"
            tile_gdf["source_path"] = str(gml_path)

            batch_parts.append(tile_gdf)
            batch_manifest.append({"source_file": gml_path.name, "source_path": str(gml_path), "status": "processed", "buildings": len(tile_gdf), "error": None})

        except Exception as error:
            batch_manifest.append({"source_file": gml_path.name, "source_path": str(gml_path), "status": "failed", "buildings": 0, "error": str(error)})

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    batch_manifest_df = pd.DataFrame(batch_manifest)
    batch_manifest_df.to_csv(manifest_path, index=False)

    ni_repair_summaries.append({
        "batch": batch_number,
        "source_files": len(batch_manifest_df),
        "processed_files": batch_manifest_df["status"].eq("processed").sum(),
        "empty_files": batch_manifest_df["status"].eq("empty").sum(),
        "failed_files": batch_manifest_df["status"].eq("failed").sum(),
        "buildings": batch_manifest_df["buildings"].sum(),
        "status": "processed"
    })

    del batch_parts, batch_manifest, batch_manifest_df
    gc.collect()

ni_repair_summary_df = pd.DataFrame(ni_repair_summaries)
ni_repair_manifest_files = sorted(NI_REPAIR_DIR.glob("lod2_manifest_NI_repair_*.csv"))
ni_repair_manifests = pd.concat([pd.read_csv(path) for path in ni_repair_manifest_files], ignore_index=True)

ni_repair_summary_path = REPORT_DIR / "lod2_repair_summary_NI.csv"
ni_repair_manifest_path = REPORT_DIR / "lod2_repair_manifest_NI.csv"

ni_repair_summary_df.to_csv(ni_repair_summary_path, index=False)
ni_repair_manifests.to_csv(ni_repair_manifest_path, index=False)

print("=" * 70)
print("LOWER SAXONY REPAIR SUMMARY")
print("=" * 70)
print(f"\nSource files repaired : {len(ni_repair_manifests):,}")
print(f"Files with buildings  : {ni_repair_manifests['status'].eq('processed').sum():,}")
print(f"Empty CityGML tiles   : {ni_repair_manifests['status'].eq('empty').sum():,}")
print(f"Failed files          : {ni_repair_manifests['status'].eq('failed').sum():,}")
print(f"Buildings recovered   : {ni_repair_manifests['buildings'].sum():,}")
print(f"Repair Parquet parts  : {len(list(NI_REPAIR_DIR.glob('*.parquet'))):,}")
print(f"Repair directory      : {NI_REPAIR_DIR}")
print(f"Combined manifest     : {ni_repair_manifest_path}")

display(ni_repair_manifests["status"].value_counts().rename_axis("status").reset_index(name="source_files"))

if ni_repair_manifests["status"].eq("failed").any():
    display(ni_repair_manifests[ni_repair_manifests["status"] == "failed"].head(30))

LOWER SAXONY REPAIR SETUP

Missing source files : 14,801
Files available      : 14,801
Repair batch size    : 500
Expected repair parts: 30


Repairing Lower Saxony tiles: 100%|████████████████████████████████████████████████| 30/30 [35:58<00:00, 71.95s/it]

LOWER SAXONY REPAIR SUMMARY

Source files repaired : 14,801
Files with buildings  : 14,800
Empty CityGML tiles   : 1
Failed files          : 0
Buildings recovered   : 2,559,818
Repair Parquet parts  : 30
Repair directory      : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/NI/parts/repair
Combined manifest     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_repair_manifest_NI.csv


,status,source_files
0,processed,14800
1,empty,1


In [54]:
# ============================================================
# 46 — FINAL LOWER SAXONY VALIDATION AND SUMMARY
# ============================================================

ni_original_parts = sorted(NI_PART_DIR.glob("lod2_buildings_NI_part_*.parquet"))
ni_repair_parts = sorted(NI_REPAIR_DIR.glob("lod2_buildings_NI_repair_*.parquet"))
ni_all_parts = ni_original_parts + ni_repair_parts

ni_expected_files = {path.name for path in NI_GML_DIR.glob("*.gml")}
ni_files_with_buildings = set()
ni_file_occurrences = Counter()
ni_summary_records = []
ni_method_counts = {}

for part_path in tqdm(ni_all_parts, desc="Validating all Lower Saxony parts"):
    part = pd.read_parquet(part_path, columns=["source_path", "measured_height_m", "footprint_method", "geometry"])
    source_files = {Path(path).name for path in part["source_path"].dropna().unique()}

    ni_files_with_buildings.update(source_files)

    for filename in source_files:
        ni_file_occurrences[filename] += 1

    ni_summary_records.append({
        "buildings": len(part),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    for method, count in part["footprint_method"].value_counts(dropna=False).items():
        ni_method_counts[method] = ni_method_counts.get(method, 0) + count

    del part
    gc.collect()

ni_empty_files = set(ni_repair_manifests.loc[ni_repair_manifests["status"] == "empty", "source_file"])
ni_failed_files = set(ni_repair_manifests.loc[ni_repair_manifests["status"] == "failed", "source_file"])
ni_accounted_files = ni_files_with_buildings | ni_empty_files

ni_missing_after_repair = sorted(ni_expected_files - ni_accounted_files)
ni_unexpected_files = sorted(ni_accounted_files - ni_expected_files)
ni_files_in_multiple_parts = sorted([filename for filename, count in ni_file_occurrences.items() if count > 1])

ni_part_stats = pd.DataFrame(ni_summary_records)

ni_summary = pd.DataFrame([{
    "state_code": "NI",
    "state_name": "Lower Saxony",
    "gml_files_expected": len(ni_expected_files),
    "files_with_buildings": len(ni_files_with_buildings),
    "empty_files": len(ni_empty_files),
    "files_accounted_for": len(ni_accounted_files),
    "missing_source_files": len(ni_missing_after_repair),
    "unexpected_source_files": len(ni_unexpected_files),
    "source_files_in_multiple_parts": len(ni_files_in_multiple_parts),
    "failed_files": len(ni_failed_files),
    "original_parquet_parts": len(ni_original_parts),
    "repair_parquet_parts": len(ni_repair_parts),
    "buildings": ni_part_stats["buildings"].sum(),
    "measured_heights": ni_part_stats["measured_heights"].sum(),
    "missing_heights": ni_part_stats["missing_heights"].sum(),
    "usable_geometries": ni_part_stats["usable_geometries"].sum(),
    "missing_geometries": ni_part_stats["missing_geometries"].sum(),
    "minimum_height_m": ni_part_stats["minimum_height_m"].min(),
    "maximum_height_m": ni_part_stats["maximum_height_m"].max(),
    "output_directory": str(NI_PART_DIR)
}])

ni_footprint_summary = pd.DataFrame([{"footprint_method": method, "building_count": count} for method, count in ni_method_counts.items()]).sort_values("building_count", ascending=False)

ni_summary_path = REPORT_DIR / "lod2_extraction_summary_NI.csv"
ni_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_NI.csv"
ni_remaining_missing_path = REPORT_DIR / "lod2_missing_source_files_after_repair_NI.csv"

ni_summary.to_csv(ni_summary_path, index=False)
ni_footprint_summary.to_csv(ni_footprint_summary_path, index=False)

if ni_missing_after_repair:
    pd.DataFrame({"missing_source_file": ni_missing_after_repair}).to_csv(ni_remaining_missing_path, index=False)

print("=" * 70)
print("LOWER SAXONY FINAL LoD2 VALIDATION")
print("=" * 70)

display(ni_summary)
display(ni_footprint_summary)

print(f"\nState summary saved to     : {ni_summary_path}")
print(f"Footprint summary saved to : {ni_footprint_summary_path}")

if ni_missing_after_repair:
    print(f"Remaining missing list     : {ni_remaining_missing_path}")
    display(pd.DataFrame({"missing_source_file": ni_missing_after_repair}).head(30))

Validating all Lower Saxony parts: 100%|███████████████████████████████████████████| 36/36 [00:39<00:00,  1.08s/it]

LOWER SAXONY FINAL LoD2 VALIDATION


,state_code,state_name,gml_files_expected,files_with_buildings,empty_files,files_accounted_for,missing_source_files,unexpected_source_files,source_files_in_multiple_parts,failed_files,original_parquet_parts,repair_parquet_parts,buildings,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,NI,Lower Saxony,37928,37927,1,37928,0,0,0,0,6,30,6828051,6828051,0,6828014,37,0.0,344.375,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


,footprint_method,building_count
0,ground_surface,6828014
1,missing,37



State summary saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_NI.csv
Footprint summary saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_NI.csv


In [55]:
# ============================================================
# 47 — DISCOVER SAXONY-ANHALT LoD2 ARCHIVES
# ============================================================

from urllib.parse import urljoin, urlparse
from lxml import html
import re

ST_RAW_DIR = RAW_DIR / "ST"
ST_RAW_DIR.mkdir(parents=True, exist_ok=True)

ST_PORTAL_URL = "https://www.geodatenportal.sachsen-anhalt.de/gfds/de/gdp-download-lod2-landesweit.html"

st_index = source_registry["state_code"].eq("ST")
source_registry.loc[st_index, ["portal_url", "download_url", "download_method", "format", "horizontal_crs", "download_status", "notes"]] = [ST_PORTAL_URL, ST_PORTAL_URL, "four statewide ZIP archives", "CityGML", "EPSG:25832", "ready", "Official statewide LoD2 split into four parts"]
source_registry.to_csv(source_registry_path, index=False)

response = requests.get(ST_PORTAL_URL, timeout=(30, 180))
response.raise_for_status()

document = html.fromstring(response.content)

candidate_urls = set()

for href in document.xpath("//a/@href"):
    url = urljoin(ST_PORTAL_URL, href)
    lower_url = url.lower()

    if ".zip" in lower_url and ("lod2" in lower_url or "3d" in lower_url or "gebaeude" in lower_url):
        candidate_urls.add(url)

for raw_url in re.findall(r'https?://[^\s"<>]+?\.zip(?:\?[^\s"<>]*)?', response.text, flags=re.IGNORECASE):
    lower_url = raw_url.lower()

    if "lod2" in lower_url or "3d" in lower_url or "gebaeude" in lower_url:
        candidate_urls.add(raw_url.replace("&amp;", "&"))

st_downloads = pd.DataFrame({"download_url": sorted(candidate_urls)})

if not st_downloads.empty:
    st_downloads["filename"] = st_downloads["download_url"].apply(lambda url: Path(urlparse(url).path).name)
    st_downloads = st_downloads.drop_duplicates(subset="download_url").reset_index(drop=True)

print("=" * 70)
print("SAXONY-ANHALT LoD2 ARCHIVE DISCOVERY")
print("=" * 70)
print(f"\nZIP archives found : {len(st_downloads):,}")

display(st_downloads)

st_download_list_path = ST_RAW_DIR / "st_download_list.csv"
st_downloads.to_csv(st_download_list_path, index=False)

print(f"\nDownload list saved to: {st_download_list_path}")

if len(st_downloads) != 4:
    print("\nThe page should provide four statewide LoD2 ZIP archives.")
    print("All ZIP-like links found on the page:")

    all_zip_links = sorted({
        urljoin(ST_PORTAL_URL, href)
        for href in document.xpath("//a/@href")
        if ".zip" in href.lower()
    })

    display(pd.DataFrame({"zip_link": all_zip_links}))

SAXONY-ANHALT LoD2 ARCHIVE DISCOVERY

ZIP archives found : 4


,download_url,filename
0,https://www.geodatenportal.sachsen-anhalt.de/g...,LoD2-1.zip
1,https://www.geodatenportal.sachsen-anhalt.de/g...,LoD2-2.zip
2,https://www.geodatenportal.sachsen-anhalt.de/g...,LoD2-3.zip
3,https://www.geodatenportal.sachsen-anhalt.de/g...,LoD2-4.zip



Download list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/ST/st_download_list.csv


In [56]:
# ============================================================
# 48 — DOWNLOAD SAXONY-ANHALT LoD2 ARCHIVES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import time

ST_ZIP_DIR = ST_RAW_DIR / "zips"
ST_ZIP_DIR.mkdir(parents=True, exist_ok=True)

ST_MAX_WORKERS = 4
ST_MAX_RETRIES = 3

def download_st_archive(url, filename):
    output_path = ST_ZIP_DIR / filename
    temp_path = ST_ZIP_DIR / f"{filename}.part"

    if output_path.exists() and output_path.stat().st_size > 0:
        try:
            with ZipFile(output_path, "r") as archive:
                bad_member = archive.testzip()

            if bad_member is None:
                return {"filename": filename, "status": "already_downloaded", "size_bytes": output_path.stat().st_size, "error": None}

            output_path.unlink()

        except Exception:
            output_path.unlink()

    for attempt in range(1, ST_MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 1800)) as response:
                response.raise_for_status()

                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            file.write(chunk)

            with ZipFile(temp_path, "r") as archive:
                bad_member = archive.testzip()

                if bad_member is not None:
                    raise ValueError(f"Corrupt ZIP member: {bad_member}")

            temp_path.replace(output_path)

            return {"filename": filename, "status": "downloaded", "size_bytes": output_path.stat().st_size, "error": None}

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == ST_MAX_RETRIES:
                return {"filename": filename, "status": "failed", "size_bytes": 0, "error": str(error)}

            time.sleep(attempt * 5)

st_download_results = []

with ThreadPoolExecutor(max_workers=ST_MAX_WORKERS) as executor:
    futures = {executor.submit(download_st_archive, row.download_url, row.filename): row.filename for row in st_downloads.itertuples(index=False)}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading Saxony-Anhalt archives"):
        st_download_results.append(future.result())

st_download_results_df = pd.DataFrame(st_download_results)
st_download_results_path = REPORT_DIR / "lod2_download_results_ST.csv"
st_download_results_df.to_csv(st_download_results_path, index=False)

st_zip_files = sorted(ST_ZIP_DIR.glob("*.zip"))

print("=" * 70)
print("SAXONY-ANHALT DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nArchives requested     : {len(st_downloads):,}")
print(f"ZIP archives present   : {len(st_zip_files):,}")
print(f"Downloaded now         : {(st_download_results_df['status'] == 'downloaded').sum():,}")
print(f"Already downloaded     : {(st_download_results_df['status'] == 'already_downloaded').sum():,}")
print(f"Failed                 : {(st_download_results_df['status'] == 'failed').sum():,}")
print(f"Total downloaded size  : {sum(path.stat().st_size for path in st_zip_files) / 1024**3:,.2f} GB")
print(f"Download report        : {st_download_results_path}")

display(st_download_results_df["status"].value_counts().rename_axis("status").reset_index(name="count"))

if (st_download_results_df["status"] == "failed").any():
    display(st_download_results_df[st_download_results_df["status"] == "failed"])

SAXONY-ANHALT DOWNLOAD SUMMARY

Archives requested     : 4
ZIP archives present   : 4
Downloaded now         : 4
Already downloaded     : 0
Failed                 : 0
Total downloaded size  : 3.39 GB
Download report        : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_download_results_ST.csv


,status,count
0,downloaded,4


In [57]:
# ============================================================
# 49 — INSPECT AND TEST SAXONY-ANHALT LoD2 ARCHIVES
# ============================================================

from io import BytesIO

st_archive_records = []

for zip_path in st_zip_files:
    with ZipFile(zip_path, "r") as archive:
        members = [info for info in archive.infolist() if not info.is_dir()]
        gml_members = [info.filename for info in members if info.filename.lower().endswith((".gml", ".xml"))]
        nested_zips = [info.filename for info in members if info.filename.lower().endswith(".zip")]

        st_archive_records.append({
            "archive": zip_path.name,
            "files": len(members),
            "gml_xml_files": len(gml_members),
            "nested_zip_files": len(nested_zips),
            "uncompressed_size_gb": sum(info.file_size for info in members) / 1024**3
        })

st_archive_summary = pd.DataFrame(st_archive_records)

print("=" * 70)
print("SAXONY-ANHALT ARCHIVE STRUCTURE")
print("=" * 70)
display(st_archive_summary)

st_test_zip = st_zip_files[len(st_zip_files) // 2]

with ZipFile(st_test_zip, "r") as archive:
    st_test_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

    if not st_test_members:
        raise FileNotFoundError(f"No CityGML files found inside {st_test_zip.name}")

    st_test_member = st_test_members[len(st_test_members) // 2]

    with archive.open(st_test_member) as stream:
        st_test_content = stream.read()

st_test_root = etree.fromstring(st_test_content)
st_bldg_namespace = st_test_root.nsmap.get("bldg")
st_srs_names = sorted({element.get("srsName") for element in st_test_root.iter() if element.get("srsName")})

st_sample_gdf = extract_citygml_stream(BytesIO(st_test_content), Path(st_test_member).name, "EPSG:25832")

print("\n" + "=" * 70)
print("SAXONY-ANHALT SAMPLE TILE TEST")
print("=" * 70)
print(f"\nArchive               : {st_test_zip.name}")
print(f"CityGML file          : {st_test_member}")
print(f"Building namespace    : {st_bldg_namespace}")
print(f"Buildings extracted   : {len(st_sample_gdf):,}")
print(f"Measured heights      : {st_sample_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights       : {st_sample_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries     : {st_sample_gdf.geometry.notna().sum():,}")
print(f"Missing geometries    : {st_sample_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs       : {st_sample_gdf['lod2_id'].nunique():,}")
print(f"Assigned CRS          : {st_sample_gdf.crs}")

print("\nCRS values in CityGML:")
for srs_name in st_srs_names:
    print(f"  {srs_name}")

print("\nFootprint methods:")
display(st_sample_gdf["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

display(st_sample_gdf.head())

SAXONY-ANHALT ARCHIVE STRUCTURE


,archive,files,gml_xml_files,nested_zip_files,uncompressed_size_gb
0,LoD2-1.zip,1381,1381,0,7.591699
1,LoD2-2.zip,1396,1396,0,10.719033
2,LoD2-3.zip,1366,1366,0,12.067075
3,LoD2-4.zip,1296,1296,0,8.738612



SAXONY-ANHALT SAMPLE TILE TEST

Archive               : LoD2-3.zip
CityGML file          : LoD23/LoD2_326985764.gml
Building namespace    : http://www.opengis.net/citygml/building/1.0
Buildings extracted   : 0
Measured heights      : 0
Missing heights       : 0
Usable geometries     : 0
Missing geometries    : 0
Unique LoD2 IDs       : 0
Assigned CRS          : EPSG:25832

CRS values in CityGML:
  urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH

Footprint methods:


,footprint_method,building_count


,lod2_id,creation_date,function,roof_type,measured_height_m,storeys_above_ground,source_file,footprint_method,geometry


In [58]:
# ============================================================
# 50 — INSPECT SAXONY-ANHALT CITYGML OBJECT TYPES
# ============================================================

from collections import Counter

st_test_records = []

for zip_path in st_zip_files:
    with ZipFile(zip_path, "r") as archive:
        gml_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

        sample_indices = sorted(set([0, len(gml_members) // 4, len(gml_members) // 2, 3 * len(gml_members) // 4, len(gml_members) - 1]))

        for index in sample_indices:
            member = gml_members[index]
            tag_counts = Counter()

            with archive.open(member) as stream:
                for event, element in etree.iterparse(stream, events=("end",), huge_tree=True):
                    local_name = etree.QName(element).localname

                    if local_name in {"Building", "BuildingPart", "cityObjectMember", "measuredHeight", "GroundSurface", "RoofSurface", "WallSurface"}:
                        tag_counts[local_name] += 1

                    element.clear()

                    while element.getprevious() is not None:
                        del element.getparent()[0]

            st_test_records.append({
                "archive": zip_path.name,
                "source_file": member,
                "buildings": tag_counts["Building"],
                "building_parts": tag_counts["BuildingPart"],
                "measured_heights": tag_counts["measuredHeight"],
                "ground_surfaces": tag_counts["GroundSurface"],
                "roof_surfaces": tag_counts["RoofSurface"],
                "wall_surfaces": tag_counts["WallSurface"],
                "city_object_members": tag_counts["cityObjectMember"]
            })

st_structure_test = pd.DataFrame(st_test_records)

print("=" * 70)
print("SAXONY-ANHALT CITYGML STRUCTURE TEST")
print("=" * 70)

display(st_structure_test)

print("\nTotals across sampled files:")
display(
    st_structure_test[
        ["buildings", "building_parts", "measured_heights", "ground_surfaces", "roof_surfaces", "wall_surfaces", "city_object_members"]
    ].sum().to_frame("count")
)

print("\nFiles containing buildings:")
display(st_structure_test[st_structure_test["buildings"] > 0])

print("\nFiles containing only BuildingPart objects:")
display(st_structure_test[(st_structure_test["buildings"] == 0) & (st_structure_test["building_parts"] > 0)])

print("\nCompletely empty sampled tiles:")
display(st_structure_test[(st_structure_test["buildings"] == 0) & (st_structure_test["building_parts"] == 0)])

SAXONY-ANHALT CITYGML STRUCTURE TEST


,archive,source_file,buildings,building_parts,measured_heights,ground_surfaces,roof_surfaces,wall_surfaces,city_object_members
0,LoD2-1.zip,LoD21/LoD2_326065760.gml,0,0,0,0,0,0,0
1,LoD2-1.zip,LoD21/LoD2_326325708.gml,0,0,0,0,0,0,0
2,LoD2-1.zip,LoD21/LoD2_326405826.gml,536,869,1154,1154,1812,6884,536
3,LoD2-1.zip,LoD21/LoD2_326485844.gml,27,63,73,73,137,448,27
4,LoD2-1.zip,LoD21/LoD2_326565862.gml,0,0,0,0,0,0,0
5,LoD2-2.zip,LoD22/LoD2_326585696.gml,26,10,33,33,47,171,26
6,LoD2-2.zip,LoD22/LoD2_326665690.gml,15,9,20,20,29,130,15
7,LoD2-2.zip,LoD22/LoD2_326725772.gml,0,0,0,0,0,0,0
8,LoD2-2.zip,LoD22/LoD2_326785818.gml,0,0,0,0,0,0,0
9,LoD2-2.zip,LoD22/LoD2_326845874.gml,0,0,0,0,0,0,0



Totals across sampled files:


,count
buildings,2420
building_parts,2832
measured_heights,4340
ground_surfaces,4340
roof_surfaces,6763
wall_surfaces,25522
city_object_members,2420



Files containing buildings:


,archive,source_file,buildings,building_parts,measured_heights,ground_surfaces,roof_surfaces,wall_surfaces,city_object_members
2,LoD2-1.zip,LoD21/LoD2_326405826.gml,536,869,1154,1154,1812,6884,536
3,LoD2-1.zip,LoD21/LoD2_326485844.gml,27,63,73,73,137,448,27
5,LoD2-2.zip,LoD22/LoD2_326585696.gml,26,10,33,33,47,171,26
6,LoD2-2.zip,LoD22/LoD2_326665690.gml,15,9,20,20,29,130,15
10,LoD2-3.zip,LoD23/LoD2_326865664.gml,122,145,219,219,348,1240,122
11,LoD2-3.zip,LoD23/LoD2_326925706.gml,1271,1323,2136,2136,3271,12454,1271
13,LoD2-3.zip,LoD23/LoD2_327045822.gml,1,0,1,1,1,1,1
17,LoD2-4.zip,LoD24/LoD2_327265664.gml,211,301,422,422,661,2499,211
18,LoD2-4.zip,LoD24/LoD2_327505726.gml,211,112,282,282,457,1695,211



Files containing only BuildingPart objects:


,archive,source_file,buildings,building_parts,measured_heights,ground_surfaces,roof_surfaces,wall_surfaces,city_object_members



Completely empty sampled tiles:


,archive,source_file,buildings,building_parts,measured_heights,ground_surfaces,roof_surfaces,wall_surfaces,city_object_members
0,LoD2-1.zip,LoD21/LoD2_326065760.gml,0,0,0,0,0,0,0
1,LoD2-1.zip,LoD21/LoD2_326325708.gml,0,0,0,0,0,0,0
4,LoD2-1.zip,LoD21/LoD2_326565862.gml,0,0,0,0,0,0,0
7,LoD2-2.zip,LoD22/LoD2_326725772.gml,0,0,0,0,0,0,0
8,LoD2-2.zip,LoD22/LoD2_326785818.gml,0,0,0,0,0,0,0
9,LoD2-2.zip,LoD22/LoD2_326845874.gml,0,0,0,0,0,0,0
12,LoD2-3.zip,LoD23/LoD2_326985764.gml,0,0,0,0,0,0,0
14,LoD2-3.zip,LoD23/LoD2_327105864.gml,0,0,0,0,0,0,0
15,LoD2-4.zip,LoD24/LoD2_327125650.gml,0,0,0,0,0,0,0
16,LoD2-4.zip,LoD24/LoD2_327185666.gml,0,0,0,0,0,0,0


In [59]:
# ============================================================
# 51 — INSPECT SAXONY-ANHALT BUILDING AND PART HEIGHTS
# ============================================================

ST_BLDG_NS = "http://www.opengis.net/citygml/building/1.0"
ST_BUILDING_TAG = f"{{{ST_BLDG_NS}}}Building"
ST_BUILDING_PART_TAG = f"{{{ST_BLDG_NS}}}BuildingPart"
ST_HEIGHT_TAG = f"{{{ST_BLDG_NS}}}measuredHeight"

def get_direct_height(element):
    height_element = element.find(ST_HEIGHT_TAG)
    return parse_number(height_element.text) if height_element is not None else np.nan

st_nonempty_samples = st_structure_test.loc[st_structure_test["buildings"] > 0, ["archive", "source_file"]]
st_height_records = []

for row in st_nonempty_samples.itertuples(index=False):
    zip_path = ST_ZIP_DIR / row.archive

    with ZipFile(zip_path, "r") as archive:
        with archive.open(row.source_file) as stream:
            for event, building in etree.iterparse(stream, events=("end",), tag=ST_BUILDING_TAG, huge_tree=True):
                parent_height = get_direct_height(building)
                building_parts = building.findall(f".//{ST_BUILDING_PART_TAG}")
                part_heights = [get_direct_height(part) for part in building_parts]
                part_heights = [height for height in part_heights if pd.notna(height)]

                st_height_records.append({
                    "archive": row.archive,
                    "source_file": row.source_file,
                    "lod2_id": building.get(GML_ID),
                    "parent_height_m": parent_height,
                    "building_parts": len(building_parts),
                    "parts_with_height": len(part_heights),
                    "minimum_part_height_m": min(part_heights) if part_heights else np.nan,
                    "maximum_part_height_m": max(part_heights) if part_heights else np.nan,
                    "first_descendant_height_m": parse_number(get_first_text(building, "measuredHeight"))
                })

                building.clear()

                while building.getprevious() is not None:
                    del building.getparent()[0]

st_height_check = pd.DataFrame(st_height_records)

st_height_check["has_parent_height"] = st_height_check["parent_height_m"].notna()
st_height_check["has_parts"] = st_height_check["building_parts"] > 0
st_height_check["parent_equals_max_part"] = np.isclose(
    st_height_check["parent_height_m"],
    st_height_check["maximum_part_height_m"],
    atol=0.01,
    equal_nan=False
)

st_height_summary = pd.DataFrame([{
    "buildings_checked": len(st_height_check),
    "buildings_with_parts": st_height_check["has_parts"].sum(),
    "buildings_without_parts": (~st_height_check["has_parts"]).sum(),
    "buildings_with_parent_height": st_height_check["has_parent_height"].sum(),
    "buildings_missing_parent_height": (~st_height_check["has_parent_height"]).sum(),
    "buildings_with_part_heights": st_height_check["parts_with_height"].gt(0).sum(),
    "parent_equals_max_part": st_height_check["parent_equals_max_part"].sum(),
    "parent_differs_from_max_part": (
        st_height_check["has_parent_height"]
        & st_height_check["parts_with_height"].gt(0)
        & ~st_height_check["parent_equals_max_part"]
    ).sum()
}])

print("=" * 70)
print("SAXONY-ANHALT BUILDING HEIGHT STRUCTURE")
print("=" * 70)

display(st_height_summary)

print("\nBuildings missing a direct parent height:")
display(
    st_height_check.loc[
        ~st_height_check["has_parent_height"],
        [
            "lod2_id",
            "building_parts",
            "parts_with_height",
            "minimum_part_height_m",
            "maximum_part_height_m",
            "first_descendant_height_m"
        ]
    ].head(30)
)

print("\nBuildings where parent and maximum part heights differ:")
display(
    st_height_check.loc[
        st_height_check["has_parent_height"]
        & st_height_check["parts_with_height"].gt(0)
        & ~st_height_check["parent_equals_max_part"],
        [
            "lod2_id",
            "parent_height_m",
            "building_parts",
            "minimum_part_height_m",
            "maximum_part_height_m",
            "first_descendant_height_m"
        ]
    ].head(30)
)

SAXONY-ANHALT BUILDING HEIGHT STRUCTURE


,buildings_checked,buildings_with_parts,buildings_without_parts,buildings_with_parent_height,buildings_missing_parent_height,buildings_with_part_heights,parent_equals_max_part,parent_differs_from_max_part
0,2420,912,1508,1508,912,912,0,0



Buildings missing a direct parent height:


,lod2_id,building_parts,parts_with_height,minimum_part_height_m,maximum_part_height_m,first_descendant_height_m
1,DEST_DESTLIKA0005fNYZ,2,2,2.567,2.590,2.567
3,DEST_DESTLIKA00055I2E,2,2,3.461,5.410,5.410
4,DEST_DESTLIKA00055I2D,2,2,3.284,4.423,4.423
6,DEST_DESTLIKA0005fNYY,4,4,3.820,9.769,9.769
9,DEST_DESTLIKA0005fNYX,9,9,0.096,8.523,8.523
12,DEST_DESTLIKA0005fNYa,3,3,2.719,8.362,8.362
14,DEST_DESTLIKA00055I2F,3,3,2.401,4.539,4.539
15,DEST_DESTLIKA0005Rt8C,3,3,2.695,6.156,6.156
18,DEST_DESTLIKA0005Rt8H,2,2,0.115,7.377,7.377
19,DEST_DESTLIKA0005Rt8K,3,3,0.101,2.498,2.498



Buildings where parent and maximum part heights differ:


,lod2_id,parent_height_m,building_parts,minimum_part_height_m,maximum_part_height_m,first_descendant_height_m


In [60]:
# ============================================================
# 52 — DEFINE AND TEST SAXONY-ANHALT BUILDING-PART PARSER
# ============================================================

def get_direct_text(element, local_name):
    for child in element:
        if etree.QName(child).localname == local_name and child.text:
            return child.text.strip()
    return None

def get_direct_or_first_text(element, local_name):
    direct_value = get_direct_text(element, local_name)
    return direct_value if direct_value is not None else get_first_text(element, local_name)

def resolve_st_height(building):
    parent_height = parse_number(get_direct_text(building, "measuredHeight"))
    building_parts = building.findall(f".//{ST_BUILDING_PART_TAG}")
    part_heights = [parse_number(get_direct_text(part, "measuredHeight")) for part in building_parts]
    part_heights = [height for height in part_heights if pd.notna(height)]
    maximum_part_height = max(part_heights) if part_heights else np.nan

    if pd.notna(parent_height):
        return parent_height, parent_height, maximum_part_height, len(building_parts), "building_measured_height"

    if pd.notna(maximum_part_height):
        return maximum_part_height, parent_height, maximum_part_height, len(building_parts), "maximum_building_part_height"

    return np.nan, parent_height, maximum_part_height, len(building_parts), "missing"

def extract_citygml_stream_st(stream, source_file, source_crs="EPSG:25832"):
    records = []

    for event, building in etree.iterparse(stream, events=("end",), tag=ST_BUILDING_TAG, huge_tree=True):
        measured_height, parent_height, maximum_part_height, part_count, height_method = resolve_st_height(building)
        geometry, footprint_method = extract_lod2_footprint(building)

        records.append({
            "lod2_id": building.get(GML_ID),
            "creation_date": get_direct_or_first_text(building, "creationDate"),
            "function": get_direct_or_first_text(building, "function"),
            "roof_type": get_direct_or_first_text(building, "roofType"),
            "measured_height_m": measured_height,
            "parent_height_m": parent_height,
            "maximum_part_height_m": maximum_part_height,
            "building_part_count": part_count,
            "height_method": height_method,
            "storeys_above_ground": parse_number(get_direct_or_first_text(building, "storeysAboveGround")),
            "source_file": source_file,
            "footprint_method": footprint_method,
            "geometry": geometry
        })

        building.clear()

        while building.getprevious() is not None:
            del building.getparent()[0]

    columns = ["lod2_id", "creation_date", "function", "roof_type", "measured_height_m", "parent_height_m", "maximum_part_height_m", "building_part_count", "height_method", "storeys_above_ground", "source_file", "footprint_method", "geometry"]

    if not records:
        return gpd.GeoDataFrame(columns=columns, geometry="geometry", crs=source_crs)

    return gpd.GeoDataFrame(records, geometry="geometry", crs=source_crs)

st_test_record = st_structure_test.loc[st_structure_test["buildings"].idxmax()]
st_parser_test_zip = ST_ZIP_DIR / st_test_record["archive"]
st_parser_test_member = st_test_record["source_file"]

with ZipFile(st_parser_test_zip, "r") as archive:
    with archive.open(st_parser_test_member) as stream:
        st_parser_test_gdf = extract_citygml_stream_st(stream, Path(st_parser_test_member).name)

print("=" * 70)
print("SAXONY-ANHALT PARSER TEST")
print("=" * 70)
print(f"\nArchive                  : {st_parser_test_zip.name}")
print(f"CityGML file             : {st_parser_test_member}")
print(f"Buildings extracted      : {len(st_parser_test_gdf):,}")
print(f"Buildings with parts     : {(st_parser_test_gdf['building_part_count'] > 0).sum():,}")
print(f"Buildings without parts  : {(st_parser_test_gdf['building_part_count'] == 0).sum():,}")
print(f"Measured heights         : {st_parser_test_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights          : {st_parser_test_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries        : {st_parser_test_gdf.geometry.notna().sum():,}")
print(f"Missing geometries       : {st_parser_test_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs          : {st_parser_test_gdf['lod2_id'].nunique():,}")

print("\nHeight methods:")
display(st_parser_test_gdf["height_method"].value_counts(dropna=False).rename_axis("height_method").reset_index(name="building_count"))

print("\nFootprint methods:")
display(st_parser_test_gdf["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

display(st_parser_test_gdf.head())

SAXONY-ANHALT PARSER TEST

Archive                  : LoD2-3.zip
CityGML file             : LoD23/LoD2_326925706.gml
Buildings extracted      : 1,271
Buildings with parts     : 458
Buildings without parts  : 813
Measured heights         : 1,271
Missing heights          : 0
Usable geometries        : 1,271
Missing geometries       : 0
Unique LoD2 IDs          : 1,271

Height methods:


,height_method,building_count
0,building_measured_height,813
1,maximum_building_part_height,458



Footprint methods:


,footprint_method,building_count
0,ground_surface,1271


,lod2_id,creation_date,function,roof_type,measured_height_m,parent_height_m,maximum_part_height_m,building_part_count,height_method,storeys_above_ground,source_file,footprint_method,geometry
0,DEST_DESTLIKA0000UN7r,2020-09-17,31001_1000,2100,7.787,NaN,7.787,2,maximum_building_part_height,NaN,LoD2_326925706.gml,ground_surface,"POLYGON ((692296.18 5707476.64, 692290.13 5707..."
1,DEST_DESTLIKA0000UN7m,2020-09-17,31001_1000,1000,2.628,2.628,NaN,0,building_measured_height,NaN,LoD2_326925706.gml,ground_surface,"POLYGON ((692268.777 5707515.929, 692268.527 5..."
2,DEST_DESTLIKA0000ULnJ,2020-09-17,31001_1000,3100,8.720,8.720,NaN,0,building_measured_height,NaN,LoD2_326925706.gml,ground_surface,"POLYGON ((692719.569 5707551.561, 692717.678 5..."
3,DEST_DESTLIKA0000UN7o,2020-09-17,31001_1000,3100,6.608,6.608,NaN,0,building_measured_height,NaN,LoD2_326925706.gml,ground_surface,"POLYGON ((692336.157 5707472.496, 692336.166 5..."
4,DEST_DESTLIKA0000UN7n,2020-09-17,31001_1000,1000,6.643,NaN,6.643,2,maximum_building_part_height,NaN,LoD2_326925706.gml,ground_surface,"POLYGON ((692626.31 5707573.57, 692626.17 5707..."


In [61]:
# ============================================================
# 53 — PROCESS ALL SAXONY-ANHALT LoD2 TILES
# ============================================================

ST_OUTPUT_DIR = EXTRACTED_DIR / "ST"
ST_PART_DIR = ST_OUTPUT_DIR / "parts"
ST_MANIFEST_DIR = ST_OUTPUT_DIR / "manifests"
ST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ST_PART_DIR.mkdir(parents=True, exist_ok=True)
ST_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

ST_BATCH_SIZE = 250
st_source_records = []

for zip_path in st_zip_files:
    with ZipFile(zip_path, "r") as archive:
        for member in archive.namelist():
            if member.lower().endswith((".gml", ".xml")):
                st_source_records.append({
                    "archive_path": str(zip_path),
                    "archive_name": zip_path.name,
                    "member": member,
                    "source_key": f"{zip_path.name}::{member}"
                })

st_sources = pd.DataFrame(st_source_records)
st_batch_starts = list(range(0, len(st_sources), ST_BATCH_SIZE))

print("=" * 70)
print("SAXONY-ANHALT PROCESSING SETUP")
print("=" * 70)
print(f"\nArchives              : {len(st_zip_files):,}")
print(f"CityGML files         : {len(st_sources):,}")
print(f"Unique source keys    : {st_sources['source_key'].nunique():,}")
print(f"Batch size            : {ST_BATCH_SIZE:,}")
print(f"Expected Parquet parts: {len(st_batch_starts):,}")

if st_sources["source_key"].duplicated().any():
    raise ValueError("Duplicate Saxony-Anhalt source keys detected.")

st_batch_summaries = []

for batch_start in tqdm(st_batch_starts, total=len(st_batch_starts), desc="Processing Saxony-Anhalt batches"):
    batch_number = batch_start // ST_BATCH_SIZE + 1
    batch_sources = st_sources.iloc[batch_start:batch_start + ST_BATCH_SIZE].copy()
    part_path = ST_PART_DIR / f"lod2_buildings_ST_part_{batch_number:04d}.parquet"
    manifest_path = ST_MANIFEST_DIR / f"lod2_manifest_ST_part_{batch_number:04d}.csv"

    if manifest_path.exists():
        existing_manifest = pd.read_csv(manifest_path)
        manifest_complete = len(existing_manifest) == len(batch_sources) and not existing_manifest["status"].eq("failed").any()
        output_complete = existing_manifest["buildings"].sum() == 0 or part_path.exists()

        if manifest_complete and output_complete:
            st_batch_summaries.append({
                "batch": batch_number,
                "source_files": len(existing_manifest),
                "processed_files": existing_manifest["status"].eq("processed").sum(),
                "empty_files": existing_manifest["status"].eq("empty").sum(),
                "failed_files": existing_manifest["status"].eq("failed").sum(),
                "buildings": existing_manifest["buildings"].sum(),
                "status": "already_processed"
            })
            continue

    if part_path.exists():
        part_path.unlink()

    batch_parts = []
    batch_manifest = []

    for archive_path, archive_sources in batch_sources.groupby("archive_path", sort=False):
        with ZipFile(archive_path, "r") as archive:
            for row in archive_sources.itertuples(index=False):
                try:
                    with archive.open(row.member) as stream:
                        tile_gdf = extract_citygml_stream_st(stream, Path(row.member).name)

                    if tile_gdf.empty:
                        batch_manifest.append({
                            "source_key": row.source_key,
                            "archive_name": row.archive_name,
                            "source_member": row.member,
                            "status": "empty",
                            "buildings": 0,
                            "error": None
                        })
                        continue

                    tile_gdf["state_code"] = "ST"
                    tile_gdf["source_state"] = "Saxony-Anhalt"
                    tile_gdf["source_crs"] = "EPSG:25832"
                    tile_gdf["citygml_version"] = "1.0"
                    tile_gdf["height_source"] = tile_gdf["height_method"]
                    tile_gdf["source_archive"] = row.archive_name
                    tile_gdf["source_member"] = row.member
                    tile_gdf["source_path"] = row.source_key
                    batch_parts.append(tile_gdf)

                    batch_manifest.append({
                        "source_key": row.source_key,
                        "archive_name": row.archive_name,
                        "source_member": row.member,
                        "status": "processed",
                        "buildings": len(tile_gdf),
                        "error": None
                    })

                except Exception as error:
                    batch_manifest.append({
                        "source_key": row.source_key,
                        "archive_name": row.archive_name,
                        "source_member": row.member,
                        "status": "failed",
                        "buildings": 0,
                        "error": str(error)
                    })

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    batch_manifest_df = pd.DataFrame(batch_manifest)
    batch_manifest_df.to_csv(manifest_path, index=False)

    st_batch_summaries.append({
        "batch": batch_number,
        "source_files": len(batch_manifest_df),
        "processed_files": batch_manifest_df["status"].eq("processed").sum(),
        "empty_files": batch_manifest_df["status"].eq("empty").sum(),
        "failed_files": batch_manifest_df["status"].eq("failed").sum(),
        "buildings": batch_manifest_df["buildings"].sum(),
        "status": "processed"
    })

    del batch_parts, batch_manifest, batch_manifest_df
    gc.collect()

st_batch_summary_df = pd.DataFrame(st_batch_summaries)
st_manifest_files = sorted(ST_MANIFEST_DIR.glob("lod2_manifest_ST_part_*.csv"))
st_manifests = pd.concat([pd.read_csv(path) for path in st_manifest_files], ignore_index=True)
st_part_files = sorted(ST_PART_DIR.glob("lod2_buildings_ST_part_*.parquet"))

st_batch_summary_path = REPORT_DIR / "lod2_batch_summary_ST.csv"
st_manifest_path = REPORT_DIR / "lod2_processing_manifest_ST.csv"

st_batch_summary_df.to_csv(st_batch_summary_path, index=False)
st_manifests.to_csv(st_manifest_path, index=False)

print("=" * 70)
print("SAXONY-ANHALT LoD2 PROCESSING SUMMARY")
print("=" * 70)
print(f"\nSource files recorded : {len(st_manifests):,}")
print(f"Files with buildings  : {st_manifests['status'].eq('processed').sum():,}")
print(f"Empty CityGML files   : {st_manifests['status'].eq('empty').sum():,}")
print(f"Failed files          : {st_manifests['status'].eq('failed').sum():,}")
print(f"Buildings extracted   : {st_manifests['buildings'].sum():,}")
print(f"Parquet parts created : {len(st_part_files):,}")
print(f"Part directory        : {ST_PART_DIR}")
print(f"Combined manifest     : {st_manifest_path}")

display(st_manifests["status"].value_counts().rename_axis("status").reset_index(name="source_files"))

if st_manifests["status"].eq("failed").any():
    display(st_manifests[st_manifests["status"] == "failed"].head(30))

SAXONY-ANHALT PROCESSING SETUP

Archives              : 4
CityGML files         : 5,439
Unique source keys    : 5,439
Batch size            : 250
Expected Parquet parts: 22


Processing Saxony-Anhalt batches: 100%|████████████████████████████████████████████| 22/22 [26:50<00:00, 73.22s/it]

SAXONY-ANHALT LoD2 PROCESSING SUMMARY

Source files recorded : 5,439
Files with buildings  : 4,804
Empty CityGML files   : 635
Failed files          : 0
Buildings extracted   : 1,775,506
Parquet parts created : 22
Part directory        : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/ST/parts
Combined manifest     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_processing_manifest_ST.csv


,status,source_files
0,processed,4804
1,empty,635


In [62]:
# ============================================================
# 54 — VALIDATE AND SUMMARISE SAXONY-ANHALT RAW LoD2 DATA
# ============================================================

st_expected_sources = set(st_sources["source_key"])
st_manifest_sources = set(st_manifests["source_key"])
st_missing_sources = sorted(st_expected_sources - st_manifest_sources)
st_unexpected_sources = sorted(st_manifest_sources - st_expected_sources)
st_duplicate_manifest_sources = st_manifests["source_key"].duplicated().sum()

st_summary_records = []
st_footprint_counts = {}
st_height_method_counts = {}

for part_path in tqdm(st_part_files, desc="Summarising Saxony-Anhalt parts"):
    part = pd.read_parquet(part_path, columns=["lod2_id", "measured_height_m", "height_method", "footprint_method", "geometry"])

    st_summary_records.append({
        "buildings": len(part),
        "unique_ids": part["lod2_id"].nunique(),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    for method, count in part["footprint_method"].value_counts(dropna=False).items():
        st_footprint_counts[method] = st_footprint_counts.get(method, 0) + count

    for method, count in part["height_method"].value_counts(dropna=False).items():
        st_height_method_counts[method] = st_height_method_counts.get(method, 0) + count

    del part
    gc.collect()

st_part_stats = pd.DataFrame(st_summary_records)

st_summary = pd.DataFrame([{
    "state_code": "ST",
    "state_name": "Saxony-Anhalt",
    "archives": len(st_zip_files),
    "source_files_expected": len(st_expected_sources),
    "source_files_recorded": len(st_manifest_sources),
    "files_with_buildings": st_manifests["status"].eq("processed").sum(),
    "empty_files": st_manifests["status"].eq("empty").sum(),
    "failed_files": st_manifests["status"].eq("failed").sum(),
    "missing_source_files": len(st_missing_sources),
    "unexpected_source_files": len(st_unexpected_sources),
    "duplicate_manifest_sources": st_duplicate_manifest_sources,
    "parquet_parts": len(st_part_files),
    "buildings": st_part_stats["buildings"].sum(),
    "unique_ids": st_part_stats["unique_ids"].sum(),
    "measured_heights": st_part_stats["measured_heights"].sum(),
    "missing_heights": st_part_stats["missing_heights"].sum(),
    "usable_geometries": st_part_stats["usable_geometries"].sum(),
    "missing_geometries": st_part_stats["missing_geometries"].sum(),
    "minimum_height_m": st_part_stats["minimum_height_m"].min(),
    "maximum_height_m": st_part_stats["maximum_height_m"].max(),
    "output_directory": str(ST_PART_DIR)
}])

st_footprint_summary = pd.DataFrame([{"footprint_method": method, "building_count": count} for method, count in st_footprint_counts.items()]).sort_values("building_count", ascending=False)
st_height_method_summary = pd.DataFrame([{"height_method": method, "building_count": count} for method, count in st_height_method_counts.items()]).sort_values("building_count", ascending=False)

st_summary_path = REPORT_DIR / "lod2_extraction_summary_ST.csv"
st_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_ST.csv"
st_height_method_summary_path = REPORT_DIR / "lod2_height_methods_ST.csv"

st_summary.to_csv(st_summary_path, index=False)
st_footprint_summary.to_csv(st_footprint_summary_path, index=False)
st_height_method_summary.to_csv(st_height_method_summary_path, index=False)

print("=" * 70)
print("SAXONY-ANHALT FINAL LoD2 VALIDATION")
print("=" * 70)

display(st_summary)
display(st_height_method_summary)
display(st_footprint_summary)

print(f"\nState summary saved to     : {st_summary_path}")
print(f"Height summary saved to    : {st_height_method_summary_path}")
print(f"Footprint summary saved to : {st_footprint_summary_path}")

if st_missing_sources:
    display(pd.DataFrame({"missing_source": st_missing_sources}).head(30))

Summarising Saxony-Anhalt parts: 100%|█████████████████████████████████████████████| 22/22 [00:20<00:00,  1.06it/s]

SAXONY-ANHALT FINAL LoD2 VALIDATION


,state_code,state_name,archives,source_files_expected,source_files_recorded,files_with_buildings,empty_files,failed_files,missing_source_files,unexpected_source_files,...,parquet_parts,buildings,unique_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,ST,Saxony-Anhalt,4,5439,5439,4804,635,0,0,0,...,22,1775506,1775506,1775506,0,1775506,0,0.001,325.0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


,height_method,building_count
0,building_measured_height,1050224
1,maximum_building_part_height,725282


,footprint_method,building_count
0,ground_surface,1775506



State summary saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_ST.csv
Height summary saved to    : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_height_methods_ST.csv
Footprint summary saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_ST.csv


In [63]:
# ============================================================
# 55 — DISCOVER SAXONY LoD2 CITYGML DOWNLOADS
# ============================================================

import json
import re
from urllib.parse import urlparse

SN_RAW_DIR = RAW_DIR / "SN"
SN_RAW_DIR.mkdir(parents=True, exist_ok=True)

SN_PORTAL_URL = "https://www.geodaten.sachsen.de/downloadbereich-digitale-3d-stadtmodelle-4875.html"
SN_BATCH_URL = "https://www.geodaten.sachsen.de/batch-download-4719.html"
SN_QUERY_URL = "https://geodienste.sachsen.de/ags-relay/ArcGISServer/guest/arcgis/rest/services/geosn/rest_geosn_downloadlinks/MapServer/3/query"

sn_index = source_registry["state_code"].eq("SN")
source_registry.loc[sn_index, ["portal_url", "download_url", "download_method", "format", "horizontal_crs", "download_status", "notes"]] = [
    SN_PORTAL_URL,
    SN_QUERY_URL,
    "ArcGIS tile catalogue and GeoCloud ZIP downloads",
    "CityGML",
    "to be confirmed from sample",
    "discovering",
    "Official statewide LoD2 provided as 2 x 2 km tiles"
]
source_registry.to_csv(source_registry_path, index=False)

# ------------------------------------------------------------
# 1. Obtain the current GeoCloud share ID from the batch page
# ------------------------------------------------------------

batch_response = requests.get(SN_BATCH_URL, timeout=(30, 180))
batch_response.raise_for_status()
batch_html = batch_response.text

products_match = re.search(r"(?:var|let|const)?\s*products\s*=\s*(\{.*?\})\s*;", batch_html, flags=re.DOTALL)

if products_match:
    products = json.loads(products_match.group(1))
    lod2_product = products.get("LoD2_CityGML", {})
    sn_share_id = lod2_product.get("share_id")
else:
    sn_share_id = None

if not sn_share_id:
    share_match = re.search(r'"LoD2_CityGML"\s*:\s*\{.*?"share_id"\s*:\s*"([^"]+)"', batch_html, flags=re.DOTALL)
    sn_share_id = share_match.group(1) if share_match else None

if not sn_share_id:
    raise RuntimeError("The current Saxony LoD2 GeoCloud share ID could not be found.")

SN_DOWNLOAD_BASE = f"https://geocloud.landesvermessung.sachsen.de/public.php/dav/files/{sn_share_id}/"

# ------------------------------------------------------------
# 2. Read the complete official ArcGIS tile catalogue
# ------------------------------------------------------------

SN_PAGE_SIZE = 1000
sn_offset = 0
sn_records = []

while True:
    params = {
        "where": "1=1",
        "outFields": "Kachel,Download_CityGML,Stand",
        "returnGeometry": "false",
        "resultOffset": sn_offset,
        "resultRecordCount": SN_PAGE_SIZE,
        "f": "json"
    }

    response = requests.get(SN_QUERY_URL, params=params, timeout=(30, 300))
    response.raise_for_status()
    page = response.json()

    if "error" in page:
        raise RuntimeError(page["error"])

    features = page.get("features", [])

    if not features:
        break

    for feature in features:
        attributes = feature.get("attributes", {})
        original_download = attributes.get("Download_CityGML")

        if not original_download:
            continue

        filename = Path(urlparse(original_download).path).name

        sn_records.append({
            "tile_id": attributes.get("Kachel"),
            "data_date": attributes.get("Stand"),
            "catalogue_download": original_download,
            "download_url": SN_DOWNLOAD_BASE + filename,
            "filename": filename
        })

    sn_offset += len(features)

    if not page.get("exceededTransferLimit", False):
        break

sn_downloads = pd.DataFrame(sn_records)
sn_downloads = sn_downloads.drop_duplicates(subset="download_url").reset_index(drop=True)

print("=" * 70)
print("SAXONY LoD2 DOWNLOAD DISCOVERY")
print("=" * 70)
print(f"\nGeoCloud share ID     : {sn_share_id}")
print(f"Catalogue records     : {len(sn_records):,}")
print(f"Unique download URLs  : {sn_downloads['download_url'].nunique():,}")
print(f"Unique filenames      : {sn_downloads['filename'].nunique():,}")
print(f"Duplicate filenames   : {sn_downloads['filename'].duplicated().sum():,}")
print(f"Missing filenames     : {sn_downloads['filename'].isna().sum():,}")

if sn_downloads.empty:
    raise RuntimeError("No Saxony LoD2 CityGML downloads were discovered.")

if sn_downloads["filename"].duplicated().any():
    raise ValueError("Duplicate Saxony filenames were discovered.")

sn_download_list_path = SN_RAW_DIR / "sn_download_list.csv"
sn_downloads.to_csv(sn_download_list_path, index=False)

display(sn_downloads.head(20))
display(sn_downloads.tail(20))

print(f"\nDownload list saved to: {sn_download_list_path}")

SAXONY LoD2 DOWNLOAD DISCOVERY

GeoCloud share ID     : AyJqXpJAZJXomCb
Catalogue records     : 4,938
Unique download URLs  : 4,938
Unique filenames      : 4,938
Duplicate filenames   : 0
Missing filenames     : 0


,tile_id,data_date,catalogue_download,download_url,filename
0,2785590,"2022( LSC: 2021, Basis-DLM: 2021, DGM: 2021 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33278_5590_2_sn_citygml.zip
1,2785592,"2022( LSC: 2021, Basis-DLM: 2021, DGM: 2021 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33278_5592_2_sn_citygml.zip
2,2785600,"2022( LSC: 2020, Basis-DLM: 2021, DGM: 2020 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33278_5600_2_sn_citygml.zip
3,2785602,"2022( LSC: 2020, Basis-DLM: 2021, DGM: 2020 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33278_5602_2_sn_citygml.zip
4,2785604,"2022( LSC: 2020, Basis-DLM: 2021, DGM: 2020 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33278_5604_2_sn_citygml.zip
5,2805588,"2024( LSC: 2021, Basis-DLM: 2022, DGM: 2021 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33280_5588_2_sn_citygml.zip
6,2805590,"2024( LSC: 2021, Basis-DLM: 2022, DGM: 2021 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33280_5590_2_sn_citygml.zip
7,2805592,"2023( LSC: 2021, Basis-DLM: 2021, DGM: 2021 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33280_5592_2_sn_citygml.zip
8,2805594,"2022( LSC: 2020, Basis-DLM: 2021, DGM: 2020 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33280_5594_2_sn_citygml.zip
9,2805600,"2022( LSC: 2020, Basis-DLM: 2021, DGM: 2020 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33280_5600_2_sn_citygml.zip


,tile_id,data_date,catalogue_download,download_url,filename
4918,4985684,"2022( LSC: 2019, Basis-DLM: 2023, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33498_5684_2_sn_citygml.zip
4919,4985686,"2023( LSC: 2019, Basis-DLM: 2023, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33498_5686_2_sn_citygml.zip
4920,4985688,"2021( LSC: 2019, Basis-DLM: 2024, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33498_5688_2_sn_citygml.zip
4921,4985690,"2021( LSC: 2019, Basis-DLM: 2024, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33498_5690_2_sn_citygml.zip
4922,4985696,"2021( LSC: 2019, Basis-DLM: 2024, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33498_5696_2_sn_citygml.zip
4923,4985698,"2021( LSC: 2019, Basis-DLM: 2024, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33498_5698_2_sn_citygml.zip
4924,5005666,"2024( LSC: 2019, Basis-DLM: 2022, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33500_5666_2_sn_citygml.zip
4925,5005668,"2023( LSC: 2019, Basis-DLM: 2020, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33500_5668_2_sn_citygml.zip
4926,5005670,"2023( LSC: 2019, Basis-DLM: 2020, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33500_5670_2_sn_citygml.zip
4927,5005672,"2023( LSC: 2019, Basis-DLM: 2024, DGM: 2019 )",https://geocloud.landesvermessung.sachsen.de/p...,https://geocloud.landesvermessung.sachsen.de/p...,lod2_33500_5672_2_sn_citygml.zip



Download list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/SN/sn_download_list.csv


In [64]:
# ============================================================
# 56 — TEST REPRESENTATIVE SAXONY LoD2 TILES
# ============================================================

from io import BytesIO
from collections import Counter

SN_TEST_DIR = SN_RAW_DIR / "test_tiles"
SN_TEST_DIR.mkdir(parents=True, exist_ok=True)

sn_sample_indices = sorted(set([0, len(sn_downloads) // 4, len(sn_downloads) // 2, 3 * len(sn_downloads) // 4, len(sn_downloads) - 1]))
sn_test_records = []

for index in tqdm(sn_sample_indices, desc="Testing Saxony tiles"):
    row = sn_downloads.iloc[index]
    zip_path = SN_TEST_DIR / row["filename"]

    response = requests.get(row["download_url"], timeout=(30, 600))
    response.raise_for_status()
    zip_path.write_bytes(response.content)

    with ZipFile(zip_path, "r") as archive:
        bad_member = archive.testzip()

        if bad_member is not None:
            raise ValueError(f"Corrupt ZIP member in {zip_path.name}: {bad_member}")

        gml_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

        if not gml_members:
            raise FileNotFoundError(f"No CityGML file found in {zip_path.name}")

        for member in gml_members:
            with archive.open(member) as stream:
                content = stream.read()

            root = etree.fromstring(content)
            bldg_namespace = root.nsmap.get("bldg")
            srs_names = sorted({element.get("srsName") for element in root.iter() if element.get("srsName")})

            tag_counts = Counter()

            for element in root.iter():
                local_name = etree.QName(element).localname

                if local_name in {"Building", "BuildingPart", "measuredHeight", "GroundSurface", "RoofSurface", "WallSurface"}:
                    tag_counts[local_name] += 1

            sample_gdf = extract_citygml_stream(BytesIO(content), Path(member).name, "EPSG:25833")

            sn_test_records.append({
                "zip_file": zip_path.name,
                "source_file": member,
                "building_namespace": bldg_namespace,
                "srs_names": " | ".join(srs_names),
                "buildings_in_xml": tag_counts["Building"],
                "building_parts": tag_counts["BuildingPart"],
                "measured_heights_in_xml": tag_counts["measuredHeight"],
                "buildings_extracted": len(sample_gdf),
                "extracted_heights": sample_gdf["measured_height_m"].notna().sum(),
                "missing_heights": sample_gdf["measured_height_m"].isna().sum(),
                "usable_geometries": sample_gdf.geometry.notna().sum(),
                "missing_geometries": sample_gdf.geometry.isna().sum(),
                "unique_ids": sample_gdf["lod2_id"].nunique()
            })

sn_structure_test = pd.DataFrame(sn_test_records)

print("=" * 70)
print("SAXONY SAMPLE TILE STRUCTURE TEST")
print("=" * 70)

display(sn_structure_test)

print("\nNamespaces:")
display(sn_structure_test["building_namespace"].value_counts(dropna=False).rename_axis("building_namespace").reset_index(name="files"))

print("\nCRS values:")
display(sn_structure_test["srs_names"].value_counts(dropna=False).rename_axis("srs_names").reset_index(name="files"))

print("\nTotals:")
display(
    sn_structure_test[
        ["buildings_in_xml", "building_parts", "measured_heights_in_xml", "buildings_extracted", "extracted_heights", "missing_heights", "usable_geometries", "missing_geometries"]
    ].sum().to_frame("count")
)

Testing Saxony tiles: 100%|██████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  2.56it/s]

SAXONY SAMPLE TILE STRUCTURE TEST


,zip_file,source_file,building_namespace,srs_names,buildings_in_xml,building_parts,measured_heights_in_xml,buildings_extracted,extracted_heights,missing_heights,usable_geometries,missing_geometries,unique_ids
0,lod2_33278_5590_2_sn_citygml.zip,lod2_33278_5590_2_sn.gml,http://www.opengis.net/citygml/building/1.0,,0,0,0,0,0,0,0,0,0
1,lod2_33336_5592_2_sn_citygml.zip,lod2_33336_5592_2_sn.gml,http://www.opengis.net/citygml/building/1.0,"urn:ogc:def:crs,crs:EPSG:6.12:25833,crs:EPSG:6...",181,78,234,181,181,0,181,0,181
2,lod2_33372_5616_2_sn_citygml.zip,lod2_33372_5616_2_sn.gml,http://www.opengis.net/citygml/building/1.0,"urn:ogc:def:crs,crs:EPSG:6.12:25833,crs:EPSG:6...",33,9,38,33,33,0,33,0,33
3,lod2_33430_5690_2_sn_citygml.zip,lod2_33430_5690_2_sn.gml,http://www.opengis.net/citygml/building/1.0,"urn:ogc:def:crs,crs:EPSG:6.12:25833,crs:EPSG:6...",2,0,2,2,2,0,2,0,2
4,lod2_33502_5682_2_sn_citygml.zip,lod2_33502_5682_2_sn.gml,http://www.opengis.net/citygml/building/1.0,,0,0,0,0,0,0,0,0,0



Namespaces:


,building_namespace,files
0,http://www.opengis.net/citygml/building/1.0,5



CRS values:


,srs_names,files
0,"urn:ogc:def:crs,crs:EPSG:6.12:25833,crs:EPSG:6...",3
1,,2



Totals:


,count
buildings_in_xml,216
building_parts,87
measured_heights_in_xml,274
buildings_extracted,216
extracted_heights,216
missing_heights,0
usable_geometries,216
missing_geometries,0


In [65]:
# ============================================================
# 57 — INSPECT SAXONY BUILDING AND PART HEIGHTS
# ============================================================

SN_BLDG_NS = "http://www.opengis.net/citygml/building/1.0"
SN_BUILDING_TAG = f"{{{SN_BLDG_NS}}}Building"
SN_BUILDING_PART_TAG = f"{{{SN_BLDG_NS}}}BuildingPart"
SN_HEIGHT_TAG = f"{{{SN_BLDG_NS}}}measuredHeight"

def get_direct_number(element, tag):
    child = element.find(tag)
    return parse_number(child.text) if child is not None and child.text else np.nan

sn_nonempty_samples = sn_structure_test.loc[sn_structure_test["buildings_in_xml"] > 0, ["zip_file", "source_file"]]
sn_height_records = []

for row in sn_nonempty_samples.itertuples(index=False):
    zip_path = SN_TEST_DIR / row.zip_file

    with ZipFile(zip_path, "r") as archive:
        with archive.open(row.source_file) as stream:
            for event, building in etree.iterparse(stream, events=("end",), tag=SN_BUILDING_TAG, huge_tree=True):
                parent_height = get_direct_number(building, SN_HEIGHT_TAG)
                building_parts = building.findall(f".//{SN_BUILDING_PART_TAG}")
                part_heights = [get_direct_number(part, SN_HEIGHT_TAG) for part in building_parts]
                part_heights = [height for height in part_heights if pd.notna(height)]

                sn_height_records.append({
                    "zip_file": row.zip_file,
                    "source_file": row.source_file,
                    "lod2_id": building.get(GML_ID),
                    "parent_height_m": parent_height,
                    "building_parts": len(building_parts),
                    "parts_with_height": len(part_heights),
                    "minimum_part_height_m": min(part_heights) if part_heights else np.nan,
                    "maximum_part_height_m": max(part_heights) if part_heights else np.nan,
                    "first_descendant_height_m": parse_number(get_first_text(building, "measuredHeight"))
                })

                building.clear()

                while building.getprevious() is not None:
                    del building.getparent()[0]

sn_height_check = pd.DataFrame(sn_height_records)
sn_height_check["has_parent_height"] = sn_height_check["parent_height_m"].notna()
sn_height_check["has_parts"] = sn_height_check["building_parts"] > 0
sn_height_check["parent_equals_max_part"] = np.isclose(sn_height_check["parent_height_m"], sn_height_check["maximum_part_height_m"], atol=0.01, equal_nan=False)

sn_height_summary = pd.DataFrame([{
    "buildings_checked": len(sn_height_check),
    "buildings_with_parts": sn_height_check["has_parts"].sum(),
    "buildings_without_parts": (~sn_height_check["has_parts"]).sum(),
    "buildings_with_parent_height": sn_height_check["has_parent_height"].sum(),
    "buildings_missing_parent_height": (~sn_height_check["has_parent_height"]).sum(),
    "buildings_with_part_heights": sn_height_check["parts_with_height"].gt(0).sum(),
    "parent_equals_max_part": sn_height_check["parent_equals_max_part"].sum(),
    "parent_differs_from_max_part": (sn_height_check["has_parent_height"] & sn_height_check["parts_with_height"].gt(0) & ~sn_height_check["parent_equals_max_part"]).sum()
}])

print("=" * 70)
print("SAXONY BUILDING HEIGHT STRUCTURE")
print("=" * 70)

display(sn_height_summary)

print("\nBuildings missing a direct parent height:")
display(sn_height_check.loc[~sn_height_check["has_parent_height"], ["lod2_id", "building_parts", "parts_with_height", "minimum_part_height_m", "maximum_part_height_m", "first_descendant_height_m"]].head(30))

print("\nCases where the first descendant is not the tallest part:")
display(sn_height_check.loc[sn_height_check["parts_with_height"].gt(0) & ~np.isclose(sn_height_check["first_descendant_height_m"], sn_height_check["maximum_part_height_m"], atol=0.01, equal_nan=False), ["lod2_id", "building_parts", "minimum_part_height_m", "maximum_part_height_m", "first_descendant_height_m"]].head(30))

SAXONY BUILDING HEIGHT STRUCTURE


,buildings_checked,buildings_with_parts,buildings_without_parts,buildings_with_parent_height,buildings_missing_parent_height,buildings_with_part_heights,parent_equals_max_part,parent_differs_from_max_part
0,216,30,186,187,29,30,1,0



Buildings missing a direct parent height:


,lod2_id,building_parts,parts_with_height,minimum_part_height_m,maximum_part_height_m,first_descendant_height_m
1,DESNATPU1000Hcg8,2,2,7.049,10.416,7.049
5,DESNATPU1000GN1z,3,3,7.118,10.245,7.118
7,DESNATP14000027Y,2,2,3.023,3.219,3.219
15,DESNATPU1000G1UV,2,2,3.658,4.703,4.703
26,DESNATPU1000DIjG,5,5,16.463,18.451,16.463
35,DESNATPU1000IkeG,2,2,14.529,14.552,14.552
51,DESNATP1Mf0000Xf,5,5,7.041,7.883,7.883
57,DESNATPU1000I40r,3,3,6.284,14.509,14.509
65,DESNATPU1000Go32,2,2,6.938,11.163,11.163
74,DESNATPU1000IKPA,2,2,5.307,12.646,12.646



Cases where the first descendant is not the tallest part:


,lod2_id,building_parts,minimum_part_height_m,maximum_part_height_m,first_descendant_height_m
1,DESNATPU1000Hcg8,2,7.049,10.416,7.049
5,DESNATPU1000GN1z,3,7.118,10.245,7.118
26,DESNATPU1000DIjG,5,16.463,18.451,16.463
75,DESNATPU1000D7ZJ,2,5.215,8.131,5.215
109,DESNATPU1000DdcD,2,2.399,5.292,2.399
121,DESNATPU1000I3cg,3,6.034,10.067,8.631
128,DESNATPU1000HVc5,2,5.541,10.243,5.541
167,DESNATPU1000GKR3,4,11.801,13.415,13.353
173,DESNATGMqH0000NZ,2,7.058,7.175,7.058
174,DESNATPU1000DUis,9,7.661,11.327,8.265


In [66]:
# ============================================================
# 58 — DEFINE AND TEST SAXONY MULTIPART PARSER
# ============================================================

def resolve_multipart_height(building, building_part_tag):
    parent_height = parse_number(get_direct_text(building, "measuredHeight"))
    building_parts = building.findall(f".//{building_part_tag}")
    part_heights = [parse_number(get_direct_text(part, "measuredHeight")) for part in building_parts]
    part_heights = [height for height in part_heights if pd.notna(height)]
    maximum_part_height = max(part_heights) if part_heights else np.nan

    if pd.notna(parent_height):
        return parent_height, parent_height, maximum_part_height, len(building_parts), "building_measured_height"

    if pd.notna(maximum_part_height):
        return maximum_part_height, parent_height, maximum_part_height, len(building_parts), "maximum_building_part_height"

    return np.nan, parent_height, maximum_part_height, len(building_parts), "missing"

def extract_citygml_stream_multipart(stream, source_file, source_crs, building_tag, building_part_tag):
    records = []

    for event, building in etree.iterparse(stream, events=("end",), tag=building_tag, huge_tree=True):
        measured_height, parent_height, maximum_part_height, part_count, height_method = resolve_multipart_height(building, building_part_tag)
        geometry, footprint_method = extract_lod2_footprint(building)

        records.append({
            "lod2_id": building.get(GML_ID),
            "creation_date": get_direct_or_first_text(building, "creationDate"),
            "function": get_direct_or_first_text(building, "function"),
            "roof_type": get_direct_or_first_text(building, "roofType"),
            "measured_height_m": measured_height,
            "parent_height_m": parent_height,
            "maximum_part_height_m": maximum_part_height,
            "building_part_count": part_count,
            "height_method": height_method,
            "storeys_above_ground": parse_number(get_direct_or_first_text(building, "storeysAboveGround")),
            "source_file": source_file,
            "footprint_method": footprint_method,
            "geometry": geometry
        })

        building.clear()

        while building.getprevious() is not None:
            del building.getparent()[0]

    columns = ["lod2_id", "creation_date", "function", "roof_type", "measured_height_m", "parent_height_m", "maximum_part_height_m", "building_part_count", "height_method", "storeys_above_ground", "source_file", "footprint_method", "geometry"]

    if not records:
        return gpd.GeoDataFrame(columns=columns, geometry="geometry", crs=source_crs)

    return gpd.GeoDataFrame(records, geometry="geometry", crs=source_crs)

sn_parser_test_parts = []

for row in sn_structure_test.itertuples(index=False):
    zip_path = SN_TEST_DIR / row.zip_file

    with ZipFile(zip_path, "r") as archive:
        with archive.open(row.source_file) as stream:
            tile_gdf = extract_citygml_stream_multipart(
                stream,
                Path(row.source_file).name,
                "EPSG:25833",
                SN_BUILDING_TAG,
                SN_BUILDING_PART_TAG
            )

    if not tile_gdf.empty:
        tile_gdf["source_zip"] = row.zip_file
        sn_parser_test_parts.append(tile_gdf)

sn_parser_test_gdf = gpd.GeoDataFrame(pd.concat(sn_parser_test_parts, ignore_index=True), geometry="geometry", crs="EPSG:25833")

print("=" * 70)
print("SAXONY MULTIPART PARSER TEST")
print("=" * 70)
print(f"\nBuildings extracted      : {len(sn_parser_test_gdf):,}")
print(f"Buildings with parts     : {(sn_parser_test_gdf['building_part_count'] > 0).sum():,}")
print(f"Buildings without parts  : {(sn_parser_test_gdf['building_part_count'] == 0).sum():,}")
print(f"Measured heights         : {sn_parser_test_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights          : {sn_parser_test_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries        : {sn_parser_test_gdf.geometry.notna().sum():,}")
print(f"Missing geometries       : {sn_parser_test_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs          : {sn_parser_test_gdf['lod2_id'].nunique():,}")

print("\nHeight methods:")
display(sn_parser_test_gdf["height_method"].value_counts(dropna=False).rename_axis("height_method").reset_index(name="building_count"))

print("\nFootprint methods:")
display(sn_parser_test_gdf["footprint_method"].value_counts(dropna=False).rename_axis("footprint_method").reset_index(name="building_count"))

SAXONY MULTIPART PARSER TEST

Buildings extracted      : 216
Buildings with parts     : 30
Buildings without parts  : 186
Measured heights         : 216
Missing heights          : 0
Usable geometries        : 216
Missing geometries       : 0
Unique LoD2 IDs          : 216

Height methods:


,height_method,building_count
0,building_measured_height,187
1,maximum_building_part_height,29



Footprint methods:


,footprint_method,building_count
0,ground_surface,216


In [67]:
# ============================================================
# 59 — DOWNLOAD ALL SAXONY LoD2 CITYGML ZIP TILES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import time

SN_ZIP_DIR = SN_RAW_DIR / "zips"
SN_ZIP_DIR.mkdir(parents=True, exist_ok=True)

SN_MAX_WORKERS = 8
SN_MAX_RETRIES = 3
SN_CHUNK_SIZE = 1024 * 1024

def validate_sn_zip(path):
    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with ZipFile(path, "r") as archive:
            if archive.testzip() is not None:
                return False

            gml_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]
            return len(gml_members) > 0

    except Exception:
        return False

def download_sn_tile(url, filename):
    output_path = SN_ZIP_DIR / filename
    temp_path = SN_ZIP_DIR / f"{filename}.part"

    if validate_sn_zip(output_path):
        return {"filename": filename, "status": "already_downloaded", "size_bytes": output_path.stat().st_size, "error": None}

    if output_path.exists():
        output_path.unlink()

    for attempt in range(1, SN_MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 900)) as response:
                response.raise_for_status()
                expected_size = int(response.headers.get("content-length", 0))

                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=SN_CHUNK_SIZE):
                        if chunk:
                            file.write(chunk)

            actual_size = temp_path.stat().st_size

            if expected_size and actual_size != expected_size:
                raise ValueError(f"Incomplete download: expected {expected_size:,} bytes, received {actual_size:,}")

            if not validate_sn_zip(temp_path):
                raise ValueError("Downloaded file is not a valid CityGML ZIP archive.")

            temp_path.replace(output_path)
            return {"filename": filename, "status": "downloaded", "size_bytes": output_path.stat().st_size, "error": None}

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == SN_MAX_RETRIES:
                return {"filename": filename, "status": "failed", "size_bytes": 0, "error": str(error)}

            time.sleep(attempt * 3)

sn_download_results = []

with ThreadPoolExecutor(max_workers=SN_MAX_WORKERS) as executor:
    futures = {executor.submit(download_sn_tile, row.download_url, row.filename): row.filename for row in sn_downloads.itertuples(index=False)}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading Saxony LoD2 tiles"):
        sn_download_results.append(future.result())

sn_download_results_df = pd.DataFrame(sn_download_results)
sn_download_results_path = REPORT_DIR / "lod2_download_results_SN.csv"
sn_download_results_df.to_csv(sn_download_results_path, index=False)

sn_zip_files = sorted(SN_ZIP_DIR.glob("*.zip"))

print("=" * 70)
print("SAXONY LoD2 DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nTiles requested       : {len(sn_downloads):,}")
print(f"ZIP files present     : {len(sn_zip_files):,}")
print(f"Downloaded now        : {(sn_download_results_df['status'] == 'downloaded').sum():,}")
print(f"Already downloaded    : {(sn_download_results_df['status'] == 'already_downloaded').sum():,}")
print(f"Failed                : {(sn_download_results_df['status'] == 'failed').sum():,}")
print(f"Total downloaded size : {sum(path.stat().st_size for path in sn_zip_files) / 1024**3:,.2f} GB")
print(f"Download report       : {sn_download_results_path}")

display(sn_download_results_df["status"].value_counts().rename_axis("status").reset_index(name="count"))

if sn_download_results_df["status"].eq("failed").any():
    display(sn_download_results_df[sn_download_results_df["status"] == "failed"].head(30))

SAXONY LoD2 DOWNLOAD SUMMARY

Tiles requested       : 4,938
ZIP files present     : 4,938
Downloaded now        : 4,938
Already downloaded    : 0
Failed                : 0
Total downloaded size : 6.34 GB
Download report       : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_download_results_SN.csv


,status,count
0,downloaded,4938


In [68]:
# ============================================================
# 60 — PROCESS ALL SAXONY LoD2 TILES
# ============================================================

SN_OUTPUT_DIR = EXTRACTED_DIR / "SN"
SN_PART_DIR = SN_OUTPUT_DIR / "parts"
SN_MANIFEST_DIR = SN_OUTPUT_DIR / "manifests"

SN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SN_PART_DIR.mkdir(parents=True, exist_ok=True)
SN_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

SN_BATCH_SIZE = 250

sn_source_records = []

for zip_path in tqdm(sn_zip_files, desc="Indexing Saxony ZIP files"):
    with ZipFile(zip_path, "r") as archive:
        gml_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

        for member in gml_members:
            sn_source_records.append({
                "archive_path": str(zip_path),
                "archive_name": zip_path.name,
                "member": member,
                "source_key": f"{zip_path.name}::{member}"
            })

sn_sources = pd.DataFrame(sn_source_records)
sn_batch_starts = list(range(0, len(sn_sources), SN_BATCH_SIZE))

print("=" * 70)
print("SAXONY PROCESSING SETUP")
print("=" * 70)
print(f"\nZIP archives          : {len(sn_zip_files):,}")
print(f"CityGML files         : {len(sn_sources):,}")
print(f"Unique source keys    : {sn_sources['source_key'].nunique():,}")
print(f"Batch size            : {SN_BATCH_SIZE:,}")
print(f"Expected Parquet parts: {len(sn_batch_starts):,}")

if sn_sources["source_key"].duplicated().any():
    raise ValueError("Duplicate Saxony source keys detected.")

sn_batch_summaries = []

for batch_start in tqdm(sn_batch_starts, total=len(sn_batch_starts), desc="Processing Saxony batches"):
    batch_number = batch_start // SN_BATCH_SIZE + 1
    batch_sources = sn_sources.iloc[batch_start:batch_start + SN_BATCH_SIZE].copy()

    part_path = SN_PART_DIR / f"lod2_buildings_SN_part_{batch_number:04d}.parquet"
    manifest_path = SN_MANIFEST_DIR / f"lod2_manifest_SN_part_{batch_number:04d}.csv"

    if manifest_path.exists():
        existing_manifest = pd.read_csv(manifest_path)
        manifest_complete = len(existing_manifest) == len(batch_sources) and not existing_manifest["status"].eq("failed").any()
        output_complete = existing_manifest["buildings"].sum() == 0 or part_path.exists()

        if manifest_complete and output_complete:
            sn_batch_summaries.append({
                "batch": batch_number,
                "source_files": len(existing_manifest),
                "processed_files": existing_manifest["status"].eq("processed").sum(),
                "empty_files": existing_manifest["status"].eq("empty").sum(),
                "failed_files": existing_manifest["status"].eq("failed").sum(),
                "buildings": existing_manifest["buildings"].sum(),
                "status": "already_processed"
            })
            continue

    if part_path.exists():
        part_path.unlink()

    batch_parts = []
    batch_manifest = []

    for archive_path, archive_sources in batch_sources.groupby("archive_path", sort=False):
        with ZipFile(archive_path, "r") as archive:
            for row in archive_sources.itertuples(index=False):
                try:
                    with archive.open(row.member) as stream:
                        tile_gdf = extract_citygml_stream_multipart(
                            stream,
                            Path(row.member).name,
                            "EPSG:25833",
                            SN_BUILDING_TAG,
                            SN_BUILDING_PART_TAG
                        )

                    if tile_gdf.empty:
                        batch_manifest.append({
                            "source_key": row.source_key,
                            "archive_name": row.archive_name,
                            "source_member": row.member,
                            "status": "empty",
                            "buildings": 0,
                            "error": None
                        })
                        continue

                    tile_gdf["state_code"] = "SN"
                    tile_gdf["source_state"] = "Saxony"
                    tile_gdf["source_crs"] = "EPSG:25833"
                    tile_gdf["citygml_version"] = "1.0"
                    tile_gdf["height_source"] = tile_gdf["height_method"]
                    tile_gdf["source_archive"] = row.archive_name
                    tile_gdf["source_member"] = row.member
                    tile_gdf["source_path"] = row.source_key

                    batch_parts.append(tile_gdf)

                    batch_manifest.append({
                        "source_key": row.source_key,
                        "archive_name": row.archive_name,
                        "source_member": row.member,
                        "status": "processed",
                        "buildings": len(tile_gdf),
                        "error": None
                    })

                except Exception as error:
                    batch_manifest.append({
                        "source_key": row.source_key,
                        "archive_name": row.archive_name,
                        "source_member": row.member,
                        "status": "failed",
                        "buildings": 0,
                        "error": str(error)
                    })

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25833")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    batch_manifest_df = pd.DataFrame(batch_manifest)
    batch_manifest_df.to_csv(manifest_path, index=False)

    sn_batch_summaries.append({
        "batch": batch_number,
        "source_files": len(batch_manifest_df),
        "processed_files": batch_manifest_df["status"].eq("processed").sum(),
        "empty_files": batch_manifest_df["status"].eq("empty").sum(),
        "failed_files": batch_manifest_df["status"].eq("failed").sum(),
        "buildings": batch_manifest_df["buildings"].sum(),
        "status": "processed"
    })

    del batch_parts, batch_manifest, batch_manifest_df
    gc.collect()

sn_batch_summary_df = pd.DataFrame(sn_batch_summaries)
sn_manifest_files = sorted(SN_MANIFEST_DIR.glob("lod2_manifest_SN_part_*.csv"))
sn_manifests = pd.concat([pd.read_csv(path) for path in sn_manifest_files], ignore_index=True)
sn_part_files = sorted(SN_PART_DIR.glob("lod2_buildings_SN_part_*.parquet"))

sn_batch_summary_path = REPORT_DIR / "lod2_batch_summary_SN.csv"
sn_manifest_path = REPORT_DIR / "lod2_processing_manifest_SN.csv"

sn_batch_summary_df.to_csv(sn_batch_summary_path, index=False)
sn_manifests.to_csv(sn_manifest_path, index=False)

print("=" * 70)
print("SAXONY LoD2 PROCESSING SUMMARY")
print("=" * 70)
print(f"\nSource files recorded : {len(sn_manifests):,}")
print(f"Files with buildings  : {sn_manifests['status'].eq('processed').sum():,}")
print(f"Empty CityGML files   : {sn_manifests['status'].eq('empty').sum():,}")
print(f"Failed files          : {sn_manifests['status'].eq('failed').sum():,}")
print(f"Buildings extracted   : {sn_manifests['buildings'].sum():,}")
print(f"Parquet parts created : {len(sn_part_files):,}")
print(f"Part directory        : {SN_PART_DIR}")
print(f"Combined manifest     : {sn_manifest_path}")

display(sn_manifests["status"].value_counts().rename_axis("status").reset_index(name="source_files"))

if sn_manifests["status"].eq("failed").any():
    display(sn_manifests[sn_manifests["status"] == "failed"].head(30))

Indexing Saxony ZIP files: 100%|██████████████████████████████████████████████| 4938/4938 [00:07<00:00, 671.70it/s]


SAXONY PROCESSING SETUP

ZIP archives          : 4,938
CityGML files         : 4,938
Unique source keys    : 4,938
Batch size            : 250
Expected Parquet parts: 20


Processing Saxony batches: 100%|███████████████████████████████████████████████████| 20/20 [29:55<00:00, 89.79s/it]

SAXONY LoD2 PROCESSING SUMMARY

Source files recorded : 4,938
Files with buildings  : 4,672
Empty CityGML files   : 266
Failed files          : 0
Buildings extracted   : 2,290,890
Parquet parts created : 20
Part directory        : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/SN/parts
Combined manifest     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_processing_manifest_SN.csv


,status,source_files
0,processed,4672
1,empty,266


In [69]:
# ============================================================
# 61 — VALIDATE AND SUMMARISE SAXONY RAW LoD2 DATA
# ============================================================

sn_expected_sources = set(sn_sources["source_key"])
sn_manifest_sources = set(sn_manifests["source_key"])
sn_missing_sources = sorted(sn_expected_sources - sn_manifest_sources)
sn_unexpected_sources = sorted(sn_manifest_sources - sn_expected_sources)
sn_duplicate_manifest_sources = sn_manifests["source_key"].duplicated().sum()

sn_summary_records = []
sn_footprint_counts = {}
sn_height_method_counts = {}
sn_id_hashes = []

for part_path in tqdm(sn_part_files, desc="Summarising Saxony parts"):
    part = pd.read_parquet(part_path, columns=["lod2_id", "measured_height_m", "height_method", "footprint_method", "geometry"])
    sn_id_hashes.append(pd.util.hash_pandas_object(part["lod2_id"], index=False).to_numpy(dtype="uint64"))

    sn_summary_records.append({
        "buildings": len(part),
        "missing_ids": part["lod2_id"].isna().sum(),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    for method, count in part["footprint_method"].value_counts(dropna=False).items():
        sn_footprint_counts[method] = sn_footprint_counts.get(method, 0) + count

    for method, count in part["height_method"].value_counts(dropna=False).items():
        sn_height_method_counts[method] = sn_height_method_counts.get(method, 0) + count

    del part
    gc.collect()

sn_part_stats = pd.DataFrame(sn_summary_records)
sn_all_id_hashes = np.concatenate(sn_id_hashes)
sn_unique_ids = np.unique(sn_all_id_hashes).size
sn_duplicate_ids = len(sn_all_id_hashes) - sn_unique_ids

sn_summary = pd.DataFrame([{
    "state_code": "SN",
    "state_name": "Saxony",
    "zip_archives": len(sn_zip_files),
    "source_files_expected": len(sn_expected_sources),
    "source_files_recorded": len(sn_manifest_sources),
    "files_with_buildings": sn_manifests["status"].eq("processed").sum(),
    "empty_files": sn_manifests["status"].eq("empty").sum(),
    "failed_files": sn_manifests["status"].eq("failed").sum(),
    "missing_source_files": len(sn_missing_sources),
    "unexpected_source_files": len(sn_unexpected_sources),
    "duplicate_manifest_sources": sn_duplicate_manifest_sources,
    "parquet_parts": len(sn_part_files),
    "buildings": sn_part_stats["buildings"].sum(),
    "unique_ids": sn_unique_ids,
    "duplicate_ids": sn_duplicate_ids,
    "missing_ids": sn_part_stats["missing_ids"].sum(),
    "measured_heights": sn_part_stats["measured_heights"].sum(),
    "missing_heights": sn_part_stats["missing_heights"].sum(),
    "usable_geometries": sn_part_stats["usable_geometries"].sum(),
    "missing_geometries": sn_part_stats["missing_geometries"].sum(),
    "minimum_height_m": sn_part_stats["minimum_height_m"].min(),
    "maximum_height_m": sn_part_stats["maximum_height_m"].max(),
    "output_directory": str(SN_PART_DIR)
}])

sn_height_method_summary = pd.DataFrame([{"height_method": method, "building_count": count} for method, count in sn_height_method_counts.items()]).sort_values("building_count", ascending=False)
sn_footprint_summary = pd.DataFrame([{"footprint_method": method, "building_count": count} for method, count in sn_footprint_counts.items()]).sort_values("building_count", ascending=False)

sn_summary_path = REPORT_DIR / "lod2_extraction_summary_SN.csv"
sn_height_method_summary_path = REPORT_DIR / "lod2_height_methods_SN.csv"
sn_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_SN.csv"

sn_summary.to_csv(sn_summary_path, index=False)
sn_height_method_summary.to_csv(sn_height_method_summary_path, index=False)
sn_footprint_summary.to_csv(sn_footprint_summary_path, index=False)

print("=" * 70)
print("SAXONY FINAL LoD2 VALIDATION")
print("=" * 70)

display(sn_summary)
display(sn_height_method_summary)
display(sn_footprint_summary)

print(f"\nState summary saved to     : {sn_summary_path}")
print(f"Height summary saved to    : {sn_height_method_summary_path}")
print(f"Footprint summary saved to : {sn_footprint_summary_path}")

if sn_missing_sources:
    display(pd.DataFrame({"missing_source": sn_missing_sources}).head(30))
    

Summarising Saxony parts: 100%|████████████████████████████████████████████████████| 20/20 [00:17<00:00,  1.12it/s]

SAXONY FINAL LoD2 VALIDATION


,state_code,state_name,zip_archives,source_files_expected,source_files_recorded,files_with_buildings,empty_files,failed_files,missing_source_files,unexpected_source_files,...,unique_ids,duplicate_ids,missing_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,SN,Saxony,4938,4938,4938,4672,266,0,0,0,...,2290450,440,0,2290890,0,2290890,0,-109.691,1232.427,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


,height_method,building_count
0,building_measured_height,1884863
1,maximum_building_part_height,406027


,footprint_method,building_count
0,ground_surface,2290890



State summary saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_SN.csv
Height summary saved to    : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_height_methods_SN.csv
Footprint summary saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_SN.csv


In [70]:
# ============================================================
# 62 — INVESTIGATE SAXONY DUPLICATES AND EXTREME HEIGHTS
# ============================================================

sn_duplicate_id_hashes = set()
sn_id_counts = Counter()

# First pass: identify duplicated IDs exactly, not only through hashes
for part_path in tqdm(sn_part_files, desc="Counting Saxony IDs"):
    ids = pd.read_parquet(part_path, columns=["lod2_id"])["lod2_id"].dropna()
    sn_id_counts.update(ids)

sn_duplicate_ids = {lod2_id for lod2_id, count in sn_id_counts.items() if count > 1}

print("=" * 70)
print("SAXONY DUPLICATE-ID DISCOVERY")
print("=" * 70)
print(f"\nDuplicated LoD2 IDs : {len(sn_duplicate_ids):,}")
print(f"Extra duplicate rows: {sum(count - 1 for count in sn_id_counts.values() if count > 1):,}")

sn_duplicate_records = []
sn_extreme_records = []

for part_path in tqdm(sn_part_files, desc="Reading Saxony anomalies"):
    columns = [
        "lod2_id",
        "measured_height_m",
        "parent_height_m",
        "maximum_part_height_m",
        "building_part_count",
        "height_method",
        "source_archive",
        "source_member",
        "source_path",
        "geometry"
    ]

    part = gpd.read_parquet(part_path, columns=columns)

    duplicate_rows = part[part["lod2_id"].isin(sn_duplicate_ids)].copy()

    if not duplicate_rows.empty:
        duplicate_rows["part_file"] = part_path.name
        duplicate_rows["geometry_wkb"] = duplicate_rows.geometry.to_wkb(hex=True)
        sn_duplicate_records.append(pd.DataFrame(duplicate_rows.drop(columns="geometry")))

    extreme_rows = part[
        (part["measured_height_m"] <= 0)
        | (part["measured_height_m"] > 300)
    ].copy()

    if not extreme_rows.empty:
        extreme_rows["part_file"] = part_path.name
        extreme_rows["geometry_area_m2"] = extreme_rows.geometry.area
        extreme_rows["geometry_wkb"] = extreme_rows.geometry.to_wkb(hex=True)
        sn_extreme_records.append(pd.DataFrame(extreme_rows.drop(columns="geometry")))

    del part
    gc.collect()

sn_duplicate_records = pd.concat(sn_duplicate_records, ignore_index=True) if sn_duplicate_records else pd.DataFrame()
sn_extreme_records = pd.concat(sn_extreme_records, ignore_index=True) if sn_extreme_records else pd.DataFrame()

if not sn_duplicate_records.empty:
    sn_duplicate_comparison = (
        sn_duplicate_records
        .groupby("lod2_id")
        .agg(
            record_count=("lod2_id", "size"),
            unique_source_files=("source_path", "nunique"),
            unique_heights=("measured_height_m", "nunique"),
            unique_geometries=("geometry_wkb", "nunique"),
            minimum_height_m=("measured_height_m", "min"),
            maximum_height_m=("measured_height_m", "max")
        )
        .reset_index()
    )

    sn_exact_duplicate_ids = sn_duplicate_comparison[
        (sn_duplicate_comparison["unique_heights"] == 1)
        & (sn_duplicate_comparison["unique_geometries"] == 1)
    ]

    sn_conflicting_duplicate_ids = sn_duplicate_comparison[
        (sn_duplicate_comparison["unique_heights"] > 1)
        | (sn_duplicate_comparison["unique_geometries"] > 1)
    ]
else:
    sn_duplicate_comparison = pd.DataFrame()
    sn_exact_duplicate_ids = pd.DataFrame()
    sn_conflicting_duplicate_ids = pd.DataFrame()

sn_duplicate_records_path = REPORT_DIR / "lod2_duplicate_records_SN.csv"
sn_duplicate_comparison_path = REPORT_DIR / "lod2_duplicate_comparison_SN.csv"
sn_extreme_heights_path = REPORT_DIR / "lod2_extreme_heights_SN.csv"

sn_duplicate_records.to_csv(sn_duplicate_records_path, index=False)
sn_duplicate_comparison.to_csv(sn_duplicate_comparison_path, index=False)
sn_extreme_records.to_csv(sn_extreme_heights_path, index=False)

print("\n" + "=" * 70)
print("SAXONY DUPLICATE-ID SUMMARY")
print("=" * 70)
print(f"\nDuplicate IDs                    : {len(sn_duplicate_comparison):,}")
print(f"Exact repeated IDs               : {len(sn_exact_duplicate_ids):,}")
print(f"IDs with conflicting data        : {len(sn_conflicting_duplicate_ids):,}")
print(f"Duplicate records                : {len(sn_duplicate_records):,}")

print("\n" + "=" * 70)
print("SAXONY EXTREME-HEIGHT SUMMARY")
print("=" * 70)
print(f"\nHeight <= 0 m                    : {(sn_extreme_records['measured_height_m'] <= 0).sum():,}")
print(f"Height > 300 m                   : {(sn_extreme_records['measured_height_m'] > 300).sum():,}")
print(f"Total extreme records            : {len(sn_extreme_records):,}")

print("\nDuplicate comparison:")
display(sn_duplicate_comparison.head(30))

print("\nConflicting duplicate IDs:")
display(sn_conflicting_duplicate_ids.head(30))

print("\nLowest heights:")
display(sn_extreme_records.sort_values("measured_height_m").head(20))

print("\nHighest heights:")
display(sn_extreme_records.sort_values("measured_height_m", ascending=False).head(20))

print(f"\nDuplicate records saved to   : {sn_duplicate_records_path}")
print(f"Duplicate comparison saved to: {sn_duplicate_comparison_path}")
print(f"Extreme heights saved to     : {sn_extreme_heights_path}")

Counting Saxony IDs: 100%|█████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 12.00it/s]


SAXONY DUPLICATE-ID DISCOVERY

Duplicated LoD2 IDs : 440
Extra duplicate rows: 440


Reading Saxony anomalies: 100%|████████████████████████████████████████████████████| 20/20 [00:18<00:00,  1.07it/s]


SAXONY DUPLICATE-ID SUMMARY

Duplicate IDs                    : 440
Exact repeated IDs               : 62
IDs with conflicting data        : 378
Duplicate records                : 880

SAXONY EXTREME-HEIGHT SUMMARY

Height <= 0 m                    : 4
Height > 300 m                   : 71
Total extreme records            : 75

Duplicate comparison:


,lod2_id,record_count,unique_source_files,unique_heights,unique_geometries,minimum_height_m,maximum_height_m
0,DESNATGMsA00013i,2,2,2,2,3.461,3.986
1,DESNATP1100004Xy,2,2,1,2,2.957,2.957
2,DESNATP110000Wvk,2,2,2,2,6.237,9.663
3,DESNATP110000Xhv,2,2,2,2,3.014,5.400
4,DESNATP110000jt4,2,2,2,2,8.984,9.675
5,DESNATP1180005FS,2,2,2,2,11.714,11.725
6,DESNATP12000AGYE,2,2,2,2,8.642,9.219
7,DESNATP12000AWQw,2,2,2,2,6.310,15.096
8,DESNATP12P0000zx,2,2,2,2,10.153,10.281
9,DESNATP12w0001Ze,2,2,2,2,3.030,3.031


...


Lowest heights:


,lod2_id,measured_height_m,parent_height_m,maximum_part_height_m,building_part_count,height_method,source_archive,source_member,source_path,part_file,geometry_area_m2,geometry_wkb
72,DESNATGMlf0000id,-109.691,-109.691,NaN,0,building_measured_height,lod2_33420_5646_2_sn_citygml.zip,lod2_33420_5646_2_sn.gml,lod2_33420_5646_2_sn_citygml.zip::lod2_33420_5...,lod2_buildings_SN_part_0015.parquet,3.101300,01030000000100000007000000CDCCCCCC2CB8194148E1...
73,DESNATPU1000HEYH,-15.447,-15.447,NaN,0,building_measured_height,lod2_33486_5638_2_sn_citygml.zip,lod2_33486_5638_2_sn.gml,lod2_33486_5638_2_sn_citygml.zip::lod2_33486_5...,lod2_buildings_SN_part_0019.parquet,37.057371,01030000000100000007000000FED478692AB21D413F35...
70,DESNATPU1000DbNe,-1.327,-1.327,NaN,0,building_measured_height,lod2_33354_5632_2_sn_citygml.zip,lod2_33354_5632_2_sn.gml,lod2_33354_5632_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,131.856207,010300000001000000150000006DE7FBA9BAB11541CFF7...
74,DESNATPU1000Bfa8,-0.374,-0.374,NaN,0,building_measured_height,lod2_33486_5638_2_sn_citygml.zip,lod2_33486_5638_2_sn.gml,lod2_33486_5638_2_sn_citygml.zip::lod2_33486_5...,lod2_buildings_SN_part_0019.parquet,72.661598,0103000000010000000800000089416065E4AD1D41508D...
27,DESNATP1GI0002Z1,303.500,303.500,NaN,0,building_measured_height,lod2_33352_5636_2_sn_citygml.zip,lod2_33352_5636_2_sn.gml,lod2_33352_5636_2_sn_citygml.zip::lod2_33352_5...,lod2_buildings_SN_part_0008.parquet,456.957300,0103000000010000001500000085EB51B82D991541D7A3...
65,DESNATP1Me0000HL,1004.545,1004.545,NaN,0,building_measured_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,41.730930,010300000001000000050000004E621058ECB2154179E9...
66,DESNATP1Me0000HK,1004.554,1004.554,NaN,0,building_measured_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,40.211069,01030000000100000005000000894160E504B31541FED4...
3,DESNATP1fl0000JF,1014.128,1014.128,NaN,0,building_measured_height,lod2_33332_5590_2_sn_citygml.zip,lod2_33332_5590_2_sn.gml,lod2_33332_5590_2_sn_citygml.zip::lod2_33332_5...,lod2_buildings_SN_part_0005.parquet,45.264062,0103000000010000000500000048E17A14AC5414415EBA...
53,DESNATP1OP0000Eo,1015.082,1015.082,NaN,0,building_measured_height,lod2_33354_5586_2_sn_citygml.zip,lod2_33354_5586_2_sn.gml,lod2_33354_5586_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,51.267566,01030000000100000005000000A01A2FDD48AD15410E2D...
4,DESNATP1fl0000JE,1015.321,1015.321,NaN,0,building_measured_height,lod2_33332_5590_2_sn_citygml.zip,lod2_33332_5590_2_sn.gml,lod2_33332_5590_2_sn_citygml.zip::lod2_33332_5...,lod2_buildings_SN_part_0005.parquet,47.855400,01030000000100000005000000B81E85EB795414419643...



Highest heights:


,lod2_id,measured_height_m,parent_height_m,maximum_part_height_m,building_part_count,height_method,source_archive,source_member,source_path,part_file,geometry_area_m2,geometry_wkb
64,DESNATPU1000FBkl,1232.427,NaN,1232.427,4,maximum_building_part_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,901.111350,010300000001000000140000000000000072A6154152B8...
59,DESNATPU1000Gwr8,1224.816,NaN,1224.816,3,maximum_building_part_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,690.738250,0103000000010000000D0000007B14AE474EA71541713D...
61,DESNATPU1000DTEU,1223.191,NaN,1223.191,4,maximum_building_part_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,245.021100,01030000000100000013000000295C8FC23CA51541A470...
56,DESNATPU1000EXEe,1219.534,1219.534,NaN,0,building_measured_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,167.193006,010300000001000000080000005C8FC2F5C4A615413D0A...
67,DESNATP1Me0000QV,1215.828,1215.828,NaN,0,building_measured_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,112.187219,0103000000010000000B00000096438BEC5BA71541448B...
40,DESNATPU1000EQpH,1209.263,NaN,1209.263,2,maximum_building_part_height,lod2_33354_5586_2_sn_citygml.zip,lod2_33354_5586_2_sn.gml,lod2_33354_5586_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,78.945500,01030000000100000008000000A4703D0A92A01541C3F5...
69,DESNATPU1000CTzU,1206.085,1206.085,NaN,0,building_measured_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,34.982200,01030000000100000005000000713D0AD7F2A71541B81E...
68,DESNATPU1000IVt3,1199.006,NaN,1199.006,4,maximum_building_part_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,946.067450,0103000000010000000F000000D7A3703DB0A415413333...
55,DESNATP1Me0000HM,1192.689,1192.689,NaN,0,building_measured_height,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,lod2_33354_5588_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,153.210771,0103000000010000000700000017D9CE7732A51541643B...
46,DESNATPU1000Ee0h,1189.758,1189.758,NaN,0,building_measured_height,lod2_33354_5586_2_sn_citygml.zip,lod2_33354_5586_2_sn.gml,lod2_33354_5586_2_sn_citygml.zip::lod2_33354_5...,lod2_buildings_SN_part_0008.parquet,35.035100,0103000000010000000500000000000000D4A31541F628...



Duplicate records saved to   : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_duplicate_records_SN.csv
Duplicate comparison saved to: /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_duplicate_comparison_SN.csv
Extreme heights saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extreme_heights_SN.csv


In [71]:
# ============================================================
# 63 — VERIFY SAXONY EXTREME HEIGHTS USING 3D COORDINATES
# ============================================================

def extract_z_values(element):
    z_values = []

    for coordinate_element in element.iter():
        local_name = etree.QName(coordinate_element).localname

        if local_name not in {"posList", "pos"} or not coordinate_element.text:
            continue

        values = [float(value) for value in coordinate_element.text.split()]

        if local_name == "pos":
            if len(values) >= 3:
                z_values.append(values[2])
            continue

        dimension_text = coordinate_element.get("srsDimension") or coordinate_element.get("dimension")
        dimension = int(dimension_text) if dimension_text and dimension_text.isdigit() else 3

        if dimension >= 3:
            z_values.extend(values[2::dimension])

    return z_values

def extract_surface_z_values(building, surface_name):
    z_values = []

    for element in building.iter():
        if etree.QName(element).localname == surface_name:
            z_values.extend(extract_z_values(element))

    return z_values

sn_extreme_source_counts = (
    sn_extreme_records
    .groupby(["source_archive", "source_member"])
    .size()
    .reset_index(name="extreme_buildings")
    .sort_values("extreme_buildings", ascending=False)
)

sn_z_check_records = []

for source in tqdm(sn_extreme_source_counts.itertuples(index=False), total=len(sn_extreme_source_counts), desc="Checking Saxony 3D heights"):
    zip_path = SN_ZIP_DIR / source.source_archive
    source_rows = sn_extreme_records[(sn_extreme_records["source_archive"] == source.source_archive) & (sn_extreme_records["source_member"] == source.source_member)]
    target_ids = set(source_rows["lod2_id"])

    with ZipFile(zip_path, "r") as archive:
        with archive.open(source.source_member) as stream:
            for event, building in etree.iterparse(stream, events=("end",), tag=SN_BUILDING_TAG, huge_tree=True):
                lod2_id = building.get(GML_ID)

                if lod2_id not in target_ids:
                    building.clear()
                    while building.getprevious() is not None:
                        del building.getparent()[0]
                    continue

                original_row = source_rows[source_rows["lod2_id"] == lod2_id].iloc[0]
                all_z = extract_z_values(building)
                ground_z = extract_surface_z_values(building, "GroundSurface")
                roof_z = extract_surface_z_values(building, "RoofSurface")

                z_min = min(all_z) if all_z else np.nan
                z_max = max(all_z) if all_z else np.nan
                z_extent = z_max - z_min if all_z else np.nan

                ground_min = min(ground_z) if ground_z else np.nan
                ground_max = max(ground_z) if ground_z else np.nan
                ground_median = float(np.median(ground_z)) if ground_z else np.nan

                roof_min = min(roof_z) if roof_z else np.nan
                roof_max = max(roof_z) if roof_z else np.nan
                roof_median = float(np.median(roof_z)) if roof_z else np.nan

                roof_max_minus_ground_min = roof_max - ground_min if ground_z and roof_z else np.nan
                roof_median_minus_ground_median = roof_median - ground_median if ground_z and roof_z else np.nan

                sn_z_check_records.append({
                    "lod2_id": lod2_id,
                    "source_archive": source.source_archive,
                    "source_member": source.source_member,
                    "source_height_m": original_row["measured_height_m"],
                    "height_method": original_row["height_method"],
                    "building_part_count": original_row["building_part_count"],
                    "z_min_m": z_min,
                    "z_max_m": z_max,
                    "z_extent_m": z_extent,
                    "ground_min_z_m": ground_min,
                    "ground_max_z_m": ground_max,
                    "roof_min_z_m": roof_min,
                    "roof_max_z_m": roof_max,
                    "roof_max_minus_ground_min_m": roof_max_minus_ground_min,
                    "roof_median_minus_ground_median_m": roof_median_minus_ground_median
                })

                building.clear()

                while building.getprevious() is not None:
                    del building.getparent()[0]

sn_z_height_check = pd.DataFrame(sn_z_check_records)

sn_z_height_check["source_matches_z_extent"] = np.isclose(sn_z_height_check["source_height_m"], sn_z_height_check["z_extent_m"], atol=0.5, equal_nan=False)
sn_z_height_check["source_matches_z_min"] = np.isclose(sn_z_height_check["source_height_m"], sn_z_height_check["z_min_m"], atol=0.5, equal_nan=False)
sn_z_height_check["source_matches_z_max"] = np.isclose(sn_z_height_check["source_height_m"], sn_z_height_check["z_max_m"], atol=0.5, equal_nan=False)
sn_z_height_check["source_matches_roof_ground"] = np.isclose(sn_z_height_check["source_height_m"], sn_z_height_check["roof_max_minus_ground_min_m"], atol=0.5, equal_nan=False)
sn_z_height_check["z_extent_plausible"] = sn_z_height_check["z_extent_m"].between(0.5, 300)
sn_z_height_check["roof_ground_plausible"] = sn_z_height_check["roof_max_minus_ground_min_m"].between(0.5, 300)

sn_z_height_check_path = REPORT_DIR / "lod2_extreme_height_3d_check_SN.csv"
sn_z_height_check.to_csv(sn_z_height_check_path, index=False)

print("=" * 70)
print("SAXONY EXTREME HEIGHT 3D VERIFICATION")
print("=" * 70)
print(f"\nExtreme records requested        : {len(sn_extreme_records):,}")
print(f"Extreme records found in XML     : {len(sn_z_height_check):,}")
print(f"Source height matches Z extent   : {sn_z_height_check['source_matches_z_extent'].sum():,}")
print(f"Source height matches Z minimum  : {sn_z_height_check['source_matches_z_min'].sum():,}")
print(f"Source height matches Z maximum  : {sn_z_height_check['source_matches_z_max'].sum():,}")
print(f"Source height matches roof-ground: {sn_z_height_check['source_matches_roof_ground'].sum():,}")
print(f"Plausible Z extents              : {sn_z_height_check['z_extent_plausible'].sum():,}")
print(f"Plausible roof-ground heights    : {sn_z_height_check['roof_ground_plausible'].sum():,}")
print(f"Results saved to                 : {sn_z_height_check_path}")

print("\nExtreme records by source tile:")
display(sn_extreme_source_counts)

print("\nLowest source heights with 3D comparison:")
display(sn_z_height_check.sort_values("source_height_m").head(20))

print("\nHighest source heights with 3D comparison:")
display(sn_z_height_check.sort_values("source_height_m", ascending=False).head(30))

Checking Saxony 3D heights: 100%|████████████████████████████████████████████████████| 9/9 [00:03<00:00,  2.32it/s]

SAXONY EXTREME HEIGHT 3D VERIFICATION

Extreme records requested        : 75
Extreme records found in XML     : 75
Source height matches Z extent   : 71
Source height matches Z minimum  : 0
Source height matches Z maximum  : 70
Source height matches roof-ground: 75
Plausible Z extents              : 4
Plausible roof-ground heights    : 0
Results saved to                 : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extreme_height_3d_check_SN.csv

Extreme records by source tile:


,source_archive,source_member,extreme_buildings
3,lod2_33354_5586_2_sn_citygml.zip,lod2_33354_5586_2_sn.gml,26
1,lod2_33352_5586_2_sn_citygml.zip,lod2_33352_5586_2_sn.gml,20
4,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,16
0,lod2_33332_5590_2_sn_citygml.zip,lod2_33332_5590_2_sn.gml,7
8,lod2_33486_5638_2_sn_citygml.zip,lod2_33486_5638_2_sn.gml,2
2,lod2_33352_5636_2_sn_citygml.zip,lod2_33352_5636_2_sn.gml,1
5,lod2_33354_5632_2_sn_citygml.zip,lod2_33354_5632_2_sn.gml,1
6,lod2_33356_5588_2_sn_citygml.zip,lod2_33356_5588_2_sn.gml,1
7,lod2_33420_5646_2_sn_citygml.zip,lod2_33420_5646_2_sn.gml,1



Lowest source heights with 3D comparison:


,lod2_id,source_archive,source_member,source_height_m,height_method,building_part_count,z_min_m,z_max_m,z_extent_m,ground_min_z_m,...,roof_min_z_m,roof_max_z_m,roof_max_minus_ground_min_m,roof_median_minus_ground_median_m,source_matches_z_extent,source_matches_z_min,source_matches_z_max,source_matches_roof_ground,z_extent_plausible,roof_ground_plausible
74,DESNATGMlf0000id,lod2_33420_5646_2_sn_citygml.zip,lod2_33420_5646_2_sn.gml,-109.691,building_measured_height,0,4.000000,113.690521,109.690521,113.690521,...,4.000000,4.000000,-109.690521,-109.690521,False,False,False,True,True,False
69,DESNATPU1000HEYH,lod2_33486_5638_2_sn_citygml.zip,lod2_33486_5638_2_sn.gml,-15.447,building_measured_height,0,239.318000,259.782213,20.464213,257.881000,...,239.318000,242.434000,-15.447000,-15.831000,False,False,False,True,True,False
72,DESNATPU1000DbNe,lod2_33354_5632_2_sn_citygml.zip,lod2_33354_5632_2_sn.gml,-1.327,building_measured_height,0,313.579000,317.173299,3.594299,317.001000,...,313.579000,315.674000,-1.327000,-2.995000,False,False,False,True,True,False
70,DESNATPU1000Bfa8,lod2_33486_5638_2_sn_citygml.zip,lod2_33486_5638_2_sn.gml,-0.374,building_measured_height,0,228.779000,234.649994,5.870994,233.987000,...,228.779000,233.613000,-0.374000,-2.857000,False,False,False,True,True,False
71,DESNATP1GI0002Z1,lod2_33352_5636_2_sn_citygml.zip,lod2_33352_5636_2_sn.gml,303.500,building_measured_height,0,288.673828,592.173828,303.500000,288.673828,...,592.173828,592.173828,303.500000,303.500000,True,False,False,True,False,False
57,DESNATP1Me0000HL,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1004.545,building_measured_height,0,0.000000,1004.545000,1004.545000,0.000000,...,1004.545000,1004.545000,1004.545000,1004.545000,True,False,True,True,False,False
58,DESNATP1Me0000HK,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1004.554,building_measured_height,0,0.000000,1004.554000,1004.554000,0.000000,...,1004.554000,1004.554000,1004.554000,1004.554000,True,False,True,True,False,False
65,DESNATP1fl0000JF,lod2_33332_5590_2_sn_citygml.zip,lod2_33332_5590_2_sn.gml,1014.128,building_measured_height,0,0.000000,1014.128000,1014.128000,0.000000,...,1012.800000,1014.128000,1014.128000,1013.345000,True,False,True,True,False,False
25,DESNATP1OP0000Eo,lod2_33354_5586_2_sn_citygml.zip,lod2_33354_5586_2_sn.gml,1015.082,building_measured_height,0,0.000000,1015.082000,1015.082000,0.000000,...,1014.800000,1015.082000,1015.082000,1014.800000,True,False,True,True,False,False
66,DESNATP1fl0000JE,lod2_33332_5590_2_sn_citygml.zip,lod2_33332_5590_2_sn.gml,1015.321,building_measured_height,0,0.000000,1015.321000,1015.321000,0.000000,...,1013.145000,1015.321000,1015.321000,1015.194000,True,False,True,True,False,False



Highest source heights with 3D comparison:


,lod2_id,source_archive,source_member,source_height_m,height_method,building_part_count,z_min_m,z_max_m,z_extent_m,ground_min_z_m,...,roof_min_z_m,roof_max_z_m,roof_max_minus_ground_min_m,roof_median_minus_ground_median_m,source_matches_z_extent,source_matches_z_min,source_matches_z_max,source_matches_roof_ground,z_extent_plausible,roof_ground_plausible
56,DESNATPU1000FBkl,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1232.427,maximum_building_part_height,4,0.0,1232.427,1232.427,0.0,...,1218.361,1232.427,1232.427,1223.609000,True,False,True,True,False,False
51,DESNATPU1000Gwr8,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1224.816,maximum_building_part_height,3,0.0,1224.816,1224.816,0.0,...,1214.241,1224.816,1224.816,1219.269000,True,False,True,True,False,False
53,DESNATPU1000DTEU,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1223.191,maximum_building_part_height,4,0.0,1223.191,1223.191,0.0,...,1215.521,1223.191,1223.191,1217.462993,True,False,True,True,False,False
48,DESNATPU1000EXEe,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1219.534,building_measured_height,0,0.0,1219.534,1219.534,0.0,...,1215.525,1219.534,1219.534,1216.224000,True,False,True,True,False,False
59,DESNATP1Me0000QV,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1215.828,building_measured_height,0,0.0,1215.828,1215.828,0.0,...,1213.482,1215.828,1215.828,1215.659000,True,False,True,True,False,False
12,DESNATPU1000EQpH,lod2_33354_5586_2_sn_citygml.zip,lod2_33354_5586_2_sn.gml,1209.263,maximum_building_part_height,2,0.0,1209.263,1209.263,0.0,...,1207.970,1209.263,1209.263,1208.275500,True,False,True,True,False,False
61,DESNATPU1000CTzU,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1206.085,building_measured_height,0,0.0,1206.085,1206.085,0.0,...,1201.649,1206.085,1206.085,1201.971000,True,False,True,True,False,False
60,DESNATPU1000IVt3,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1199.006,maximum_building_part_height,4,0.0,1199.006,1199.006,0.0,...,1191.211,1199.006,1199.006,1196.113000,True,False,True,True,False,False
47,DESNATP1Me0000HM,lod2_33354_5588_2_sn_citygml.zip,lod2_33354_5588_2_sn.gml,1192.689,building_measured_height,0,0.0,1192.689,1192.689,0.0,...,1191.079,1192.689,1192.689,1191.890000,True,False,True,True,False,False
18,DESNATPU1000Ee0h,lod2_33354_5586_2_sn_citygml.zip,lod2_33354_5586_2_sn.gml,1189.758,building_measured_height,0,0.0,1189.758,1189.758,0.0,...,1188.771,1189.758,1189.758,1189.263000,True,False,True,True,False,False


In [72]:
# ============================================================
# 64 — CLASSIFY SAXONY EXTREME-HEIGHT QUALITY ISSUES
# ============================================================

sn_height_quality = sn_z_height_check.copy()

negative_mask = sn_height_quality["source_height_m"] <= 0
zero_ground_mask = (
    (sn_height_quality["source_height_m"] > 300)
    & np.isclose(sn_height_quality["z_min_m"], 0, atol=0.05)
    & sn_height_quality["source_matches_z_max"]
)
consistent_tall_mask = (
    (sn_height_quality["source_height_m"] > 300)
    & (sn_height_quality["z_min_m"] > 0.05)
    & sn_height_quality["source_matches_z_extent"]
)

sn_height_quality["height_issue"] = np.select(
    [negative_mask, zero_ground_mask, consistent_tall_mask],
    ["non_positive_inverted_surfaces", "zero_ground_absolute_elevation", "tall_geometry_consistent"],
    default="unresolved_extreme"
)

sn_height_quality["height_quality"] = np.select(
    [
        sn_height_quality["height_issue"].isin(["non_positive_inverted_surfaces", "zero_ground_absolute_elevation"]),
        sn_height_quality["height_issue"].eq("tall_geometry_consistent")
    ],
    ["invalid", "valid_but_extreme"],
    default="review"
)

sn_height_quality["candidate_height_m"] = np.nan
sn_height_quality.loc[negative_mask, "candidate_height_m"] = sn_height_quality.loc[negative_mask, "z_extent_m"]
sn_height_quality.loc[consistent_tall_mask, "candidate_height_m"] = sn_height_quality.loc[consistent_tall_mask, "source_height_m"]

sn_height_quality["recommended_action"] = np.select(
    [
        sn_height_quality["height_issue"].eq("non_positive_inverted_surfaces"),
        sn_height_quality["height_issue"].eq("zero_ground_absolute_elevation"),
        sn_height_quality["height_issue"].eq("tall_geometry_consistent")
    ],
    [
        "retain raw; review z_extent as correction candidate during national cleaning",
        "retain raw; set cleaned height missing and impute during national cleaning",
        "retain source height; review only as national upper-tail outlier"
    ],
    default="manual review during national cleaning"
)

sn_height_quality_summary = (
    sn_height_quality
    .groupby(["height_issue", "height_quality", "recommended_action"], dropna=False)
    .agg(
        building_count=("lod2_id", "size"),
        minimum_source_height_m=("source_height_m", "min"),
        maximum_source_height_m=("source_height_m", "max"),
        minimum_candidate_height_m=("candidate_height_m", "min"),
        maximum_candidate_height_m=("candidate_height_m", "max")
    )
    .reset_index()
    .sort_values("building_count", ascending=False)
)

sn_height_quality_path = REPORT_DIR / "lod2_height_quality_flags_SN.csv"
sn_height_quality_summary_path = REPORT_DIR / "lod2_height_quality_summary_SN.csv"

sn_height_quality.to_csv(sn_height_quality_path, index=False)
sn_height_quality_summary.to_csv(sn_height_quality_summary_path, index=False)

print("=" * 70)
print("SAXONY HEIGHT-QUALITY CLASSIFICATION")
print("=" * 70)
print(f"\nExtreme records assessed : {len(sn_height_quality):,}")
print(f"Invalid source heights   : {sn_height_quality['height_quality'].eq('invalid').sum():,}")
print(f"Valid but extreme        : {sn_height_quality['height_quality'].eq('valid_but_extreme').sum():,}")
print(f"Still unresolved         : {sn_height_quality['height_quality'].eq('review').sum():,}")

display(sn_height_quality_summary)

print("\nNon-positive records and correction candidates:")
display(
    sn_height_quality.loc[
        sn_height_quality["height_issue"].eq("non_positive_inverted_surfaces"),
        ["lod2_id", "source_height_m", "z_extent_m", "candidate_height_m", "source_archive"]
    ]
)

print("\nGeometrically consistent tall records:")
display(
    sn_height_quality.loc[
        sn_height_quality["height_issue"].eq("tall_geometry_consistent"),
        ["lod2_id", "source_height_m", "z_min_m", "z_max_m", "z_extent_m", "source_archive"]
    ]
)

print(f"\nDetailed quality flags saved to: {sn_height_quality_path}")
print(f"Quality summary saved to       : {sn_height_quality_summary_path}")

SAXONY HEIGHT-QUALITY CLASSIFICATION

Extreme records assessed : 75
Invalid source heights   : 74
Valid but extreme        : 1
Still unresolved         : 0


,height_issue,height_quality,recommended_action,building_count,minimum_source_height_m,maximum_source_height_m,minimum_candidate_height_m,maximum_candidate_height_m
2,zero_ground_absolute_elevation,invalid,retain raw; set cleaned height missing and imp...,70,1004.545,1232.427,NaN,NaN
0,non_positive_inverted_surfaces,invalid,retain raw; review z_extent as correction cand...,4,-109.691,-0.374,3.594299,109.690521
1,tall_geometry_consistent,valid_but_extreme,retain source height; review only as national ...,1,303.500,303.500,303.500000,303.500000



Non-positive records and correction candidates:


,lod2_id,source_height_m,z_extent_m,candidate_height_m,source_archive
69,DESNATPU1000HEYH,-15.447,20.464213,20.464213,lod2_33486_5638_2_sn_citygml.zip
70,DESNATPU1000Bfa8,-0.374,5.870994,5.870994,lod2_33486_5638_2_sn_citygml.zip
72,DESNATPU1000DbNe,-1.327,3.594299,3.594299,lod2_33354_5632_2_sn_citygml.zip
74,DESNATGMlf0000id,-109.691,109.690521,109.690521,lod2_33420_5646_2_sn_citygml.zip



Geometrically consistent tall records:


,lod2_id,source_height_m,z_min_m,z_max_m,z_extent_m,source_archive
71,DESNATP1GI0002Z1,303.5,288.673828,592.173828,303.5,lod2_33352_5636_2_sn_citygml.zip



Detailed quality flags saved to: /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_height_quality_flags_SN.csv
Quality summary saved to       : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_height_quality_summary_SN.csv


In [73]:
# ============================================================
# 65 — DISCOVER THURINGIA LoD2 DOWNLOADS
# ============================================================

from collections import deque
from urllib.parse import urljoin, urlparse

TH_RAW_DIR = RAW_DIR / "TH"
TH_RAW_DIR.mkdir(parents=True, exist_ok=True)

TH_PORTAL_URL = "https://geoportal.thueringen.de/gdi-th/download-offene-geodaten/download-3d-gebaeudedaten"
TH_ATOM_URL = "https://geoportal.geoportal-th.de/dienste/atom_th_gebaeude"

th_index = source_registry["state_code"].eq("TH")
source_registry.loc[th_index, ["portal_url", "download_url", "download_method", "format", "horizontal_crs", "download_status", "notes"]] = [
    TH_PORTAL_URL,
    TH_ATOM_URL,
    "Official INSPIRE Atom feed",
    "CityGML ZIP",
    "to be confirmed from sample",
    "discovering",
    "Statewide LoD1 and LoD2 building models"
]
source_registry.to_csv(source_registry_path, index=False)

ATOM_NS = {"atom": "http://www.w3.org/2005/Atom"}
th_feed_queue = deque([TH_ATOM_URL])
th_visited_feeds = set()
th_feed_records = []
th_link_records = []

while th_feed_queue:
    feed_url = th_feed_queue.popleft()

    if feed_url in th_visited_feeds:
        continue

    response = requests.get(feed_url, timeout=(30, 300))
    response.raise_for_status()

    root = etree.fromstring(response.content)
    root_name = etree.QName(root).localname

    if root_name != "feed":
        continue

    th_visited_feeds.add(feed_url)

    feed_title = root.findtext("atom:title", namespaces=ATOM_NS)
    feed_id = root.findtext("atom:id", namespaces=ATOM_NS)

    th_feed_records.append({
        "feed_title": feed_title,
        "feed_id": feed_id,
        "feed_url": feed_url
    })

    for entry in root.findall("atom:entry", namespaces=ATOM_NS):
        entry_title = entry.findtext("atom:title", namespaces=ATOM_NS)
        entry_id = entry.findtext("atom:id", namespaces=ATOM_NS)
        updated = entry.findtext("atom:updated", namespaces=ATOM_NS)

        for link in entry.findall("atom:link", namespaces=ATOM_NS):
            href = link.get("href")

            if not href:
                continue

            absolute_url = urljoin(feed_url, href)
            relation = link.get("rel", "")
            media_type = link.get("type", "")
            combined_text = " ".join([
                str(feed_title or ""),
                str(entry_title or ""),
                str(entry_id or ""),
                str(absolute_url or ""),
                str(media_type or "")
            ]).lower()

            th_link_records.append({
                "feed_title": feed_title,
                "entry_title": entry_title,
                "entry_id": entry_id,
                "updated": updated,
                "relation": relation,
                "media_type": media_type,
                "url": absolute_url
            })

            is_atom_feed = "atom+xml" in media_type.lower() or (
                "atom" in absolute_url.lower()
                and not absolute_url.lower().endswith((".zip", ".gml", ".xml"))
            )

            if is_atom_feed and absolute_url not in th_visited_feeds:
                th_feed_queue.append(absolute_url)

th_feeds = pd.DataFrame(th_feed_records).drop_duplicates(subset="feed_url")
th_atom_links = pd.DataFrame(th_link_records).drop_duplicates(subset=["entry_id", "url"])

if th_atom_links.empty:
    raise RuntimeError("No links were discovered in the Thuringia Atom feed.")

th_atom_links["filename"] = th_atom_links["url"].map(lambda url: Path(urlparse(url).path).name)
th_atom_links["is_zip"] = th_atom_links["url"].str.lower().str.contains(r"\.zip(?:$|\?)", regex=True)
th_atom_links["mentions_lod2"] = th_atom_links[["feed_title", "entry_title", "entry_id", "url"]].fillna("").agg(" ".join, axis=1).str.contains(r"\blod[\s_-]?2\b", case=False, regex=True)

th_zip_links = th_atom_links[th_atom_links["is_zip"]].copy()
th_lod2_candidates = th_zip_links[th_zip_links["mentions_lod2"]].copy()
th_lod2_candidates = th_lod2_candidates.drop_duplicates(subset="url").reset_index(drop=True)

th_feeds_path = TH_RAW_DIR / "th_atom_feeds.csv"
th_links_path = TH_RAW_DIR / "th_atom_links.csv"
th_candidates_path = TH_RAW_DIR / "th_lod2_download_candidates.csv"

th_feeds.to_csv(th_feeds_path, index=False)
th_atom_links.to_csv(th_links_path, index=False)
th_lod2_candidates.to_csv(th_candidates_path, index=False)

print("=" * 70)
print("THURINGIA LoD2 ATOM DISCOVERY")
print("=" * 70)
print(f"\nAtom feeds visited      : {len(th_feeds):,}")
print(f"All entry links         : {len(th_atom_links):,}")
print(f"All ZIP links           : {len(th_zip_links):,}")
print(f"LoD2 ZIP candidates     : {len(th_lod2_candidates):,}")
print(f"Unique candidate URLs   : {th_lod2_candidates['url'].nunique():,}")
print(f"Unique filenames        : {th_lod2_candidates['filename'].nunique():,}")
print(f"Duplicate filenames     : {th_lod2_candidates['filename'].duplicated().sum():,}")

print("\nDiscovered feeds:")
display(th_feeds)

print("\nZIP links grouped by feed and entry title:")
display(
    th_zip_links
    .groupby(["feed_title", "entry_title"], dropna=False)
    .size()
    .reset_index(name="zip_count")
    .sort_values("zip_count", ascending=False)
    .head(30)
)

print("\nLoD2 candidate examples:")
display(th_lod2_candidates.head(30))

print(f"\nCandidate list saved to: {th_candidates_path}")

THURINGIA LoD2 ATOM DISCOVERY

Atom feeds visited      : 2
All entry links         : 8,782
All ZIP links           : 8,780
LoD2 ZIP candidates     : 8,780
Unique candidate URLs   : 8,780
Unique filenames        : 8,780
Duplicate filenames     : 0

Discovered feeds:


,feed_title,feed_id,feed_url
0,3D Gebäude (LoD1 und LoD2) Thüringen,https://geoportal.geoportal-th.de/dienste/atom...,https://geoportal.geoportal-th.de/dienste/atom...
1,3D Gebäudedaten LoD1 & LoD2,https://geoportal.geoportal-th.de/dienste/atom...,https://geoportal.geoportal-th.de/dienste/atom...



ZIP links grouped by feed and entry title:


,feed_title,entry_title,zip_count
0,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,4390
1,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,4390



LoD2 candidate examples:


,feed_title,entry_title,entry_id,updated,relation,media_type,url,filename,is_zip,mentions_lod2
0,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5696_2_TH.zip,True,True
1,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5692_2_TH.zip,True,True
2,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5694_2_TH.zip,True,True
3,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5698_2_TH.zip,True,True
4,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5700_2_TH.zip,True,True
5,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5702_2_TH.zip,True,True
6,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5704_2_TH.zip,True,True
7,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5706_2_TH.zip,True,True
8,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5708_2_TH.zip,True,True
9,3D Gebäudedaten LoD1 & LoD2,LOD1 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD1_32_598_5710_2_TH.zip,True,True



Candidate list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/TH/th_lod2_download_candidates.csv


In [74]:
# ============================================================
# 66 — CORRECT THURINGIA LoD2 DOWNLOAD LIST
# ============================================================

th_lod2_downloads = th_zip_links[
    th_zip_links["filename"].str.match(r"(?i)^lod2[_-]", na=False)
    | th_zip_links["entry_title"].fillna("").str.match(r"(?i)^lod2\b")
].copy()

th_lod2_downloads = (
    th_lod2_downloads
    .drop_duplicates(subset="url")
    .sort_values("filename")
    .reset_index(drop=True)
)

th_lod1_downloads = th_zip_links[
    th_zip_links["filename"].str.match(r"(?i)^lod1[_-]", na=False)
].drop_duplicates(subset="url")

th_unclassified_zips = th_zip_links[
    ~th_zip_links["url"].isin(th_lod1_downloads["url"])
    & ~th_zip_links["url"].isin(th_lod2_downloads["url"])
].copy()

th_lod2_downloads_path = TH_RAW_DIR / "th_lod2_download_list.csv"
th_lod2_downloads.to_csv(th_lod2_downloads_path, index=False)

print("=" * 70)
print("THURINGIA CORRECTED LoD2 DOWNLOAD LIST")
print("=" * 70)
print(f"\nAll ZIP links           : {len(th_zip_links):,}")
print(f"LoD1 ZIP files          : {len(th_lod1_downloads):,}")
print(f"LoD2 ZIP files          : {len(th_lod2_downloads):,}")
print(f"Unclassified ZIP files  : {len(th_unclassified_zips):,}")
print(f"Unique LoD2 URLs        : {th_lod2_downloads['url'].nunique():,}")
print(f"Unique LoD2 filenames   : {th_lod2_downloads['filename'].nunique():,}")
print(f"Duplicate LoD2 filenames: {th_lod2_downloads['filename'].duplicated().sum():,}")
print(f"LoD1 files in LoD2 list : {th_lod2_downloads['filename'].str.match(r'(?i)^lod1[_-]', na=False).sum():,}")

display(th_lod2_downloads.head(20))
display(th_lod2_downloads.tail(20))

print(f"\nCorrected list saved to: {th_lod2_downloads_path}")

if len(th_lod2_downloads) != 4390:
    raise ValueError(f"Expected 4,390 Thuringia LoD2 tiles, found {len(th_lod2_downloads):,}.")

if th_lod2_downloads["filename"].duplicated().any():
    raise ValueError("Duplicate Thuringia LoD2 filenames detected.")

if not th_unclassified_zips.empty:
    display(th_unclassified_zips.head(30))

THURINGIA CORRECTED LoD2 DOWNLOAD LIST

All ZIP links           : 8,780
LoD1 ZIP files          : 4,390
LoD2 ZIP files          : 4,390
Unclassified ZIP files  : 0
Unique LoD2 URLs        : 4,390
Unique LoD2 filenames   : 4,390
Duplicate LoD2 filenames: 0
LoD1 files in LoD2 list : 0


,feed_title,entry_title,entry_id,updated,relation,media_type,url,filename,is_zip,mentions_lod2
0,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_560_5608_2_TH.zip,True,True
1,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_560_5610_2_TH.zip,True,True
2,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_562_5608_2_TH.zip,True,True
3,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_562_5610_2_TH.zip,True,True
4,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_562_5612_2_TH.zip,True,True
5,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_562_5614_2_TH.zip,True,True
6,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_564_5608_2_TH.zip,True,True
7,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_564_5610_2_TH.zip,True,True
8,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_564_5612_2_TH.zip,True,True
9,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_564_5614_2_TH.zip,True,True


,feed_title,entry_title,entry_id,updated,relation,media_type,url,filename,is_zip,mentions_lod2
4370,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_750_5646_2_TH.zip,True,True
4371,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_750_5648_2_TH.zip,True,True
4372,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_750_5650_2_TH.zip,True,True
4373,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_750_5652_2_TH.zip,True,True
4374,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_750_5654_2_TH.zip,True,True
4375,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_752_5644_2_TH.zip,True,True
4376,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_752_5646_2_TH.zip,True,True
4377,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_752_5648_2_TH.zip,True,True
4378,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_752_5650_2_TH.zip,True,True
4379,3D Gebäudedaten LoD1 & LoD2,LOD2 - 3D Gebäudedaten im CRS EPSG:25832 und F...,https://geoportal.geoportal-th.de/dienste/atom...,2026-05-04T08:00:00+02:00,section,application/zip,https://geoportal.geoportal-th.de/3dgebaeude/L...,LoD2_32_752_5652_2_TH.zip,True,True



Corrected list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/TH/th_lod2_download_list.csv


In [75]:
# ============================================================
# 67 — TEST REPRESENTATIVE THURINGIA LoD2 TILES
# ============================================================

from io import BytesIO
from collections import Counter

TH_TEST_DIR = TH_RAW_DIR / "test_tiles"
TH_TEST_DIR.mkdir(parents=True, exist_ok=True)

th_sample_indices = sorted(set([0, len(th_lod2_downloads) // 4, len(th_lod2_downloads) // 2, 3 * len(th_lod2_downloads) // 4, len(th_lod2_downloads) - 1]))
th_test_records = []

for index in tqdm(th_sample_indices, desc="Testing Thuringia tiles"):
    row = th_lod2_downloads.iloc[index]
    zip_path = TH_TEST_DIR / row["filename"]

    if not zip_path.exists():
        response = requests.get(row["url"], timeout=(30, 600))
        response.raise_for_status()
        zip_path.write_bytes(response.content)

    with ZipFile(zip_path, "r") as archive:
        bad_member = archive.testzip()

        if bad_member is not None:
            raise ValueError(f"Corrupt ZIP member in {zip_path.name}: {bad_member}")

        gml_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

        if not gml_members:
            raise FileNotFoundError(f"No CityGML file found in {zip_path.name}")

        for member in gml_members:
            with archive.open(member) as stream:
                content = stream.read()

            root = etree.fromstring(content)
            building_namespace = root.nsmap.get("bldg")
            srs_names = sorted({element.get("srsName") for element in root.iter() if element.get("srsName")})

            tag_counts = Counter()

            for element in root.iter():
                local_name = etree.QName(element).localname

                if local_name in {"Building", "BuildingPart", "measuredHeight", "GroundSurface", "RoofSurface", "WallSurface"}:
                    tag_counts[local_name] += 1

            sample_gdf = extract_citygml_stream(BytesIO(content), Path(member).name, "EPSG:25832")

            th_test_records.append({
                "zip_file": zip_path.name,
                "source_file": member,
                "building_namespace": building_namespace,
                "srs_names": " | ".join(srs_names),
                "buildings_in_xml": tag_counts["Building"],
                "building_parts": tag_counts["BuildingPart"],
                "measured_heights_in_xml": tag_counts["measuredHeight"],
                "ground_surfaces": tag_counts["GroundSurface"],
                "buildings_extracted": len(sample_gdf),
                "extracted_heights": sample_gdf["measured_height_m"].notna().sum(),
                "missing_heights": sample_gdf["measured_height_m"].isna().sum(),
                "usable_geometries": sample_gdf.geometry.notna().sum(),
                "missing_geometries": sample_gdf.geometry.isna().sum(),
                "unique_ids": sample_gdf["lod2_id"].nunique()
            })

th_structure_test = pd.DataFrame(th_test_records)

print("=" * 70)
print("THURINGIA SAMPLE TILE STRUCTURE TEST")
print("=" * 70)

display(th_structure_test)

print("\nNamespaces:")
display(th_structure_test["building_namespace"].value_counts(dropna=False).rename_axis("building_namespace").reset_index(name="files"))

print("\nCRS values:")
display(th_structure_test["srs_names"].value_counts(dropna=False).rename_axis("srs_names").reset_index(name="files"))

print("\nTotals:")
display(
    th_structure_test[
        [
            "buildings_in_xml",
            "building_parts",
            "measured_heights_in_xml",
            "ground_surfaces",
            "buildings_extracted",
            "extracted_heights",
            "missing_heights",
            "usable_geometries",
            "missing_geometries"
        ]
    ].sum().to_frame("count")
)

Testing Thuringia tiles: 100%|███████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.47it/s]

THURINGIA SAMPLE TILE STRUCTURE TEST


,zip_file,source_file,building_namespace,srs_names,buildings_in_xml,building_parts,measured_heights_in_xml,ground_surfaces,buildings_extracted,extracted_heights,missing_heights,usable_geometries,missing_geometries,unique_ids
0,LoD2_32_560_5608_2_TH.zip,LoD2_32_560_5608_2_TH.gml,http://www.opengis.net/citygml/building/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,0,0,0,0,0,0,0,0,0,0
1,LoD2_32_606_5678_2_TH.zip,LoD2_32_606_5678_2_TH.gml,http://www.opengis.net/citygml/building/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,2,0,2,2,2,2,0,2,0,2
2,LoD2_32_636_5626_2_TH.zip,LoD2_32_636_5626_2_TH.gml,http://www.opengis.net/citygml/building/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,329,63,364,364,329,329,0,329,0,329
3,LoD2_32_674_5608_2_TH.zip,LoD2_32_674_5608_2_TH.gml,http://www.opengis.net/citygml/building/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,197,41,220,220,197,197,0,197,0,197
4,LoD2_32_756_5648_2_TH.zip,LoD2_32_756_5648_2_TH.gml,http://www.opengis.net/citygml/building/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,0,0,0,0,0,0,0,0,0,0



Namespaces:


,building_namespace,files
0,http://www.opengis.net/citygml/building/1.0,5



CRS values:


,srs_names,files
0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,5



Totals:


,count
buildings_in_xml,528
building_parts,104
measured_heights_in_xml,586
ground_surfaces,586
buildings_extracted,528
extracted_heights,528
missing_heights,0
usable_geometries,528
missing_geometries,0


In [76]:
# ============================================================
# 68 — INSPECT THURINGIA BUILDING AND PART HEIGHTS
# ============================================================

TH_BLDG_NS = "http://www.opengis.net/citygml/building/1.0"
TH_BUILDING_TAG = f"{{{TH_BLDG_NS}}}Building"
TH_BUILDING_PART_TAG = f"{{{TH_BLDG_NS}}}BuildingPart"
TH_HEIGHT_TAG = f"{{{TH_BLDG_NS}}}measuredHeight"

th_nonempty_samples = th_structure_test.loc[th_structure_test["buildings_in_xml"] > 0, ["zip_file", "source_file"]]
th_height_records = []

for row in th_nonempty_samples.itertuples(index=False):
    zip_path = TH_TEST_DIR / row.zip_file

    with ZipFile(zip_path, "r") as archive:
        with archive.open(row.source_file) as stream:
            for event, building in etree.iterparse(stream, events=("end",), tag=TH_BUILDING_TAG, huge_tree=True):
                parent_height = get_direct_number(building, TH_HEIGHT_TAG)
                building_parts = building.findall(f".//{TH_BUILDING_PART_TAG}")
                part_heights = [get_direct_number(part, TH_HEIGHT_TAG) for part in building_parts]
                part_heights = [height for height in part_heights if pd.notna(height)]

                th_height_records.append({
                    "zip_file": row.zip_file,
                    "source_file": row.source_file,
                    "lod2_id": building.get(GML_ID),
                    "parent_height_m": parent_height,
                    "building_parts": len(building_parts),
                    "parts_with_height": len(part_heights),
                    "minimum_part_height_m": min(part_heights) if part_heights else np.nan,
                    "maximum_part_height_m": max(part_heights) if part_heights else np.nan,
                    "first_descendant_height_m": parse_number(get_first_text(building, "measuredHeight"))
                })

                building.clear()

                while building.getprevious() is not None:
                    del building.getparent()[0]

th_height_check = pd.DataFrame(th_height_records)
th_height_check["has_parent_height"] = th_height_check["parent_height_m"].notna()
th_height_check["has_parts"] = th_height_check["building_parts"] > 0
th_height_check["parent_equals_max_part"] = np.isclose(
    th_height_check["parent_height_m"],
    th_height_check["maximum_part_height_m"],
    atol=0.01,
    equal_nan=False
)

th_height_summary = pd.DataFrame([{
    "buildings_checked": len(th_height_check),
    "buildings_with_parts": th_height_check["has_parts"].sum(),
    "buildings_without_parts": (~th_height_check["has_parts"]).sum(),
    "buildings_with_parent_height": th_height_check["has_parent_height"].sum(),
    "buildings_missing_parent_height": (~th_height_check["has_parent_height"]).sum(),
    "buildings_with_part_heights": th_height_check["parts_with_height"].gt(0).sum(),
    "multipart_missing_parent_height": (th_height_check["has_parts"] & ~th_height_check["has_parent_height"]).sum(),
    "multipart_missing_all_heights": (th_height_check["has_parts"] & ~th_height_check["has_parent_height"] & th_height_check["parts_with_height"].eq(0)).sum(),
    "parent_equals_max_part": th_height_check["parent_equals_max_part"].sum(),
    "parent_differs_from_max_part": (
        th_height_check["has_parent_height"]
        & th_height_check["parts_with_height"].gt(0)
        & ~th_height_check["parent_equals_max_part"]
    ).sum()
}])

first_not_max = (
    th_height_check["parts_with_height"].gt(0)
    & ~np.isclose(
        th_height_check["first_descendant_height_m"],
        th_height_check["maximum_part_height_m"],
        atol=0.01,
        equal_nan=False
    )
)

print("=" * 70)
print("THURINGIA BUILDING HEIGHT STRUCTURE")
print("=" * 70)

display(th_height_summary)

print("\nBuildings missing a direct parent height:")
display(
    th_height_check.loc[
        ~th_height_check["has_parent_height"],
        [
            "lod2_id",
            "building_parts",
            "parts_with_height",
            "minimum_part_height_m",
            "maximum_part_height_m",
            "first_descendant_height_m"
        ]
    ].head(30)
)

print("\nCases where the first descendant is not the tallest part:")
display(
    th_height_check.loc[
        first_not_max,
        [
            "lod2_id",
            "building_parts",
            "minimum_part_height_m",
            "maximum_part_height_m",
            "first_descendant_height_m"
        ]
    ].head(30)
)

print(f"\nFirst-descendant height differs from maximum part height: {first_not_max.sum():,}")

THURINGIA BUILDING HEIGHT STRUCTURE


,buildings_checked,buildings_with_parts,buildings_without_parts,buildings_with_parent_height,buildings_missing_parent_height,buildings_with_part_heights,multipart_missing_parent_height,multipart_missing_all_heights,parent_equals_max_part,parent_differs_from_max_part
0,528,46,482,482,46,46,46,0,0,0



Buildings missing a direct parent height:


,lod2_id,building_parts,parts_with_height,minimum_part_height_m,maximum_part_height_m,first_descendant_height_m
13,DETHL56P00007epG,2,2,5.548,7.144,7.144
27,DETHL56P00007epk,2,2,7.579,7.579,7.579
39,DETHL56P00007eoe,2,2,9.309,10.263,10.263
68,DETHL56P00007epA,2,2,4.463,6.877,6.877
76,DETHL56P00007emw,2,2,5.144,9.246,9.246
80,DETHL56P00007emH,2,2,6.866,8.084,6.866
98,DETHL56P00007enf,2,2,4.688,10.111,10.111
100,DETHL56P00007enq,2,2,2.310,2.439,2.439
117,DETHL56P00007enl,3,3,4.083,10.083,10.083
126,DETHL56P00007emv,2,2,8.756,11.753,11.753



Cases where the first descendant is not the tallest part:


,lod2_id,building_parts,minimum_part_height_m,maximum_part_height_m,first_descendant_height_m
80,DETHL56P00007emH,2,6.866,8.084,6.866
173,DETHL56P00007emj,7,5.692,19.666,15.972
228,DETHL56P00007emA,2,5.022,7.189,5.022



First-descendant height differs from maximum part height: 3


In [77]:
# ============================================================
# 69 — TEST MULTIPART PARSER ON THURINGIA
# ============================================================

th_parser_test_parts = []

for row in th_structure_test.itertuples(index=False):
    zip_path = TH_TEST_DIR / row.zip_file

    with ZipFile(zip_path, "r") as archive:
        with archive.open(row.source_file) as stream:
            tile_gdf = extract_citygml_stream_multipart(
                stream,
                Path(row.source_file).name,
                "EPSG:25832",
                TH_BUILDING_TAG,
                TH_BUILDING_PART_TAG
            )

    if not tile_gdf.empty:
        tile_gdf["source_zip"] = row.zip_file
        th_parser_test_parts.append(tile_gdf)

th_parser_test_gdf = gpd.GeoDataFrame(
    pd.concat(th_parser_test_parts, ignore_index=True),
    geometry="geometry",
    crs="EPSG:25832"
)

print("=" * 70)
print("THURINGIA MULTIPART PARSER TEST")
print("=" * 70)
print(f"\nBuildings extracted      : {len(th_parser_test_gdf):,}")
print(f"Buildings with parts     : {(th_parser_test_gdf['building_part_count'] > 0).sum():,}")
print(f"Buildings without parts  : {(th_parser_test_gdf['building_part_count'] == 0).sum():,}")
print(f"Measured heights         : {th_parser_test_gdf['measured_height_m'].notna().sum():,}")
print(f"Missing heights          : {th_parser_test_gdf['measured_height_m'].isna().sum():,}")
print(f"Usable geometries        : {th_parser_test_gdf.geometry.notna().sum():,}")
print(f"Missing geometries       : {th_parser_test_gdf.geometry.isna().sum():,}")
print(f"Unique LoD2 IDs          : {th_parser_test_gdf['lod2_id'].nunique():,}")

print("\nHeight methods:")
display(
    th_parser_test_gdf["height_method"]
    .value_counts(dropna=False)
    .rename_axis("height_method")
    .reset_index(name="building_count")
)

print("\nFootprint methods:")
display(
    th_parser_test_gdf["footprint_method"]
    .value_counts(dropna=False)
    .rename_axis("footprint_method")
    .reset_index(name="building_count")
)

THURINGIA MULTIPART PARSER TEST

Buildings extracted      : 528
Buildings with parts     : 46
Buildings without parts  : 482
Measured heights         : 528
Missing heights          : 0
Usable geometries        : 528
Missing geometries       : 0
Unique LoD2 IDs          : 528

Height methods:


,height_method,building_count
0,building_measured_height,482
1,maximum_building_part_height,46



Footprint methods:


,footprint_method,building_count
0,ground_surface,528


In [78]:
# ============================================================
# 70 — DOWNLOAD ALL THURINGIA LoD2 CITYGML ZIP TILES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import time

TH_ZIP_DIR = TH_RAW_DIR / "zips"
TH_ZIP_DIR.mkdir(parents=True, exist_ok=True)

TH_MAX_WORKERS = 8
TH_MAX_RETRIES = 3
TH_CHUNK_SIZE = 1024 * 1024

def validate_th_zip(path):
    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with ZipFile(path, "r") as archive:
            if archive.testzip() is not None:
                return False

            gml_members = [
                name for name in archive.namelist()
                if name.lower().endswith((".gml", ".xml"))
            ]

            return len(gml_members) > 0

    except Exception:
        return False

def download_th_tile(url, filename):
    output_path = TH_ZIP_DIR / filename
    temp_path = TH_ZIP_DIR / f"{filename}.part"

    if validate_th_zip(output_path):
        return {
            "filename": filename,
            "status": "already_downloaded",
            "size_bytes": output_path.stat().st_size,
            "error": None
        }

    if output_path.exists():
        output_path.unlink()

    for attempt in range(1, TH_MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 900)) as response:
                response.raise_for_status()
                expected_size = int(response.headers.get("content-length", 0))

                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=TH_CHUNK_SIZE):
                        if chunk:
                            file.write(chunk)

            actual_size = temp_path.stat().st_size

            if expected_size and actual_size != expected_size:
                raise ValueError(
                    f"Incomplete download: expected {expected_size:,} bytes, "
                    f"received {actual_size:,}"
                )

            if not validate_th_zip(temp_path):
                raise ValueError("Downloaded file is not a valid CityGML ZIP archive.")

            temp_path.replace(output_path)

            return {
                "filename": filename,
                "status": "downloaded",
                "size_bytes": output_path.stat().st_size,
                "error": None
            }

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == TH_MAX_RETRIES:
                return {
                    "filename": filename,
                    "status": "failed",
                    "size_bytes": 0,
                    "error": str(error)
                }

            time.sleep(attempt * 3)

th_download_results = []

with ThreadPoolExecutor(max_workers=TH_MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_th_tile, row.url, row.filename): row.filename
        for row in th_lod2_downloads.itertuples(index=False)
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Downloading Thuringia LoD2 tiles"
    ):
        th_download_results.append(future.result())

th_download_results_df = pd.DataFrame(th_download_results)
th_download_results_path = REPORT_DIR / "lod2_download_results_TH.csv"
th_download_results_df.to_csv(th_download_results_path, index=False)

th_zip_files = sorted(TH_ZIP_DIR.glob("*.zip"))

print("=" * 70)
print("THURINGIA LoD2 DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nTiles requested       : {len(th_lod2_downloads):,}")
print(f"ZIP files present     : {len(th_zip_files):,}")
print(f"Downloaded now        : {(th_download_results_df['status'] == 'downloaded').sum():,}")
print(f"Already downloaded    : {(th_download_results_df['status'] == 'already_downloaded').sum():,}")
print(f"Failed                : {(th_download_results_df['status'] == 'failed').sum():,}")
print(f"Total downloaded size : {sum(path.stat().st_size for path in th_zip_files) / 1024**3:,.2f} GB")
print(f"Download report       : {th_download_results_path}")

display(
    th_download_results_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

if th_download_results_df["status"].eq("failed").any():
    display(
        th_download_results_df[
            th_download_results_df["status"] == "failed"
        ].head(30)
    )

THURINGIA LoD2 DOWNLOAD SUMMARY

Tiles requested       : 4,390
ZIP files present     : 4,390
Downloaded now        : 4,390
Already downloaded    : 0
Failed                : 0
Total downloaded size : 2.41 GB
Download report       : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_download_results_TH.csv


,status,count
0,downloaded,4390


In [79]:
# ============================================================
# 71 — PROCESS ALL THURINGIA LoD2 TILES
# ============================================================

TH_OUTPUT_DIR = EXTRACTED_DIR / "TH"
TH_PART_DIR = TH_OUTPUT_DIR / "parts"
TH_MANIFEST_DIR = TH_OUTPUT_DIR / "manifests"

TH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TH_PART_DIR.mkdir(parents=True, exist_ok=True)
TH_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

TH_BATCH_SIZE = 250

# ------------------------------------------------------------
# 1. Build the complete source-file index
# ------------------------------------------------------------

th_source_records = []

for zip_path in tqdm(th_zip_files, desc="Indexing Thuringia ZIP files"):
    with ZipFile(zip_path, "r") as archive:
        gml_members = [name for name in archive.namelist() if name.lower().endswith((".gml", ".xml"))]

        for member in gml_members:
            th_source_records.append({
                "archive_path": str(zip_path),
                "archive_name": zip_path.name,
                "member": member,
                "source_key": f"{zip_path.name}::{member}"
            })

th_sources = pd.DataFrame(th_source_records)
th_batch_starts = list(range(0, len(th_sources), TH_BATCH_SIZE))

print("=" * 70)
print("THURINGIA PROCESSING SETUP")
print("=" * 70)
print(f"\nZIP archives          : {len(th_zip_files):,}")
print(f"CityGML files         : {len(th_sources):,}")
print(f"Unique source keys    : {th_sources['source_key'].nunique():,}")
print(f"Batch size            : {TH_BATCH_SIZE:,}")
print(f"Expected Parquet parts: {len(th_batch_starts):,}")

if len(th_sources) != len(th_zip_files):
    print("\nNote: Some ZIP archives contain more than one CityGML file.")

if th_sources["source_key"].duplicated().any():
    raise ValueError("Duplicate Thuringia source keys detected.")

# ------------------------------------------------------------
# 2. Process each source batch
# ------------------------------------------------------------

th_batch_summaries = []

for batch_start in tqdm(th_batch_starts, desc="Processing Thuringia batches"):
    batch_number = batch_start // TH_BATCH_SIZE + 1
    batch_sources = th_sources.iloc[batch_start:batch_start + TH_BATCH_SIZE].copy()

    part_path = TH_PART_DIR / f"lod2_buildings_TH_part_{batch_number:04d}.parquet"
    manifest_path = TH_MANIFEST_DIR / f"lod2_manifest_TH_part_{batch_number:04d}.csv"

    # Resume only when the saved manifest exactly matches this batch
    if manifest_path.exists():
        existing_manifest = pd.read_csv(manifest_path)
        expected_keys = set(batch_sources["source_key"])
        existing_keys = set(existing_manifest["source_key"])

        manifest_complete = (
            expected_keys == existing_keys
            and len(existing_manifest) == len(batch_sources)
            and not existing_manifest["status"].eq("failed").any()
        )

        output_complete = existing_manifest["buildings"].sum() == 0 or part_path.exists()

        if manifest_complete and output_complete:
            th_batch_summaries.append({
                "batch": batch_number,
                "source_files": len(existing_manifest),
                "processed_files": existing_manifest["status"].eq("processed").sum(),
                "empty_files": existing_manifest["status"].eq("empty").sum(),
                "failed_files": existing_manifest["status"].eq("failed").sum(),
                "buildings": existing_manifest["buildings"].sum(),
                "status": "already_processed"
            })
            continue

    if part_path.exists():
        part_path.unlink()

    batch_parts = []
    batch_manifest = []

    for archive_path, archive_sources in batch_sources.groupby("archive_path", sort=False):
        with ZipFile(archive_path, "r") as archive:
            for row in archive_sources.itertuples(index=False):
                try:
                    with archive.open(row.member) as stream:
                        tile_gdf = extract_citygml_stream_multipart(
                            stream,
                            Path(row.member).name,
                            "EPSG:25832",
                            TH_BUILDING_TAG,
                            TH_BUILDING_PART_TAG
                        )

                    if tile_gdf.empty:
                        batch_manifest.append({
                            "source_key": row.source_key,
                            "archive_name": row.archive_name,
                            "source_member": row.member,
                            "status": "empty",
                            "buildings": 0,
                            "error": None
                        })
                        continue

                    tile_gdf["state_code"] = "TH"
                    tile_gdf["source_state"] = "Thuringia"
                    tile_gdf["source_crs"] = "EPSG:25832"
                    tile_gdf["citygml_version"] = "1.0"
                    tile_gdf["height_source"] = tile_gdf["height_method"]
                    tile_gdf["source_archive"] = row.archive_name
                    tile_gdf["source_member"] = row.member
                    tile_gdf["source_path"] = row.source_key

                    batch_parts.append(tile_gdf)

                    batch_manifest.append({
                        "source_key": row.source_key,
                        "archive_name": row.archive_name,
                        "source_member": row.member,
                        "status": "processed",
                        "buildings": len(tile_gdf),
                        "error": None
                    })

                except Exception as error:
                    batch_manifest.append({
                        "source_key": row.source_key,
                        "archive_name": row.archive_name,
                        "source_member": row.member,
                        "status": "failed",
                        "buildings": 0,
                        "error": str(error)
                    })

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    batch_manifest_df = pd.DataFrame(batch_manifest)
    batch_manifest_df.to_csv(manifest_path, index=False)

    th_batch_summaries.append({
        "batch": batch_number,
        "source_files": len(batch_manifest_df),
        "processed_files": batch_manifest_df["status"].eq("processed").sum(),
        "empty_files": batch_manifest_df["status"].eq("empty").sum(),
        "failed_files": batch_manifest_df["status"].eq("failed").sum(),
        "buildings": batch_manifest_df["buildings"].sum(),
        "status": "processed"
    })

    del batch_parts, batch_manifest, batch_manifest_df
    gc.collect()

# ------------------------------------------------------------
# 3. Combine processing manifests
# ------------------------------------------------------------

th_batch_summary_df = pd.DataFrame(th_batch_summaries)
th_manifest_files = sorted(TH_MANIFEST_DIR.glob("lod2_manifest_TH_part_*.csv"))
th_manifests = pd.concat([pd.read_csv(path) for path in th_manifest_files], ignore_index=True)
th_part_files = sorted(TH_PART_DIR.glob("lod2_buildings_TH_part_*.parquet"))

th_batch_summary_path = REPORT_DIR / "lod2_batch_summary_TH.csv"
th_manifest_path = REPORT_DIR / "lod2_processing_manifest_TH.csv"

th_batch_summary_df.to_csv(th_batch_summary_path, index=False)
th_manifests.to_csv(th_manifest_path, index=False)

print("=" * 70)
print("THURINGIA LoD2 PROCESSING SUMMARY")
print("=" * 70)
print(f"\nSource files recorded : {len(th_manifests):,}")
print(f"Files with buildings  : {th_manifests['status'].eq('processed').sum():,}")
print(f"Empty CityGML files   : {th_manifests['status'].eq('empty').sum():,}")
print(f"Failed files          : {th_manifests['status'].eq('failed').sum():,}")
print(f"Buildings extracted   : {th_manifests['buildings'].sum():,}")
print(f"Parquet parts created : {len(th_part_files):,}")
print(f"Part directory        : {TH_PART_DIR}")
print(f"Combined manifest     : {th_manifest_path}")

display(
    th_manifests["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="source_files")
)

if th_manifests["status"].eq("failed").any():
    display(th_manifests[th_manifests["status"] == "failed"].head(30))

Indexing Thuringia ZIP files: 100%|██████████████████████████████████████████| 4390/4390 [00:03<00:00, 1291.12it/s]


THURINGIA PROCESSING SETUP

ZIP archives          : 4,390
CityGML files         : 4,390
Unique source keys    : 4,390
Batch size            : 250
Expected Parquet parts: 18


Processing Thuringia batches: 100%|████████████████████████████████████████████████| 18/18 [19:23<00:00, 64.66s/it]

THURINGIA LoD2 PROCESSING SUMMARY

Source files recorded : 4,390
Files with buildings  : 4,085
Empty CityGML files   : 305
Failed files          : 0
Buildings extracted   : 2,333,261
Parquet parts created : 18
Part directory        : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/TH/parts
Combined manifest     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_processing_manifest_TH.csv


,status,source_files
0,processed,4085
1,empty,305


In [80]:
# ============================================================
# 72 — VALIDATE AND SUMMARISE THURINGIA RAW LoD2 DATA
# ============================================================

th_expected_sources = set(th_sources["source_key"])
th_manifest_sources = set(th_manifests["source_key"])
th_missing_sources = sorted(th_expected_sources - th_manifest_sources)
th_unexpected_sources = sorted(th_manifest_sources - th_expected_sources)
th_duplicate_manifest_sources = th_manifests["source_key"].duplicated().sum()

th_summary_records = []
th_footprint_counts = Counter()
th_height_method_counts = Counter()
th_id_counts = Counter()

for part_path in tqdm(th_part_files, desc="Summarising Thuringia parts"):
    part = pd.read_parquet(
        part_path,
        columns=["lod2_id", "measured_height_m", "height_method", "footprint_method", "geometry"]
    )

    th_id_counts.update(part["lod2_id"].dropna().astype(str))

    th_summary_records.append({
        "buildings": len(part),
        "missing_ids": part["lod2_id"].isna().sum(),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    th_footprint_counts.update(part["footprint_method"].fillna("missing"))
    th_height_method_counts.update(part["height_method"].fillna("missing"))

    del part
    gc.collect()

th_part_stats = pd.DataFrame(th_summary_records)
th_duplicate_ids = {lod2_id: count for lod2_id, count in th_id_counts.items() if count > 1}
th_extra_duplicate_rows = sum(count - 1 for count in th_duplicate_ids.values())
th_unique_ids = len(th_id_counts)

th_summary = pd.DataFrame([{
    "state_code": "TH",
    "state_name": "Thuringia",
    "zip_archives": len(th_zip_files),
    "source_files_expected": len(th_expected_sources),
    "source_files_recorded": len(th_manifest_sources),
    "files_with_buildings": th_manifests["status"].eq("processed").sum(),
    "empty_files": th_manifests["status"].eq("empty").sum(),
    "failed_files": th_manifests["status"].eq("failed").sum(),
    "missing_source_files": len(th_missing_sources),
    "unexpected_source_files": len(th_unexpected_sources),
    "duplicate_manifest_sources": th_duplicate_manifest_sources,
    "parquet_parts": len(th_part_files),
    "buildings": th_part_stats["buildings"].sum(),
    "unique_ids": th_unique_ids,
    "duplicated_ids": len(th_duplicate_ids),
    "extra_duplicate_rows": th_extra_duplicate_rows,
    "missing_ids": th_part_stats["missing_ids"].sum(),
    "measured_heights": th_part_stats["measured_heights"].sum(),
    "missing_heights": th_part_stats["missing_heights"].sum(),
    "usable_geometries": th_part_stats["usable_geometries"].sum(),
    "missing_geometries": th_part_stats["missing_geometries"].sum(),
    "minimum_height_m": th_part_stats["minimum_height_m"].min(),
    "maximum_height_m": th_part_stats["maximum_height_m"].max(),
    "output_directory": str(TH_PART_DIR)
}])

th_height_method_summary = pd.DataFrame(
    [{"height_method": method, "building_count": count} for method, count in th_height_method_counts.items()]
).sort_values("building_count", ascending=False)

th_footprint_summary = pd.DataFrame(
    [{"footprint_method": method, "building_count": count} for method, count in th_footprint_counts.items()]
).sort_values("building_count", ascending=False)

th_summary_path = REPORT_DIR / "lod2_extraction_summary_TH.csv"
th_height_method_summary_path = REPORT_DIR / "lod2_height_methods_TH.csv"
th_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_TH.csv"

th_summary.to_csv(th_summary_path, index=False)
th_height_method_summary.to_csv(th_height_method_summary_path, index=False)
th_footprint_summary.to_csv(th_footprint_summary_path, index=False)

print("=" * 70)
print("THURINGIA FINAL LoD2 VALIDATION")
print("=" * 70)

display(th_summary)
display(th_height_method_summary)
display(th_footprint_summary)

print(f"\nState summary saved to     : {th_summary_path}")
print(f"Height summary saved to    : {th_height_method_summary_path}")
print(f"Footprint summary saved to : {th_footprint_summary_path}")

if th_missing_sources:
    display(pd.DataFrame({"missing_source": th_missing_sources}).head(30))

Summarising Thuringia parts: 100%|█████████████████████████████████████████████████| 18/18 [00:17<00:00,  1.05it/s]

THURINGIA FINAL LoD2 VALIDATION


,state_code,state_name,zip_archives,source_files_expected,source_files_recorded,files_with_buildings,empty_files,failed_files,missing_source_files,unexpected_source_files,...,duplicated_ids,extra_duplicate_rows,missing_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,TH,Thuringia,4390,4390,4390,4085,305,0,0,0,...,935,935,0,2333261,0,2333261,0,0.001,197.637,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


,height_method,building_count
0,building_measured_height,2059404
1,maximum_building_part_height,273857


,footprint_method,building_count
0,ground_surface,2333261



State summary saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_TH.csv
Height summary saved to    : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_height_methods_TH.csv
Footprint summary saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_TH.csv


In [81]:
# ============================================================
# 73 — DISCOVER HESSE LoD2 REGIONAL ARCHIVES
# ============================================================

import json
from urllib.parse import urljoin, urlparse, unquote

HE_RAW_DIR = RAW_DIR / "HE"
HE_METADATA_DIR = HE_RAW_DIR / "metadata"
HE_RAW_DIR.mkdir(parents=True, exist_ok=True)
HE_METADATA_DIR.mkdir(parents=True, exist_ok=True)

HE_SERVER = "https://gds.hessen.de"
HE_PORTAL_URL = "https://gds.hessen.de/INTERSHOP/web/WFS/HLBG-Geodaten-Site/de_DE/-/EUR/ViewDownloadcenter-Start?path=3D-Daten/3D-Geb%C3%A4udemodelle/3D-Geb%C3%A4udemodelle%20LoD2"
HE_API_URL = "https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-Geodaten-Site/-/downloadcenter?path=3D-Daten/3D-Geb%C3%A4udemodelle/3D-Geb%C3%A4udemodelle%20LoD2&navigation=all"

he_index = source_registry["state_code"].eq("HE")
source_registry.loc[he_index, ["portal_url", "download_url", "download_method", "format", "horizontal_crs", "download_status", "notes"]] = [
    HE_PORTAL_URL,
    HE_API_URL,
    "Official regional download-centre archives",
    "CityGML ZIP",
    "EPSG:25832",
    "discovering",
    "LoD2 organised by Hessian districts and independent cities"
]
source_registry.to_csv(source_registry_path, index=False)

# ------------------------------------------------------------
# 1. Read the top-level LoD2 catalogue
# ------------------------------------------------------------

response = requests.get(HE_API_URL, timeout=(30, 300))
response.raise_for_status()
he_catalogue = response.json()

he_catalogue_path = HE_METADATA_DIR / "he_lod2_catalogue.json"
he_catalogue_path.write_text(json.dumps(he_catalogue, ensure_ascii=False, indent=2), encoding="utf-8")

he_region_nodes = pd.DataFrame([
    node for node in he_catalogue.get("navigation", [])
    if node.get("level") == 4 and "LoD2" in str(node.get("uri", ""))
])

if he_region_nodes.empty:
    raise RuntimeError("No Hesse LoD2 regional folders were discovered.")

# ------------------------------------------------------------
# 2. Open every regional folder and collect its packages
# ------------------------------------------------------------

he_package_records = []
he_region_status = []

for region in tqdm(he_region_nodes.itertuples(index=False), total=len(he_region_nodes), desc="Reading Hesse regional folders"):
    region_url = requests.utils.requote_uri(urljoin(HE_SERVER, region.uri))

    try:
        region_response = requests.get(region_url, timeout=(30, 300))
        region_response.raise_for_status()
        region_data = region_response.json()

        safe_region_name = re.sub(r"[^A-Za-z0-9_-]+", "_", region.name).strip("_")
        region_json_path = HE_METADATA_DIR / f"{safe_region_name}.json"
        region_json_path.write_text(json.dumps(region_data, ensure_ascii=False, indent=2), encoding="utf-8")

        packages = region_data.get("searchresult", {}).get("packages", [])

        he_region_status.append({
            "region_name": region.name,
            "region_id": region.id,
            "region_url": region_url,
            "packages": len(packages),
            "status": "processed",
            "error": None
        })

        for package_number, package in enumerate(packages, start=1):
            download_link = package.get("downloadLink", {})
            download_uri = download_link.get("uri") if isinstance(download_link, dict) else download_link

            if not download_uri:
                continue

            download_url = requests.utils.requote_uri(urljoin(HE_SERVER, download_uri))
            filename = Path(unquote(urlparse(download_url).path)).name

            if not filename.lower().endswith(".zip"):
                filename_candidates = [
                    package.get("filename"),
                    package.get("fileName"),
                    package.get("name"),
                    download_link.get("name") if isinstance(download_link, dict) else None
                ]
                filename = next((str(value) for value in filename_candidates if value and str(value).lower().endswith(".zip")), filename)

            he_package_records.append({
                "region_name": region.name,
                "region_id": region.id,
                "package_number": package_number,
                "package_name": package.get("name"),
                "creation_date": package.get("creationDate"),
                "download_url": download_url,
                "filename": filename,
                "region_url": region_url
            })

    except Exception as error:
        he_region_status.append({
            "region_name": region.name,
            "region_id": region.id,
            "region_url": region_url,
            "packages": 0,
            "status": "failed",
            "error": str(error)
        })

he_regions = pd.DataFrame(he_region_status)
he_downloads = pd.DataFrame(he_package_records)

if he_downloads.empty:
    raise RuntimeError("No downloadable Hesse LoD2 packages were discovered.")

he_downloads = he_downloads.drop_duplicates(subset="download_url").reset_index(drop=True)

he_regions_path = HE_RAW_DIR / "he_region_discovery.csv"
he_downloads_path = HE_RAW_DIR / "he_lod2_download_list.csv"

he_regions.to_csv(he_regions_path, index=False)
he_downloads.to_csv(he_downloads_path, index=False)

print("=" * 70)
print("HESSE LoD2 ARCHIVE DISCOVERY")
print("=" * 70)
print(f"\nRegional folders found : {len(he_region_nodes):,}")
print(f"Folders processed      : {he_regions['status'].eq('processed').sum():,}")
print(f"Folders failed         : {he_regions['status'].eq('failed').sum():,}")
print(f"Packages discovered    : {len(he_package_records):,}")
print(f"Unique download URLs   : {he_downloads['download_url'].nunique():,}")
print(f"Unique filenames       : {he_downloads['filename'].nunique():,}")
print(f"Duplicate filenames    : {he_downloads['filename'].duplicated().sum():,}")
print(f"Missing filenames      : {he_downloads['filename'].isna().sum():,}")

print("\nRegional folder summary:")
display(he_regions)

print("\nDiscovered download packages:")
display(he_downloads)

print(f"\nDownload list saved to: {he_downloads_path}")

if he_regions["status"].eq("failed").any():
    display(he_regions[he_regions["status"] == "failed"])

Reading Hesse regional folders: 100%|██████████████████████████████████████████████| 27/27 [00:03<00:00,  8.91it/s]

HESSE LoD2 ARCHIVE DISCOVERY

Regional folders found : 27
Folders processed      : 27
Folders failed         : 0
Packages discovered    : 27
Unique download URLs   : 27
Unique filenames       : 27
Duplicate filenames    : 0
Missing filenames      : 0

Regional folder summary:


,region_name,region_id,region_url,packages,status,error
0,Hochtaunus,DC0102050,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
1,Kreisfreie Stadt Darmstadt,DC0100004,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
2,Kreisfreie Stadt Frankfurt,DC0102051,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
3,Kreisfreie Stadt Hanau,DC0101850,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
4,Kreisfreie Stadt Kassel,DC0100006,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
5,Kreisfreie Stadt Offenbach-am-Main,DC0100007,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
6,Kreisfreie Stadt Wiesbaden,DC0100008,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
7,Lahn-Dill-Kreis,DC0100009,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
8,Landkreis Bergstrasse,DC0100010,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None
9,Landkreis Darmstadt-Dieburg,DC0100011,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...,1,processed,None



Discovered download packages:


,region_name,region_id,package_number,package_name,creation_date,download_url,filename,region_url
0,Hochtaunus,DC0102050,1,Alle Dateien zu Hochtaunus herunterladen,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Hochtaunus.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
1,Kreisfreie Stadt Darmstadt,DC0100004,1,Alle Dateien zu Kreisfreie Stadt Darmstadt her...,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Kreisfreie Stadt Darmstadt.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
2,Kreisfreie Stadt Frankfurt,DC0102051,1,Alle Dateien zu Kreisfreie Stadt Frankfurt her...,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Kreisfreie Stadt Frankfurt.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
3,Kreisfreie Stadt Hanau,DC0101850,1,Alle Dateien zu Kreisfreie Stadt Hanau herunte...,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Kreisfreie Stadt Hanau.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
4,Kreisfreie Stadt Kassel,DC0100006,1,Alle Dateien zu Kreisfreie Stadt Kassel herunt...,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Kreisfreie Stadt Kassel.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
5,Kreisfreie Stadt Offenbach-am-Main,DC0100007,1,Alle Dateien zu Kreisfreie Stadt Offenbach-am-...,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Kreisfreie Stadt Offenbach-am-Main.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
6,Kreisfreie Stadt Wiesbaden,DC0100008,1,Alle Dateien zu Kreisfreie Stadt Wiesbaden her...,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Kreisfreie Stadt Wiesbaden.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
7,Lahn-Dill-Kreis,DC0100009,1,Alle Dateien zu Lahn-Dill-Kreis herunterladen,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Lahn-Dill-Kreis.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
8,Landkreis Bergstrasse,DC0100010,1,Alle Dateien zu Landkreis Bergstrasse herunter...,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Landkreis Bergstrasse.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...
9,Landkreis Darmstadt-Dieburg,DC0100011,1,Alle Dateien zu Landkreis Darmstadt-Dieburg he...,29.06.2026,https://gds.hessen.de/downloadcenter/20260726/...,PKT_Landkreis Darmstadt-Dieburg.zip,https://gds.hessen.de/INTERSHOP/rest/WFS/HLBG-...



Download list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/HE/he_lod2_download_list.csv


In [82]:
# ============================================================
# 74 — INSPECT HESSE PACKAGE SIZES AND ARCHIVE STRUCTURE
# ============================================================

from io import BytesIO

HE_TEST_DIR = HE_RAW_DIR / "test_archive"
HE_TEST_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Read HTTP metadata without downloading the archives
# ------------------------------------------------------------

he_http_records = []

for row in tqdm(he_downloads.itertuples(index=False), total=len(he_downloads), desc="Checking Hesse package sizes"):
    try:
        with requests.get(row.download_url, stream=True, allow_redirects=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            size_bytes = parse_number(response.headers.get("content-length"))

            he_http_records.append({
                "region_name": row.region_name,
                "filename": row.filename,
                "download_url": row.download_url,
                "http_status": response.status_code,
                "content_type": response.headers.get("content-type"),
                "content_disposition": response.headers.get("content-disposition"),
                "size_bytes": size_bytes,
                "size_gb": size_bytes / 1024**3 if pd.notna(size_bytes) else np.nan,
                "final_url": response.url,
                "error": None
            })

    except Exception as error:
        he_http_records.append({
            "region_name": row.region_name,
            "filename": row.filename,
            "download_url": row.download_url,
            "http_status": np.nan,
            "content_type": None,
            "content_disposition": None,
            "size_bytes": np.nan,
            "size_gb": np.nan,
            "final_url": None,
            "error": str(error)
        })

he_package_metadata = pd.DataFrame(he_http_records)
he_package_metadata_path = HE_RAW_DIR / "he_package_metadata.csv"
he_package_metadata.to_csv(he_package_metadata_path, index=False)

print("=" * 70)
print("HESSE LoD2 PACKAGE SIZE SUMMARY")
print("=" * 70)
print(f"\nPackages checked       : {len(he_package_metadata):,}")
print(f"Successful responses   : {he_package_metadata['error'].isna().sum():,}")
print(f"Failed responses       : {he_package_metadata['error'].notna().sum():,}")
print(f"Packages with size     : {he_package_metadata['size_bytes'].notna().sum():,}")
print(f"Estimated total size   : {he_package_metadata['size_gb'].sum():,.2f} GB")
print(f"Smallest package       : {he_package_metadata['size_gb'].min():,.3f} GB")
print(f"Largest package        : {he_package_metadata['size_gb'].max():,.3f} GB")

display(
    he_package_metadata[
        ["region_name", "filename", "content_type", "size_gb", "error"]
    ].sort_values("size_gb")
)

# ------------------------------------------------------------
# 2. Select and download a median-sized representative archive
# ------------------------------------------------------------

he_sized_packages = he_package_metadata.dropna(subset=["size_bytes"]).sort_values("size_bytes").reset_index(drop=True)

if he_sized_packages.empty:
    he_sample_row = he_package_metadata.iloc[0]
else:
    he_sample_row = he_sized_packages.iloc[len(he_sized_packages) // 2]

he_sample_path = HE_TEST_DIR / he_sample_row["filename"]

if not he_sample_path.exists():
    temp_path = he_sample_path.with_suffix(".zip.part")

    with requests.get(he_sample_row["download_url"], stream=True, timeout=(30, 1800)) as response:
        response.raise_for_status()

        with open(temp_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    temp_path.replace(he_sample_path)

with ZipFile(he_sample_path, "r") as archive:
    bad_member = archive.testzip()

    if bad_member is not None:
        raise ValueError(f"Corrupt member in representative Hesse archive: {bad_member}")

    outer_infos = archive.infolist()
    outer_gml_members = [info for info in outer_infos if info.filename.lower().endswith((".gml", ".xml"))]
    nested_zip_members = [info for info in outer_infos if info.filename.lower().endswith(".zip")]
    other_members = [info for info in outer_infos if info not in outer_gml_members and info not in nested_zip_members]

    he_outer_structure = pd.DataFrame([{
        "region_name": he_sample_row["region_name"],
        "archive": he_sample_path.name,
        "archive_size_gb": he_sample_path.stat().st_size / 1024**3,
        "all_members": len(outer_infos),
        "direct_gml_xml": len(outer_gml_members),
        "nested_zip_files": len(nested_zip_members),
        "other_files": len(other_members),
        "uncompressed_size_gb": sum(info.file_size for info in outer_infos) / 1024**3
    }])

    # Select up to three representative direct GML files or nested ZIPs
    if nested_zip_members:
        selected_indices = sorted(set([0, len(nested_zip_members) // 2, len(nested_zip_members) - 1]))
        selected_containers = [nested_zip_members[index] for index in selected_indices]
        source_mode = "nested_zip"
    else:
        selected_indices = sorted(set([0, len(outer_gml_members) // 2, len(outer_gml_members) - 1])) if outer_gml_members else []
        selected_containers = [outer_gml_members[index] for index in selected_indices]
        source_mode = "direct_gml"

    he_structure_records = []

    for selected_info in selected_containers:
        if source_mode == "nested_zip":
            nested_bytes = archive.read(selected_info.filename)

            with ZipFile(BytesIO(nested_bytes), "r") as nested_archive:
                nested_bad_member = nested_archive.testzip()

                if nested_bad_member is not None:
                    raise ValueError(f"Corrupt nested ZIP member: {selected_info.filename}::{nested_bad_member}")

                nested_gml_infos = [
                    info for info in nested_archive.infolist()
                    if info.filename.lower().endswith((".gml", ".xml"))
                ]

                if not nested_gml_infos:
                    he_structure_records.append({
                        "outer_archive": he_sample_path.name,
                        "container": selected_info.filename,
                        "gml_file": None,
                        "building_namespace": None,
                        "srs_names": None,
                        "buildings": 0,
                        "building_parts": 0,
                        "measured_heights": 0,
                        "ground_surfaces": 0,
                        "nested_gml_files": 0,
                        "error": "No GML/XML file in nested ZIP"
                    })
                    continue

                gml_info = nested_gml_infos[len(nested_gml_infos) // 2]
                content = nested_archive.read(gml_info.filename)
                container_name = selected_info.filename
                gml_name = gml_info.filename

        else:
            content = archive.read(selected_info.filename)
            container_name = he_sample_path.name
            gml_name = selected_info.filename
            nested_gml_infos = outer_gml_members

        root = etree.fromstring(content)
        building_namespace = root.nsmap.get("bldg")
        srs_names = sorted({element.get("srsName") for element in root.iter() if element.get("srsName")})
        tag_counts = Counter(etree.QName(element).localname for element in root.iter())

        he_structure_records.append({
            "outer_archive": he_sample_path.name,
            "container": container_name,
            "gml_file": gml_name,
            "building_namespace": building_namespace,
            "srs_names": " | ".join(srs_names),
            "buildings": tag_counts["Building"],
            "building_parts": tag_counts["BuildingPart"],
            "measured_heights": tag_counts["measuredHeight"],
            "ground_surfaces": tag_counts["GroundSurface"],
            "nested_gml_files": len(nested_gml_infos),
            "error": None
        })

he_sample_structure = pd.DataFrame(he_structure_records)

print("\n" + "=" * 70)
print("HESSE REPRESENTATIVE ARCHIVE STRUCTURE")
print("=" * 70)

display(he_outer_structure)
display(he_sample_structure)

print(f"\nRepresentative region : {he_sample_row['region_name']}")
print(f"Representative archive: {he_sample_path}")
print(f"Metadata saved to     : {he_package_metadata_path}")

Checking Hesse package sizes: 100%|████████████████████████████████████████████████| 27/27 [00:02<00:00, 13.44it/s]

HESSE LoD2 PACKAGE SIZE SUMMARY

Packages checked       : 27
Successful responses   : 27
Failed responses       : 0
Packages with size     : 27
Estimated total size   : 9.89 GB
Smallest package       : 0.094 GB
Largest package        : 0.564 GB


,region_name,filename,content_type,size_gb,error
5,Kreisfreie Stadt Offenbach-am-Main,PKT_Kreisfreie Stadt Offenbach-am-Main.zip,application/zip,0.093751,None
3,Kreisfreie Stadt Hanau,PKT_Kreisfreie Stadt Hanau.zip,application/zip,0.108296,None
1,Kreisfreie Stadt Darmstadt,PKT_Kreisfreie Stadt Darmstadt.zip,application/zip,0.155048,None
4,Kreisfreie Stadt Kassel,PKT_Kreisfreie Stadt Kassel.zip,application/zip,0.206246,None
21,Odenwaldkreis,PKT_Odenwaldkreis.zip,application/zip,0.238734,None
6,Kreisfreie Stadt Wiesbaden,PKT_Kreisfreie Stadt Wiesbaden.zip,application/zip,0.265335,None
25,Werra-Meissner-Kreis,PKT_Werra-Meissner-Kreis.zip,application/zip,0.274113,None
20,Main-Taunus-Kreis,PKT_Main-Taunus-Kreis.zip,application/zip,0.292517,None
13,Landkreis Hersfeld-Rotenburg,PKT_Landkreis Hersfeld-Rotenburg.zip,application/zip,0.292837,None
24,Vogelsbergkreis,PKT_Vogelsbergkreis.zip,application/zip,0.293506,None



HESSE REPRESENTATIVE ARCHIVE STRUCTURE


,region_name,archive,archive_size_gb,all_members,direct_gml_xml,nested_zip_files,other_files,uncompressed_size_gb
0,Landkreis Gross-Gerau,PKT_Landkreis Gross-Gerau.zip,0.385354,14,0,14,0,0.385352


,outer_archive,container,gml_file,building_namespace,srs_names,buildings,building_parts,measured_heights,ground_surfaces,nested_gml_files,error
0,PKT_Landkreis Gross-Gerau.zip,Biebesheim_am_Rhein-LoD2.zip,Biebesheim am Rhein-LoD2.gml,http://www.opengis.net/citygml/building/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,8101,4034,10660,10660,1,None
1,PKT_Landkreis Gross-Gerau.zip,Moerfelden-Walldorf-LoD2.zip,Moerfelden-Walldorf-LoD2.gml,http://www.opengis.net/citygml/building/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,25854,7093,30407,30402,1,None
2,PKT_Landkreis Gross-Gerau.zip,Trebur-LoD2.zip,Trebur-LoD2.gml,http://www.opengis.net/citygml/building/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,14857,4320,17447,17442,1,None



Representative region : Landkreis Gross-Gerau
Representative archive: /fast/home/o-olajuyigbe/data/germany_lod2/raw/HE/test_archive/PKT_Landkreis Gross-Gerau.zip
Metadata saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/raw/HE/he_package_metadata.csv


In [83]:
# ============================================================
# 74b — PRINT COMPACT HESSE ARCHIVE STRUCTURE
# ============================================================

print("=" * 70)
print("HESSE REPRESENTATIVE ARCHIVE STRUCTURE")
print("=" * 70)

print("\nOuter archive:")
print(he_outer_structure.to_string(index=False))

print("\nSample CityGML containers:")
print(he_sample_structure.to_string(index=False))

print(f"\nRepresentative region : {he_sample_row['region_name']}")
print(f"Representative archive: {he_sample_path}")

HESSE REPRESENTATIVE ARCHIVE STRUCTURE

Outer archive:
          region_name                       archive  archive_size_gb  all_members  direct_gml_xml  nested_zip_files  other_files  uncompressed_size_gb
Landkreis Gross-Gerau PKT_Landkreis Gross-Gerau.zip         0.385354           14               0                14            0              0.385352

Sample CityGML containers:
                outer_archive                    container                     gml_file                          building_namespace                               srs_names  buildings  building_parts  measured_heights  ground_surfaces  nested_gml_files error
PKT_Landkreis Gross-Gerau.zip Biebesheim_am_Rhein-LoD2.zip Biebesheim am Rhein-LoD2.gml http://www.opengis.net/citygml/building/1.0 urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH       8101            4034             10660            10660                 1  None
PKT_Landkreis Gross-Gerau.zip Moerfelden-Walldorf-LoD2.zip Moerfelden-Walldorf-LoD2.gml http://www.

In [84]:
# ============================================================
# 75 — TEST HESSE MULTIPART HEIGHT AND FOOTPRINT PARSING
# ============================================================

HE_BLDG_NS = "http://www.opengis.net/citygml/building/1.0"
HE_BUILDING_TAG = f"{{{HE_BLDG_NS}}}Building"
HE_BUILDING_PART_TAG = f"{{{HE_BLDG_NS}}}BuildingPart"

he_parser_records = []
he_height_records = []

with ZipFile(he_sample_path, "r") as outer_archive:
    for container_name in he_sample_structure["container"].dropna().unique():
        nested_content = outer_archive.read(container_name)

        with ZipFile(BytesIO(nested_content), "r") as nested_archive:
            gml_members = [name for name in nested_archive.namelist() if name.lower().endswith((".gml", ".xml"))]

            for gml_member in gml_members:
                content = nested_archive.read(gml_member)

                # ----------------------------------------------------
                # Inspect direct parent and BuildingPart heights
                # ----------------------------------------------------
                for event, building in etree.iterparse(BytesIO(content), events=("end",), tag=HE_BUILDING_TAG, huge_tree=True):
                    parent_height = parse_number(get_direct_text(building, "measuredHeight"))
                    building_parts = building.findall(f".//{HE_BUILDING_PART_TAG}")
                    part_heights = [parse_number(get_direct_text(part, "measuredHeight")) for part in building_parts]
                    part_heights = [height for height in part_heights if pd.notna(height)]

                    he_height_records.append({
                        "municipality_zip": container_name,
                        "source_file": gml_member,
                        "lod2_id": building.get(GML_ID),
                        "parent_height_m": parent_height,
                        "building_parts": len(building_parts),
                        "parts_with_height": len(part_heights),
                        "minimum_part_height_m": min(part_heights) if part_heights else np.nan,
                        "maximum_part_height_m": max(part_heights) if part_heights else np.nan,
                        "first_descendant_height_m": parse_number(get_first_text(building, "measuredHeight"))
                    })

                    building.clear()
                    while building.getprevious() is not None:
                        del building.getparent()[0]

                # ----------------------------------------------------
                # Test the multipart extraction parser
                # ----------------------------------------------------
                tile_gdf = extract_citygml_stream_multipart(
                    BytesIO(content),
                    Path(gml_member).name,
                    "EPSG:25832",
                    HE_BUILDING_TAG,
                    HE_BUILDING_PART_TAG
                )

                he_parser_records.append({
                    "municipality_zip": container_name,
                    "source_file": gml_member,
                    "buildings_extracted": len(tile_gdf),
                    "unique_ids": tile_gdf["lod2_id"].nunique(),
                    "missing_ids": tile_gdf["lod2_id"].isna().sum(),
                    "measured_heights": tile_gdf["measured_height_m"].notna().sum(),
                    "missing_heights": tile_gdf["measured_height_m"].isna().sum(),
                    "buildings_with_parts": tile_gdf["building_part_count"].gt(0).sum(),
                    "parent_height_method": tile_gdf["height_method"].eq("building_measured_height").sum(),
                    "part_height_method": tile_gdf["height_method"].eq("maximum_building_part_height").sum(),
                    "usable_geometries": tile_gdf.geometry.notna().sum(),
                    "missing_geometries": tile_gdf.geometry.isna().sum(),
                    "ground_surface": tile_gdf["footprint_method"].eq("ground_surface").sum(),
                    "other_footprint_method": tile_gdf["footprint_method"].ne("ground_surface").sum()
                })

he_parser_test = pd.DataFrame(he_parser_records)
he_height_check = pd.DataFrame(he_height_records)

he_height_check["has_parent_height"] = he_height_check["parent_height_m"].notna()
he_height_check["has_parts"] = he_height_check["building_parts"] > 0
he_height_check["first_differs_from_maximum"] = (
    he_height_check["parts_with_height"].gt(0)
    & ~np.isclose(
        he_height_check["first_descendant_height_m"],
        he_height_check["maximum_part_height_m"],
        atol=0.01,
        equal_nan=False
    )
)

he_height_summary = pd.DataFrame([{
    "buildings_checked": len(he_height_check),
    "buildings_with_parts": he_height_check["has_parts"].sum(),
    "buildings_without_parts": (~he_height_check["has_parts"]).sum(),
    "buildings_with_parent_height": he_height_check["has_parent_height"].sum(),
    "buildings_missing_parent_height": (~he_height_check["has_parent_height"]).sum(),
    "multipart_missing_parent_height": (he_height_check["has_parts"] & ~he_height_check["has_parent_height"]).sum(),
    "multipart_missing_all_heights": (
        he_height_check["has_parts"]
        & ~he_height_check["has_parent_height"]
        & he_height_check["parts_with_height"].eq(0)
    ).sum(),
    "first_descendant_differs_from_maximum_part": he_height_check["first_differs_from_maximum"].sum()
}])

print("=" * 70)
print("HESSE MUNICIPAL PARSER TEST")
print("=" * 70)

display(he_parser_test)

print("\nCombined parser totals:")
display(
    he_parser_test[
        [
            "buildings_extracted",
            "unique_ids",
            "missing_ids",
            "measured_heights",
            "missing_heights",
            "buildings_with_parts",
            "parent_height_method",
            "part_height_method",
            "usable_geometries",
            "missing_geometries"
        ]
    ].sum().to_frame("count")
)

print("\nHeight structure:")
display(he_height_summary)

print("\nCases where the first descendant is not the tallest BuildingPart:")
display(
    he_height_check.loc[
        he_height_check["first_differs_from_maximum"],
        [
            "lod2_id",
            "building_parts",
            "minimum_part_height_m",
            "maximum_part_height_m",
            "first_descendant_height_m",
            "municipality_zip"
        ]
    ].head(30)
)

HESSE MUNICIPAL PARSER TEST


,municipality_zip,source_file,buildings_extracted,unique_ids,missing_ids,measured_heights,missing_heights,buildings_with_parts,parent_height_method,part_height_method,usable_geometries,missing_geometries,ground_surface,other_footprint_method
0,Biebesheim_am_Rhein-LoD2.zip,Biebesheim am Rhein-LoD2.gml,8101,8101,0,8101,0,1475,6626,1475,8101,0,8101,0
1,Moerfelden-Walldorf-LoD2.zip,Moerfelden-Walldorf-LoD2.gml,25854,25854,0,25854,0,2540,23314,2540,25850,4,25849,5
2,Trebur-LoD2.zip,Trebur-LoD2.gml,14857,14857,0,14857,0,1730,13127,1730,14852,5,14852,5



Combined parser totals:


,count
buildings_extracted,48812
unique_ids,48812
missing_ids,0
measured_heights,48812
missing_heights,0
buildings_with_parts,5745
parent_height_method,43067
part_height_method,5745
usable_geometries,48803
missing_geometries,9



Height structure:


,buildings_checked,buildings_with_parts,buildings_without_parts,buildings_with_parent_height,buildings_missing_parent_height,multipart_missing_parent_height,multipart_missing_all_heights,first_descendant_differs_from_maximum_part
0,48812,5745,43067,43067,5745,5745,0,858



Cases where the first descendant is not the tallest BuildingPart:


,lod2_id,building_parts,minimum_part_height_m,maximum_part_height_m,first_descendant_height_m,municipality_zip
43,DEHE06120000243F,3,7.013,9.063,8.923,Biebesheim_am_Rhein-LoD2.zip
60,DEHE0612000024BS,3,3.557,4.867,3.557,Biebesheim_am_Rhein-LoD2.zip
72,DEHE0612000024wy,4,3.038,4.228,4.198,Biebesheim_am_Rhein-LoD2.zip
85,DEHE0612000024Ar,2,2.496,5.416,2.496,Biebesheim_am_Rhein-LoD2.zip
92,DEHE0612000024BG,3,5.638,6.738,6.358,Biebesheim_am_Rhein-LoD2.zip
153,DEHE0612000023RX,2,2.424,2.444,2.424,Biebesheim_am_Rhein-LoD2.zip
243,DEHE0612000022BV,3,3.874,8.704,4.574,Biebesheim_am_Rhein-LoD2.zip
278,DEHE0612000022XK,2,4.868,7.658,4.868,Biebesheim_am_Rhein-LoD2.zip
382,DEHE0612000023Ho,4,13.116,16.046,15.986,Biebesheim_am_Rhein-LoD2.zip
384,DEHE0612000024rm,3,5.428,7.468,5.428,Biebesheim_am_Rhein-LoD2.zip


In [85]:
# ============================================================
# 76 — DOWNLOAD ALL HESSE LoD2 REGIONAL ARCHIVES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import time

HE_ZIP_DIR = HE_RAW_DIR / "regional_archives"
HE_ZIP_DIR.mkdir(parents=True, exist_ok=True)

HE_MAX_WORKERS = 4
HE_MAX_RETRIES = 3
HE_CHUNK_SIZE = 1024 * 1024

def validate_he_archive(path):
    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with ZipFile(path, "r") as archive:
            members = archive.infolist()
            nested_zips = [info for info in members if info.filename.lower().endswith(".zip")]
            direct_gml = [info for info in members if info.filename.lower().endswith((".gml", ".xml"))]
            return bool(nested_zips or direct_gml)
    except Exception:
        return False

def download_he_archive(url, filename, region_name):
    output_path = HE_ZIP_DIR / filename
    temp_path = HE_ZIP_DIR / f"{filename}.part"

    if validate_he_archive(output_path):
        return {
            "region_name": region_name,
            "filename": filename,
            "status": "already_downloaded",
            "size_bytes": output_path.stat().st_size,
            "error": None
        }

    if output_path.exists():
        output_path.unlink()

    for attempt in range(1, HE_MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 1800)) as response:
                response.raise_for_status()
                expected_size = int(response.headers.get("content-length", 0))

                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=HE_CHUNK_SIZE):
                        if chunk:
                            file.write(chunk)

            actual_size = temp_path.stat().st_size

            if expected_size and actual_size != expected_size:
                raise ValueError(f"Incomplete download: expected {expected_size:,} bytes, received {actual_size:,}")

            if not validate_he_archive(temp_path):
                raise ValueError("Downloaded file is not a valid Hesse LoD2 archive.")

            temp_path.replace(output_path)

            return {
                "region_name": region_name,
                "filename": filename,
                "status": "downloaded",
                "size_bytes": output_path.stat().st_size,
                "error": None
            }

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == HE_MAX_RETRIES:
                return {
                    "region_name": region_name,
                    "filename": filename,
                    "status": "failed",
                    "size_bytes": 0,
                    "error": str(error)
                }

            time.sleep(attempt * 5)

he_download_results = []

with ThreadPoolExecutor(max_workers=HE_MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_he_archive, row.download_url, row.filename, row.region_name): row.filename
        for row in he_downloads.itertuples(index=False)
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading Hesse regional archives"):
        he_download_results.append(future.result())

he_download_results_df = pd.DataFrame(he_download_results).sort_values("region_name").reset_index(drop=True)
he_download_results_path = REPORT_DIR / "lod2_download_results_HE.csv"
he_download_results_df.to_csv(he_download_results_path, index=False)

he_regional_archives = sorted(HE_ZIP_DIR.glob("*.zip"))

print("=" * 70)
print("HESSE LoD2 DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nRegions requested      : {len(he_downloads):,}")
print(f"Archives present       : {len(he_regional_archives):,}")
print(f"Downloaded now         : {he_download_results_df['status'].eq('downloaded').sum():,}")
print(f"Already downloaded     : {he_download_results_df['status'].eq('already_downloaded').sum():,}")
print(f"Failed                 : {he_download_results_df['status'].eq('failed').sum():,}")
print(f"Total downloaded size  : {sum(path.stat().st_size for path in he_regional_archives) / 1024**3:,.2f} GB")
print(f"Download report        : {he_download_results_path}")

display(
    he_download_results_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

if he_download_results_df["status"].eq("failed").any():
    display(he_download_results_df[he_download_results_df["status"] == "failed"])

HESSE LoD2 DOWNLOAD SUMMARY

Regions requested      : 27
Archives present       : 27
Downloaded now         : 27
Already downloaded     : 0
Failed                 : 0
Total downloaded size  : 9.89 GB
Download report        : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_download_results_HE.csv


,status,count
0,downloaded,27


In [86]:
# ============================================================
# 77 — INDEX ALL HESSE MUNICIPALITY LoD2 ARCHIVES
# ============================================================

he_region_lookup = he_downloads.set_index("filename")["region_name"].to_dict()
he_municipality_records = []
he_outer_archive_records = []

for outer_path in tqdm(he_regional_archives, desc="Indexing Hesse regional archives"):
    region_name = he_region_lookup.get(outer_path.name)

    with ZipFile(outer_path, "r") as outer_archive:
        bad_member = outer_archive.testzip()

        if bad_member is not None:
            raise ValueError(f"Corrupt file in {outer_path.name}: {bad_member}")

        nested_zip_infos = [info for info in outer_archive.infolist() if info.filename.lower().endswith(".zip")]
        direct_gml_infos = [info for info in outer_archive.infolist() if info.filename.lower().endswith((".gml", ".xml"))]

        he_outer_archive_records.append({
            "region_name": region_name,
            "regional_archive": outer_path.name,
            "municipality_zips": len(nested_zip_infos),
            "direct_gml_files": len(direct_gml_infos),
            "archive_size_gb": outer_path.stat().st_size / 1024**3,
            "nested_uncompressed_size_gb": sum(info.file_size for info in nested_zip_infos) / 1024**3
        })

        for info in nested_zip_infos:
            municipality_name = re.sub(r"(?i)-lod2\.zip$", "", Path(info.filename).name).replace("_", " ").strip()

            he_municipality_records.append({
                "region_name": region_name,
                "regional_archive": outer_path.name,
                "regional_archive_path": str(outer_path),
                "municipality_zip": info.filename,
                "municipality_name": municipality_name,
                "nested_zip_size_bytes": info.file_size,
                "nested_zip_compressed_bytes": info.compress_size,
                "source_key": f"{outer_path.name}::{info.filename}"
            })

he_outer_archives_index = pd.DataFrame(he_outer_archive_records).sort_values("region_name").reset_index(drop=True)
he_municipalities = pd.DataFrame(he_municipality_records).sort_values(["region_name", "municipality_zip"]).reset_index(drop=True)

he_outer_index_path = HE_RAW_DIR / "he_regional_archive_index.csv"
he_municipality_index_path = HE_RAW_DIR / "he_municipality_archive_index.csv"

he_outer_archives_index.to_csv(he_outer_index_path, index=False)
he_municipalities.to_csv(he_municipality_index_path, index=False)

print("=" * 70)
print("HESSE MUNICIPALITY ARCHIVE INDEX")
print("=" * 70)
print(f"\nRegional archives       : {len(he_outer_archives_index):,}")
print(f"Municipality ZIP files  : {len(he_municipalities):,}")
print(f"Unique source keys      : {he_municipalities['source_key'].nunique():,}")
print(f"Unique ZIP filenames    : {he_municipalities['municipality_zip'].nunique():,}")
print(f"Duplicate source keys   : {he_municipalities['source_key'].duplicated().sum():,}")
print(f"Direct outer GML files  : {he_outer_archives_index['direct_gml_files'].sum():,}")
print(f"Nested ZIP total size   : {he_municipalities['nested_zip_size_bytes'].sum() / 1024**3:,.2f} GB")

print("\nMunicipality packages by region:")
display(
    he_municipalities.groupby("region_name", as_index=False)
    .agg(
        municipality_zips=("source_key", "size"),
        nested_size_gb=("nested_zip_size_bytes", lambda values: values.sum() / 1024**3)
    )
    .sort_values("region_name")
)

print("\nLargest municipality ZIP files:")
display(
    he_municipalities[
        ["region_name", "municipality_name", "municipality_zip", "nested_zip_size_bytes"]
    ]
    .assign(size_gb=lambda df: df["nested_zip_size_bytes"] / 1024**3)
    .sort_values("nested_zip_size_bytes", ascending=False)
    .head(20)
)

print(f"\nRegional index saved to    : {he_outer_index_path}")
print(f"Municipality index saved to: {he_municipality_index_path}")

if he_municipalities["source_key"].duplicated().any():
    raise ValueError("Duplicate Hesse municipality source keys detected.")

if he_outer_archives_index["direct_gml_files"].sum() > 0:
    print("\nWarning: Direct GML/XML files were found in one or more regional archives.")

Indexing Hesse regional archives: 100%|████████████████████████████████████████████| 27/27 [00:21<00:00,  1.26it/s]

HESSE MUNICIPALITY ARCHIVE INDEX

Regional archives       : 27
Municipality ZIP files  : 483
Unique source keys      : 483
Unique ZIP filenames    : 481
Duplicate source keys   : 0
Direct outer GML files  : 0
Nested ZIP total size   : 9.89 GB

Municipality packages by region:


,region_name,municipality_zips,nested_size_gb
0,Hochtaunus,13,0.329214
1,Kreisfreie Stadt Darmstadt,1,0.155048
2,Kreisfreie Stadt Frankfurt,59,0.502074
3,Kreisfreie Stadt Hanau,1,0.108296
4,Kreisfreie Stadt Kassel,1,0.206246
5,Kreisfreie Stadt Offenbach-am-Main,1,0.093751
6,Kreisfreie Stadt Wiesbaden,1,0.265335
7,Lahn-Dill-Kreis,23,0.513773
8,Landkreis Bergstrasse,23,0.517606
9,Landkreis Darmstadt-Dieburg,23,0.520315



Largest municipality ZIP files:


,region_name,municipality_name,municipality_zip,nested_zip_size_bytes,size_gb
76,Kreisfreie Stadt Wiesbaden,Wiesbaden,Wiesbaden-LoD2.zip,284900827,0.265335
74,Kreisfreie Stadt Kassel,Kassel,Kassel-LoD2.zip,221454426,0.206246
13,Kreisfreie Stadt Darmstadt,Darmstadt,Darmstadt-LoD2.zip,166481099,0.155048
73,Kreisfreie Stadt Hanau,Hanau,Hanau-LoD2.zip,116281666,0.108296
75,Kreisfreie Stadt Offenbach-am-Main,Offenbach am Main,Offenbach_am_Main-LoD2.zip,100664524,0.093751
173,Landkreis Gießen,Giessen,Giessen-LoD2.zip,99905584,0.093044
282,Landkreis Marburg-Biedenkopf,Marburg,Marburg-LoD2.zip,97427265,0.090736
154,Landkreis Fulda,Fulda,Fulda-LoD2.zip,92295662,0.085957
99,Lahn-Dill-Kreis,Wetzlar,Wetzlar-LoD2.zip,86642049,0.080692
101,Landkreis Bergstrasse,Bensheim,Bensheim-LoD2.zip,74963451,0.069815



Regional index saved to    : /fast/home/o-olajuyigbe/data/germany_lod2/raw/HE/he_regional_archive_index.csv
Municipality index saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/HE/he_municipality_archive_index.csv


In [87]:
# ============================================================
# 78 — INVESTIGATE DUPLICATE HESSE MUNICIPALITY FILENAMES
# ============================================================

he_duplicate_filenames = he_municipalities[
    he_municipalities["municipality_zip"].duplicated(keep=False)
].copy().sort_values(["municipality_zip", "region_name"])

he_duplicate_details = []

for row in he_duplicate_filenames.itertuples(index=False):
    outer_path = Path(row.regional_archive_path)

    with ZipFile(outer_path, "r") as outer_archive:
        info = outer_archive.getinfo(row.municipality_zip)
        nested_content = outer_archive.read(row.municipality_zip)

    with ZipFile(BytesIO(nested_content), "r") as nested_archive:
        nested_bad_member = nested_archive.testzip()
        gml_members = [name for name in nested_archive.namelist() if name.lower().endswith((".gml", ".xml"))]

        he_duplicate_details.append({
            "municipality_zip": row.municipality_zip,
            "municipality_name": row.municipality_name,
            "region_name": row.region_name,
            "regional_archive": row.regional_archive,
            "source_key": row.source_key,
            "nested_zip_size_bytes": info.file_size,
            "nested_zip_crc": info.CRC,
            "nested_zip_members": len(nested_archive.namelist()),
            "gml_xml_files": len(gml_members),
            "gml_filenames": " | ".join(gml_members),
            "nested_zip_valid": nested_bad_member is None
        })

he_duplicate_details = pd.DataFrame(he_duplicate_details)

he_duplicate_comparison = (
    he_duplicate_details
    .groupby("municipality_zip", as_index=False)
    .agg(
        occurrences=("source_key", "size"),
        regions=("region_name", lambda values: " | ".join(values)),
        unique_sizes=("nested_zip_size_bytes", "nunique"),
        unique_crc_values=("nested_zip_crc", "nunique"),
        all_valid=("nested_zip_valid", "all"),
        total_gml_files=("gml_xml_files", "sum")
    )
)

he_duplicate_details_path = REPORT_DIR / "lod2_duplicate_municipality_filenames_HE.csv"
he_duplicate_details.to_csv(he_duplicate_details_path, index=False)

print("=" * 70)
print("HESSE DUPLICATE MUNICIPALITY-FILENAME CHECK")
print("=" * 70)
print(f"\nMunicipality packages       : {len(he_municipalities):,}")
print(f"Unique ZIP filenames        : {he_municipalities['municipality_zip'].nunique():,}")
print(f"Duplicated filename values  : {he_duplicate_details['municipality_zip'].nunique():,}")
print(f"Records using those names   : {len(he_duplicate_details):,}")

print("\nDuplicate package details:")
display(he_duplicate_details)

print("\nFilename-level comparison:")
display(he_duplicate_comparison)

print(f"\nReport saved to: {he_duplicate_details_path}")

HESSE DUPLICATE MUNICIPALITY-FILENAME CHECK

Municipality packages       : 483
Unique ZIP filenames        : 481
Duplicated filename values  : 2
Records using those names   : 4

Duplicate package details:


,municipality_zip,municipality_name,region_name,regional_archive,source_key,nested_zip_size_bytes,nested_zip_crc,nested_zip_members,gml_xml_files,gml_filenames,nested_zip_valid
0,Griesheim-LoD2.zip,Griesheim,Kreisfreie Stadt Frankfurt,PKT_Kreisfreie Stadt Frankfurt.zip,PKT_Kreisfreie Stadt Frankfurt.zip::Griesheim-...,15793217,2792948290,2,1,Griesheim-LoD2.gml,True
1,Griesheim-LoD2.zip,Griesheim,Landkreis Darmstadt-Dieburg,PKT_Landkreis Darmstadt-Dieburg.zip,PKT_Landkreis Darmstadt-Dieburg.zip::Griesheim...,40766588,2538775088,2,1,Griesheim-LoD2.gml,True
2,Kalbach-LoD2.zip,Kalbach,Kreisfreie Stadt Frankfurt,PKT_Kreisfreie Stadt Frankfurt.zip,PKT_Kreisfreie Stadt Frankfurt.zip::Kalbach-Lo...,14791412,3521124241,2,1,Kalbach-LoD2.gml,True
3,Kalbach-LoD2.zip,Kalbach,Landkreis Fulda,PKT_Landkreis Fulda.zip,PKT_Landkreis Fulda.zip::Kalbach-LoD2.zip,17015219,2181407538,2,1,Kalbach-LoD2.gml,True



Filename-level comparison:


,municipality_zip,occurrences,regions,unique_sizes,unique_crc_values,all_valid,total_gml_files
0,Griesheim-LoD2.zip,2,Kreisfreie Stadt Frankfurt | Landkreis Darmsta...,2,2,True,2
1,Kalbach-LoD2.zip,2,Kreisfreie Stadt Frankfurt | Landkreis Fulda,2,2,True,2



Report saved to: /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_duplicate_municipality_filenames_HE.csv


In [88]:
# ============================================================
# 79 — PROCESS ALL HESSE LoD2 MUNICIPALITY ARCHIVES
# ============================================================

HE_OUTPUT_DIR = EXTRACTED_DIR / "HE"
HE_PART_DIR = HE_OUTPUT_DIR / "parts"
HE_MANIFEST_DIR = HE_OUTPUT_DIR / "manifests"

HE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HE_PART_DIR.mkdir(parents=True, exist_ok=True)
HE_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

HE_BATCH_SIZE = 15
he_batch_starts = list(range(0, len(he_municipalities), HE_BATCH_SIZE))

print("=" * 70)
print("HESSE PROCESSING SETUP")
print("=" * 70)
print(f"\nRegional archives       : {len(he_regional_archives):,}")
print(f"Municipality ZIP files  : {len(he_municipalities):,}")
print(f"Unique source keys      : {he_municipalities['source_key'].nunique():,}")
print(f"Batch size              : {HE_BATCH_SIZE:,}")
print(f"Expected Parquet parts  : {len(he_batch_starts):,}")

if he_municipalities["source_key"].duplicated().any():
    raise ValueError("Duplicate Hesse source keys detected.")

he_batch_summaries = []

for batch_start in tqdm(he_batch_starts, desc="Processing Hesse batches"):
    batch_number = batch_start // HE_BATCH_SIZE + 1
    batch_sources = he_municipalities.iloc[batch_start:batch_start + HE_BATCH_SIZE].copy()

    part_path = HE_PART_DIR / f"lod2_buildings_HE_part_{batch_number:04d}.parquet"
    manifest_path = HE_MANIFEST_DIR / f"lod2_manifest_HE_part_{batch_number:04d}.csv"

    # Resume only when this exact batch has already been completed
    if manifest_path.exists():
        existing_manifest = pd.read_csv(manifest_path)
        expected_keys = set(batch_sources["source_key"])
        existing_keys = set(existing_manifest["source_key"])

        manifest_complete = (
            expected_keys == existing_keys
            and len(existing_manifest) == len(batch_sources)
            and not existing_manifest["status"].eq("failed").any()
        )

        output_complete = existing_manifest["buildings"].sum() == 0 or part_path.exists()

        if manifest_complete and output_complete:
            he_batch_summaries.append({
                "batch": batch_number,
                "municipality_packages": len(existing_manifest),
                "processed_packages": existing_manifest["status"].eq("processed").sum(),
                "empty_packages": existing_manifest["status"].eq("empty").sum(),
                "failed_packages": existing_manifest["status"].eq("failed").sum(),
                "gml_files": existing_manifest["gml_files"].sum(),
                "buildings": existing_manifest["buildings"].sum(),
                "status": "already_processed"
            })
            continue

    if part_path.exists():
        part_path.unlink()

    batch_parts = []
    batch_manifest = []

    for outer_path, outer_sources in batch_sources.groupby("regional_archive_path", sort=False):
        with ZipFile(outer_path, "r") as outer_archive:
            for row in outer_sources.itertuples(index=False):
                try:
                    nested_content = outer_archive.read(row.municipality_zip)

                    with ZipFile(BytesIO(nested_content), "r") as nested_archive:
                        gml_members = [name for name in nested_archive.namelist() if name.lower().endswith((".gml", ".xml"))]

                        if not gml_members:
                            batch_manifest.append({
                                "source_key": row.source_key,
                                "region_name": row.region_name,
                                "municipality_name": row.municipality_name,
                                "regional_archive": row.regional_archive,
                                "municipality_zip": row.municipality_zip,
                                "status": "empty",
                                "gml_files": 0,
                                "buildings": 0,
                                "missing_heights": 0,
                                "missing_geometries": 0,
                                "error": None
                            })
                            continue

                        municipality_parts = []

                        for gml_member in gml_members:
                            with nested_archive.open(gml_member) as stream:
                                tile_gdf = extract_citygml_stream_multipart(
                                    stream,
                                    Path(gml_member).name,
                                    "EPSG:25832",
                                    HE_BUILDING_TAG,
                                    HE_BUILDING_PART_TAG
                                )

                            if tile_gdf.empty:
                                continue

                            tile_gdf["state_code"] = "HE"
                            tile_gdf["source_state"] = "Hesse"
                            tile_gdf["source_crs"] = "EPSG:25832"
                            tile_gdf["citygml_version"] = "1.0"
                            tile_gdf["height_source"] = tile_gdf["height_method"]
                            tile_gdf["source_region"] = row.region_name
                            tile_gdf["source_municipality"] = row.municipality_name
                            tile_gdf["source_archive"] = row.regional_archive
                            tile_gdf["source_nested_archive"] = row.municipality_zip
                            tile_gdf["source_member"] = gml_member
                            tile_gdf["source_path"] = f"{row.regional_archive}::{row.municipality_zip}::{gml_member}"

                            municipality_parts.append(tile_gdf)

                        if municipality_parts:
                            municipality_gdf = gpd.GeoDataFrame(pd.concat(municipality_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
                            batch_parts.append(municipality_gdf)

                            batch_manifest.append({
                                "source_key": row.source_key,
                                "region_name": row.region_name,
                                "municipality_name": row.municipality_name,
                                "regional_archive": row.regional_archive,
                                "municipality_zip": row.municipality_zip,
                                "status": "processed",
                                "gml_files": len(gml_members),
                                "buildings": len(municipality_gdf),
                                "missing_heights": municipality_gdf["measured_height_m"].isna().sum(),
                                "missing_geometries": municipality_gdf.geometry.isna().sum(),
                                "error": None
                            })
                        else:
                            batch_manifest.append({
                                "source_key": row.source_key,
                                "region_name": row.region_name,
                                "municipality_name": row.municipality_name,
                                "regional_archive": row.regional_archive,
                                "municipality_zip": row.municipality_zip,
                                "status": "empty",
                                "gml_files": len(gml_members),
                                "buildings": 0,
                                "missing_heights": 0,
                                "missing_geometries": 0,
                                "error": None
                            })

                except Exception as error:
                    batch_manifest.append({
                        "source_key": row.source_key,
                        "region_name": row.region_name,
                        "municipality_name": row.municipality_name,
                        "regional_archive": row.regional_archive,
                        "municipality_zip": row.municipality_zip,
                        "status": "failed",
                        "gml_files": 0,
                        "buildings": 0,
                        "missing_heights": 0,
                        "missing_geometries": 0,
                        "error": str(error)
                    })

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    batch_manifest_df = pd.DataFrame(batch_manifest)
    batch_manifest_df.to_csv(manifest_path, index=False)

    he_batch_summaries.append({
        "batch": batch_number,
        "municipality_packages": len(batch_manifest_df),
        "processed_packages": batch_manifest_df["status"].eq("processed").sum(),
        "empty_packages": batch_manifest_df["status"].eq("empty").sum(),
        "failed_packages": batch_manifest_df["status"].eq("failed").sum(),
        "gml_files": batch_manifest_df["gml_files"].sum(),
        "buildings": batch_manifest_df["buildings"].sum(),
        "status": "processed"
    })

    del batch_parts, batch_manifest, batch_manifest_df
    gc.collect()

# ------------------------------------------------------------
# Combine batch manifests
# ------------------------------------------------------------

he_batch_summary_df = pd.DataFrame(he_batch_summaries)
he_manifest_files = sorted(HE_MANIFEST_DIR.glob("lod2_manifest_HE_part_*.csv"))
he_manifests = pd.concat([pd.read_csv(path) for path in he_manifest_files], ignore_index=True)
he_part_files = sorted(HE_PART_DIR.glob("lod2_buildings_HE_part_*.parquet"))

he_batch_summary_path = REPORT_DIR / "lod2_batch_summary_HE.csv"
he_manifest_path = REPORT_DIR / "lod2_processing_manifest_HE.csv"

he_batch_summary_df.to_csv(he_batch_summary_path, index=False)
he_manifests.to_csv(he_manifest_path, index=False)

print("=" * 70)
print("HESSE LoD2 PROCESSING SUMMARY")
print("=" * 70)
print(f"\nMunicipality packages recorded : {len(he_manifests):,}")
print(f"Packages with buildings        : {he_manifests['status'].eq('processed').sum():,}")
print(f"Empty packages                 : {he_manifests['status'].eq('empty').sum():,}")
print(f"Failed packages                : {he_manifests['status'].eq('failed').sum():,}")
print(f"CityGML files processed        : {he_manifests['gml_files'].sum():,}")
print(f"Buildings extracted            : {he_manifests['buildings'].sum():,}")
print(f"Missing heights                : {he_manifests['missing_heights'].sum():,}")
print(f"Missing geometries             : {he_manifests['missing_geometries'].sum():,}")
print(f"Parquet parts created          : {len(he_part_files):,}")
print(f"Part directory                 : {HE_PART_DIR}")
print(f"Combined manifest              : {he_manifest_path}")

display(he_manifests["status"].value_counts().rename_axis("status").reset_index(name="packages"))

if he_manifests["status"].eq("failed").any():
    display(he_manifests[he_manifests["status"] == "failed"].head(30))

HESSE PROCESSING SETUP

Regional archives       : 27
Municipality ZIP files  : 483
Unique source keys      : 483
Batch size              : 15
Expected Parquet parts  : 33


Processing Hesse batches: 100%|█████████████████████████████████████████████████| 33/33 [1:04:50<00:00, 117.88s/it]

HESSE LoD2 PROCESSING SUMMARY

Municipality packages recorded : 483
Packages with buildings        : 483
Empty packages                 : 0
Failed packages                : 0
CityGML files processed        : 483
Buildings extracted            : 4,981,859
Missing heights                : 0
Missing geometries             : 72
Parquet parts created          : 33
Part directory                 : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/HE/parts
Combined manifest              : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_processing_manifest_HE.csv


,status,packages
0,processed,483


In [89]:
# ============================================================
# 80 — VALIDATE AND SUMMARISE HESSE RAW LoD2 DATA
# ============================================================

he_expected_sources = set(he_municipalities["source_key"])
he_manifest_sources = set(he_manifests["source_key"])
he_missing_sources = sorted(he_expected_sources - he_manifest_sources)
he_unexpected_sources = sorted(he_manifest_sources - he_expected_sources)
he_duplicate_manifest_sources = he_manifests["source_key"].duplicated().sum()

he_summary_records = []
he_height_method_counts = Counter()
he_footprint_counts = Counter()
he_id_counts = Counter()
he_missing_geometry_records = []

for part_path in tqdm(he_part_files, desc="Summarising Hesse parts"):
    part = pd.read_parquet(
        part_path,
        columns=[
            "lod2_id",
            "measured_height_m",
            "height_method",
            "footprint_method",
            "source_region",
            "source_municipality",
            "source_archive",
            "source_nested_archive",
            "source_member",
            "source_path",
            "geometry"
        ]
    )

    he_id_counts.update(part["lod2_id"].dropna().astype(str))
    he_height_method_counts.update(part["height_method"].fillna("missing"))
    he_footprint_counts.update(part["footprint_method"].fillna("missing"))

    missing_geometry = part[part["geometry"].isna()].drop(columns="geometry").copy()

    if not missing_geometry.empty:
        missing_geometry["part_file"] = part_path.name
        he_missing_geometry_records.append(missing_geometry)

    he_summary_records.append({
        "buildings": len(part),
        "missing_ids": part["lod2_id"].isna().sum(),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    del part, missing_geometry
    gc.collect()

he_part_stats = pd.DataFrame(he_summary_records)

he_duplicate_ids = {lod2_id: count for lod2_id, count in he_id_counts.items() if count > 1}
he_duplicate_id_counts = pd.DataFrame(
    [{"lod2_id": lod2_id, "record_count": count} for lod2_id, count in he_duplicate_ids.items()]
).sort_values(["record_count", "lod2_id"], ascending=[False, True]) if he_duplicate_ids else pd.DataFrame(columns=["lod2_id", "record_count"])

he_unique_ids = len(he_id_counts)
he_extra_duplicate_rows = sum(count - 1 for count in he_duplicate_ids.values())

he_height_method_summary = pd.DataFrame(
    [{"height_method": method, "building_count": count} for method, count in he_height_method_counts.items()]
).sort_values("building_count", ascending=False)

he_footprint_summary = pd.DataFrame(
    [{"footprint_method": method, "building_count": count} for method, count in he_footprint_counts.items()]
).sort_values("building_count", ascending=False)

he_summary = pd.DataFrame([{
    "state_code": "HE",
    "state_name": "Hesse",
    "regional_archives": len(he_regional_archives),
    "municipality_packages_expected": len(he_expected_sources),
    "municipality_packages_recorded": len(he_manifest_sources),
    "packages_with_buildings": he_manifests["status"].eq("processed").sum(),
    "empty_packages": he_manifests["status"].eq("empty").sum(),
    "failed_packages": he_manifests["status"].eq("failed").sum(),
    "missing_source_packages": len(he_missing_sources),
    "unexpected_source_packages": len(he_unexpected_sources),
    "duplicate_manifest_sources": he_duplicate_manifest_sources,
    "citygml_files": he_manifests["gml_files"].sum(),
    "parquet_parts": len(he_part_files),
    "buildings": he_part_stats["buildings"].sum(),
    "unique_ids": he_unique_ids,
    "duplicated_ids": len(he_duplicate_ids),
    "extra_duplicate_rows": he_extra_duplicate_rows,
    "missing_ids": he_part_stats["missing_ids"].sum(),
    "measured_heights": he_part_stats["measured_heights"].sum(),
    "missing_heights": he_part_stats["missing_heights"].sum(),
    "usable_geometries": he_part_stats["usable_geometries"].sum(),
    "missing_geometries": he_part_stats["missing_geometries"].sum(),
    "minimum_height_m": he_part_stats["minimum_height_m"].min(),
    "maximum_height_m": he_part_stats["maximum_height_m"].max(),
    "output_directory": str(HE_PART_DIR)
}])

he_summary_path = REPORT_DIR / "lod2_extraction_summary_HE.csv"
he_height_summary_path = REPORT_DIR / "lod2_height_methods_HE.csv"
he_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_HE.csv"
he_duplicate_ids_path = REPORT_DIR / "lod2_duplicate_id_counts_HE.csv"
he_missing_geometry_path = REPORT_DIR / "lod2_missing_geometry_records_HE.csv"

he_summary.to_csv(he_summary_path, index=False)
he_height_method_summary.to_csv(he_height_summary_path, index=False)
he_footprint_summary.to_csv(he_footprint_summary_path, index=False)
he_duplicate_id_counts.to_csv(he_duplicate_ids_path, index=False)

if he_missing_geometry_records:
    he_missing_geometry_df = pd.concat(he_missing_geometry_records, ignore_index=True)
else:
    he_missing_geometry_df = pd.DataFrame()

he_missing_geometry_df.to_csv(he_missing_geometry_path, index=False)

print("=" * 70)
print("HESSE FINAL LoD2 VALIDATION")
print("=" * 70)

display(he_summary)

print("\nHeight methods:")
display(he_height_method_summary)

print("\nFootprint methods:")
display(he_footprint_summary)

print(f"\nState summary saved to           : {he_summary_path}")
print(f"Height summary saved to          : {he_height_summary_path}")
print(f"Footprint summary saved to       : {he_footprint_summary_path}")
print(f"Duplicate-ID counts saved to     : {he_duplicate_ids_path}")
print(f"Missing-geometry records saved to: {he_missing_geometry_path}")

if he_missing_sources:
    print("\nMissing source packages:")
    display(pd.DataFrame({"source_key": he_missing_sources}).head(30))

if not he_duplicate_id_counts.empty:
    print("\nMost frequent duplicate IDs:")
    display(he_duplicate_id_counts.head(30))

if not he_missing_geometry_df.empty:
    print("\nBuildings with missing geometry:")
    display(he_missing_geometry_df.head(30))

Summarising Hesse parts: 100%|█████████████████████████████████████████████████████| 33/33 [00:37<00:00,  1.15s/it]


HESSE FINAL LoD2 VALIDATION


,state_code,state_name,regional_archives,municipality_packages_expected,municipality_packages_recorded,packages_with_buildings,empty_packages,failed_packages,missing_source_packages,unexpected_source_packages,...,duplicated_ids,extra_duplicate_rows,missing_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,HE,Hesse,27,483,483,483,0,0,0,0,...,1389,1402,0,4981859,0,4981787,72,-0.38,261.813,/fast/home/o-olajuyigbe/data/germany_lod2/extr...



Height methods:


,height_method,building_count
0,building_measured_height,4280268
1,maximum_building_part_height,701591


...

,footprint_method,building_count
0,ground_surface,4981772
2,missing,72
1,wall_bottom_convex_hull,15



State summary saved to           : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_HE.csv
Height summary saved to          : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_height_methods_HE.csv
Footprint summary saved to       : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_HE.csv
Duplicate-ID counts saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_duplicate_id_counts_HE.csv
Missing-geometry records saved to: /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_missing_geometry_records_HE.csv

Most frequent duplicate IDs:


,lod2_id,record_count
1051,DEHE06060000NDC7,3
614,DEHE06200001MjsH,3
1315,DEHE06200001OQzl,3
93,DEHE06200001Uu2S,3
326,DEHE06200001c3RK,3
260,DEHE06200001cS9U,3
507,DEHE06200001dRJ2,3
274,DEHE06200001en5n,3
720,DEHE06200001gQ76,3
363,DEHE06200001hXmc,3



Buildings with missing geometry:


,lod2_id,measured_height_m,height_method,footprint_method,source_region,source_municipality,source_archive,source_nested_archive,source_member,source_path,part_file
0,DEHE06200002ja1m,6.459,building_measured_height,missing,Hochtaunus,Kronberg im Taunus,PKT_Hochtaunus.zip,Kronberg_im_Taunus-LoD2.zip,Kronberg im Taunus-LoD2.gml,PKT_Hochtaunus.zip::Kronberg_im_Taunus-LoD2.zi...,lod2_buildings_HE_part_0001.parquet
1,DEHE06200000nc5m,5.101,building_measured_height,missing,Hochtaunus,Kronberg im Taunus,PKT_Hochtaunus.zip,Kronberg_im_Taunus-LoD2.zip,Kronberg im Taunus-LoD2.gml,PKT_Hochtaunus.zip::Kronberg_im_Taunus-LoD2.zi...,lod2_buildings_HE_part_0001.parquet
2,DEHE06200002lo7j,4.639,building_measured_height,missing,Hochtaunus,Kronberg im Taunus,PKT_Hochtaunus.zip,Kronberg_im_Taunus-LoD2.zip,Kronberg im Taunus-LoD2.gml,PKT_Hochtaunus.zip::Kronberg_im_Taunus-LoD2.zi...,lod2_buildings_HE_part_0001.parquet
3,DEHE06120001lOzI,0.009,building_measured_height,missing,Kreisfreie Stadt Darmstadt,Darmstadt,PKT_Kreisfreie Stadt Darmstadt.zip,Darmstadt-LoD2.zip,Darmstadt-LoD2.gml,PKT_Kreisfreie Stadt Darmstadt.zip::Darmstadt-...,lod2_buildings_HE_part_0001.parquet
4,DEHE06120001kGoN,0.009,building_measured_height,missing,Kreisfreie Stadt Darmstadt,Darmstadt,PKT_Kreisfreie Stadt Darmstadt.zip,Darmstadt-LoD2.zip,Darmstadt-LoD2.gml,PKT_Kreisfreie Stadt Darmstadt.zip::Darmstadt-...,lod2_buildings_HE_part_0001.parquet
5,DEHE06120001kNCj,0.003,building_measured_height,missing,Kreisfreie Stadt Darmstadt,Darmstadt,PKT_Kreisfreie Stadt Darmstadt.zip,Darmstadt-LoD2.zip,Darmstadt-LoD2.gml,PKT_Kreisfreie Stadt Darmstadt.zip::Darmstadt-...,lod2_buildings_HE_part_0001.parquet
6,DEHE06120001kHrU,0.010,building_measured_height,missing,Kreisfreie Stadt Darmstadt,Darmstadt,PKT_Kreisfreie Stadt Darmstadt.zip,Darmstadt-LoD2.zip,Darmstadt-LoD2.gml,PKT_Kreisfreie Stadt Darmstadt.zip::Darmstadt-...,lod2_buildings_HE_part_0001.parquet
7,DEHE06120001kPqM,0.004,building_measured_height,missing,Kreisfreie Stadt Darmstadt,Darmstadt,PKT_Kreisfreie Stadt Darmstadt.zip,Darmstadt-LoD2.zip,Darmstadt-LoD2.gml,PKT_Kreisfreie Stadt Darmstadt.zip::Darmstadt-...,lod2_buildings_HE_part_0001.parquet
8,DEHE06120001mJJL,0.004,building_measured_height,missing,Kreisfreie Stadt Darmstadt,Darmstadt,PKT_Kreisfreie Stadt Darmstadt.zip,Darmstadt-LoD2.zip,Darmstadt-LoD2.gml,PKT_Kreisfreie Stadt Darmstadt.zip::Darmstadt-...,lod2_buildings_HE_part_0001.parquet
9,DEHE06120001nJww,0.003,building_measured_height,missing,Kreisfreie Stadt Darmstadt,Darmstadt,PKT_Kreisfreie Stadt Darmstadt.zip,Darmstadt-LoD2.zip,Darmstadt-LoD2.gml,PKT_Kreisfreie Stadt Darmstadt.zip::Darmstadt-...,lod2_buildings_HE_part_0001.parquet


In [90]:
# ============================================================
# 81 — SAVE DETAILED HESSE ANOMALY REPORTS
# ============================================================

HE_LOW_HEIGHT_LIMIT = 1.0
HE_EXTREME_HEIGHT_LIMIT = 300.0

he_duplicate_id_set = set(he_duplicate_id_counts["lod2_id"])
he_duplicate_records = []
he_height_anomalies = []

columns_needed = [
    "lod2_id",
    "creation_date",
    "function",
    "roof_type",
    "measured_height_m",
    "parent_height_m",
    "maximum_part_height_m",
    "building_part_count",
    "height_method",
    "storeys_above_ground",
    "source_region",
    "source_municipality",
    "source_archive",
    "source_nested_archive",
    "source_member",
    "source_path",
    "footprint_method",
    "geometry"
]

for part_path in tqdm(he_part_files, desc="Collecting Hesse anomaly records"):
    part = gpd.read_parquet(part_path, columns=columns_needed)

    duplicate_part = part[part["lod2_id"].isin(he_duplicate_id_set)].copy()

    if not duplicate_part.empty:
        duplicate_part["part_file"] = part_path.name
        duplicate_part["geometry_area_m2"] = duplicate_part.geometry.area
        duplicate_part["geometry_wkb"] = duplicate_part.geometry.apply(
            lambda geom: geom.wkb_hex if geom is not None else None
        )
        he_duplicate_records.append(pd.DataFrame(duplicate_part.drop(columns="geometry")))

    anomalous_part = part[
        part["measured_height_m"].isna()
        | (part["measured_height_m"] < HE_LOW_HEIGHT_LIMIT)
        | (part["measured_height_m"] > HE_EXTREME_HEIGHT_LIMIT)
        | part.geometry.isna()
    ].copy()

    if not anomalous_part.empty:
        anomalous_part["part_file"] = part_path.name
        anomalous_part["geometry_area_m2"] = anomalous_part.geometry.area
        anomalous_part["height_issue"] = np.select(
            [
                anomalous_part["measured_height_m"].isna(),
                anomalous_part["measured_height_m"] <= 0,
                anomalous_part["measured_height_m"].between(0, HE_LOW_HEIGHT_LIMIT, inclusive="left"),
                anomalous_part["measured_height_m"] > HE_EXTREME_HEIGHT_LIMIT
            ],
            [
                "missing_height",
                "non_positive_height",
                "height_below_1m",
                "height_above_300m"
            ],
            default="height_not_flagged"
        )
        anomalous_part["geometry_issue"] = np.where(
            anomalous_part.geometry.isna(),
            "missing_geometry",
            "geometry_present"
        )
        anomalous_part["geometry_wkb"] = anomalous_part.geometry.apply(
            lambda geom: geom.wkb_hex if geom is not None else None
        )
        he_height_anomalies.append(pd.DataFrame(anomalous_part.drop(columns="geometry")))

    del part, duplicate_part, anomalous_part
    gc.collect()

he_duplicate_records_df = (
    pd.concat(he_duplicate_records, ignore_index=True)
    if he_duplicate_records
    else pd.DataFrame()
)

he_height_anomalies_df = (
    pd.concat(he_height_anomalies, ignore_index=True)
    if he_height_anomalies
    else pd.DataFrame()
)

# ------------------------------------------------------------
# Compare records belonging to each duplicated ID
# ------------------------------------------------------------

if not he_duplicate_records_df.empty:
    he_duplicate_comparison = (
        he_duplicate_records_df
        .groupby("lod2_id", as_index=False)
        .agg(
            record_count=("lod2_id", "size"),
            unique_regions=("source_region", "nunique"),
            unique_municipalities=("source_municipality", "nunique"),
            unique_source_paths=("source_path", "nunique"),
            unique_heights=("measured_height_m", "nunique"),
            unique_geometries=("geometry_wkb", "nunique"),
            minimum_height_m=("measured_height_m", "min"),
            maximum_height_m=("measured_height_m", "max")
        )
    )

    he_duplicate_comparison["exact_repeated_record"] = (
        he_duplicate_comparison["unique_heights"].eq(1)
        & he_duplicate_comparison["unique_geometries"].eq(1)
    )
else:
    he_duplicate_comparison = pd.DataFrame()

# ------------------------------------------------------------
# Summarise anomaly categories
# ------------------------------------------------------------

if not he_height_anomalies_df.empty:
    he_anomaly_summary = (
        he_height_anomalies_df
        .groupby(["height_issue", "geometry_issue"], dropna=False)
        .agg(
            building_records=("lod2_id", "size"),
            unique_ids=("lod2_id", "nunique"),
            minimum_height_m=("measured_height_m", "min"),
            maximum_height_m=("measured_height_m", "max")
        )
        .reset_index()
    )
else:
    he_anomaly_summary = pd.DataFrame()

# ------------------------------------------------------------
# Save reports
# ------------------------------------------------------------

he_duplicate_records_path = REPORT_DIR / "lod2_duplicate_records_HE.csv"
he_duplicate_comparison_path = REPORT_DIR / "lod2_duplicate_comparison_HE.csv"
he_height_anomalies_path = REPORT_DIR / "lod2_height_geometry_anomalies_HE.csv"
he_anomaly_summary_path = REPORT_DIR / "lod2_anomaly_summary_HE.csv"

he_duplicate_records_df.to_csv(he_duplicate_records_path, index=False)
he_duplicate_comparison.to_csv(he_duplicate_comparison_path, index=False)
he_height_anomalies_df.to_csv(he_height_anomalies_path, index=False)
he_anomaly_summary.to_csv(he_anomaly_summary_path, index=False)

print("=" * 70)
print("HESSE ANOMALY REPORT SUMMARY")
print("=" * 70)
print(f"\nDuplicated IDs              : {len(he_duplicate_id_set):,}")
print(f"Duplicate records collected : {len(he_duplicate_records_df):,}")
print(f"Exact repeated IDs          : {he_duplicate_comparison['exact_repeated_record'].sum() if not he_duplicate_comparison.empty else 0:,}")
print(f"Conflicting duplicate IDs   : {(~he_duplicate_comparison['exact_repeated_record']).sum() if not he_duplicate_comparison.empty else 0:,}")
print(f"Anomalous records collected : {len(he_height_anomalies_df):,}")

print("\nHeight and geometry anomaly summary:")
display(he_anomaly_summary)

print("\nDuplicate comparison:")
display(he_duplicate_comparison.sort_values(["record_count", "lod2_id"], ascending=[False, True]).head(30))

print(f"\nDuplicate records saved to : {he_duplicate_records_path}")
print(f"Duplicate comparison saved : {he_duplicate_comparison_path}")
print(f"Anomaly records saved to   : {he_height_anomalies_path}")
print(f"Anomaly summary saved to   : {he_anomaly_summary_path}")

HESSE ANOMALY REPORT SUMMARY

Duplicated IDs              : 1,389
Duplicate records collected : 2,791
Exact repeated IDs          : 1,389
Conflicting duplicate IDs   : 0
Anomalous records collected : 18,307

Height and geometry anomaly summary:


,height_issue,geometry_issue,building_records,unique_ids,minimum_height_m,maximum_height_m
0,height_below_1m,geometry_present,18233,18225,0.003,0.999
1,height_below_1m,missing_geometry,66,66,0.001,0.010
2,height_not_flagged,missing_geometry,5,5,2.786,6.459
3,non_positive_height,geometry_present,2,2,-0.380,-0.133
4,non_positive_height,missing_geometry,1,1,0.000,0.000



Duplicate comparison:


,lod2_id,record_count,unique_regions,unique_municipalities,unique_source_paths,unique_heights,unique_geometries,minimum_height_m,maximum_height_m,exact_repeated_record
70,DEHE06060000NDC7,3,1,3,3,1,1,7.743,7.743,True
419,DEHE06200001MjsH,3,1,3,3,1,1,18.203,18.203,True
428,DEHE06200001OQzl,3,1,3,3,1,1,6.828,6.828,True
451,DEHE06200001Uu2S,3,1,3,3,1,1,6.494,6.494,True
570,DEHE06200001c3RK,3,1,3,3,1,1,18.445,18.445,True
628,DEHE06200001cS9U,3,1,3,3,1,1,23.214,23.214,True
798,DEHE06200001dRJ2,3,1,3,3,1,1,21.581,21.581,True
963,DEHE06200001en5n,3,1,3,3,1,1,23.634,23.634,True
995,DEHE06200001gQ76,3,1,3,3,1,1,24.581,24.581,True
1017,DEHE06200001hXmc,3,1,3,3,1,1,10.090,10.090,True



Duplicate records saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_duplicate_records_HE.csv
Duplicate comparison saved : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_duplicate_comparison_HE.csv
Anomaly records saved to   : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_height_geometry_anomalies_HE.csv
Anomaly summary saved to   : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_anomaly_summary_HE.csv


In [92]:
# ============================================================
# 82b — CORRECT BAVARIA METALINK FILE DISCOVERY
# ============================================================

from urllib.parse import urlparse, unquote
from email.message import Message

def filename_from_content_disposition(value):
    if not value:
        return None

    message = Message()
    message["content-disposition"] = value
    filename = message.get_filename()
    return Path(filename).name if filename else None

# ------------------------------------------------------------
# 1. Rebuild the Metalink table without assuming .zip names
# ------------------------------------------------------------

by_file_records = []

for file_element in by_meta4_root.xpath(".//*[local-name()='file']"):
    metalink_name = file_element.get("name")

    size_values = file_element.xpath("./*[local-name()='size']/text()")
    size_bytes = parse_number(size_values[0]) if size_values else np.nan

    urls = [
        str(value).strip()
        for value in file_element.xpath("./*[local-name()='url']/text()")
        if str(value).strip()
    ]

    hashes = {}

    for hash_element in file_element.xpath("./*[local-name()='hash']"):
        hash_type = str(hash_element.get("type", "unknown")).lower()
        hash_value = (hash_element.text or "").strip()

        if hash_value:
            hashes[hash_type] = hash_value

    https_urls = [url for url in urls if url.lower().startswith("https://")]
    selected_url = https_urls[0] if https_urls else (urls[0] if urls else None)

    url_filename = (
        Path(unquote(urlparse(selected_url).path)).name
        if selected_url
        else None
    )

    by_file_records.append({
        "metalink_name": metalink_name,
        "metalink_filename": Path(metalink_name).name if metalink_name else None,
        "download_url": selected_url,
        "url_filename": url_filename,
        "mirror_count": len(urls),
        "size_bytes": size_bytes,
        "size_mb": size_bytes / 1024**2 if pd.notna(size_bytes) else np.nan,
        "sha256": hashes.get("sha-256") or hashes.get("sha256"),
        "sha1": hashes.get("sha-1") or hashes.get("sha1"),
        "md5": hashes.get("md5")
    })

by_downloads = pd.DataFrame(by_file_records)

by_downloads["metalink_suffix"] = (
    by_downloads["metalink_filename"]
    .fillna("")
    .map(lambda value: Path(value).suffix.lower())
)

by_downloads["url_suffix"] = (
    by_downloads["url_filename"]
    .fillna("")
    .map(lambda value: Path(value).suffix.lower())
)

by_lod2_downloads = (
    by_downloads[
        by_downloads["download_url"].notna()
        & by_downloads["metalink_filename"].notna()
    ]
    .drop_duplicates(subset="download_url")
    .sort_values("metalink_filename")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Inspect five representative HTTP responses without
#    downloading the complete files
# ------------------------------------------------------------

sample_indices = sorted(set([
    0,
    len(by_lod2_downloads) // 4,
    len(by_lod2_downloads) // 2,
    3 * len(by_lod2_downloads) // 4,
    len(by_lod2_downloads) - 1
]))

by_response_records = []

for index in tqdm(sample_indices, desc="Inspecting Bavaria tile responses"):
    row = by_lod2_downloads.iloc[index]

    try:
        with requests.get(
            row["download_url"],
            stream=True,
            timeout=(30, 300)
        ) as response:
            response.raise_for_status()

            first_chunk = next(
                response.iter_content(chunk_size=4096),
                b""
            )

            content_type = response.headers.get("content-type", "")
            content_disposition = response.headers.get(
                "content-disposition"
            )
            response_filename = filename_from_content_disposition(
                content_disposition
            )

            if first_chunk.startswith(b"PK\x03\x04"):
                detected_type = "zip"
            elif first_chunk.lstrip().startswith(b"<?xml") or b"<core:CityModel" in first_chunk:
                detected_type = "xml_citygml"
            else:
                detected_type = "unknown"

            by_response_records.append({
                "metalink_filename": row["metalink_filename"],
                "url_filename": row["url_filename"],
                "response_filename": response_filename,
                "content_type": content_type,
                "content_length": parse_number(
                    response.headers.get("content-length")
                ),
                "detected_type": detected_type,
                "first_bytes": repr(first_chunk[:20])
            })

    except Exception as error:
        by_response_records.append({
            "metalink_filename": row["metalink_filename"],
            "url_filename": row["url_filename"],
            "response_filename": None,
            "content_type": None,
            "content_length": np.nan,
            "detected_type": "failed",
            "first_bytes": None,
            "error": str(error)
        })

by_response_test = pd.DataFrame(by_response_records)

# ------------------------------------------------------------
# 3. Assign safe local filenames based on actual delivery type
# ------------------------------------------------------------

detected_types = set(
    by_response_test.loc[
        by_response_test["detected_type"].ne("failed"),
        "detected_type"
    ]
)

if detected_types == {"zip"}:
    by_lod2_downloads["filename"] = (
        by_lod2_downloads["url_filename"]
        .where(
            by_lod2_downloads["url_filename"]
            .fillna("")
            .str.lower()
            .str.endswith(".zip"),
            by_lod2_downloads["metalink_filename"]
            .map(lambda value: f"{Path(value).stem}.zip")
        )
    )
    BY_DELIVERY_TYPE = "zip"

elif detected_types == {"xml_citygml"}:
    by_lod2_downloads["filename"] = by_lod2_downloads[
        "metalink_filename"
    ]
    BY_DELIVERY_TYPE = "xml_citygml"

else:
    by_lod2_downloads["filename"] = by_lod2_downloads[
        "metalink_filename"
    ]
    BY_DELIVERY_TYPE = "mixed_or_unresolved"

by_download_list_path = BY_RAW_DIR / "by_lod2_download_list.csv"
by_lod2_downloads.to_csv(by_download_list_path, index=False)
by_response_test.to_csv(
    BY_RAW_DIR / "by_sample_response_types.csv",
    index=False
)

print("=" * 70)
print("BAVARIA CORRECTED METALINK DISCOVERY")
print("=" * 70)
print(f"\nAll Metalink entries   : {len(by_downloads):,}")
print(f"Usable download entries: {len(by_lod2_downloads):,}")
print(f"Unique download URLs   : {by_lod2_downloads['download_url'].nunique():,}")
print(f"Unique local filenames : {by_lod2_downloads['filename'].nunique():,}")
print(f"Duplicate filenames    : {by_lod2_downloads['filename'].duplicated().sum():,}")
print(f"Missing URLs           : {by_lod2_downloads['download_url'].isna().sum():,}")
print(f"Entries with size      : {by_lod2_downloads['size_bytes'].notna().sum():,}")
print(f"Estimated total size   : {by_lod2_downloads['size_bytes'].sum() / 1024**3:,.2f} GB")
print(f"Detected delivery type : {BY_DELIVERY_TYPE}")

print("\nMetalink filename suffixes:")
display(
    by_downloads["metalink_suffix"]
    .value_counts(dropna=False)
    .rename_axis("suffix")
    .reset_index(name="files")
)

print("\nURL filename suffixes:")
display(
    by_downloads["url_suffix"]
    .value_counts(dropna=False)
    .rename_axis("suffix")
    .reset_index(name="files")
)

print("\nRepresentative response types:")
display(by_response_test)

print("\nCorrected download examples:")
display(
    by_lod2_downloads[
        [
            "metalink_filename",
            "url_filename",
            "filename",
            "size_mb",
            "download_url"
        ]
    ].head(20)
)

print(f"\nCorrected list saved to: {by_download_list_path}")

Inspecting Bavaria tile responses: 100%|█████████████████████████████████████████████| 5/5 [00:00<00:00, 13.97it/s]


BAVARIA CORRECTED METALINK DISCOVERY

All Metalink entries   : 18,297
Usable download entries: 18,297
Unique download URLs   : 18,297
Unique local filenames : 18,297
Duplicate filenames    : 0
Missing URLs           : 0
Entries with size      : 18,297
Estimated total size   : 141.66 GB
Detected delivery type : xml_citygml

Metalink filename suffixes:


,suffix,files
0,.gml,18297



URL filename suffixes:


,suffix,files
0,.gml,18297



Representative response types:


,metalink_filename,url_filename,response_filename,content_type,content_length,detected_type,first_bytes
0,498_5542.gml,498_5542.gml,None,application/octet-stream,2470727.0,xml_citygml,"b'<?xml version=""1.0"" '"
1,618_5388.gml,618_5388.gml,None,application/octet-stream,20697508.0,xml_citygml,"b'<?xml version=""1.0"" '"
2,674_5362.gml,674_5362.gml,None,application/octet-stream,2884563.0,xml_citygml,"b'<?xml version=""1.0"" '"
3,732_5404.gml,732_5404.gml,None,application/octet-stream,695206.0,xml_citygml,"b'<?xml version=""1.0"" '"
4,854_5412.gml,854_5412.gml,None,application/octet-stream,1085.0,xml_citygml,"b'<?xml version=""1.0"" '"



Corrected download examples:


,metalink_filename,url_filename,filename,size_mb,download_url
0,498_5542.gml,498_5542.gml,498_5542.gml,2.356269,https://download1.bayernwolke.de/a/lod2/citygm...
1,498_5544.gml,498_5544.gml,498_5544.gml,15.458280,https://download1.bayernwolke.de/a/lod2/citygm...
2,498_5546.gml,498_5546.gml,498_5546.gml,3.222115,https://download1.bayernwolke.de/a/lod2/citygm...
3,498_5548.gml,498_5548.gml,498_5548.gml,0.023314,https://download1.bayernwolke.de/a/lod2/citygm...
4,500_5528.gml,500_5528.gml,500_5528.gml,0.001035,https://download1.bayernwolke.de/a/lod2/citygm...
5,500_5530.gml,500_5530.gml,500_5530.gml,0.155829,https://download1.bayernwolke.de/a/lod2/citygm...
6,500_5532.gml,500_5532.gml,500_5532.gml,0.001035,https://download1.bayernwolke.de/a/lod2/citygm...
7,500_5534.gml,500_5534.gml,500_5534.gml,0.001035,https://download1.bayernwolke.de/a/lod2/citygm...
8,500_5536.gml,500_5536.gml,500_5536.gml,0.371448,https://download1.bayernwolke.de/a/lod2/citygm...
9,500_5538.gml,500_5538.gml,500_5538.gml,0.001035,https://download1.bayernwolke.de/a/lod2/citygm...



Corrected list saved to: /fast/home/o-olajuyigbe/data/germany_lod2/raw/BY/by_lod2_download_list.csv


In [93]:
# ============================================================
# 83 — TEST REPRESENTATIVE BAVARIA LoD2 CITYGML TILES
# ============================================================

BY_TEST_DIR = BY_RAW_DIR / "test_tiles"
BY_TEST_DIR.mkdir(parents=True, exist_ok=True)

by_sample_indices = sorted(set([0, len(by_lod2_downloads) // 4, len(by_lod2_downloads) // 2, 3 * len(by_lod2_downloads) // 4, len(by_lod2_downloads) - 1]))
by_test_records = []

for index in tqdm(by_sample_indices, desc="Testing Bavaria CityGML tiles"):
    row = by_lod2_downloads.iloc[index]
    filename = row["filename"]

    if not Path(filename).suffix:
        filename = f"{filename}.gml"

    file_path = BY_TEST_DIR / filename

    if not file_path.exists():
        temp_path = file_path.with_suffix(file_path.suffix + ".part")

        with requests.get(row["download_url"], stream=True, timeout=(30, 1200)) as response:
            response.raise_for_status()

            with open(temp_path, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)

        temp_path.replace(file_path)

    with open(file_path, "rb") as stream:
        context = etree.iterparse(stream, events=("start",), huge_tree=True)
        event, root = next(context)
        building_namespace = root.nsmap.get("bldg")
        core_namespace = root.nsmap.get("core")
        del context

    srs_names = set()
    tag_counts = Counter()

    for event, element in etree.iterparse(file_path, events=("end",), huge_tree=True):
        local_name = etree.QName(element).localname

        if element.get("srsName"):
            srs_names.add(element.get("srsName"))

        if local_name in {"Building", "BuildingPart", "measuredHeight", "GroundSurface", "RoofSurface", "WallSurface", "cityObjectMember"}:
            tag_counts[local_name] += 1

        element.clear()

        while element.getprevious() is not None:
            del element.getparent()[0]

    by_test_records.append({
        "filename": file_path.name,
        "file_size_mb": file_path.stat().st_size / 1024**2,
        "building_namespace": building_namespace,
        "core_namespace": core_namespace,
        "srs_names": " | ".join(sorted(srs_names)),
        "buildings": tag_counts["Building"],
        "building_parts": tag_counts["BuildingPart"],
        "measured_heights": tag_counts["measuredHeight"],
        "ground_surfaces": tag_counts["GroundSurface"],
        "roof_surfaces": tag_counts["RoofSurface"],
        "wall_surfaces": tag_counts["WallSurface"],
        "city_object_members": tag_counts["cityObjectMember"]
    })

by_structure_test = pd.DataFrame(by_test_records)

print("=" * 70)
print("BAVARIA SAMPLE CITYGML STRUCTURE TEST")
print("=" * 70)

display(by_structure_test)

print("\nBuilding namespaces:")
display(
    by_structure_test["building_namespace"]
    .value_counts(dropna=False)
    .rename_axis("building_namespace")
    .reset_index(name="files")
)

print("\nCRS values:")
display(
    by_structure_test["srs_names"]
    .value_counts(dropna=False)
    .rename_axis("srs_names")
    .reset_index(name="files")
)

print("\nTotals:")
display(
    by_structure_test[
        [
            "buildings",
            "building_parts",
            "measured_heights",
            "ground_surfaces",
            "roof_surfaces",
            "wall_surfaces",
            "city_object_members"
        ]
    ].sum().to_frame("count")
)

Testing Bavaria CityGML tiles: 100%|█████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.64it/s]


BAVARIA SAMPLE CITYGML STRUCTURE TEST


,filename,file_size_mb,building_namespace,core_namespace,srs_names,buildings,building_parts,measured_heights,ground_surfaces,roof_surfaces,wall_surfaces,city_object_members
0,498_5542.gml,2.356269,http://www.opengis.net/citygml/building/1.0,None,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,159,0,159,159,314,1254,159
1,618_5388.gml,19.738682,http://www.opengis.net/citygml/building/1.0,None,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,1300,0,1300,1300,3164,9736,1300
2,674_5362.gml,2.750934,http://www.opengis.net/citygml/building/1.0,None,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,208,0,208,208,436,1307,208
3,732_5404.gml,0.663000,http://www.opengis.net/citygml/building/1.0,None,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,50,0,50,50,106,320,50
4,854_5412.gml,0.001035,http://www.opengis.net/citygml/building/1.0,http://www.opengis.net/citygml/1.0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,0,0,0,0,0,0,0



Building namespaces:


,building_namespace,files
0,http://www.opengis.net/citygml/building/1.0,5



CRS values:


,srs_names,files
0,urn:adv:crs:ETRS89_UTM32*DE_DHHN2016_NH,5



Totals:


,count
buildings,1717
building_parts,0
measured_heights,1717
ground_surfaces,1717
roof_surfaces,4020
wall_surfaces,12617
city_object_members,1717


In [94]:
# ============================================================
# 84 — TEST BAVARIA HEIGHT AND FOOTPRINT EXTRACTION
# ============================================================

BY_BLDG_NS = "http://www.opengis.net/citygml/building/1.0"
BY_BUILDING_TAG = f"{{{BY_BLDG_NS}}}Building"
BY_BUILDING_PART_TAG = f"{{{BY_BLDG_NS}}}BuildingPart"

by_parser_records = []
by_parser_parts = []

for row in tqdm(by_structure_test.itertuples(index=False), total=len(by_structure_test), desc="Parsing Bavaria sample tiles"):
    file_path = BY_TEST_DIR / row.filename

    with open(file_path, "rb") as stream:
        tile_gdf = extract_citygml_stream_multipart(
            stream,
            file_path.name,
            "EPSG:25832",
            BY_BUILDING_TAG,
            BY_BUILDING_PART_TAG
        )

    by_parser_records.append({
        "filename": file_path.name,
        "buildings_in_xml": row.buildings,
        "buildings_extracted": len(tile_gdf),
        "unique_ids": tile_gdf["lod2_id"].nunique(),
        "missing_ids": tile_gdf["lod2_id"].isna().sum(),
        "measured_heights": tile_gdf["measured_height_m"].notna().sum(),
        "missing_heights": tile_gdf["measured_height_m"].isna().sum(),
        "buildings_with_parts": tile_gdf["building_part_count"].gt(0).sum(),
        "parent_height_method": tile_gdf["height_method"].eq("building_measured_height").sum(),
        "part_height_method": tile_gdf["height_method"].eq("maximum_building_part_height").sum(),
        "usable_geometries": tile_gdf.geometry.notna().sum(),
        "missing_geometries": tile_gdf.geometry.isna().sum(),
        "minimum_height_m": tile_gdf["measured_height_m"].min(),
        "maximum_height_m": tile_gdf["measured_height_m"].max()
    })

    if not tile_gdf.empty:
        tile_gdf["sample_file"] = file_path.name
        by_parser_parts.append(tile_gdf)

by_parser_test = pd.DataFrame(by_parser_records)

if by_parser_parts:
    by_parser_test_gdf = gpd.GeoDataFrame(
        pd.concat(by_parser_parts, ignore_index=True),
        geometry="geometry",
        crs="EPSG:25832"
    )
else:
    by_parser_test_gdf = gpd.GeoDataFrame(
        columns=["geometry"],
        geometry="geometry",
        crs="EPSG:25832"
    )

print("=" * 70)
print("BAVARIA SAMPLE PARSER TEST")
print("=" * 70)

display(by_parser_test)

print("\nCombined totals:")
display(
    by_parser_test[
        [
            "buildings_in_xml",
            "buildings_extracted",
            "unique_ids",
            "missing_ids",
            "measured_heights",
            "missing_heights",
            "buildings_with_parts",
            "parent_height_method",
            "part_height_method",
            "usable_geometries",
            "missing_geometries"
        ]
    ].sum().to_frame("count")
)

print("\nHeight methods:")
display(
    by_parser_test_gdf["height_method"]
    .value_counts(dropna=False)
    .rename_axis("height_method")
    .reset_index(name="building_count")
)

print("\nFootprint methods:")
display(
    by_parser_test_gdf["footprint_method"]
    .value_counts(dropna=False)
    .rename_axis("footprint_method")
    .reset_index(name="building_count")
)

if not by_parser_test.empty:
    mismatch = by_parser_test["buildings_in_xml"] != by_parser_test["buildings_extracted"]
    print(f"\nTiles with building-count mismatch: {mismatch.sum():,}")

Parsing Bavaria sample tiles: 100%|██████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.38it/s]

BAVARIA SAMPLE PARSER TEST


,filename,buildings_in_xml,buildings_extracted,unique_ids,missing_ids,measured_heights,missing_heights,buildings_with_parts,parent_height_method,part_height_method,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m
0,498_5542.gml,159,159,159,0,159,0,0,159,0,159,0,0.200,31.509
1,618_5388.gml,1300,1300,1300,0,1300,0,0,1300,0,1300,0,1.000,31.647
2,674_5362.gml,208,208,208,0,208,0,0,208,0,208,0,1.323,17.140
3,732_5404.gml,50,50,50,0,50,0,0,50,0,50,0,2.500,14.630
4,854_5412.gml,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN



Combined totals:


,count
buildings_in_xml,1717
buildings_extracted,1717
unique_ids,1717
missing_ids,0
measured_heights,1717
missing_heights,0
buildings_with_parts,0
parent_height_method,1717
part_height_method,0
usable_geometries,1717



Height methods:


,height_method,building_count
0,building_measured_height,1717



Footprint methods:


,footprint_method,building_count
0,ground_surface,1717



Tiles with building-count mismatch: 0


In [95]:
# ============================================================
# 85 — DOWNLOAD ALL BAVARIA LoD2 CITYGML TILES
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import time

BY_GML_DIR = BY_RAW_DIR / "gml"
BY_GML_DIR.mkdir(parents=True, exist_ok=True)

BY_MAX_WORKERS = 8
BY_MAX_RETRIES = 3
BY_CHUNK_SIZE = 1024 * 1024

def validate_by_gml(path, expected_size=None):
    if not path.exists() or path.stat().st_size == 0:
        return False

    if pd.notna(expected_size) and path.stat().st_size != int(expected_size):
        return False

    try:
        with open(path, "rb") as stream:
            context = etree.iterparse(stream, events=("start",), huge_tree=True)
            event, root = next(context)
            root_name = etree.QName(root).localname
            del context
        return root_name == "CityModel"
    except Exception:
        return False

def download_by_tile(row):
    filename = str(row.filename)
    output_path = BY_GML_DIR / filename
    temp_path = BY_GML_DIR / f"{filename}.part"

    expected_size = int(row.size_bytes) if pd.notna(row.size_bytes) else None
    expected_sha256 = str(row.sha256).lower() if pd.notna(row.sha256) else None
    expected_sha1 = str(row.sha1).lower() if pd.notna(row.sha1) else None
    expected_md5 = str(row.md5).lower() if pd.notna(row.md5) else None

    if validate_by_gml(output_path, expected_size):
        return {
            "filename": filename,
            "status": "already_downloaded",
            "size_bytes": output_path.stat().st_size,
            "error": None
        }

    if output_path.exists():
        output_path.unlink()

    if temp_path.exists():
        temp_path.unlink()

    for attempt in range(1, BY_MAX_RETRIES + 1):
        try:
            sha256_hasher = hashlib.sha256() if expected_sha256 else None
            sha1_hasher = hashlib.sha1() if expected_sha1 else None
            md5_hasher = hashlib.md5() if expected_md5 else None

            with requests.get(row.download_url, stream=True, timeout=(30, 1800)) as response:
                response.raise_for_status()

                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=BY_CHUNK_SIZE):
                        if not chunk:
                            continue

                        file.write(chunk)

                        if sha256_hasher:
                            sha256_hasher.update(chunk)

                        if sha1_hasher:
                            sha1_hasher.update(chunk)

                        if md5_hasher:
                            md5_hasher.update(chunk)

            actual_size = temp_path.stat().st_size

            if expected_size is not None and actual_size != expected_size:
                raise ValueError(f"Size mismatch: expected {expected_size:,}, received {actual_size:,}")

            if expected_sha256 and sha256_hasher.hexdigest().lower() != expected_sha256:
                raise ValueError("SHA-256 checksum mismatch.")

            if expected_sha1 and sha1_hasher.hexdigest().lower() != expected_sha1:
                raise ValueError("SHA-1 checksum mismatch.")

            if expected_md5 and md5_hasher.hexdigest().lower() != expected_md5:
                raise ValueError("MD5 checksum mismatch.")

            if not validate_by_gml(temp_path, expected_size):
                raise ValueError("Downloaded file is not a valid CityGML document.")

            temp_path.replace(output_path)

            return {
                "filename": filename,
                "status": "downloaded",
                "size_bytes": output_path.stat().st_size,
                "error": None
            }

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == BY_MAX_RETRIES:
                return {
                    "filename": filename,
                    "status": "failed",
                    "size_bytes": 0,
                    "error": str(error)
                }

            time.sleep(attempt * 5)

by_download_results = []

with ThreadPoolExecutor(max_workers=BY_MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_by_tile, row): row.filename
        for row in by_lod2_downloads.itertuples(index=False)
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading Bavaria CityGML tiles"):
        by_download_results.append(future.result())

by_download_results_df = pd.DataFrame(by_download_results).sort_values("filename").reset_index(drop=True)
by_download_results_path = REPORT_DIR / "lod2_download_results_BY.csv"
by_download_results_df.to_csv(by_download_results_path, index=False)

by_gml_files = sorted(BY_GML_DIR.glob("*.gml")) + sorted(BY_GML_DIR.glob("*.xml"))

print("=" * 70)
print("BAVARIA LoD2 DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nTiles requested       : {len(by_lod2_downloads):,}")
print(f"CityGML files present : {len(by_gml_files):,}")
print(f"Downloaded now        : {by_download_results_df['status'].eq('downloaded').sum():,}")
print(f"Already downloaded    : {by_download_results_df['status'].eq('already_downloaded').sum():,}")
print(f"Failed                : {by_download_results_df['status'].eq('failed').sum():,}")
print(f"Total downloaded size : {sum(path.stat().st_size for path in by_gml_files) / 1024**3:,.2f} GB")
print(f"Download report       : {by_download_results_path}")

display(
    by_download_results_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

if by_download_results_df["status"].eq("failed").any():
    print("\nFailed downloads:")
    display(by_download_results_df[by_download_results_df["status"] == "failed"].head(50))

BAVARIA LoD2 DOWNLOAD SUMMARY

Tiles requested       : 18,297
CityGML files present : 18,297
Downloaded now        : 18,297
Already downloaded    : 0
Failed                : 0
Total downloaded size : 141.66 GB
Download report       : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_download_results_BY.csv


,status,count
0,downloaded,18297


In [96]:
# ============================================================
# 86 — PROCESS AND VALIDATE ALL BAVARIA LoD2 FILES
# ============================================================

BY_OUTPUT_DIR = EXTRACTED_DIR / "BY"
BY_PART_DIR = BY_OUTPUT_DIR / "parts"
BY_MANIFEST_DIR = BY_OUTPUT_DIR / "manifests"

BY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BY_PART_DIR.mkdir(parents=True, exist_ok=True)
BY_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

BY_BATCH_SIZE = 300

by_sources = pd.DataFrame({
    "source_file": [path.name for path in by_gml_files],
    "source_path": [str(path) for path in by_gml_files]
}).sort_values("source_file").reset_index(drop=True)

by_batch_starts = list(range(0, len(by_sources), BY_BATCH_SIZE))

print("=" * 70)
print("BAVARIA PROCESSING SETUP")
print("=" * 70)
print(f"\nCityGML files          : {len(by_sources):,}")
print(f"Unique filenames       : {by_sources['source_file'].nunique():,}")
print(f"Batch size             : {BY_BATCH_SIZE:,}")
print(f"Expected Parquet parts : {len(by_batch_starts):,}")

if by_sources["source_file"].duplicated().any():
    raise ValueError("Duplicate Bavaria CityGML filenames detected.")

# ------------------------------------------------------------
# 1. Extract all CityGML files
# ------------------------------------------------------------

by_batch_summaries = []

for batch_start in tqdm(by_batch_starts, desc="Processing Bavaria batches"):
    batch_number = batch_start // BY_BATCH_SIZE + 1
    batch_sources = by_sources.iloc[batch_start:batch_start + BY_BATCH_SIZE].copy()

    part_path = BY_PART_DIR / f"lod2_buildings_BY_part_{batch_number:04d}.parquet"
    manifest_path = BY_MANIFEST_DIR / f"lod2_manifest_BY_part_{batch_number:04d}.csv"

    # Resume only if this precise batch was completed successfully
    if manifest_path.exists():
        existing_manifest = pd.read_csv(manifest_path)
        expected_files = set(batch_sources["source_file"])
        recorded_files = set(existing_manifest["source_file"])

        manifest_complete = (
            expected_files == recorded_files
            and len(existing_manifest) == len(batch_sources)
            and not existing_manifest["status"].eq("failed").any()
        )

        output_complete = existing_manifest["buildings"].sum() == 0 or part_path.exists()

        if manifest_complete and output_complete:
            by_batch_summaries.append({
                "batch": batch_number,
                "source_files": len(existing_manifest),
                "processed_files": existing_manifest["status"].eq("processed").sum(),
                "empty_files": existing_manifest["status"].eq("empty").sum(),
                "failed_files": existing_manifest["status"].eq("failed").sum(),
                "buildings": existing_manifest["buildings"].sum(),
                "status": "already_processed"
            })
            continue

    if part_path.exists():
        part_path.unlink()

    batch_parts = []
    batch_manifest = []

    for row in batch_sources.itertuples(index=False):
        file_path = Path(row.source_path)

        try:
            with open(file_path, "rb") as stream:
                tile_gdf = extract_citygml_stream_multipart(
                    stream,
                    file_path.name,
                    "EPSG:25832",
                    BY_BUILDING_TAG,
                    BY_BUILDING_PART_TAG
                )

            if tile_gdf.empty:
                batch_manifest.append({
                    "source_file": row.source_file,
                    "source_path": row.source_path,
                    "status": "empty",
                    "buildings": 0,
                    "missing_ids": 0,
                    "missing_heights": 0,
                    "missing_geometries": 0,
                    "minimum_height_m": np.nan,
                    "maximum_height_m": np.nan,
                    "error": None
                })
                continue

            tile_gdf["state_code"] = "BY"
            tile_gdf["source_state"] = "Bavaria"
            tile_gdf["source_crs"] = "EPSG:25832"
            tile_gdf["citygml_version"] = "1.0"
            tile_gdf["height_source"] = tile_gdf["height_method"]
            tile_gdf["source_path"] = row.source_path

            batch_parts.append(tile_gdf)

            batch_manifest.append({
                "source_file": row.source_file,
                "source_path": row.source_path,
                "status": "processed",
                "buildings": len(tile_gdf),
                "missing_ids": tile_gdf["lod2_id"].isna().sum(),
                "missing_heights": tile_gdf["measured_height_m"].isna().sum(),
                "missing_geometries": tile_gdf.geometry.isna().sum(),
                "minimum_height_m": tile_gdf["measured_height_m"].min(),
                "maximum_height_m": tile_gdf["measured_height_m"].max(),
                "error": None
            })

        except Exception as error:
            batch_manifest.append({
                "source_file": row.source_file,
                "source_path": row.source_path,
                "status": "failed",
                "buildings": 0,
                "missing_ids": 0,
                "missing_heights": 0,
                "missing_geometries": 0,
                "minimum_height_m": np.nan,
                "maximum_height_m": np.nan,
                "error": str(error)
            })

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    batch_manifest_df = pd.DataFrame(batch_manifest)
    batch_manifest_df.to_csv(manifest_path, index=False)

    by_batch_summaries.append({
        "batch": batch_number,
        "source_files": len(batch_manifest_df),
        "processed_files": batch_manifest_df["status"].eq("processed").sum(),
        "empty_files": batch_manifest_df["status"].eq("empty").sum(),
        "failed_files": batch_manifest_df["status"].eq("failed").sum(),
        "buildings": batch_manifest_df["buildings"].sum(),
        "status": "processed"
    })

    del batch_parts, batch_manifest, batch_manifest_df
    gc.collect()

# ------------------------------------------------------------
# 2. Combine manifests
# ------------------------------------------------------------

by_manifest_files = sorted(BY_MANIFEST_DIR.glob("lod2_manifest_BY_part_*.csv"))
by_manifests = pd.concat([pd.read_csv(path) for path in by_manifest_files], ignore_index=True)
by_part_files = sorted(BY_PART_DIR.glob("lod2_buildings_BY_part_*.parquet"))
by_batch_summary_df = pd.DataFrame(by_batch_summaries)

by_manifest_path = REPORT_DIR / "lod2_processing_manifest_BY.csv"
by_batch_summary_path = REPORT_DIR / "lod2_batch_summary_BY.csv"

by_manifests.to_csv(by_manifest_path, index=False)
by_batch_summary_df.to_csv(by_batch_summary_path, index=False)

# ------------------------------------------------------------
# 3. Validate all generated Parquet parts
# ------------------------------------------------------------

by_expected_files = set(by_sources["source_file"])
by_recorded_files = set(by_manifests["source_file"])

by_missing_source_files = sorted(by_expected_files - by_recorded_files)
by_unexpected_source_files = sorted(by_recorded_files - by_expected_files)
by_duplicate_manifest_files = by_manifests["source_file"].duplicated().sum()

by_part_stats = []
by_height_method_counts = Counter()
by_footprint_counts = Counter()

for part_path in tqdm(by_part_files, desc="Validating Bavaria parts"):
    part = pd.read_parquet(
        part_path,
        columns=[
            "lod2_id",
            "measured_height_m",
            "height_method",
            "footprint_method",
            "geometry"
        ]
    )

    by_height_method_counts.update(part["height_method"].fillna("missing"))
    by_footprint_counts.update(part["footprint_method"].fillna("missing"))

    by_part_stats.append({
        "buildings": len(part),
        "missing_ids": part["lod2_id"].isna().sum(),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    del part
    gc.collect()

by_part_stats = pd.DataFrame(by_part_stats)

by_height_method_summary = pd.DataFrame(
    [{"height_method": method, "building_count": count} for method, count in by_height_method_counts.items()]
).sort_values("building_count", ascending=False)

by_footprint_summary = pd.DataFrame(
    [{"footprint_method": method, "building_count": count} for method, count in by_footprint_counts.items()]
).sort_values("building_count", ascending=False)

by_summary = pd.DataFrame([{
    "state_code": "BY",
    "state_name": "Bavaria",
    "source_files_expected": len(by_expected_files),
    "source_files_recorded": len(by_recorded_files),
    "files_with_buildings": by_manifests["status"].eq("processed").sum(),
    "empty_files": by_manifests["status"].eq("empty").sum(),
    "failed_files": by_manifests["status"].eq("failed").sum(),
    "missing_source_files": len(by_missing_source_files),
    "unexpected_source_files": len(by_unexpected_source_files),
    "duplicate_manifest_files": by_duplicate_manifest_files,
    "parquet_parts": len(by_part_files),
    "buildings": by_part_stats["buildings"].sum(),
    "missing_ids": by_part_stats["missing_ids"].sum(),
    "measured_heights": by_part_stats["measured_heights"].sum(),
    "missing_heights": by_part_stats["missing_heights"].sum(),
    "usable_geometries": by_part_stats["usable_geometries"].sum(),
    "missing_geometries": by_part_stats["missing_geometries"].sum(),
    "minimum_height_m": by_part_stats["minimum_height_m"].min(),
    "maximum_height_m": by_part_stats["maximum_height_m"].max(),
    "output_directory": str(BY_PART_DIR)
}])

# ------------------------------------------------------------
# 4. Save final Bavaria reports
# ------------------------------------------------------------

by_summary_path = REPORT_DIR / "lod2_extraction_summary_BY.csv"
by_height_summary_path = REPORT_DIR / "lod2_height_methods_BY.csv"
by_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_BY.csv"

by_summary.to_csv(by_summary_path, index=False)
by_height_method_summary.to_csv(by_height_summary_path, index=False)
by_footprint_summary.to_csv(by_footprint_summary_path, index=False)

source_registry.loc[source_registry["state_code"].eq("BY"), "download_status"] = "complete"
source_registry.to_csv(source_registry_path, index=False)

print("=" * 70)
print("BAVARIA FINAL LoD2 VALIDATION")
print("=" * 70)

display(by_summary)

print("\nHeight methods:")
display(by_height_method_summary)

print("\nFootprint methods:")
display(by_footprint_summary)

print(f"\nState summary saved to     : {by_summary_path}")
print(f"Height summary saved to    : {by_height_summary_path}")
print(f"Footprint summary saved to : {by_footprint_summary_path}")

if by_missing_source_files:
    display(pd.DataFrame({"missing_source_file": by_missing_source_files}).head(30))

if by_manifests["status"].eq("failed").any():
    print("\nFailed files:")
    display(by_manifests[by_manifests["status"] == "failed"].head(30))

BAVARIA PROCESSING SETUP

CityGML files          : 18,297
Unique filenames       : 18,297
Batch size             : 300
Expected Parquet parts : 61


Validating Bavaria parts: 100%|████████████████████████████████████████████████████| 61/61 [01:02<00:00,  1.02s/it]

BAVARIA FINAL LoD2 VALIDATION


,state_code,state_name,source_files_expected,source_files_recorded,files_with_buildings,empty_files,failed_files,missing_source_files,unexpected_source_files,duplicate_manifest_files,parquet_parts,buildings,missing_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,BY,Bavaria,18297,18297,17606,691,0,0,0,0,61,10111920,0,10111920,0,10111920,0,-0.03,292.98,/fast/home/o-olajuyigbe/data/germany_lod2/extr...



Height methods:


,height_method,building_count
0,building_measured_height,10111920



Footprint methods:


,footprint_method,building_count
0,ground_surface,10111920



State summary saved to     : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_extraction_summary_BY.csv
Height summary saved to    : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_height_methods_BY.csv
Footprint summary saved to : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_footprint_methods_BY.csv


In [ ]:
# ============================================================
# 87 — BADEN-WÜRTTEMBERG: DISCOVER, DOWNLOAD, PROCESS, VALIDATE
# ============================================================

import re
import time
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
import gc

BW_RAW_DIR = RAW_DIR / "BW"
BW_ZIP_DIR = BW_RAW_DIR / "zips"
BW_OUTPUT_DIR = EXTRACTED_DIR / "BW"
BW_PART_DIR = BW_OUTPUT_DIR / "parts"
BW_MANIFEST_DIR = BW_OUTPUT_DIR / "manifests"

for path in [BW_RAW_DIR, BW_ZIP_DIR, BW_OUTPUT_DIR, BW_PART_DIR, BW_MANIFEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BW_PORTAL_URL = "https://opengeodata.lgl-bw.de/#/(sidenav:product/lod2)"
BW_WFS_URL = "https://owsproxy.lgl-bw.de/owsproxy/wfs/WFS_LGL-BW_LoD2_Aktualitaet?SERVICE=WFS&VERSION=1.1.0&REQUEST=GetFeature&TYPENAME=verm:v_lod2_aktualitaet&MAXFEATURES=50000&OUTPUTFORMAT=application/json"
BW_DOWNLOAD_ROOT = "https://opengeodata.lgl-bw.de/data/lod2"

BW_MAX_WORKERS = 8
BW_MAX_RETRIES = 3
BW_BATCH_SIZE = 300

# ------------------------------------------------------------
# 1. Discover official 2 km x 2 km LoD2 archive packages
# ------------------------------------------------------------

bw_response = requests.get(BW_WFS_URL, timeout=(30, 600))
bw_response.raise_for_status()
bw_wfs_data = bw_response.json()

(BW_RAW_DIR / "bw_lod2_tile_index.json").write_text(
    json.dumps(bw_wfs_data, ensure_ascii=False),
    encoding="utf-8"
)

bw_records = []

for feature in bw_wfs_data.get("features", []):
    properties = feature.get("properties", {})
    tile_name = str(properties.get("kachelname", "")).strip()
    numbers = re.findall(r"\d+", tile_name)

    if len(numbers) < 2:
        continue

    x, y = int(numbers[-2]), int(numbers[-1])

    # One 2 km package represents four 1 km freshness-index tiles.
    if x % 2 == 0 or y % 2 == 1:
        continue

    filename = f"LoD2_32_{x}_{y}_2_bw.zip"

    bw_records.append({
        "tile_name": tile_name,
        "x": x,
        "y": y,
        "production_date": properties.get("produktionsdatum"),
        "filename": filename,
        "download_url": f"{BW_DOWNLOAD_ROOT}/{filename}"
    })

bw_downloads = (
    pd.DataFrame(bw_records)
    .drop_duplicates(subset="filename")
    .sort_values(["x", "y"])
    .reset_index(drop=True)
)

if bw_downloads.empty:
    raise RuntimeError("No Baden-Württemberg LoD2 archives were discovered.")

bw_download_list_path = BW_RAW_DIR / "bw_lod2_download_list.csv"
bw_downloads.to_csv(bw_download_list_path, index=False)

source_registry.loc[
    source_registry["state_code"].eq("BW"),
    ["portal_url", "download_url", "download_method", "format", "horizontal_crs", "download_status", "notes"]
] = [
    BW_PORTAL_URL,
    BW_WFS_URL,
    "Official WFS tile index and direct 2 km CityGML ZIP archives",
    "CityGML ZIP",
    "EPSG:25832",
    "downloading",
    "Processed directly from ZIP archives"
]
source_registry.to_csv(source_registry_path, index=False)

print("=" * 70)
print("BADEN-WÜRTTEMBERG LoD2 DISCOVERY")
print("=" * 70)
print(f"\nWFS features returned : {len(bw_wfs_data.get('features', [])):,}")
print(f"LoD2 ZIP packages     : {len(bw_downloads):,}")
print(f"Unique URLs           : {bw_downloads['download_url'].nunique():,}")
print(f"Unique filenames      : {bw_downloads['filename'].nunique():,}")
print(f"Duplicate filenames   : {bw_downloads['filename'].duplicated().sum():,}")

# ------------------------------------------------------------
# 2. Download all archives
# ------------------------------------------------------------

def validate_bw_zip(path):
    if not path.exists() or path.stat().st_size == 0:
        return False

    try:
        with ZipFile(path, "r") as archive:
            return (
                archive.testzip() is None
                and any(name.lower().endswith((".gml", ".xml")) for name in archive.namelist())
            )
    except Exception:
        return False

def download_bw_archive(url, filename):
    output_path = BW_ZIP_DIR / filename
    temp_path = BW_ZIP_DIR / f"{filename}.part"

    if validate_bw_zip(output_path):
        return {"filename": filename, "status": "already_downloaded", "size_bytes": output_path.stat().st_size, "error": None}

    if output_path.exists():
        output_path.unlink()

    for attempt in range(1, BW_MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 1800)) as response:
                response.raise_for_status()
                expected_size = int(response.headers.get("content-length", 0))

                with open(temp_path, "wb") as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            file.write(chunk)

            actual_size = temp_path.stat().st_size

            if expected_size and actual_size != expected_size:
                raise ValueError(f"Expected {expected_size:,} bytes, received {actual_size:,}.")

            if not validate_bw_zip(temp_path):
                raise ValueError("Downloaded file is not a valid CityGML ZIP archive.")

            temp_path.replace(output_path)
            return {"filename": filename, "status": "downloaded", "size_bytes": actual_size, "error": None}

        except Exception as error:
            if temp_path.exists():
                temp_path.unlink()

            if attempt == BW_MAX_RETRIES:
                return {"filename": filename, "status": "failed", "size_bytes": 0, "error": str(error)}

            time.sleep(attempt * 5)

bw_download_results = []

with ThreadPoolExecutor(max_workers=BW_MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_bw_archive, row.download_url, row.filename): row.filename
        for row in bw_downloads.itertuples(index=False)
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading Baden-Württemberg LoD2"):
        bw_download_results.append(future.result())

bw_download_results_df = pd.DataFrame(bw_download_results).sort_values("filename").reset_index(drop=True)
bw_download_results_path = REPORT_DIR / "lod2_download_results_BW.csv"
bw_download_results_df.to_csv(bw_download_results_path, index=False)

bw_zip_files = sorted(BW_ZIP_DIR.glob("*.zip"))
bw_failed_downloads = bw_download_results_df["status"].eq("failed").sum()

print("\n" + "=" * 70)
print("BADEN-WÜRTTEMBERG DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nPackages requested    : {len(bw_downloads):,}")
print(f"ZIP archives present  : {len(bw_zip_files):,}")
print(f"Downloaded now        : {bw_download_results_df['status'].eq('downloaded').sum():,}")
print(f"Already downloaded    : {bw_download_results_df['status'].eq('already_downloaded').sum():,}")
print(f"Failed                : {bw_failed_downloads:,}")
print(f"Downloaded size       : {sum(path.stat().st_size for path in bw_zip_files) / 1024**3:,.2f} GB")

if bw_failed_downloads:
    display(bw_download_results_df[bw_download_results_df["status"] == "failed"].head(30))
    raise RuntimeError("Some Baden-Württemberg downloads failed. Re-run this cell to retry only missing archives.")

# ------------------------------------------------------------
# 3. Index every CityGML member
# ------------------------------------------------------------

bw_source_records = []

for zip_path in tqdm(bw_zip_files, desc="Indexing Baden-Württemberg archives"):
    with ZipFile(zip_path, "r") as archive:
        for member in archive.namelist():
            if member.lower().endswith((".gml", ".xml")):
                bw_source_records.append({
                    "archive_name": zip_path.name,
                    "archive_path": str(zip_path),
                    "source_member": member,
                    "source_file": Path(member).name,
                    "source_key": f"{zip_path.name}::{member}"
                })

bw_sources = (
    pd.DataFrame(bw_source_records)
    .sort_values("source_key")
    .reset_index(drop=True)
)

if bw_sources.empty:
    raise RuntimeError("No Baden-Württemberg CityGML files were found.")

if bw_sources["source_key"].duplicated().any():
    raise ValueError("Duplicate Baden-Württemberg source keys detected.")

bw_sources.to_csv(BW_RAW_DIR / "bw_citygml_source_index.csv", index=False)
bw_batch_starts = list(range(0, len(bw_sources), BW_BATCH_SIZE))

print("\n" + "=" * 70)
print("BADEN-WÜRTTEMBERG PROCESSING SETUP")
print("=" * 70)
print(f"\nZIP archives          : {len(bw_zip_files):,}")
print(f"CityGML files         : {len(bw_sources):,}")
print(f"Unique source keys    : {bw_sources['source_key'].nunique():,}")
print(f"Batch size            : {BW_BATCH_SIZE:,}")
print(f"Expected Parquet parts: {len(bw_batch_starts):,}")

# ------------------------------------------------------------
# 4. Process all CityGML members
# ------------------------------------------------------------

def detect_building_namespace(archive, member):
    with archive.open(member) as stream:
        for event, root in etree.iterparse(stream, events=("start",), huge_tree=True):
            namespace = root.nsmap.get("bldg")

            if namespace is None:
                namespace = next(
                    (value for value in root.nsmap.values() if value and "citygml/building" in value),
                    None
                )

            return namespace

    return None

bw_batch_summaries = []

for batch_start in tqdm(bw_batch_starts, desc="Processing Baden-Württemberg batches"):
    batch_number = batch_start // BW_BATCH_SIZE + 1
    batch_sources = bw_sources.iloc[batch_start:batch_start + BW_BATCH_SIZE].copy()

    part_path = BW_PART_DIR / f"lod2_buildings_BW_part_{batch_number:04d}.parquet"
    manifest_path = BW_MANIFEST_DIR / f"lod2_manifest_BW_part_{batch_number:04d}.csv"

    if manifest_path.exists():
        existing_manifest = pd.read_csv(manifest_path)
        expected_keys = set(batch_sources["source_key"])
        recorded_keys = set(existing_manifest["source_key"])

        manifest_complete = (
            expected_keys == recorded_keys
            and len(existing_manifest) == len(batch_sources)
            and not existing_manifest["status"].eq("failed").any()
        )

        output_complete = existing_manifest["buildings"].sum() == 0 or part_path.exists()

        if manifest_complete and output_complete:
            bw_batch_summaries.append({
                "batch": batch_number,
                "source_files": len(existing_manifest),
                "processed_files": existing_manifest["status"].eq("processed").sum(),
                "empty_files": existing_manifest["status"].eq("empty").sum(),
                "failed_files": existing_manifest["status"].eq("failed").sum(),
                "buildings": existing_manifest["buildings"].sum(),
                "status": "already_processed"
            })
            continue

    if part_path.exists():
        part_path.unlink()

    batch_parts = []
    batch_manifest = []

    for archive_path, archive_sources in batch_sources.groupby("archive_path", sort=False):
        with ZipFile(archive_path, "r") as archive:
            for row in archive_sources.itertuples(index=False):
                try:
                    building_namespace = detect_building_namespace(archive, row.source_member)

                    if not building_namespace:
                        raise ValueError("CityGML building namespace could not be detected.")

                    building_tag = f"{{{building_namespace}}}Building"
                    building_part_tag = f"{{{building_namespace}}}BuildingPart"
                    citygml_version = building_namespace.rstrip("/").split("/")[-1]

                    with archive.open(row.source_member) as stream:
                        tile_gdf = extract_citygml_stream_multipart(
                            stream,
                            row.source_file,
                            "EPSG:25832",
                            building_tag,
                            building_part_tag
                        )

                    if tile_gdf.empty:
                        batch_manifest.append({
                            "source_key": row.source_key,
                            "archive_name": row.archive_name,
                            "source_member": row.source_member,
                            "status": "empty",
                            "buildings": 0,
                            "missing_ids": 0,
                            "missing_heights": 0,
                            "missing_geometries": 0,
                            "minimum_height_m": np.nan,
                            "maximum_height_m": np.nan,
                            "error": None
                        })
                        continue

                    tile_gdf["state_code"] = "BW"
                    tile_gdf["source_state"] = "Baden-Württemberg"
                    tile_gdf["source_crs"] = "EPSG:25832"
                    tile_gdf["citygml_version"] = citygml_version
                    tile_gdf["height_source"] = tile_gdf["height_method"]
                    tile_gdf["source_archive"] = row.archive_name
                    tile_gdf["source_member"] = row.source_member
                    tile_gdf["source_path"] = row.source_key

                    batch_parts.append(tile_gdf)

                    batch_manifest.append({
                        "source_key": row.source_key,
                        "archive_name": row.archive_name,
                        "source_member": row.source_member,
                        "status": "processed",
                        "buildings": len(tile_gdf),
                        "missing_ids": tile_gdf["lod2_id"].isna().sum(),
                        "missing_heights": tile_gdf["measured_height_m"].isna().sum(),
                        "missing_geometries": tile_gdf.geometry.isna().sum(),
                        "minimum_height_m": tile_gdf["measured_height_m"].min(),
                        "maximum_height_m": tile_gdf["measured_height_m"].max(),
                        "error": None
                    })

                except Exception as error:
                    batch_manifest.append({
                        "source_key": row.source_key,
                        "archive_name": row.archive_name,
                        "source_member": row.source_member,
                        "status": "failed",
                        "buildings": 0,
                        "missing_ids": 0,
                        "missing_heights": 0,
                        "missing_geometries": 0,
                        "minimum_height_m": np.nan,
                        "maximum_height_m": np.nan,
                        "error": str(error)
                    })

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    batch_manifest_df = pd.DataFrame(batch_manifest)
    batch_manifest_df.to_csv(manifest_path, index=False)

    bw_batch_summaries.append({
        "batch": batch_number,
        "source_files": len(batch_manifest_df),
        "processed_files": batch_manifest_df["status"].eq("processed").sum(),
        "empty_files": batch_manifest_df["status"].eq("empty").sum(),
        "failed_files": batch_manifest_df["status"].eq("failed").sum(),
        "buildings": batch_manifest_df["buildings"].sum(),
        "status": "processed"
    })

    del batch_parts, batch_manifest, batch_manifest_df
    gc.collect()

# ------------------------------------------------------------
# 5. Combine manifests and validate output
# ------------------------------------------------------------

bw_manifest_files = sorted(BW_MANIFEST_DIR.glob("lod2_manifest_BW_part_*.csv"))
bw_manifests = pd.concat([pd.read_csv(path) for path in bw_manifest_files], ignore_index=True)
bw_part_files = sorted(BW_PART_DIR.glob("lod2_buildings_BW_part_*.parquet"))
bw_batch_summary_df = pd.DataFrame(bw_batch_summaries)

bw_manifests.to_csv(REPORT_DIR / "lod2_processing_manifest_BW.csv", index=False)
bw_batch_summary_df.to_csv(REPORT_DIR / "lod2_batch_summary_BW.csv", index=False)

bw_expected_sources = set(bw_sources["source_key"])
bw_recorded_sources = set(bw_manifests["source_key"])

bw_part_stats = []
bw_height_counts = Counter()
bw_footprint_counts = Counter()

for part_path in tqdm(bw_part_files, desc="Validating Baden-Württemberg parts"):
    part = pd.read_parquet(
        part_path,
        columns=["lod2_id", "measured_height_m", "height_method", "footprint_method", "geometry"]
    )

    bw_height_counts.update(part["height_method"].fillna("missing"))
    bw_footprint_counts.update(part["footprint_method"].fillna("missing"))

    bw_part_stats.append({
        "buildings": len(part),
        "missing_ids": part["lod2_id"].isna().sum(),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })

    del part
    gc.collect()

bw_part_stats = pd.DataFrame(bw_part_stats)

bw_height_summary = pd.DataFrame(
    [{"height_method": key, "building_count": value} for key, value in bw_height_counts.items()]
).sort_values("building_count", ascending=False)

bw_footprint_summary = pd.DataFrame(
    [{"footprint_method": key, "building_count": value} for key, value in bw_footprint_counts.items()]
).sort_values("building_count", ascending=False)

bw_summary = pd.DataFrame([{
    "state_code": "BW",
    "state_name": "Baden-Württemberg",
    "zip_archives": len(bw_zip_files),
    "source_files_expected": len(bw_expected_sources),
    "source_files_recorded": len(bw_recorded_sources),
    "files_with_buildings": bw_manifests["status"].eq("processed").sum(),
    "empty_files": bw_manifests["status"].eq("empty").sum(),
    "failed_files": bw_manifests["status"].eq("failed").sum(),
    "missing_source_files": len(bw_expected_sources - bw_recorded_sources),
    "unexpected_source_files": len(bw_recorded_sources - bw_expected_sources),
    "duplicate_manifest_sources": bw_manifests["source_key"].duplicated().sum(),
    "parquet_parts": len(bw_part_files),
    "buildings": bw_part_stats["buildings"].sum(),
    "missing_ids": bw_part_stats["missing_ids"].sum(),
    "measured_heights": bw_part_stats["measured_heights"].sum(),
    "missing_heights": bw_part_stats["missing_heights"].sum(),
    "usable_geometries": bw_part_stats["usable_geometries"].sum(),
    "missing_geometries": bw_part_stats["missing_geometries"].sum(),
    "minimum_height_m": bw_part_stats["minimum_height_m"].min(),
    "maximum_height_m": bw_part_stats["maximum_height_m"].max(),
    "output_directory": str(BW_PART_DIR)
}])

bw_summary_path = REPORT_DIR / "lod2_extraction_summary_BW.csv"
bw_height_summary_path = REPORT_DIR / "lod2_height_methods_BW.csv"
bw_footprint_summary_path = REPORT_DIR / "lod2_footprint_methods_BW.csv"

bw_summary.to_csv(bw_summary_path, index=False)
bw_height_summary.to_csv(bw_height_summary_path, index=False)
bw_footprint_summary.to_csv(bw_footprint_summary_path, index=False)

source_registry.loc[source_registry["state_code"].eq("BW"), "download_status"] = "complete"
source_registry.to_csv(source_registry_path, index=False)

print("\n" + "=" * 70)
print("BADEN-WÜRTTEMBERG FINAL LoD2 VALIDATION")
print("=" * 70)

display(bw_summary)

print("\nHeight methods:")
display(bw_height_summary)

print("\nFootprint methods:")
display(bw_footprint_summary)

print(f"\nState summary saved to     : {bw_summary_path}")
print(f"Height summary saved to    : {bw_height_summary_path}")
print(f"Footprint summary saved to : {bw_footprint_summary_path}")

if bw_manifests["status"].eq("failed").any():
    print("\nFailed source files:")
    display(bw_manifests[bw_manifests["status"] == "failed"].head(30))

BADEN-WÜRTTEMBERG LoD2 DISCOVERY

WFS features returned : 36,650
LoD2 ZIP packages     : 9,157
Unique URLs           : 9,157
Unique filenames      : 9,157
Duplicate filenames   : 0



BADEN-WÜRTTEMBERG DOWNLOAD SUMMARY

Packages requested    : 9,157
ZIP archives present  : 9,157
Downloaded now        : 0
Already downloaded    : 9,157
Failed                : 0
Downloaded size       : 15.32 GB


Indexing Baden-Württemberg archives: 100%|█████████████████████████████| 9157/9157 [00:02<00:00, 3636.41it/s]



BADEN-WÜRTTEMBERG PROCESSING SETUP

ZIP archives          : 9,157
CityGML files         : 36,486
Unique source keys    : 36,486
Batch size            : 300
Expected Parquet parts: 122


Processing Baden-Württemberg batches: 100%|████████████████████████████████| 122/122 [00:46<00:00,  2.65it/s]
Validating Baden-Württemberg parts: 0it [00:00, ?it/s]


KeyError: 'building_count'

In [20]:
# ============================================================
# 87B — RESTORE PARSER AND REPROCESS BADEN-WÜRTTEMBERG
# ============================================================

import gc
import re
import geopandas as gpd
from collections import Counter
from lxml import etree
from shapely.geometry import Polygon, MultiPoint
from shapely.ops import unary_union

BW_BLDG_NS = "http://www.opengis.net/citygml/building/1.0"
BW_BUILDING_TAG = f"{{{BW_BLDG_NS}}}Building"
BW_BUILDING_PART_TAG = f"{{{BW_BLDG_NS}}}BuildingPart"

def parse_number(value):
    if value is None:
        return np.nan
    match = re.search(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)", str(value).replace(",", "."))
    return float(match.group()) if match else np.nan

def get_direct_text(element, local_name):
    for child in element:
        if etree.QName(child).localname == local_name and child.text:
            value = child.text.strip()
            return value if value else None
    return None

def get_direct_number(element, local_name):
    return parse_number(get_direct_text(element, local_name))

def parse_gml_ring(ring):
    pos_lists = ring.xpath(".//*[local-name()='posList']")
    if pos_lists:
        pos_list = pos_lists[0]
        values = [float(value) for value in pos_list.text.split()] if pos_list.text else []
        dimension = int(pos_list.get("srsDimension") or ring.get("srsDimension") or 3)
        if dimension not in {2, 3} or len(values) % dimension:
            dimension = 3 if len(values) % 3 == 0 else 2
        coordinates = [tuple(values[i:i + dimension]) for i in range(0, len(values), dimension)]
    else:
        coordinates = []
        for pos in ring.xpath(".//*[local-name()='pos']"):
            values = [float(value) for value in pos.text.split()] if pos.text else []
            if len(values) >= 2:
                coordinates.append(tuple(values))

    coordinates = [coordinate for i, coordinate in enumerate(coordinates) if i == 0 or coordinate != coordinates[i - 1]]
    if len(coordinates) >= 3 and coordinates[0][:2] != coordinates[-1][:2]:
        coordinates.append(coordinates[0])
    return coordinates

def polygons_from_surface(surface):
    polygons = []

    for polygon_element in surface.xpath(".//*[local-name()='Polygon']"):
        exterior_rings = polygon_element.xpath("./*[local-name()='exterior']//*[local-name()='LinearRing']")
        if not exterior_rings:
            continue

        exterior_coordinates = parse_gml_ring(exterior_rings[0])
        exterior_xy = [(coordinate[0], coordinate[1]) for coordinate in exterior_coordinates]

        if len(exterior_xy) < 4:
            continue

        interior_xy = []

        for interior_ring in polygon_element.xpath("./*[local-name()='interior']//*[local-name()='LinearRing']"):
            interior_coordinates = parse_gml_ring(interior_ring)
            ring_xy = [(coordinate[0], coordinate[1]) for coordinate in interior_coordinates]
            if len(ring_xy) >= 4:
                interior_xy.append(ring_xy)

        try:
            polygon = Polygon(exterior_xy, interior_xy)
            if not polygon.is_valid:
                polygon = polygon.buffer(0)
            if not polygon.is_empty:
                polygons.append(polygon)
        except Exception:
            continue

    return polygons

def merge_polygonal_geometries(polygons):
    if not polygons:
        return None

    try:
        geometry = unary_union(polygons)

        if not geometry.is_valid:
            geometry = geometry.buffer(0)

        if geometry.geom_type in {"Polygon", "MultiPolygon"} and not geometry.is_empty:
            return geometry

        if hasattr(geometry, "geoms"):
            polygonal = [item for item in geometry.geoms if item.geom_type in {"Polygon", "MultiPolygon"} and not item.is_empty]
            if polygonal:
                return unary_union(polygonal)

    except Exception:
        return None

    return None

def extract_lod2_footprint(building):
    ground_polygons = []

    for surface in building.xpath(".//*[local-name()='GroundSurface']"):
        ground_polygons.extend(polygons_from_surface(surface))

    geometry = merge_polygonal_geometries(ground_polygons)

    if geometry is not None:
        return geometry, "ground_surface"

    wall_points = []

    for surface in building.xpath(".//*[local-name()='WallSurface']"):
        for ring in surface.xpath(".//*[local-name()='LinearRing']"):
            coordinates = parse_gml_ring(ring)
            wall_points.extend((coordinate[0], coordinate[1]) for coordinate in coordinates if len(coordinate) >= 2)

    if len(set(wall_points)) >= 3:
        try:
            geometry = MultiPoint(list(set(wall_points))).convex_hull
            if geometry.geom_type == "Polygon" and not geometry.is_empty:
                return geometry, "wall_bottom_convex_hull"
        except Exception:
            pass

    return None, "missing"

def extract_citygml_stream_multipart(stream, source_file, crs, building_tag, building_part_tag):
    records = []

    for event, building in etree.iterparse(stream, events=("end",), tag=building_tag, huge_tree=True, recover=True):
        lod2_id = building.get(f"{{http://www.opengis.net/gml}}id")
        parent_height = get_direct_number(building, "measuredHeight")
        parts = building.xpath(".//*[local-name()='BuildingPart']")
        part_heights = [get_direct_number(part, "measuredHeight") for part in parts]
        part_heights = [height for height in part_heights if pd.notna(height)]
        maximum_part_height = max(part_heights) if part_heights else np.nan

        if pd.notna(parent_height):
            measured_height = parent_height
            height_method = "building_measured_height"
        elif pd.notna(maximum_part_height):
            measured_height = maximum_part_height
            height_method = "maximum_building_part_height"
        else:
            measured_height = np.nan
            height_method = "missing"

        geometry, footprint_method = extract_lod2_footprint(building)

        records.append({
            "lod2_id": lod2_id,
            "creation_date": get_direct_text(building, "creationDate"),
            "function": get_direct_text(building, "function"),
            "roof_type": get_direct_text(building, "roofType"),
            "measured_height_m": measured_height,
            "parent_height_m": parent_height,
            "maximum_part_height_m": maximum_part_height,
            "building_part_count": len(parts),
            "height_method": height_method,
            "storeys_above_ground": get_direct_number(building, "storeysAboveGround"),
            "source_file": source_file,
            "footprint_method": footprint_method,
            "geometry": geometry
        })

        building.clear()
        while building.getprevious() is not None:
            del building.getparent()[0]

    columns = [
        "lod2_id", "creation_date", "function", "roof_type", "measured_height_m",
        "parent_height_m", "maximum_part_height_m", "building_part_count",
        "height_method", "storeys_above_ground", "source_file",
        "footprint_method", "geometry"
    ]

    if not records:
        return gpd.GeoDataFrame(columns=columns, geometry="geometry", crs=crs)

    return gpd.GeoDataFrame(records, geometry="geometry", crs=crs)

# ------------------------------------------------------------
# 1. Test restored parser before statewide processing
# ------------------------------------------------------------

bw_sources = pd.read_csv(BW_RAW_DIR / "bw_citygml_source_index.csv")
sample_indices = sorted(set([0, len(bw_sources) // 4, len(bw_sources) // 2, 3 * len(bw_sources) // 4, len(bw_sources) - 1]))
bw_parser_tests = []

for row in tqdm(bw_sources.iloc[sample_indices].itertuples(index=False), total=len(sample_indices), desc="Testing restored BW parser"):
    with ZipFile(row.archive_path, "r") as archive:
        with archive.open(row.source_member) as stream:
            sample_gdf = extract_citygml_stream_multipart(stream, row.source_file, "EPSG:25832", BW_BUILDING_TAG, BW_BUILDING_PART_TAG)

    bw_parser_tests.append({
        "archive_name": row.archive_name,
        "source_member": row.source_member,
        "buildings_extracted": len(sample_gdf),
        "measured_heights": sample_gdf["measured_height_m"].notna().sum(),
        "buildings_with_parts": sample_gdf["building_part_count"].gt(0).sum(),
        "usable_geometries": sample_gdf.geometry.notna().sum(),
        "missing_geometries": sample_gdf.geometry.isna().sum()
    })

bw_parser_tests = pd.DataFrame(bw_parser_tests)

print("=" * 70)
print("RESTORED BADEN-WÜRTTEMBERG PARSER TEST")
print("=" * 70)
display(bw_parser_tests)

if bw_parser_tests["buildings_extracted"].sum() == 0:
    raise RuntimeError("The restored parser extracted no buildings. Statewide processing was stopped.")

# ------------------------------------------------------------
# 2. Remove only the failed manifests from the previous run
# ------------------------------------------------------------

for manifest_path in BW_MANIFEST_DIR.glob("lod2_manifest_BW_part_*.csv"):
    manifest = pd.read_csv(manifest_path)
    if manifest["status"].eq("failed").all():
        manifest_path.unlink()

bw_batch_starts = list(range(0, len(bw_sources), BW_BATCH_SIZE))
bw_batch_summaries = []

# ------------------------------------------------------------
# 3. Reprocess statewide data
# ------------------------------------------------------------

for batch_start in tqdm(bw_batch_starts, desc="Processing Baden-Württemberg batches"):
    batch_number = batch_start // BW_BATCH_SIZE + 1
    batch_sources = bw_sources.iloc[batch_start:batch_start + BW_BATCH_SIZE]
    part_path = BW_PART_DIR / f"lod2_buildings_BW_part_{batch_number:04d}.parquet"
    manifest_path = BW_MANIFEST_DIR / f"lod2_manifest_BW_part_{batch_number:04d}.csv"

    if manifest_path.exists():
        existing_manifest = pd.read_csv(manifest_path)
        expected_keys = set(batch_sources["source_key"])
        recorded_keys = set(existing_manifest["source_key"])
        manifest_complete = expected_keys == recorded_keys and len(existing_manifest) == len(batch_sources) and not existing_manifest["status"].eq("failed").any()
        output_complete = existing_manifest["buildings"].sum() == 0 or part_path.exists()

        if manifest_complete and output_complete:
            bw_batch_summaries.append({
                "batch": batch_number,
                "source_files": len(existing_manifest),
                "processed_files": existing_manifest["status"].eq("processed").sum(),
                "empty_files": existing_manifest["status"].eq("empty").sum(),
                "failed_files": 0,
                "buildings": existing_manifest["buildings"].sum(),
                "status": "already_processed"
            })
            continue

    part_path.unlink(missing_ok=True)
    batch_parts, batch_manifest = [], []

    for archive_path, archive_sources in batch_sources.groupby("archive_path", sort=False):
        with ZipFile(archive_path, "r") as archive:
            for row in archive_sources.itertuples(index=False):
                try:
                    with archive.open(row.source_member) as stream:
                        tile_gdf = extract_citygml_stream_multipart(stream, row.source_file, "EPSG:25832", BW_BUILDING_TAG, BW_BUILDING_PART_TAG)

                    if tile_gdf.empty:
                        batch_manifest.append({
                            "source_key": row.source_key, "archive_name": row.archive_name,
                            "source_member": row.source_member, "status": "empty",
                            "buildings": 0, "missing_ids": 0, "missing_heights": 0,
                            "missing_geometries": 0, "minimum_height_m": np.nan,
                            "maximum_height_m": np.nan, "error": None
                        })
                        continue

                    tile_gdf["state_code"] = "BW"
                    tile_gdf["source_state"] = "Baden-Württemberg"
                    tile_gdf["source_crs"] = "EPSG:25832"
                    tile_gdf["citygml_version"] = "1.0"
                    tile_gdf["height_source"] = tile_gdf["height_method"]
                    tile_gdf["source_archive"] = row.archive_name
                    tile_gdf["source_member"] = row.source_member
                    tile_gdf["source_path"] = row.source_key
                    batch_parts.append(tile_gdf)

                    batch_manifest.append({
                        "source_key": row.source_key, "archive_name": row.archive_name,
                        "source_member": row.source_member, "status": "processed",
                        "buildings": len(tile_gdf),
                        "missing_ids": tile_gdf["lod2_id"].isna().sum(),
                        "missing_heights": tile_gdf["measured_height_m"].isna().sum(),
                        "missing_geometries": tile_gdf.geometry.isna().sum(),
                        "minimum_height_m": tile_gdf["measured_height_m"].min(),
                        "maximum_height_m": tile_gdf["measured_height_m"].max(),
                        "error": None
                    })

                except Exception as error:
                    batch_manifest.append({
                        "source_key": row.source_key, "archive_name": row.archive_name,
                        "source_member": row.source_member, "status": "failed",
                        "buildings": 0, "missing_ids": 0, "missing_heights": 0,
                        "missing_geometries": 0, "minimum_height_m": np.nan,
                        "maximum_height_m": np.nan, "error": str(error)
                    })

    if batch_parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(batch_parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    batch_manifest_df = pd.DataFrame(batch_manifest)
    batch_manifest_df.to_csv(manifest_path, index=False)

    bw_batch_summaries.append({
        "batch": batch_number,
        "source_files": len(batch_manifest_df),
        "processed_files": batch_manifest_df["status"].eq("processed").sum(),
        "empty_files": batch_manifest_df["status"].eq("empty").sum(),
        "failed_files": batch_manifest_df["status"].eq("failed").sum(),
        "buildings": batch_manifest_df["buildings"].sum(),
        "status": "processed"
    })

    del batch_parts, batch_manifest, batch_manifest_df
    gc.collect()

# ------------------------------------------------------------
# 4. Validate completed output
# ------------------------------------------------------------

bw_manifest_files = sorted(BW_MANIFEST_DIR.glob("lod2_manifest_BW_part_*.csv"))
bw_part_files = sorted(BW_PART_DIR.glob("lod2_buildings_BW_part_*.parquet"))
bw_manifests = pd.concat([pd.read_csv(path) for path in bw_manifest_files], ignore_index=True)
bw_batch_summary_df = pd.DataFrame(bw_batch_summaries)

bw_manifests.to_csv(REPORT_DIR / "lod2_processing_manifest_BW.csv", index=False)
bw_batch_summary_df.to_csv(REPORT_DIR / "lod2_batch_summary_BW.csv", index=False)

if not bw_part_files:
    raise RuntimeError("No Baden-Württemberg Parquet parts were produced.")

bw_part_stats, bw_height_counts, bw_footprint_counts = [], Counter(), Counter()

for part_path in tqdm(bw_part_files, desc="Validating Baden-Württemberg parts"):
    part = pd.read_parquet(part_path, columns=["lod2_id", "measured_height_m", "height_method", "footprint_method", "geometry"])
    bw_height_counts.update(part["height_method"].fillna("missing"))
    bw_footprint_counts.update(part["footprint_method"].fillna("missing"))
    bw_part_stats.append({
        "buildings": len(part),
        "missing_ids": part["lod2_id"].isna().sum(),
        "measured_heights": part["measured_height_m"].notna().sum(),
        "missing_heights": part["measured_height_m"].isna().sum(),
        "usable_geometries": part["geometry"].notna().sum(),
        "missing_geometries": part["geometry"].isna().sum(),
        "minimum_height_m": part["measured_height_m"].min(),
        "maximum_height_m": part["measured_height_m"].max()
    })
    del part
    gc.collect()

bw_part_stats = pd.DataFrame(bw_part_stats)
bw_height_summary = pd.DataFrame([{"height_method": key, "building_count": value} for key, value in bw_height_counts.items()], columns=["height_method", "building_count"]).sort_values("building_count", ascending=False)
bw_footprint_summary = pd.DataFrame([{"footprint_method": key, "building_count": value} for key, value in bw_footprint_counts.items()], columns=["footprint_method", "building_count"]).sort_values("building_count", ascending=False)

bw_expected_sources = set(bw_sources["source_key"])
bw_recorded_sources = set(bw_manifests["source_key"])

bw_summary = pd.DataFrame([{
    "state_code": "BW",
    "state_name": "Baden-Württemberg",
    "zip_archives": len(list(BW_ZIP_DIR.glob("*.zip"))),
    "source_files_expected": len(bw_expected_sources),
    "source_files_recorded": len(bw_recorded_sources),
    "files_with_buildings": bw_manifests["status"].eq("processed").sum(),
    "empty_files": bw_manifests["status"].eq("empty").sum(),
    "failed_files": bw_manifests["status"].eq("failed").sum(),
    "missing_source_files": len(bw_expected_sources - bw_recorded_sources),
    "unexpected_source_files": len(bw_recorded_sources - bw_expected_sources),
    "duplicate_manifest_sources": bw_manifests["source_key"].duplicated().sum(),
    "parquet_parts": len(bw_part_files),
    "buildings": bw_part_stats["buildings"].sum(),
    "missing_ids": bw_part_stats["missing_ids"].sum(),
    "measured_heights": bw_part_stats["measured_heights"].sum(),
    "missing_heights": bw_part_stats["missing_heights"].sum(),
    "usable_geometries": bw_part_stats["usable_geometries"].sum(),
    "missing_geometries": bw_part_stats["missing_geometries"].sum(),
    "minimum_height_m": bw_part_stats["minimum_height_m"].min(),
    "maximum_height_m": bw_part_stats["maximum_height_m"].max(),
    "output_directory": str(BW_PART_DIR)
}])

bw_summary.to_csv(REPORT_DIR / "lod2_extraction_summary_BW.csv", index=False)
bw_height_summary.to_csv(REPORT_DIR / "lod2_height_methods_BW.csv", index=False)
bw_footprint_summary.to_csv(REPORT_DIR / "lod2_footprint_methods_BW.csv", index=False)

source_registry.loc[source_registry["state_code"].eq("BW"), "download_status"] = "complete"
source_registry.to_csv(source_registry_path, index=False)

print("=" * 70)
print("BADEN-WÜRTTEMBERG FINAL LoD2 VALIDATION")
print("=" * 70)
display(bw_summary)

print("\nHeight methods:")
display(bw_height_summary)

print("\nFootprint methods:")
display(bw_footprint_summary)

if bw_manifests["status"].eq("failed").any():
    print("\nFailed source files:")
    display(bw_manifests[bw_manifests["status"] == "failed"].head(30))

Testing restored BW parser: 100%|██████████████████████████████████████████████| 5/5 [00:00<00:00, 25.41it/s]

RESTORED BADEN-WÜRTTEMBERG PARSER TEST


,archive_name,source_member,buildings_extracted,measured_heights,buildings_with_parts,usable_geometries,missing_geometries
0,LoD2_32_389_5278_2_bw.zip,LoD2_32_389_5278_2_bw/LoD2_32_389_5278_1_BW.gml,0,0,0,0,0
1,LoD2_32_461_5416_2_bw.zip,LoD2_32_461_5416_2_bw/LoD2_32_462_5417_1_BW.gml,7,7,1,7,0
2,LoD2_32_507_5298_2_bw.zip,LoD2_32_507_5298_2_bw/LoD2_32_508_5299_1_BW.gml,273,273,31,273,0
3,LoD2_32_547_5292_2_bw.zip,LoD2_32_547_5292_2_bw/LoD2_32_548_5292_1_BW.gml,24,24,6,24,0
4,LoD2_32_609_5396_2_bw.zip,LoD2_32_609_5396_2_bw/LoD2_32_609_5397_1_BW.gml,0,0,0,0,0


Validating Baden-Württemberg parts: 100%|██████████████████████████████████| 122/122 [00:36<00:00,  3.32it/s]

BADEN-WÜRTTEMBERG FINAL LoD2 VALIDATION


,state_code,state_name,zip_archives,source_files_expected,source_files_recorded,files_with_buildings,empty_files,failed_files,missing_source_files,unexpected_source_files,...,parquet_parts,buildings,missing_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,output_directory
0,BW,Baden-Württemberg,9157,36486,36486,30731,5755,0,0,0,...,122,6465296,0,6465296,0,6465292,4,0.143,269.0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...



Height methods:


,height_method,building_count
0,building_measured_height,5654392
1,maximum_building_part_height,810904



Footprint methods:


,footprint_method,building_count
0,ground_surface,6465292
1,missing,4


In [21]:
# ============================================================
# 88 — INVENTORY SAFE RAW-DATA DELETION CANDIDATES
# ============================================================

import subprocess
from pathlib import Path

COMPLETED_STATES = ["BW"]

def directory_size_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    return int(subprocess.check_output(["du", "-sb", str(path)]).decode().split()[0]) / 1024**3

cleanup_candidates = []

for state in COMPLETED_STATES:
    raw_state = RAW_DIR / state
    extracted_state = EXTRACTED_DIR / state
    summary_path = REPORT_DIR / f"lod2_extraction_summary_{state}.csv"
    parquet_files = list(extracted_state.rglob("*.parquet"))

    cleanup_candidates.append({
        "state_code": state,
        "raw_exists": raw_state.exists(),
        "raw_size_gb": directory_size_gb(raw_state),
        "summary_exists": summary_path.exists(),
        "parquet_files": len(parquet_files),
        "extracted_size_gb": directory_size_gb(extracted_state),
        "safe_to_delete_raw": raw_state.exists() and summary_path.exists() and len(parquet_files) > 0
    })

cleanup_candidates = pd.DataFrame(cleanup_candidates).sort_values("raw_size_gb", ascending=False).reset_index(drop=True)

print("=" * 70)
print("SAFE RAW-DATA CLEANUP CANDIDATES")
print("=" * 70)
display(cleanup_candidates)

safe_space_gb = cleanup_candidates.loc[cleanup_candidates["safe_to_delete_raw"], "raw_size_gb"].sum()
print(f"\nPotential space recovery: {safe_space_gb:,.2f} GB")

SAFE RAW-DATA CLEANUP CANDIDATES


,state_code,raw_exists,raw_size_gb,summary_exists,parquet_files,extracted_size_gb,safe_to_delete_raw
0,BW,True,15.347604,True,122,0.634176,True



Potential space recovery: 15.35 GB


In [22]:
# ============================================================
# 89 — DELETE RAW DATA FOR ALL COMPLETED STATES
# ============================================================

import shutil

SOURCE_METADATA_DIR = REPORT_DIR / "source_metadata"
SOURCE_METADATA_DIR.mkdir(parents=True, exist_ok=True)

PRESERVE_EXTENSIONS = {".csv", ".json", ".geojson", ".meta4", ".txt", ".html"}
MAX_METADATA_SIZE = 50 * 1024**2

deleted_records = []

for row in cleanup_candidates.itertuples(index=False):
    state = row.state_code
    raw_state = RAW_DIR / state

    if not row.safe_to_delete_raw:
        print(f"Skipping {state}: validation files or Parquet outputs are missing.")
        continue

    metadata_state = SOURCE_METADATA_DIR / state

    for source_path in raw_state.rglob("*"):
        if not source_path.is_file():
            continue

        if source_path.suffix.lower() in PRESERVE_EXTENSIONS and source_path.stat().st_size <= MAX_METADATA_SIZE:
            relative_path = source_path.relative_to(raw_state)
            destination = metadata_state / relative_path
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination)

    raw_size_gb = directory_size_gb(raw_state)
    shutil.rmtree(raw_state)

    deleted_records.append({
        "state_code": state,
        "deleted_raw_size_gb": raw_size_gb,
        "metadata_saved_to": str(metadata_state)
    })

deleted_records = pd.DataFrame(deleted_records)

print("=" * 70)
print("COMPLETED-STATE RAW DATA DELETED")
print("=" * 70)

display(deleted_records)

print(f"\nTotal space recovered: {deleted_records['deleted_raw_size_gb'].sum() if not deleted_records.empty else 0:,.2f} GB")
print(f"Metadata retained in : {SOURCE_METADATA_DIR}")

COMPLETED-STATE RAW DATA DELETED


,state_code,deleted_raw_size_gb,metadata_saved_to
0,BW,15.347604,/fast/home/o-olajuyigbe/data/germany_lod2/qual...



Total space recovered: 15.35 GB
Metadata retained in : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/source_metadata


In [27]:
# ============================================================
# 88A — NRW LoD2 DISCOVERY + SAMPLE TEST
# ============================================================

import re, html
from urllib.parse import urljoin, urlparse, parse_qs, unquote
from zipfile import ZipFile

NW_RAW_DIR, NW_TEST_DIR = RAW_DIR/"NW", RAW_DIR/"NW"/"test_tiles"
NW_RAW_DIR.mkdir(parents=True, exist_ok=True); NW_TEST_DIR.mkdir(parents=True, exist_ok=True)

NW_URL = "https://www.opengeodata.nrw.de/produkte/geobasis/3dg/lod2_gml/lod2_gml/"
r = requests.get(NW_URL, headers={"User-Agent":"Mozilla/5.0"}, timeout=(30,600)); r.raise_for_status()
text = html.unescape(r.text)

def get_filename(url):
    q = parse_qs(urlparse(url).query)
    for key in ("file","filename","name"):
        if q.get(key): return Path(unquote(q[key][0])).name
    return Path(unquote(urlparse(url).path)).name

# Extract URLs, relative paths and filenames from all XML/HTML text
tokens = re.findall(r'''(?i)(?:https?://|/)?[^\s<>"']*lod2[^\s<>"']*\.(?:zip|gml|citygml|xml)(?:\?[^\s<>"']*)?''', text)

records = []
for token in tokens:
    url = token if token.startswith("http") else urljoin(NW_URL, token.lstrip("/"))
    filename = get_filename(url)
    if filename.lower().endswith((".zip",".gml",".citygml",".xml")):
        records.append({"download_url":url, "filename":filename})

nw_downloads = pd.DataFrame(records, columns=["download_url","filename"]).drop_duplicates("download_url").sort_values("filename").reset_index(drop=True)

if nw_downloads.empty:
    print("Content-Type:", r.headers.get("content-type"))
    print("Response size:", len(r.content))
    print(text[:2000])
    raise RuntimeError("No NRW LoD2 filenames found; response preview printed above.")

nw_downloads.to_csv(NW_RAW_DIR/"nw_lod2_download_list.csv", index=False)

print("="*70)
print("NORTH RHINE-WESTPHALIA LoD2 DISCOVERY")
print("="*70)
print(f"\nResponse size      : {len(r.content)/1024**2:,.2f} MB")
print(f"References matched : {len(tokens):,}")
print(f"Download files     : {len(nw_downloads):,}")
print(f"Unique filenames   : {nw_downloads.filename.nunique():,}")
print(f"Duplicate filenames: {nw_downloads.filename.duplicated().sum():,}")
display(nw_downloads.head())

rows = nw_downloads.iloc[np.linspace(0, len(nw_downloads)-1, min(5,len(nw_downloads)), dtype=int)]
results = []

for row in tqdm(rows.itertuples(index=False), total=len(rows), desc="Testing NRW files"):
    path = NW_TEST_DIR/row.filename

    try:
        if not path.exists() or path.stat().st_size == 0:
            with requests.get(row.download_url, stream=True, timeout=(30,1800)) as response:
                response.raise_for_status()
                with open(path,"wb") as f:
                    for chunk in response.iter_content(1024**2):
                        if chunk: f.write(chunk)

        if path.suffix.lower() == ".zip":
            with ZipFile(path) as z:
                members = [x for x in z.namelist() if x.lower().endswith((".gml",".xml",".citygml"))]
                if not members: raise ValueError("No CityGML member in ZIP.")
                member = members[0]
                with z.open(member) as stream:
                    gdf = extract_citygml_stream_multipart(stream, Path(member).name, "EPSG:25832",
                        "{http://www.opengis.net/citygml/building/1.0}Building",
                        "{http://www.opengis.net/citygml/building/1.0}BuildingPart")
        else:
            member = path.name
            with open(path,"rb") as stream:
                gdf = extract_citygml_stream_multipart(stream, path.name, "EPSG:25832",
                    "{http://www.opengis.net/citygml/building/1.0}Building",
                    "{http://www.opengis.net/citygml/building/1.0}BuildingPart")

        results.append({
            "filename":row.filename, "source_member":member,
            "size_mb":path.stat().st_size/1024**2, "buildings":len(gdf),
            "heights":gdf.measured_height_m.notna().sum(),
            "missing_heights":gdf.measured_height_m.isna().sum(),
            "geometries":gdf.geometry.notna().sum(),
            "missing_geometries":gdf.geometry.isna().sum(), "error":None
        })

    except Exception as error:
        results.append({
            "filename":row.filename, "source_member":None,
            "size_mb":path.stat().st_size/1024**2 if path.exists() else 0,
            "buildings":-1, "heights":0, "missing_heights":0,
            "geometries":0, "missing_geometries":0, "error":str(error)
        })

nw_sample_results = pd.DataFrame(results)

print("\n"+"="*70)
print("NORTH RHINE-WESTPHALIA SAMPLE TEST")
print("="*70)
display(nw_sample_results)

NORTH RHINE-WESTPHALIA LoD2 DISCOVERY

Response size      : 3.09 MB
References matched : 35,022
Download files     : 35,022
Unique filenames   : 35,022
Duplicate filenames: 0


,download_url,filename
0,https://www.opengeodata.nrw.de/produkte/geobas...,LoD2_32_280_5657_1_NW.gml
1,https://www.opengeodata.nrw.de/produkte/geobas...,LoD2_32_280_5658_1_NW.gml
2,https://www.opengeodata.nrw.de/produkte/geobas...,LoD2_32_280_5659_1_NW.gml
3,https://www.opengeodata.nrw.de/produkte/geobas...,LoD2_32_280_5660_1_NW.gml
4,https://www.opengeodata.nrw.de/produkte/geobas...,LoD2_32_281_5652_1_NW.gml


Testing NRW files: 100%|███████████████████████████████████████████████████████| 5/5 [00:00<00:00,  7.05it/s]


NORTH RHINE-WESTPHALIA SAMPLE TEST


,filename,source_member,size_mb,buildings,heights,missing_heights,geometries,missing_geometries,error
0,LoD2_32_280_5657_1_NW.gml,LoD2_32_280_5657_1_NW.gml,1.449156,86,86,0,86,0,None
1,LoD2_32_348_5694_1_NW.gml,LoD2_32_348_5694_1_NW.gml,0.176940,3,3,0,3,0,None
2,LoD2_32_396_5777_1_NW.gml,LoD2_32_396_5777_1_NW.gml,0.203449,11,11,0,11,0,None
3,LoD2_32_450_5640_1_NW.gml,LoD2_32_450_5640_1_NW.gml,0.001481,0,0,0,0,0,None
4,LoD2_32_531_5745_1_NW.gml,LoD2_32_531_5745_1_NW.gml,0.001481,0,0,0,0,0,None


In [30]:
# ============================================================
# 89 — NORTH RHINE-WESTPHALIA: STREAM, PROCESS, VALIDATE
# ============================================================

import time, gc
from io import BytesIO
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

NW_OUT = EXTRACTED_DIR/"NW"
NW_PART = NW_OUT/"parts"
NW_MANIFEST = NW_OUT/"manifests"
for p in [NW_OUT, NW_PART, NW_MANIFEST]: p.mkdir(parents=True, exist_ok=True)

NW_BATCH_SIZE, NW_WORKERS, NW_RETRIES = 300, 6, 3
NW_BLDG = "{http://www.opengis.net/citygml/building/1.0}Building"
NW_PART_TAG = "{http://www.opengis.net/citygml/building/1.0}BuildingPart"

if "nw_downloads" not in globals():
    nw_downloads = pd.read_csv(RAW_DIR/"NW"/"nw_lod2_download_list.csv")

nw_downloads = nw_downloads.drop_duplicates("download_url").sort_values("filename").reset_index(drop=True)
batch_starts = list(range(0, len(nw_downloads), NW_BATCH_SIZE))

def process_nw_tile(url, filename):
    for attempt in range(1, NW_RETRIES + 1):
        try:
            r = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}, timeout=(30,1800))
            r.raise_for_status()

            gdf = extract_citygml_stream_multipart(
                BytesIO(r.content), filename, "EPSG:25832", NW_BLDG, NW_PART_TAG
            )

            if not gdf.empty:
                gdf["state_code"] = "NW"
                gdf["source_state"] = "North Rhine-Westphalia"
                gdf["source_crs"] = "EPSG:25832"
                gdf["citygml_version"] = "1.0"
                gdf["height_source"] = gdf["height_method"]
                gdf["source_url"] = url
                gdf["source_path"] = url

            manifest = {
                "source_url":url, "source_file":filename,
                "status":"empty" if gdf.empty else "processed",
                "buildings":len(gdf),
                "missing_ids":0 if gdf.empty else gdf["lod2_id"].isna().sum(),
                "missing_heights":0 if gdf.empty else gdf["measured_height_m"].isna().sum(),
                "missing_geometries":0 if gdf.empty else gdf.geometry.isna().sum(),
                "minimum_height_m":np.nan if gdf.empty else gdf["measured_height_m"].min(),
                "maximum_height_m":np.nan if gdf.empty else gdf["measured_height_m"].max(),
                "download_size_bytes":len(r.content), "error":None
            }
            return gdf, manifest

        except Exception as e:
            if attempt == NW_RETRIES:
                return None, {
                    "source_url":url, "source_file":filename, "status":"failed",
                    "buildings":0, "missing_ids":0, "missing_heights":0,
                    "missing_geometries":0, "minimum_height_m":np.nan,
                    "maximum_height_m":np.nan, "download_size_bytes":0,
                    "error":str(e)
                }
            time.sleep(attempt * 3)

batch_summaries = []

for start in tqdm(batch_starts, desc="Processing NRW batches"):
    number = start // NW_BATCH_SIZE + 1
    sources = nw_downloads.iloc[start:start+NW_BATCH_SIZE]
    part_path = NW_PART/f"lod2_buildings_NW_part_{number:04d}.parquet"
    manifest_path = NW_MANIFEST/f"lod2_manifest_NW_part_{number:04d}.csv"

    if manifest_path.exists():
        old = pd.read_csv(manifest_path)
        complete_manifest = (
            set(old["source_url"]) == set(sources["download_url"])
            and len(old) == len(sources)
            and not old["status"].eq("failed").any()
        )
        complete_output = old["buildings"].sum() == 0 or part_path.exists()

        if complete_manifest and complete_output:
            batch_summaries.append({
                "batch":number, "source_files":len(old),
                "processed_files":old["status"].eq("processed").sum(),
                "empty_files":old["status"].eq("empty").sum(),
                "failed_files":0, "buildings":old["buildings"].sum(),
                "status":"already_processed"
            })
            continue

    part_path.unlink(missing_ok=True)
    parts, records = [], []

    with ThreadPoolExecutor(max_workers=NW_WORKERS) as executor:
        futures = {
            executor.submit(process_nw_tile, row.download_url, row.filename): row.filename
            for row in sources.itertuples(index=False)
        }

        for future in as_completed(futures):
            gdf, record = future.result()
            records.append(record)
            if gdf is not None and not gdf.empty: parts.append(gdf)

    if parts:
        batch_gdf = gpd.GeoDataFrame(
            pd.concat(parts, ignore_index=True), geometry="geometry", crs="EPSG:25832"
        )
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    manifest = pd.DataFrame(records).sort_values("source_file").reset_index(drop=True)
    manifest.to_csv(manifest_path, index=False)

    batch_summaries.append({
        "batch":number, "source_files":len(manifest),
        "processed_files":manifest["status"].eq("processed").sum(),
        "empty_files":manifest["status"].eq("empty").sum(),
        "failed_files":manifest["status"].eq("failed").sum(),
        "buildings":manifest["buildings"].sum(), "status":"processed"
    })

    del parts, records, manifest
    gc.collect()

# Validation
manifest_files = sorted(NW_MANIFEST.glob("lod2_manifest_NW_part_*.csv"))
part_files = sorted(NW_PART.glob("lod2_buildings_NW_part_*.parquet"))

manifests = pd.concat([pd.read_csv(p) for p in manifest_files], ignore_index=True)
batch_summary = pd.DataFrame(batch_summaries)

manifests.to_csv(REPORT_DIR/"lod2_processing_manifest_NW.csv", index=False)
batch_summary.to_csv(REPORT_DIR/"lod2_batch_summary_NW.csv", index=False)

stats, height_counts, footprint_counts = [], Counter(), Counter()

for path in tqdm(part_files, desc="Validating NRW parts"):
    df = pd.read_parquet(
        path,
        columns=["lod2_id","measured_height_m","height_method","footprint_method","geometry"]
    )

    height_counts.update(df["height_method"].fillna("missing"))
    footprint_counts.update(df["footprint_method"].fillna("missing"))

    stats.append({
        "buildings":len(df),
        "missing_ids":df["lod2_id"].isna().sum(),
        "measured_heights":df["measured_height_m"].notna().sum(),
        "missing_heights":df["measured_height_m"].isna().sum(),
        "usable_geometries":df["geometry"].notna().sum(),
        "missing_geometries":df["geometry"].isna().sum(),
        "minimum_height_m":df["measured_height_m"].min(),
        "maximum_height_m":df["measured_height_m"].max()
    })
    del df
    gc.collect()

stats = pd.DataFrame(stats)
expected, recorded = set(nw_downloads["download_url"]), set(manifests["source_url"])

height_summary = pd.DataFrame(
    height_counts.items(), columns=["height_method","building_count"]
).sort_values("building_count", ascending=False)

footprint_summary = pd.DataFrame(
    footprint_counts.items(), columns=["footprint_method","building_count"]
).sort_values("building_count", ascending=False)

summary = pd.DataFrame([{
    "state_code":"NW",
    "state_name":"North Rhine-Westphalia",
    "source_files_expected":len(expected),
    "source_files_recorded":len(recorded),
    "files_with_buildings":manifests["status"].eq("processed").sum(),
    "empty_files":manifests["status"].eq("empty").sum(),
    "failed_files":manifests["status"].eq("failed").sum(),
    "missing_source_files":len(expected-recorded),
    "unexpected_source_files":len(recorded-expected),
    "duplicate_manifest_sources":manifests["source_url"].duplicated().sum(),
    "parquet_parts":len(part_files),
    "buildings":stats["buildings"].sum(),
    "missing_ids":stats["missing_ids"].sum(),
    "measured_heights":stats["measured_heights"].sum(),
    "missing_heights":stats["missing_heights"].sum(),
    "usable_geometries":stats["usable_geometries"].sum(),
    "missing_geometries":stats["missing_geometries"].sum(),
    "minimum_height_m":stats["minimum_height_m"].min(),
    "maximum_height_m":stats["maximum_height_m"].max(),
    "raw_gml_saved":False,
    "output_directory":str(NW_PART)
}])

summary.to_csv(REPORT_DIR/"lod2_extraction_summary_NW.csv", index=False)
height_summary.to_csv(REPORT_DIR/"lod2_height_methods_NW.csv", index=False)
footprint_summary.to_csv(REPORT_DIR/"lod2_footprint_methods_NW.csv", index=False)

source_registry.loc[source_registry["state_code"].eq("NW"), [
    "portal_url","download_url","download_method","format",
    "citygml_version","horizontal_crs","download_status","notes"
]] = [
    NW_URL, NW_URL, "Direct streamed CityGML tiles",
    "CityGML", "1.0", "EPSG:25832", "complete",
    "Processed directly from URLs; raw GML not retained"
]
source_registry.to_csv(source_registry_path, index=False)

print("="*70)
print("NORTH RHINE-WESTPHALIA FINAL LoD2 VALIDATION")
print("="*70)
display(summary)

print("\nHeight methods:")
display(height_summary)

print("\nFootprint methods:")
display(footprint_summary)

if manifests["status"].eq("failed").any():
    print("\nFailed source files:")
    display(manifests[manifests["status"].eq("failed")].head(30))

Validating NRW parts: 100%|████████████████████████████████████████████████| 117/117 [00:50<00:00,  2.30it/s]

NORTH RHINE-WESTPHALIA FINAL LoD2 VALIDATION


,state_code,state_name,source_files_expected,source_files_recorded,files_with_buildings,empty_files,failed_files,missing_source_files,unexpected_source_files,duplicate_manifest_sources,...,buildings,missing_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,raw_gml_saved,output_directory
0,NW,North Rhine-Westphalia,35022,35022,32646,2376,0,0,0,0,...,11870213,0,11870213,0,11870213,0,0.0,320.657,False,/fast/home/o-olajuyigbe/data/germany_lod2/extr...



Height methods:


,height_method,building_count
0,building_measured_height,10443722
1,maximum_building_part_height,1426491



Footprint methods:


,footprint_method,building_count
0,ground_surface,11870212
1,wall_bottom_convex_hull,1


In [32]:
# ============================================================
# 90 — CLEAN VALIDATED NRW RAW FILES
# ============================================================

import shutil

NW_RAW_DIR = RAW_DIR/"NW"
NW_SUMMARY_PATH = REPORT_DIR/"lod2_extraction_summary_NW.csv"
NW_PART_DIR = EXTRACTED_DIR/"NW"/"parts"
NW_METADATA_DIR = REPORT_DIR/"source_metadata"/"NW"

summary = pd.read_csv(NW_SUMMARY_PATH).iloc[0]
parts = list(NW_PART_DIR.glob("lod2_buildings_NW_part_*.parquet"))

valid = (
    summary["source_files_expected"] == summary["source_files_recorded"]
    and summary["failed_files"] == 0
    and summary["missing_source_files"] == 0
    and summary["missing_heights"] == 0
    and summary["missing_geometries"] == 0
    and len(parts) == summary["parquet_parts"]
)

if not valid:
    raise RuntimeError("NRW validation is incomplete. Raw files were not deleted.")

size_gb = sum(p.stat().st_size for p in NW_RAW_DIR.rglob("*") if p.is_file())/1024**3 if NW_RAW_DIR.exists() else 0

if NW_RAW_DIR.exists():
    NW_METADATA_DIR.mkdir(parents=True, exist_ok=True)
    for p in NW_RAW_DIR.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".csv",".json",".geojson",".txt",".xml"} and p.stat().st_size < 50*1024**2:
            dst = NW_METADATA_DIR/p.relative_to(NW_RAW_DIR)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
    shutil.rmtree(NW_RAW_DIR)

print("="*70)
print("NRW RAW CLEANUP COMPLETE")
print("="*70)
print(f"\nSpace recovered : {size_gb:,.3f} GB")
print(f"Parquet retained : {NW_PART_DIR}")
print(f"Metadata retained: {NW_METADATA_DIR}")

NRW RAW CLEANUP COMPLETE

Space recovered : 0.012 GB
Parquet retained : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/NW/parts
Metadata retained: /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/source_metadata/NW


In [6]:
# ============================================================
# 90 — RHINELAND-PALATINATE LoD2 MASS-DOWNLOAD DISCOVERY
# ============================================================

import re, json
from urllib.parse import urljoin, urlparse

RP_RAW_DIR = RAW_DIR/"RP"
RP_TEST_DIR = RP_RAW_DIR/"test_tiles"
RP_RAW_DIR.mkdir(parents=True, exist_ok=True)
RP_TEST_DIR.mkdir(parents=True, exist_ok=True)

RP_PORTAL_URL = "https://geoshop.rlp.de/files/anpassungen/hvd/index.html?config=./products/geb3dlo.json"
RP_CONFIG_URL = "https://geoshop.rlp.de/files/anpassungen/hvd/products/geb3dlo.json"

def collect_strings(obj, path="root"):
    rows = []
    if isinstance(obj, dict):
        for key, value in obj.items(): rows.extend(collect_strings(value, f"{path}.{key}"))
    elif isinstance(obj, list):
        for i, value in enumerate(obj): rows.extend(collect_strings(value, f"{path}[{i}]"))
    elif isinstance(obj, str):
        rows.append({"path":path, "value":obj})
    return rows

def resolve_reference(value, base_url):
    value = value.strip()
    if value.startswith(("http://","https://")): return value
    if value.startswith(("./","../","/")) or re.search(r"\.(json|geojson|csv|xml|zip|gml|citygml)(?:\?|$)", value, re.I): return urljoin(base_url, value)
    return None

queue, visited, references = [RP_CONFIG_URL], set(), []

while queue and len(visited) < 30:
    url = queue.pop(0)
    if url in visited: continue
    visited.add(url)

    response = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}, timeout=(30,300))
    response.raise_for_status()

    if len(response.content) > 25*1024**2:
        references.append({"source_url":url, "path":"response", "value":url, "resolved_url":url, "status":"skipped_large_metadata"})
        continue

    try:
        data = response.json()
        if url == RP_CONFIG_URL: (RP_RAW_DIR/"rp_lod2_config.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        strings = collect_strings(data)
    except Exception:
        text = response.text
        strings = [{"path":"text", "value":value} for value in re.findall(r'''https?://[^\s<>"']+|(?:\.\.?/|/)?[^\s<>"']+\.(?:json|geojson|csv|xml|zip|gml|citygml)(?:\?[^\s<>"']*)?''', text, re.I)]

    for item in strings:
        resolved = resolve_reference(item["value"], url)
        if not resolved: continue
        references.append({"source_url":url, "path":item["path"], "value":item["value"], "resolved_url":resolved, "status":"discovered"})
        suffix = Path(urlparse(resolved).path).suffix.lower()
        if suffix in {".json",".geojson"} and resolved not in visited and resolved not in queue: queue.append(resolved)

rp_references = pd.DataFrame(references).drop_duplicates(["source_url","path","resolved_url"]).reset_index(drop=True)
rp_references.to_csv(RP_RAW_DIR/"rp_lod2_discovered_references.csv", index=False)

rp_downloads = rp_references[
    rp_references["resolved_url"].str.lower().str.contains(r"\.(zip|gml|citygml)(?:\?|$)", regex=True, na=False)
].copy()

rp_downloads["filename"] = rp_downloads["resolved_url"].map(lambda x: Path(urlparse(x).path).name)
rp_downloads = rp_downloads[["resolved_url","filename"]].rename(columns={"resolved_url":"download_url"}).drop_duplicates("download_url").sort_values("filename").reset_index(drop=True)
rp_downloads.to_csv(RP_RAW_DIR/"rp_lod2_download_list.csv", index=False)

print("="*70)
print("RHINELAND-PALATINATE LoD2 DISCOVERY")
print("="*70)
print(f"\nMetadata documents read : {len(visited):,}")
print(f"References discovered   : {len(rp_references):,}")
print(f"Direct LoD2 files found : {len(rp_downloads):,}")
print(f"Unique filenames        : {rp_downloads.filename.nunique() if not rp_downloads.empty else 0:,}")

if not rp_downloads.empty:
    display(rp_downloads.head(10))
else:
    print("\nNo direct files were embedded in the metadata. Relevant configuration entries:")
    display(rp_references[rp_references["path"].str.contains("url|download|layer|source|service|index|grid|tile", case=False, regex=True)].head(50))

RHINELAND-PALATINATE LoD2 DISCOVERY

Metadata documents read : 1
References discovered   : 15
Direct LoD2 files found : 1
Unique filenames        : 1


/tmp/ipykernel_109491/1655913917.py:65: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  rp_references["resolved_url"].str.lower().str.contains(r"\.(zip|gml|citygml)(?:\?|$)", regex=True, na=False)


,download_url,filename
0,https://geobasis-rlp.de/data/geb3dlo/current/g...,"LoD2_32_{kachel,0,3}_{kachel,3}_2_RP.gml"


In [7]:
# ============================================================
# 90A — INSPECT RP CONFIGURATION AND TILE-INDEX SOURCES
# ============================================================

import json, re
from urllib.parse import urljoin

RP_CONFIG_URL = "https://geoshop.rlp.de/files/anpassungen/hvd/products/geb3dlo.json"
config = json.loads((RP_RAW_DIR/"rp_lod2_config.json").read_text(encoding="utf-8"))

def flatten(obj, path="root"):
    rows = []
    if isinstance(obj, dict):
        for k, v in obj.items(): rows.extend(flatten(v, f"{path}.{k}"))
    elif isinstance(obj, list):
        for i, v in enumerate(obj): rows.extend(flatten(v, f"{path}[{i}]"))
    else: rows.append({"path":path, "value":obj})
    return rows

rp_config_fields = pd.DataFrame(flatten(config))
rp_config_fields["value_text"] = rp_config_fields["value"].astype(str)
mask = rp_config_fields["path"].str.contains("url|source|layer|service|feature|tile|kachel|grid|download|template|index|wfs|geojson", case=False, regex=True) | rp_config_fields["value_text"].str.contains("http|kachel|wfs|geojson|gml|shape", case=False, regex=True)
rp_relevant_fields = rp_config_fields[mask].copy()

urls = []
for row in rp_relevant_fields.itertuples(index=False):
    for value in re.findall(r"https?://[^\s'\"}]+|(?:\.\.?/|/)[^\s'\"}]+", row.value_text):
        url = urljoin(RP_CONFIG_URL, value)
        if "{" not in url: urls.append({"path":row.path, "url":url})

url_checks = []
for item in pd.DataFrame(urls).drop_duplicates("url").itertuples(index=False):
    try:
        r = requests.get(item.url, headers={"User-Agent":"Mozilla/5.0"}, timeout=(20,120), stream=True)
        preview = next(r.iter_content(1000), b"").decode("utf-8", errors="ignore").replace("\n"," ")[:250]
        url_checks.append({"path":item.path, "url":item.url, "status":r.status_code, "content_type":r.headers.get("content-type"), "content_length":r.headers.get("content-length"), "preview":preview})
        r.close()
    except Exception as e:
        url_checks.append({"path":item.path, "url":item.url, "status":None, "content_type":None, "content_length":None, "preview":str(e)})

rp_url_checks = pd.DataFrame(url_checks)
rp_relevant_fields.to_csv(RP_RAW_DIR/"rp_relevant_config_fields.csv", index=False)
rp_url_checks.to_csv(RP_RAW_DIR/"rp_config_url_checks.csv", index=False)

print("="*70)
print("RHINELAND-PALATINATE CONFIGURATION FIELDS")
print("="*70)
display(rp_relevant_fields[["path","value_text"]])

print("\n"+"="*70)
print("RHINELAND-PALATINATE SOURCE URL CHECKS")
print("="*70)
display(rp_url_checks)

RHINELAND-PALATINATE CONFIGURATION FIELDS


,path,value_text
1,root.in_update,https://geobasis-rlp.de/data/geb3dlo/in_update...
2,root.metadata.csw_ResourceIdentifier,https://registry.gdi-de.org/id/de.rp.vermkv/0b...
4,root.selection_hierarchy[0].info_template,Sie haben gerade <br /> {name} <br /> ausgewäh...
5,root.selection_hierarchy[0].urls.data.scheme,https://geobasis-rlp.de/data/geb3dlo/current/m...
6,root.selection_hierarchy[0].urls.data.label,Datensatz herunterladen
8,root.selection_hierarchy[1].info_template,Sie haben gerade den Landkreis <br /> {ldkreis...
9,root.selection_hierarchy[1].urls.data.scheme,https://geobasis-rlp.de/data/geb3dlo/current/m...
10,root.selection_hierarchy[1].urls.data.label,Datensatz herunterladen
12,root.selection_hierarchy[2].info_template,Sie haben gerade die Verbandsgemeinde <br /> {...
13,root.selection_hierarchy[2].urls.data.scheme,https://geobasis-rlp.de/data/geb3dlo/current/m...



RHINELAND-PALATINATE SOURCE URL CHECKS


,path,url,status,content_type,content_length,preview
0,root.in_update,https://geobasis-rlp.de/data/geb3dlo/in_update...,404,text/html,1595,"<!doctype html><html lang=""en""><head><meta cha..."
1,root.metadata.csw_ResourceIdentifier,https://registry.gdi-de.org/id/de.rp.vermkv/0b...,404,text/html;charset=UTF-8,88,Anfrage ungültig: es existiert kein passender ...
2,root.selection_hierarchy[0].info_template,https://geoshop.rlp.de/>,404,text/html; charset=UTF-8,42715,"<!DOCTYPE html> <html lang=""de"" > <head> <meta..."
3,root.selection_hierarchy[0].info_template,https://geoshop.rlp.de/div>,404,text/html; charset=UTF-8,42745,"<!DOCTYPE html> <html lang=""de"" > <head> <meta..."
4,root.selection_hierarchy[0].urls.data.scheme,https://geobasis-rlp.de/data/geb3dlo/current/m...,200,application/metalink4+xml,4416164,"<?xml version=""1.0"" encoding=""UTF-8""?>\r <meta..."
5,root.base_layers[0].source_conf.url,https://sgx.geodatenzentrum.de/wms_basemapde,403,text/xml;charset=UTF-8,None,"<?xml version=""1.0"" encoding=""utf-8"" ?> <Servi..."
6,root.base_layers[1].source_conf.url,https://geo4.service24.rlp.de/wms/shade1m.fcgi,200,text/html,None,<HTML> <HEAD><TITLE>MapServer Message</TITLE><...
7,root.optional_layers[1].source_conf.url,https://geo5.service24.rlp.de/wms/karte_rp.fcgi,200,text/html,195,<HTML> <HEAD><TITLE>MapServer Message</TITLE><...
8,root.optional_layers[3].title,https://geoshop.rlp.de/Bauwerke,404,text/html; charset=UTF-8,42765,"<!DOCTYPE html> <html lang=""de"" > <head> <meta..."
9,root.optional_layers[3].source_conf.url,https://geo5.service24.rlp.de/wms/liegenschaft...,200,text/html,195,<HTML> <HEAD><TITLE>MapServer Message</TITLE><...


In [8]:
# ============================================================
# 90B — RP METALINK DISCOVERY + SAMPLE TEST
# ============================================================

from urllib.parse import urlparse
from io import BytesIO
from zipfile import ZipFile
from lxml import etree

RP_RAW_DIR, RP_TEST_DIR = RAW_DIR/"RP", RAW_DIR/"RP"/"test_tiles"
RP_TEST_DIR.mkdir(parents=True, exist_ok=True)

config = json.loads((RP_RAW_DIR/"rp_lod2_config.json").read_text(encoding="utf-8"))
RP_METALINK_URL = config["selection_hierarchy"][0]["urls"]["data"]["scheme"]

r = requests.get(RP_METALINK_URL, headers={"User-Agent":"Mozilla/5.0"}, timeout=(30,600))
r.raise_for_status()
metalink_path = RP_RAW_DIR/"rp_lod2_download.meta4"
metalink_path.write_bytes(r.content)

root = etree.fromstring(r.content, etree.XMLParser(recover=True, huge_tree=True))
records = []

for file_el in root.xpath("//*[local-name()='file']"):
    filename = file_el.get("name")
    urls = [u.text.strip() for u in file_el.xpath("./*[local-name()='url']") if u.text and u.text.strip()]

    for url in urls:
        name = filename or Path(urlparse(url).path).name
        if name.lower().endswith((".gml",".xml",".citygml",".zip")):
            records.append({"download_url":url, "filename":name})

rp_downloads = pd.DataFrame(records, columns=["download_url","filename"])
rp_downloads = rp_downloads.drop_duplicates("filename").sort_values("filename").reset_index(drop=True)

if rp_downloads.empty:
    raise RuntimeError("No Rheinland-Pfalz CityGML files were found in the Metalink.")

rp_downloads.to_csv(RP_RAW_DIR/"rp_lod2_download_list.csv", index=False)

print("="*70)
print("RHINELAND-PALATINATE LoD2 DISCOVERY")
print("="*70)
print(f"\nMetalink size      : {len(r.content)/1024**2:,.2f} MB")
print(f"Download files     : {len(rp_downloads):,}")
print(f"Unique URLs        : {rp_downloads.download_url.nunique():,}")
print(f"Unique filenames   : {rp_downloads.filename.nunique():,}")
print(f"Duplicate filenames: {rp_downloads.filename.duplicated().sum():,}")
print(f"ZIP files          : {rp_downloads.filename.str.lower().str.endswith('.zip').sum():,}")
print(f"GML/XML files      : {rp_downloads.filename.str.lower().str.endswith(('.gml','.xml','.citygml')).sum():,}")
display(rp_downloads.head())

rows = rp_downloads.iloc[np.linspace(0, len(rp_downloads)-1, min(5,len(rp_downloads)), dtype=int)]
results = []

for row in tqdm(rows.itertuples(index=False), total=len(rows), desc="Testing RP files"):
    path = RP_TEST_DIR/row.filename

    try:
        if not path.exists() or path.stat().st_size == 0:
            with requests.get(row.download_url, headers={"User-Agent":"Mozilla/5.0"}, stream=True, timeout=(30,1800)) as response:
                response.raise_for_status()
                with open(path,"wb") as f:
                    for chunk in response.iter_content(1024**2):
                        if chunk: f.write(chunk)

        if path.suffix.lower() == ".zip":
            with ZipFile(path) as z:
                members = [m for m in z.namelist() if m.lower().endswith((".gml",".xml",".citygml"))]
                if not members: raise ValueError("No CityGML member found in ZIP.")
                member = members[0]
                with z.open(member) as stream:
                    gdf = extract_citygml_stream_multipart(
                        stream, Path(member).name, "EPSG:25832",
                        "{http://www.opengis.net/citygml/building/1.0}Building",
                        "{http://www.opengis.net/citygml/building/1.0}BuildingPart"
                    )
        else:
            member = path.name
            with open(path,"rb") as stream:
                gdf = extract_citygml_stream_multipart(
                    stream, path.name, "EPSG:25832",
                    "{http://www.opengis.net/citygml/building/1.0}Building",
                    "{http://www.opengis.net/citygml/building/1.0}BuildingPart"
                )

        results.append({
            "filename":row.filename,
            "source_member":member,
            "size_mb":path.stat().st_size/1024**2,
            "buildings":len(gdf),
            "heights":gdf.measured_height_m.notna().sum(),
            "missing_heights":gdf.measured_height_m.isna().sum(),
            "geometries":gdf.geometry.notna().sum(),
            "missing_geometries":gdf.geometry.isna().sum(),
            "error":None
        })

    except Exception as e:
        results.append({
            "filename":row.filename,
            "source_member":None,
            "size_mb":path.stat().st_size/1024**2 if path.exists() else 0,
            "buildings":-1,
            "heights":0,
            "missing_heights":0,
            "geometries":0,
            "missing_geometries":0,
            "error":str(e)
        })

rp_sample_results = pd.DataFrame(results)

print("\n"+"="*70)
print("RHINELAND-PALATINATE SAMPLE TEST")
print("="*70)
display(rp_sample_results)

RHINELAND-PALATINATE LoD2 DISCOVERY

Metalink size      : 4.21 MB
Download files     : 10,526
Unique URLs        : 10,526
Unique filenames   : 10,526
Duplicate filenames: 0
ZIP files          : 0
GML/XML files      : 10,526


,download_url,filename
0,https://geobasis-rlp.de/data/geb3dlo/current/g...,LoD2_32_292_5548_2_RP.gml
1,https://geobasis-rlp.de/data/geb3dlo/current/g...,LoD2_32_292_5548_2_RP_meta.xml
2,https://geobasis-rlp.de/data/geb3dlo/current/g...,LoD2_32_292_5550_2_RP.gml
3,https://geobasis-rlp.de/data/geb3dlo/current/g...,LoD2_32_292_5550_2_RP_meta.xml
4,https://geobasis-rlp.de/data/geb3dlo/current/g...,LoD2_32_292_5552_2_RP.gml


Testing RP files: 100%|████████████████████████████████████████| 5/5 [00:00<00:00, 84.33it/s]


RHINELAND-PALATINATE SAMPLE TEST


,filename,source_member,size_mb,buildings,heights,missing_heights,geometries,missing_geometries,error
0,LoD2_32_292_5548_2_RP.gml,None,0.084129,-1,0,0,0,0,name 'extract_citygml_stream_multipart' is not...
1,LoD2_32_358_5512_2_RP.gml,None,4.866364,-1,0,0,0,0,name 'extract_citygml_stream_multipart' is not...
2,LoD2_32_394_5474_2_RP_meta.xml,None,0.000000,-1,0,0,0,0,404 Client Error: Not Found for url: https://g...
3,LoD2_32_420_5514_2_RP.gml,None,25.005703,-1,0,0,0,0,name 'extract_citygml_stream_multipart' is not...
4,LoD2_32_464_5476_2_RP_meta.xml,None,0.060479,-1,0,0,0,0,name 'extract_citygml_stream_multipart' is not...


In [9]:
# ============================================================
# 91 — RHINELAND-PALATINATE: STREAM, PROCESS, VALIDATE
# ============================================================

import time, gc
from io import BytesIO
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

RP_OUT, RP_PART, RP_MANIFEST = EXTRACTED_DIR/"RP", EXTRACTED_DIR/"RP"/"parts", EXTRACTED_DIR/"RP"/"manifests"
for p in [RP_OUT, RP_PART, RP_MANIFEST]: p.mkdir(parents=True, exist_ok=True)

RP_BATCH_SIZE, RP_WORKERS, RP_RETRIES = 300, 6, 3
RP_BLDG = "{http://www.opengis.net/citygml/building/1.0}Building"
RP_PART_TAG = "{http://www.opengis.net/citygml/building/1.0}BuildingPart"

if "rp_downloads" not in globals(): rp_downloads = pd.read_csv(RAW_DIR/"RP"/"rp_lod2_download_list.csv")

rp_downloads = rp_downloads[rp_downloads["filename"].str.lower().str.endswith(".gml")].drop_duplicates("filename").sort_values("filename").reset_index(drop=True)
rp_downloads.to_csv(RAW_DIR/"RP"/"rp_lod2_gml_download_list.csv", index=False)
batch_starts = list(range(0, len(rp_downloads), RP_BATCH_SIZE))

print("="*70)
print("RHINELAND-PALATINATE PROCESSING SETUP")
print("="*70)
print(f"\nCityGML files         : {len(rp_downloads):,}")
print(f"Metadata files removed: {10_526-len(rp_downloads):,}")
print(f"Batch size            : {RP_BATCH_SIZE:,}")
print(f"Expected Parquet parts: {len(batch_starts):,}")

def process_rp_tile(url, filename):
    for attempt in range(1, RP_RETRIES+1):
        try:
            r = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}, timeout=(30,1800))
            r.raise_for_status()
            gdf = extract_citygml_stream_multipart(BytesIO(r.content), filename, "EPSG:25832", RP_BLDG, RP_PART_TAG)

            if not gdf.empty:
                gdf["state_code"] = "RP"
                gdf["source_state"] = "Rhineland-Palatinate"
                gdf["source_crs"] = "EPSG:25832"
                gdf["citygml_version"] = "1.0"
                gdf["height_source"] = gdf["height_method"]
                gdf["source_url"] = url
                gdf["source_path"] = url

            return gdf, {
                "source_url":url, "source_file":filename, "status":"empty" if gdf.empty else "processed",
                "buildings":len(gdf), "missing_ids":0 if gdf.empty else gdf["lod2_id"].isna().sum(),
                "missing_heights":0 if gdf.empty else gdf["measured_height_m"].isna().sum(),
                "missing_geometries":0 if gdf.empty else gdf.geometry.isna().sum(),
                "minimum_height_m":np.nan if gdf.empty else gdf["measured_height_m"].min(),
                "maximum_height_m":np.nan if gdf.empty else gdf["measured_height_m"].max(),
                "download_size_bytes":len(r.content), "error":None
            }

        except Exception as e:
            if attempt == RP_RETRIES:
                return None, {
                    "source_url":url, "source_file":filename, "status":"failed", "buildings":0,
                    "missing_ids":0, "missing_heights":0, "missing_geometries":0,
                    "minimum_height_m":np.nan, "maximum_height_m":np.nan,
                    "download_size_bytes":0, "error":str(e)
                }
            time.sleep(attempt*3)

batch_summaries = []

for start in tqdm(batch_starts, desc="Processing Rhineland-Palatinate batches"):
    number = start//RP_BATCH_SIZE+1
    sources = rp_downloads.iloc[start:start+RP_BATCH_SIZE]
    part_path = RP_PART/f"lod2_buildings_RP_part_{number:04d}.parquet"
    manifest_path = RP_MANIFEST/f"lod2_manifest_RP_part_{number:04d}.csv"

    if manifest_path.exists():
        old = pd.read_csv(manifest_path)
        manifest_ok = set(old["source_url"]) == set(sources["download_url"]) and len(old) == len(sources) and not old["status"].eq("failed").any()
        output_ok = old["buildings"].sum() == 0 or part_path.exists()

        if manifest_ok and output_ok:
            batch_summaries.append({
                "batch":number, "source_files":len(old), "processed_files":old["status"].eq("processed").sum(),
                "empty_files":old["status"].eq("empty").sum(), "failed_files":0,
                "buildings":old["buildings"].sum(), "status":"already_processed"
            })
            continue

    part_path.unlink(missing_ok=True)
    parts, records = [], []

    with ThreadPoolExecutor(max_workers=RP_WORKERS) as executor:
        futures = {executor.submit(process_rp_tile, row.download_url, row.filename):row.filename for row in sources.itertuples(index=False)}
        for future in as_completed(futures):
            gdf, record = future.result()
            records.append(record)
            if gdf is not None and not gdf.empty: parts.append(gdf)

    if parts:
        batch_gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry="geometry", crs="EPSG:25832")
        batch_gdf.to_parquet(part_path, index=False)
        del batch_gdf

    manifest = pd.DataFrame(records).sort_values("source_file").reset_index(drop=True)
    manifest.to_csv(manifest_path, index=False)

    batch_summaries.append({
        "batch":number, "source_files":len(manifest), "processed_files":manifest["status"].eq("processed").sum(),
        "empty_files":manifest["status"].eq("empty").sum(), "failed_files":manifest["status"].eq("failed").sum(),
        "buildings":manifest["buildings"].sum(), "status":"processed"
    })

    del parts, records, manifest
    gc.collect()

manifest_files = sorted(RP_MANIFEST.glob("lod2_manifest_RP_part_*.csv"))
part_files = sorted(RP_PART.glob("lod2_buildings_RP_part_*.parquet"))
manifests = pd.concat([pd.read_csv(p) for p in manifest_files], ignore_index=True)
batch_summary = pd.DataFrame(batch_summaries)

manifests.to_csv(REPORT_DIR/"lod2_processing_manifest_RP.csv", index=False)
batch_summary.to_csv(REPORT_DIR/"lod2_batch_summary_RP.csv", index=False)

stats, height_counts, footprint_counts = [], Counter(), Counter()

for path in tqdm(part_files, desc="Validating Rhineland-Palatinate parts"):
    df = pd.read_parquet(path, columns=["lod2_id","measured_height_m","height_method","footprint_method","geometry"])
    height_counts.update(df["height_method"].fillna("missing"))
    footprint_counts.update(df["footprint_method"].fillna("missing"))
    stats.append({
        "buildings":len(df), "missing_ids":df["lod2_id"].isna().sum(),
        "measured_heights":df["measured_height_m"].notna().sum(),
        "missing_heights":df["measured_height_m"].isna().sum(),
        "usable_geometries":df["geometry"].notna().sum(),
        "missing_geometries":df["geometry"].isna().sum(),
        "minimum_height_m":df["measured_height_m"].min(),
        "maximum_height_m":df["measured_height_m"].max()
    })
    del df
    gc.collect()

stats = pd.DataFrame(stats)
expected, recorded = set(rp_downloads["download_url"]), set(manifests["source_url"])
height_summary = pd.DataFrame(height_counts.items(), columns=["height_method","building_count"]).sort_values("building_count", ascending=False)
footprint_summary = pd.DataFrame(footprint_counts.items(), columns=["footprint_method","building_count"]).sort_values("building_count", ascending=False)

summary = pd.DataFrame([{
    "state_code":"RP", "state_name":"Rhineland-Palatinate",
    "source_files_expected":len(expected), "source_files_recorded":len(recorded),
    "files_with_buildings":manifests["status"].eq("processed").sum(),
    "empty_files":manifests["status"].eq("empty").sum(),
    "failed_files":manifests["status"].eq("failed").sum(),
    "missing_source_files":len(expected-recorded),
    "unexpected_source_files":len(recorded-expected),
    "duplicate_manifest_sources":manifests["source_url"].duplicated().sum(),
    "parquet_parts":len(part_files), "buildings":stats["buildings"].sum(),
    "missing_ids":stats["missing_ids"].sum(),
    "measured_heights":stats["measured_heights"].sum(),
    "missing_heights":stats["missing_heights"].sum(),
    "usable_geometries":stats["usable_geometries"].sum(),
    "missing_geometries":stats["missing_geometries"].sum(),
    "minimum_height_m":stats["minimum_height_m"].min(),
    "maximum_height_m":stats["maximum_height_m"].max(),
    "raw_gml_saved":False, "output_directory":str(RP_PART)
}])

summary.to_csv(REPORT_DIR/"lod2_extraction_summary_RP.csv", index=False)
height_summary.to_csv(REPORT_DIR/"lod2_height_methods_RP.csv", index=False)
footprint_summary.to_csv(REPORT_DIR/"lod2_footprint_methods_RP.csv", index=False)

source_registry.loc[source_registry["state_code"].eq("RP"), ["portal_url","download_url","download_method","format","citygml_version","horizontal_crs","download_status","notes"]] = [
    RP_PORTAL_URL, RP_METALINK_URL, "Official Metalink; streamed CityGML tiles",
    "CityGML", "1.0", "EPSG:25832", "complete", "Metadata XML excluded; raw GML not retained"
]
source_registry.to_csv(source_registry_path, index=False)

print("="*70)
print("RHINELAND-PALATINATE FINAL LoD2 VALIDATION")
print("="*70)
display(summary)

print("\nHeight methods:")
display(height_summary)

print("\nFootprint methods:")
display(footprint_summary)

if manifests["status"].eq("failed").any():
    print("\nFailed source files:")
    display(manifests[manifests["status"].eq("failed")].head(30))

RHINELAND-PALATINATE PROCESSING SETUP

CityGML files         : 5,266
Metadata files removed: 5,260
Batch size            : 300
Expected Parquet parts: 18


Validating Rhineland-Palatinate parts: 100%|█████████████████| 18/18 [00:05<00:00,  3.01it/s]

RHINELAND-PALATINATE FINAL LoD2 VALIDATION


,state_code,state_name,source_files_expected,source_files_recorded,files_with_buildings,empty_files,failed_files,missing_source_files,unexpected_source_files,duplicate_manifest_sources,...,buildings,missing_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,raw_gml_saved,output_directory
0,RP,Rhineland-Palatinate,5266,5266,4876,390,0,0,0,0,...,3265854,0,3265854,0,3265853,1,0.0,246.5,False,/fast/home/o-olajuyigbe/data/germany_lod2/extr...



Height methods:


,height_method,building_count
0,building_measured_height,2716469
1,maximum_building_part_height,549385



Footprint methods:


,footprint_method,building_count
0,ground_surface,3265639
1,wall_bottom_convex_hull,214
2,missing,1


In [11]:
# ============================================================
# 92A — SAARLAND ORIGINAL DOWNLOAD-LINK INSPECTION
# ============================================================

import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse, parse_qs

if "sl_links" not in globals(): sl_links = pd.read_csv(SL_RAW_DIR/"sl_atom_links.csv")

sl_sections = sl_links[(sl_links["rel"].eq("section")) & sl_links["media_type"].str.contains("gml", case=False, na=False)].copy()
sl_sections["crs"] = sl_sections["title"].str.extract(r"EPSG:(\d+)", expand=False)
sl_sections["filename"] = sl_sections["href"].map(lambda x: Path(urlparse(x).path).name or parse_qs(urlparse(x).query).get("typeNames", ["saarland_lod2.gml"])[0].replace(":","_")+".gml")
sl_sections = sl_sections.drop_duplicates("href").reset_index(drop=True)

def probe_sl(row):
    try:
        r = requests.get(row.href, headers={"User-Agent":"Mozilla/5.0"}, stream=True, timeout=(30,180))
        preview = next(r.iter_content(1000), b"")
        result = {"href":row.href,"title":row.title,"crs":row.crs,"filename":row.filename,"status":r.status_code,"content_type":r.headers.get("content-type"),"content_length_mb":int(r.headers["content-length"])/1024**2 if r.headers.get("content-length","").isdigit() else np.nan,"xml_response":preview.lstrip().startswith(b"<?xml") or preview.lstrip().startswith(b"<"),"preview":preview.decode("utf-8", errors="ignore").replace("\n"," ")[:180]}
        r.close()
        return result
    except Exception as e:
        return {"href":row.href,"title":row.title,"crs":row.crs,"filename":row.filename,"status":None,"content_type":None,"content_length_mb":np.nan,"xml_response":False,"preview":str(e)}

results = []
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(probe_sl, row) for row in sl_sections.itertuples(index=False)]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Probing Saarland links"): results.append(future.result())

sl_probe = pd.DataFrame(results).sort_values(["crs","filename"]).reset_index(drop=True)
sl_probe.to_csv(SL_RAW_DIR/"sl_lod2_link_probe.csv", index=False)

print("="*70)
print("SAARLAND LoD2 ORIGINAL-LINK SUMMARY")
print("="*70)
print(f"\nSection links : {len(sl_sections):,}")
print(f"Unique URLs   : {sl_sections.href.nunique():,}")
print(f"Successful    : {sl_probe.status.eq(200).sum():,}")
print(f"Failed        : {sl_probe.status.ne(200).sum():,}")

print("\nLinks by CRS:")
display(sl_sections.groupby("crs", dropna=False).size().rename("files").reset_index())

print("\nResponse status by CRS:")
display(sl_probe.groupby(["crs","status"], dropna=False).size().rename("files").reset_index())

print("\nEPSG:25832 candidates:")
display(sl_probe[sl_probe["crs"].eq("25832")][["filename","status","content_type","content_length_mb","xml_response","title","href"]].head(20))

Probing Saarland links: 100%|████████████████████████████████| 72/72 [00:10<00:00,  6.91it/s]

SAARLAND LoD2 ORIGINAL-LINK SUMMARY

Section links : 72
Unique URLs   : 72
Successful    : 4
Failed        : 68

Links by CRS:


,crs,files
0,4258,72



Response status by CRS:


,crs,status,files
0,4258,200,4
1,4258,500,68



EPSG:25832 candidates:


,filename,status,content_type,content_length_mb,xml_response,title,href


In [12]:
# ============================================================
# 92B — INSPECT VALID SAARLAND LoD2 LAYERS
# ============================================================

from collections import Counter
from lxml import etree

if "sl_probe" not in globals(): sl_probe = pd.read_csv(SL_RAW_DIR/"sl_lod2_link_probe.csv")
sl_valid = sl_probe[sl_probe["status"].eq(200)].reset_index(drop=True)

print("="*70)
print("SAARLAND VALID DOWNLOAD LINKS")
print("="*70)
display(sl_valid[["title","content_type","content_length_mb","href"]])

results = []

for i, row in tqdm(sl_valid.iterrows(), total=len(sl_valid), desc="Inspecting Saarland layers"):
    r = requests.get(row["href"], headers={"User-Agent":"Mozilla/5.0"}, stream=True, timeout=(30,600))
    r.raise_for_status()

    chunks, size = [], 0
    for chunk in r.iter_content(1024**2):
        if not chunk: continue
        chunks.append(chunk); size += len(chunk)
        if size >= 5*1024**2: break
    r.close()

    content = b"".join(chunks)
    root = etree.fromstring(content, etree.XMLParser(recover=True, huge_tree=True))
    counts = Counter(etree.QName(el).localname for el in root.iter())
    namespaces = {k or "default":v for k,v in root.nsmap.items() if v}
    srs = sorted(set(root.xpath("//@srsName")))

    results.append({
        "layer":i+1,
        "title":row["title"],
        "preview_mb":len(content)/1024**2,
        "root_tag":etree.QName(root).localname,
        "feature_members":counts["featureMember"]+counts["member"],
        "Building":counts["Building"],
        "BuildingPart":counts["BuildingPart"],
        "Building3D":counts["Building3D"],
        "BuildingPart3D":counts["BuildingPart3D"],
        "measuredHeight":counts["measuredHeight"],
        "heightAboveGround":counts["heightAboveGround"],
        "GroundSurface":counts["GroundSurface"],
        "WallSurface":counts["WallSurface"],
        "RoofSurface":counts["RoofSurface"],
        "srs_names":"; ".join(srs[:5]),
        "building_namespaces":"; ".join(v for v in namespaces.values() if "building" in v.lower()),
        "href":row["href"]
    })

sl_structure = pd.DataFrame(results)
sl_structure.to_csv(SL_RAW_DIR/"sl_valid_layer_structure.csv", index=False)

print("\n"+"="*70)
print("SAARLAND VALID-LAYER STRUCTURE")
print("="*70)
display(sl_structure)

SAARLAND VALID DOWNLOAD LINKS


,title,content_type,content_length_mb,href
0,INSPIRE SL Gebäude - 3D LoD2 im CRS EPSG:4258 ...,application/gml+xml; version=3.2,NaN,https://geoportal.saarland.de/gdi-sl/inspirewf...
1,INSPIRE SL Gebäude - 3D LoD2 im CRS EPSG:4258 ...,application/gml+xml; version=3.2,NaN,https://geoportal.saarland.de/gdi-sl/inspirewf...
2,INSPIRE SL Gebäude - 3D LoD2 im CRS EPSG:4258 ...,application/gml+xml; version=3.2,NaN,https://geoportal.saarland.de/gdi-sl/inspirewf...
3,INSPIRE SL Gebäude - 3D LoD2 im CRS EPSG:4258 ...,application/gml+xml; version=3.2,NaN,https://geoportal.saarland.de/gdi-sl/inspirewf...


Inspecting Saarland layers:   0%|                                      | 0/4 [00:00<?, ?it/s]


HTTPError: 500 Server Error: Internal Server Error for url: https://geoportal.saarland.de/gdi-sl/inspirewfs_Gebaeude_3D_LoD2?SERVICE=WFS&REQUEST=GetFeature&VERSION=2.0.0&typeNames=bu-core3d:Building&outputFormat=application%2Fgml%2Bxml%3B%20version%3D3.2&BBOX=49.263493119121,6.74598165099,49.317990825495,6.8549770637375&EPSG=7423

In [13]:
# ============================================================
# 92C — SAARLAND WFS CAPABILITIES
# ============================================================

import time
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
from lxml import etree

if "sl_links" not in globals(): sl_links = pd.read_csv(SL_RAW_DIR/"sl_atom_links.csv")

service_urls = sl_links.loc[
    sl_links["rel"].eq("related") & sl_links["href"].str.contains("inspirewfs", case=False, na=False),
    "href"
].drop_duplicates().tolist()

if not service_urls: raise RuntimeError("No Saarland WFS service URL was found in the Atom feed.")

def add_params(url, **params):
    p = urlsplit(url); query = dict(parse_qsl(p.query)); query.update(params)
    return urlunsplit((p.scheme, p.netloc, p.path, urlencode(query), p.fragment))

session = requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0"})
responses = []

for service_url in service_urls:
    capabilities_url = add_params(service_url, SERVICE="WFS", REQUEST="GetCapabilities", VERSION="2.0.0")

    for attempt in range(1, 8):
        try:
            r = session.get(capabilities_url, timeout=(30,300))
            if r.status_code == 200 and b"FeatureType" in r.content:
                responses.append((service_url, capabilities_url, r.content))
                break
            error = f"HTTP {r.status_code}: {r.text[:200]}"
        except Exception as e:
            error = str(e)

        time.sleep(min(10*attempt, 60))
    else:
        print(f"Failed service: {service_url}\n{error}\n")

if not responses: raise RuntimeError("The Saarland WFS remained unavailable after sequential retries.")

records = []

for service_url, capabilities_url, content in responses:
    root = etree.fromstring(content, etree.XMLParser(recover=True, huge_tree=True))

    for ft in root.xpath("//*[local-name()='FeatureType']"):
        name = ft.xpath("string(./*[local-name()='Name'])").strip()
        title = ft.xpath("string(./*[local-name()='Title'])").strip()
        default_crs = ft.xpath("string(./*[local-name()='DefaultCRS' or local-name()='DefaultSRS'])").strip()
        other_crs = [x.strip() for x in ft.xpath("./*[local-name()='OtherCRS' or local-name()='OtherSRS']/text()") if x.strip()]
        records.append({
            "service_url":service_url,
            "capabilities_url":capabilities_url,
            "feature_type":name,
            "title":title,
            "default_crs":default_crs,
            "other_crs":"; ".join(other_crs)
        })

sl_feature_types = pd.DataFrame(records).drop_duplicates(["service_url","feature_type"]).reset_index(drop=True)
sl_feature_types.to_csv(SL_RAW_DIR/"sl_wfs_feature_types.csv", index=False)

print("="*70)
print("SAARLAND WFS CAPABILITIES")
print("="*70)
print(f"\nServices available : {len(responses):,}")
print(f"Feature types found: {len(sl_feature_types):,}")
display(sl_feature_types)

SAARLAND WFS CAPABILITIES

Services available : 1
Feature types found: 2


,service_url,capabilities_url,feature_type,title,default_crs,other_crs
0,https://geoportal.saarland.de/gdi-sl/inspirewf...,https://geoportal.saarland.de/gdi-sl/inspirewf...,bu-core3d:Building,Gebäude,urn:ogc:def:crs:EPSG::7423,urn:ogc:def:crs:EPSG::4258; urn:ogc:def:crs:EP...
1,https://geoportal.saarland.de/gdi-sl/inspirewf...,https://geoportal.saarland.de/gdi-sl/inspirewf...,bu-core3d:BuildingPart,Gebäudeteil,urn:ogc:def:crs:EPSG::7423,urn:ogc:def:crs:EPSG::4258; urn:ogc:def:crs:EP...


In [15]:
# ============================================================
# 92D — SAARLAND WFS COUNTS + SAMPLE STRUCTURE
# ============================================================

import time
from collections import Counter
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
from lxml import etree

if "sl_feature_types" not in globals(): sl_feature_types = pd.read_csv(SL_RAW_DIR/"sl_wfs_feature_types.csv")

service_url = sl_feature_types.iloc[0]["service_url"]

def wfs_url(**params):
    p = urlsplit(service_url); q = dict(parse_qsl(p.query)); q.update(params)
    return urlunsplit((p.scheme,p.netloc,p.path,urlencode(q),p.fragment))

def get_wfs(params, retries=6):
    error = None
    for attempt in range(1,retries+1):
        r = requests.get(wfs_url(**params), headers={"User-Agent":"Mozilla/5.0"}, timeout=(30,600))
        if r.status_code == 200 and b"ExceptionReport" not in r.content: return r
        error = f"HTTP {r.status_code}: {r.text[:250]}"
        time.sleep(min(attempt*10,60))
    raise RuntimeError(error)

results, tag_rows = [], []

for row in sl_feature_types.itertuples(index=False):
    typename = row.feature_type

    hits = get_wfs({"SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature","TYPENAMES":typename,"RESULTTYPE":"hits"})
    hits_root = etree.fromstring(hits.content, etree.XMLParser(recover=True, huge_tree=True))
    matched = hits_root.get("numberMatched") or hits_root.get("numberOfFeatures")

    sample = get_wfs({"SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature","TYPENAMES":typename,"COUNT":"20","SRSNAME":"urn:ogc:def:crs:EPSG::25832"})
    sample_path = SL_TEST_DIR/f"{typename.replace(':','_')}_sample.gml"
    sample_path.write_bytes(sample.content)

    root = etree.fromstring(sample.content, etree.XMLParser(recover=True, huge_tree=True))
    counts = Counter(etree.QName(el.tag).localname for el in root.iter() if isinstance(el.tag,str))
    srs_names = sorted(set(root.xpath("//@srsName")))
    namespaces = {k or "default":v for k,v in root.nsmap.items() if v}

    for tag,count in counts.most_common(30): tag_rows.append({"feature_type":typename,"tag":tag,"count":count})

    results.append({
        "feature_type":typename,
        "number_matched":matched,
        "sample_features":counts["member"]+counts["featureMember"],
        "Building":counts["Building"],
        "BuildingPart":counts["BuildingPart"],
        "heightAboveGround":counts["heightAboveGround"],
        "HeightAboveGround":counts["HeightAboveGround"],
        "value":counts["value"],
        "geometry3DLoD2":counts["geometry3DLoD2"],
        "MultiSurface":counts["MultiSurface"],
        "Polygon":counts["Polygon"],
        "posList":counts["posList"],
        "srs_names":"; ".join(srs_names),
        "building_namespaces":"; ".join(v for v in namespaces.values() if "building" in v.lower()),
        "sample_file":str(sample_path)
    })

sl_sample_summary = pd.DataFrame(results)
sl_sample_tags = pd.DataFrame(tag_rows)
sl_sample_summary.to_csv(SL_RAW_DIR/"sl_wfs_sample_summary.csv", index=False)
sl_sample_tags.to_csv(SL_RAW_DIR/"sl_wfs_sample_tags.csv", index=False)

print("="*70)
print("SAARLAND WFS FEATURE COUNTS AND SAMPLE STRUCTURE")
print("="*70)
display(sl_sample_summary)

print("\nMost frequent sample tags:")
display(sl_sample_tags)

SAARLAND WFS FEATURE COUNTS AND SAMPLE STRUCTURE


,feature_type,number_matched,sample_features,Building,BuildingPart,heightAboveGround,HeightAboveGround,value,geometry3DLoD2,MultiSurface,Polygon,posList,srs_names,building_namespaces,sample_file
0,bu-core3d:Building,738006,20,20,0,20,20,19,19,19,181,200,urn:ogc:def:crs:EPSG::25832,,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...
1,bu-core3d:BuildingPart,93547,20,0,20,20,20,20,20,20,255,275,urn:ogc:def:crs:EPSG::25832,,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...



Most frequent sample tags:


,feature_type,tag,count
0,bu-core3d:Building,posList,200
1,bu-core3d:Building,surfaceMember,181
2,bu-core3d:Building,Polygon,181
3,bu-core3d:Building,exterior,181
4,bu-core3d:Building,LinearRing,181
5,bu-core3d:Building,currentUse,40
6,bu-core3d:Building,member,20
7,bu-core3d:Building,Building,20
8,bu-core3d:Building,identifier,20
9,bu-core3d:Building,beginLifespanVersion,20


In [16]:
# ============================================================
# 92E — INSPECT SAARLAND INSPIRE BUILDING STRUCTURE
# ============================================================

from lxml import etree
from collections import Counter

def local_name(el): return etree.QName(el.tag).localname if isinstance(el.tag,str) else None
def first_text(el, name):
    values = el.xpath(f".//*[local-name()='{name}']/text()")
    return values[0].strip() if values and values[0].strip() else None

files = {
    "Building":SL_TEST_DIR/"bu-core3d_Building_sample.gml",
    "BuildingPart":SL_TEST_DIR/"bu-core3d_BuildingPart_sample.gml"
}

feature_rows, height_rows, reference_rows, tag_rows = [], [], [], []

for feature_type, path in files.items():
    root = etree.parse(path, etree.XMLParser(recover=True, huge_tree=True)).getroot()
    features = root.xpath(f"//*[local-name()='{feature_type}']")

    for i, feature in enumerate(features[:5], 1):
        gml_id = next((v for k,v in feature.attrib.items() if k.endswith("}id")), None)
        geometry = feature.xpath(".//*[local-name()='geometry3DLoD2']/*[local-name()='MultiSurface']")
        polygons = geometry[0].xpath(".//*[local-name()='Polygon']") if geometry else []
        polygon_stats = []

        for polygon in polygons:
            pos = polygon.xpath(".//*[local-name()='exterior']//*[local-name()='posList'][1]")
            if not pos or not pos[0].text: continue
            values = np.fromstring(pos[0].text, sep=" ")
            dim = int(pos[0].get("srsDimension") or 3)
            if dim >= 3 and len(values)%dim == 0:
                xyz = values.reshape(-1,dim)
                polygon_stats.append((xyz[:,2].min(), xyz[:,2].max(), xyz[:,2].ptp()))

        z_min = min((x[0] for x in polygon_stats), default=np.nan)
        z_max = max((x[1] for x in polygon_stats), default=np.nan)

        feature_rows.append({
            "feature_type":feature_type, "sample_number":i, "gml_id":gml_id,
            "local_id":first_text(feature,"localId"), "identifier":first_text(feature,"identifier"),
            "begin_lifespan":first_text(feature,"beginLifespanVersion"),
            "condition":first_text(feature,"conditionOfConstruction"),
            "current_use":first_text(feature,"currentUse"),
            "polygon_count":len(polygons), "z_min_m":z_min, "z_max_m":z_max,
            "z_extent_m":z_max-z_min if pd.notna(z_min) and pd.notna(z_max) else np.nan,
            "horizontal_polygons":sum(x[2]<=0.05 for x in polygon_stats),
            "minimum_z_polygons":sum(abs(x[0]-z_min)<=0.05 and x[2]<=0.05 for x in polygon_stats) if pd.notna(z_min) else 0
        })

        heights = feature.xpath(".//*[local-name()='HeightAboveGround']")
        if not heights: height_rows.append({"feature_type":feature_type,"sample_number":i,"gml_id":gml_id,"height_m":np.nan,"uom":None,"height_reference":None,"low_reference":None,"status":None})

        for height in heights:
            value_el = height.xpath(".//*[local-name()='value'][1]")
            height_rows.append({
                "feature_type":feature_type, "sample_number":i, "gml_id":gml_id,
                "height_m":float(value_el[0].text) if value_el and value_el[0].text else np.nan,
                "uom":value_el[0].get("uom") if value_el else None,
                "height_reference":first_text(height,"heightReference"),
                "low_reference":first_text(height,"lowReference"),
                "status":first_text(height,"status")
            })

        for el in feature.iter():
            if not isinstance(el.tag,str): continue
            tag_rows.append({"feature_type":feature_type,"tag":local_name(el)})
            for key,value in el.attrib.items():
                if key.endswith("}href") or key=="href":
                    reference_rows.append({"feature_type":feature_type,"sample_number":i,"gml_id":gml_id,"relation_tag":local_name(el),"href":value})

feature_summary = pd.DataFrame(feature_rows)
height_summary = pd.DataFrame(height_rows)
reference_summary = pd.DataFrame(reference_rows)
tag_summary = pd.DataFrame(tag_rows).value_counts(["feature_type","tag"]).rename("count").reset_index()

feature_summary.to_csv(SL_RAW_DIR/"sl_feature_structure.csv", index=False)
height_summary.to_csv(SL_RAW_DIR/"sl_height_structure.csv", index=False)
reference_summary.to_csv(SL_RAW_DIR/"sl_reference_structure.csv", index=False)

print("="*70)
print("SAARLAND INSPIRE FEATURE STRUCTURE")
print("="*70)
display(feature_summary)

print("\nHeight structure:")
display(height_summary)

print("\nReference relationships:")
display(reference_summary)

print("\nRelevant feature tags:")
display(tag_summary[tag_summary["tag"].str.contains("building|part|geometry|height|reference|local|identifier", case=False, regex=True)])

SAARLAND INSPIRE FEATURE STRUCTURE


,feature_type,sample_number,gml_id,local_id,identifier,begin_lifespan,condition,current_use,polygon_count,z_min_m,z_max_m,z_extent_m,horizontal_polygons,minimum_z_polygons
0,Building,1,Building_DESL64685862E1EB,Building_DESL64685862E1EB,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-02-21T00:00:00,None,None,0,NaN,NaN,NaN,0,0
1,Building,2,Building_DESL6471706EC4A5,Building_DESL6471706EC4A5,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-02-20T00:00:00,None,None,0,NaN,NaN,NaN,0,0
2,Building,3,Building_DESL647171EEED51,Building_DESL647171EEED51,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-02-23T00:00:00,None,None,0,NaN,NaN,NaN,0,0
3,Building,4,Building_DESL6471722BDD9E,Building_DESL6471722BDD9E,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-02-21T00:00:00,None,None,0,NaN,NaN,NaN,0,0
4,Building,5,Building_DESL647173EAAB87,Building_DESL647173EAAB87,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-02-22T00:00:00,None,None,0,NaN,NaN,NaN,0,0
5,BuildingPart,1,BuildingPart_UUID_00012ff2-f29f-4f93-8f18-e065...,BuildingPart_UUID_00012ff2-f29f-4f93-8f18-e065...,https://registry.gdi-de.org/id/de.sl.inspire.c...,2025-11-28T00:00:00,None,None,0,NaN,NaN,NaN,0,0
6,BuildingPart,2,BuildingPart_UUID_0002b3c5-a808-4ae8-bcfc-0da8...,BuildingPart_UUID_0002b3c5-a808-4ae8-bcfc-0da8...,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-02-28T00:00:00,None,None,0,NaN,NaN,NaN,0,0
7,BuildingPart,3,BuildingPart_UUID_0002c43c-af71-4852-8dfd-0c9f...,BuildingPart_UUID_0002c43c-af71-4852-8dfd-0c9f...,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-02-29T00:00:00,None,None,0,NaN,NaN,NaN,0,0
8,BuildingPart,4,BuildingPart_UUID_0002c49b-744e-49db-a831-aa7d...,BuildingPart_UUID_0002c49b-744e-49db-a831-aa7d...,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-03-04T00:00:00,None,None,0,NaN,NaN,NaN,0,0
9,BuildingPart,5,BuildingPart_UUID_0004073b-8d2d-4a03-824f-7b7f...,BuildingPart_UUID_0004073b-8d2d-4a03-824f-7b7f...,https://registry.gdi-de.org/id/de.sl.inspire.c...,2024-02-19T00:00:00,None,None,0,NaN,NaN,NaN,0,0



Height structure:


,feature_type,sample_number,gml_id,height_m,uom,height_reference,low_reference,status
0,Building,1,Building_DESL64685862E1EB,1.908,m,None,None,None
1,Building,2,Building_DESL6471706EC4A5,2.450,m,None,None,None
2,Building,3,Building_DESL647171EEED51,17.573,m,None,None,None
3,Building,4,Building_DESL6471722BDD9E,1.694,m,None,None,None
4,Building,5,Building_DESL647173EAAB87,2.470,m,None,None,None
5,BuildingPart,1,BuildingPart_UUID_00012ff2-f29f-4f93-8f18-e065...,3.967,m,None,None,None
6,BuildingPart,2,BuildingPart_UUID_0002b3c5-a808-4ae8-bcfc-0da8...,10.532,m,None,None,None
7,BuildingPart,3,BuildingPart_UUID_0002c43c-af71-4852-8dfd-0c9f...,5.755,m,None,None,None
8,BuildingPart,4,BuildingPart_UUID_0002c49b-744e-49db-a831-aa7d...,9.707,m,None,None,None
9,BuildingPart,5,BuildingPart_UUID_0004073b-8d2d-4a03-824f-7b7f...,7.517,m,None,None,None



Reference relationships:


,feature_type,sample_number,gml_id,relation_tag,href
0,Building,1,Building_DESL64685862E1EB,status,http://inspire.ec.europa.eu/codelist/HeightSta...
1,Building,1,Building_DESL64685862E1EB,currentUse,https://registry.gdi-de.org/codelist/de.adv-on...
2,Building,2,Building_DESL6471706EC4A5,status,http://inspire.ec.europa.eu/codelist/HeightSta...
3,Building,2,Building_DESL6471706EC4A5,currentUse,https://registry.gdi-de.org/codelist/de.adv-on...
4,Building,3,Building_DESL647171EEED51,status,http://inspire.ec.europa.eu/codelist/HeightSta...
5,Building,3,Building_DESL647171EEED51,currentUse,https://registry.gdi-de.org/codelist/de.adv-on...
6,Building,4,Building_DESL6471722BDD9E,status,http://inspire.ec.europa.eu/codelist/HeightSta...
7,Building,4,Building_DESL6471722BDD9E,currentUse,https://registry.gdi-de.org/codelist/de.adv-on...
8,Building,5,Building_DESL647173EAAB87,status,http://inspire.ec.europa.eu/codelist/HeightSta...
9,Building,5,Building_DESL647173EAAB87,currentUse,https://registry.gdi-de.org/codelist/de.adv-on...



Relevant feature tags:


,feature_type,tag,count
12,Building,HeightAboveGround,5
19,Building,Identifier,5
20,Building,BuildingGeometry3DLoD2,5
24,Building,ExternalReference,5
25,Building,Building,5
26,Building,identifier,5
27,Building,horizontalGeometryEstimatedAccuracy,5
28,Building,heightReference,5
29,Building,heightAboveGround,5
30,Building,geometryMultiSurface,5


In [18]:
# ============================================================
# 92F — SAARLAND INSPIRE BUILDING PARSER TEST
# ============================================================
import geopandas as gpd
import time
from io import BytesIO
from lxml import etree
from shapely.geometry import Polygon
from shapely.ops import unary_union
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode

SL_TYPENAME = "bu-core3d:Building"
SL_CRS = "EPSG:25832"
service_url = sl_feature_types.iloc[0]["service_url"]

def sl_wfs_url(**params):
    p = urlsplit(service_url); q = dict(parse_qsl(p.query)); q.update(params)
    return urlunsplit((p.scheme,p.netloc,p.path,urlencode(q),p.fragment))

def sl_get_wfs(params, retries=6):
    for attempt in range(1,retries+1):
        r = requests.get(sl_wfs_url(**params), headers={"User-Agent":"Mozilla/5.0"}, timeout=(30,600))
        if r.status_code == 200 and b"ExceptionReport" not in r.content: return r
        error = f"HTTP {r.status_code}: {r.text[:250]}"
        time.sleep(min(attempt*10,60))
    raise RuntimeError(error)

def sl_text(element, name):
    values = element.xpath(f".//*[local-name()='{name}']/text()")
    return values[0].strip() if values and values[0].strip() else None

def sl_href(element, name):
    matches = element.xpath(f".//*[local-name()='{name}'][1]")
    if not matches: return None
    return next((value for key,value in matches[0].attrib.items() if key.endswith("}href") or key=="href"), sl_text(element,name))

def sl_ring(ring):
    pos = ring.xpath(".//*[local-name()='posList'][1]")
    if not pos or not pos[0].text: return []
    values = np.fromstring(pos[0].text, sep=" ")
    dim = int(pos[0].get("srsDimension") or 3)
    if dim not in {2,3} or len(values)%dim: dim = 3 if len(values)%3 == 0 else 2
    coords = values.reshape(-1,dim)[:,:2]
    if len(coords) >= 3 and not np.array_equal(coords[0],coords[-1]): coords = np.vstack([coords,coords[0]])
    return [tuple(x) for x in coords]

def sl_footprint(building):
    polygons = []

    for polygon_el in building.xpath(".//*[local-name()='geometry3DLoD2']//*[local-name()='Polygon']"):
        exterior = polygon_el.xpath("./*[local-name()='exterior']//*[local-name()='LinearRing'][1]")
        if not exterior: continue
        shell = sl_ring(exterior[0])
        if len(shell) < 4: continue

        holes = []
        for interior in polygon_el.xpath("./*[local-name()='interior']//*[local-name()='LinearRing']"):
            ring = sl_ring(interior)
            if len(ring) >= 4: holes.append(ring)

        try:
            polygon = Polygon(shell, holes)
            if not polygon.is_valid: polygon = polygon.buffer(0)
            if not polygon.is_empty and polygon.area > 0.01: polygons.append(polygon)
        except Exception:
            continue

    if not polygons: return None

    geometry = unary_union(polygons)
    if not geometry.is_valid: geometry = geometry.buffer(0)
    return geometry if not geometry.is_empty and geometry.geom_type in {"Polygon","MultiPolygon"} else None

def extract_sl_buildings(content):
    records = []
    root = etree.fromstring(content, etree.XMLParser(recover=True, huge_tree=True))

    for building in root.xpath("//*[local-name()='Building']"):
        gml_id = next((value for key,value in building.attrib.items() if key.endswith("}id")), None)
        height_el = building.xpath(".//*[local-name()='heightAboveGround']/*[local-name()='HeightAboveGround'][1]")
        value_el = height_el[0].xpath(".//*[local-name()='value'][1]") if height_el else []
        height = float(value_el[0].text) if value_el and value_el[0].text else np.nan

        records.append({
            "lod2_id":sl_text(building,"localId") or gml_id,
            "gml_id":gml_id,
            "creation_date":sl_text(building,"beginLifespanVersion"),
            "function":sl_href(building,"currentUse"),
            "condition":sl_href(building,"conditionOfConstruction"),
            "measured_height_m":height,
            "height_method":"height_above_ground" if pd.notna(height) else "missing",
            "footprint_method":"projected_3d_surfaces",
            "geometry":sl_footprint(building)
        })

    return gpd.GeoDataFrame(records, geometry="geometry", crs=SL_CRS)

sample = sl_get_wfs({
    "SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature",
    "TYPENAMES":SL_TYPENAME,"COUNT":"200","STARTINDEX":"0",
    "SRSNAME":"urn:ogc:def:crs:EPSG::25832"
})

sl_parser_test = extract_sl_buildings(sample.content)
sl_parser_test.to_parquet(SL_TEST_DIR/"sl_inspire_parser_test.parquet", index=False)

print("="*70)
print("SAARLAND INSPIRE PARSER TEST")
print("="*70)
print(f"\nBuildings extracted : {len(sl_parser_test):,}")
print(f"Unique IDs          : {sl_parser_test.lod2_id.nunique():,}")
print(f"Measured heights    : {sl_parser_test.measured_height_m.notna().sum():,}")
print(f"Missing heights     : {sl_parser_test.measured_height_m.isna().sum():,}")
print(f"Usable geometries   : {sl_parser_test.geometry.notna().sum():,}")
print(f"Missing geometries  : {sl_parser_test.geometry.isna().sum():,}")
print(f"Minimum height      : {sl_parser_test.measured_height_m.min():,.3f} m")
print(f"Maximum height      : {sl_parser_test.measured_height_m.max():,.3f} m")
display(sl_parser_test.head())

SAARLAND INSPIRE PARSER TEST

Buildings extracted : 200
Unique IDs          : 200
Measured heights    : 186
Missing heights     : 14
Usable geometries   : 0
Missing geometries  : 200
Minimum height      : 1.694 m
Maximum height      : 17.573 m


,lod2_id,gml_id,creation_date,function,condition,measured_height_m,height_method,footprint_method,geometry
0,Building_DESL64685862E1EB,Building_DESL64685862E1EB,2024-02-21T00:00:00,None,None,1.908,height_above_ground,projected_3d_surfaces,None
1,Building_DESL6471706EC4A5,Building_DESL6471706EC4A5,2024-02-20T00:00:00,None,None,2.450,height_above_ground,projected_3d_surfaces,None
2,Building_DESL647171EEED51,Building_DESL647171EEED51,2024-02-23T00:00:00,None,None,17.573,height_above_ground,projected_3d_surfaces,None
3,Building_DESL6471722BDD9E,Building_DESL6471722BDD9E,2024-02-21T00:00:00,None,None,1.694,height_above_ground,projected_3d_surfaces,None
4,Building_DESL647173EAAB87,Building_DESL647173EAAB87,2024-02-22T00:00:00,None,None,2.470,height_above_ground,projected_3d_surfaces,None


In [19]:
# ============================================================
# 92G — SAARLAND REFERENCED-GEOMETRY PARSER TEST
# ============================================================

from shapely.geometry import Polygon
from shapely.ops import unary_union

def sl_gml_id(el): return next((v for k,v in el.attrib.items() if k.endswith("}id")), None)
def sl_xlink(el): return next((v for k,v in el.attrib.items() if k.endswith("}href") or k=="href"), None)

def sl_referenced_nodes(el, id_map):
    nodes, queue, seen = [el], [el], set()
    while queue:
        node = queue.pop()
        for child in node.iter():
            href = sl_xlink(child)
            if href:
                target = id_map.get(href.lstrip("#"))
                if target is not None and id(target) not in seen:
                    seen.add(id(target)); nodes.append(target); queue.append(target)
    return nodes

def sl_polygon(polygon_el):
    exterior = polygon_el.xpath("./*[local-name()='exterior']//*[local-name()='LinearRing'][1]")
    if not exterior: return None
    shell = sl_ring(exterior[0])
    if len(shell) < 4: return None
    holes = [ring for ring in (sl_ring(x) for x in polygon_el.xpath("./*[local-name()='interior']//*[local-name()='LinearRing']")) if len(ring)>=4]
    try:
        polygon = Polygon(shell, holes)
        if not polygon.is_valid: polygon = polygon.buffer(0)
        return polygon if not polygon.is_empty and polygon.area>0.01 else None
    except Exception:
        return None

def sl_footprint(building, id_map):
    polygons = []
    geometry_links = building.xpath(".//*[local-name()='geometry3DLoD2']")
    for link in geometry_links:
        for node in sl_referenced_nodes(link, id_map):
            for polygon_el in node.xpath(".//*[local-name()='Polygon']"):
                polygon = sl_polygon(polygon_el)
                if polygon is not None: polygons.append(polygon)
    if not polygons: return None
    geometry = unary_union(polygons)
    if not geometry.is_valid: geometry = geometry.buffer(0)
    return geometry if not geometry.is_empty and geometry.geom_type in {"Polygon","MultiPolygon"} else None

def extract_sl_buildings(content):
    root = etree.fromstring(content, etree.XMLParser(recover=True, huge_tree=True))
    id_map = {sl_gml_id(el):el for el in root.iter() if isinstance(el.tag,str) and sl_gml_id(el)}
    records = []

    for building in root.xpath("//*[local-name()='Building']"):
        gml_id = sl_gml_id(building)
        height_el = building.xpath(".//*[local-name()='heightAboveGround']/*[local-name()='HeightAboveGround'][1]")
        value_el = height_el[0].xpath(".//*[local-name()='value'][1]") if height_el else []
        height = float(value_el[0].text) if value_el and value_el[0].text else np.nan
        geometry = sl_footprint(building, id_map)

        records.append({
            "lod2_id":sl_text(building,"localId") or gml_id,
            "gml_id":gml_id,
            "creation_date":sl_text(building,"beginLifespanVersion"),
            "function":sl_href(building,"currentUse"),
            "condition":sl_href(building,"conditionOfConstruction"),
            "measured_height_m":height,
            "height_method":"height_above_ground" if pd.notna(height) else "missing",
            "footprint_method":"referenced_3d_surfaces" if geometry is not None else "missing",
            "geometry":geometry
        })

    return gpd.GeoDataFrame(records, geometry="geometry", crs=SL_CRS)

sample = sl_get_wfs({
    "SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature",
    "TYPENAMES":SL_TYPENAME,"COUNT":"200","STARTINDEX":"0",
    "SRSNAME":"urn:ogc:def:crs:EPSG::25832"
})

sl_parser_test = extract_sl_buildings(sample.content)

print("="*70)
print("SAARLAND REFERENCED-GEOMETRY PARSER TEST")
print("="*70)
print(f"\nBuildings extracted : {len(sl_parser_test):,}")
print(f"Unique IDs          : {sl_parser_test.lod2_id.nunique():,}")
print(f"Measured heights    : {sl_parser_test.measured_height_m.notna().sum():,}")
print(f"Missing heights     : {sl_parser_test.measured_height_m.isna().sum():,}")
print(f"Usable geometries   : {sl_parser_test.geometry.notna().sum():,}")
print(f"Missing geometries  : {sl_parser_test.geometry.isna().sum():,}")

print("\nMissing height/geometry combinations:")
display(sl_parser_test.assign(missing_height=sl_parser_test.measured_height_m.isna(), missing_geometry=sl_parser_test.geometry.isna()).groupby(["missing_height","missing_geometry"]).size().rename("buildings").reset_index())

SAARLAND REFERENCED-GEOMETRY PARSER TEST

Buildings extracted : 200
Unique IDs          : 200
Measured heights    : 186
Missing heights     : 14
Usable geometries   : 0
Missing geometries  : 200

Missing height/geometry combinations:


,missing_height,missing_geometry,buildings
0,False,True,186
1,True,True,14


In [20]:
# ============================================================
# 92H — SAARLAND DIRECT-POLYGON FOOTPRINT TEST
# ============================================================

def sl_polygon_2d(polygon_el):
    exterior = polygon_el.xpath("./*[local-name()='exterior']//*[local-name()='LinearRing'][1]")
    if not exterior: return None

    shell = sl_ring(exterior[0])
    if len(shell) < 4: return None

    holes = []
    for interior in polygon_el.xpath("./*[local-name()='interior']//*[local-name()='LinearRing']"):
        ring = sl_ring(interior)
        if len(ring) >= 4: holes.append(ring)

    try:
        polygon = Polygon(shell, holes)
        if not polygon.is_valid: polygon = polygon.buffer(0)
        return polygon if not polygon.is_empty and polygon.area > 0.01 else None
    except Exception:
        return None

def sl_footprint(building, id_map):
    polygon_elements = list(building.xpath(".//*[local-name()='Polygon']"))

    for el in building.xpath(".//*[@*[local-name()='href']]"):
        href = sl_xlink(el)
        target = id_map.get(href.split("#")[-1]) if href else None
        if target is not None: polygon_elements.extend(target.xpath(".//*[local-name()='Polygon']"))

    polygons, seen = [], set()

    for polygon_el in polygon_elements:
        polygon_id = sl_gml_id(polygon_el) or id(polygon_el)
        if polygon_id in seen: continue
        seen.add(polygon_id)

        polygon = sl_polygon_2d(polygon_el)
        if polygon is not None: polygons.append(polygon)

    if not polygons: return None

    geometry = unary_union(polygons)
    if not geometry.is_valid: geometry = geometry.buffer(0)

    if geometry.geom_type == "GeometryCollection":
        geometry = unary_union([g for g in geometry.geoms if g.geom_type in {"Polygon","MultiPolygon"}])

    return geometry if not geometry.is_empty and geometry.geom_type in {"Polygon","MultiPolygon"} else None

def extract_sl_buildings(content):
    root = etree.fromstring(content, etree.XMLParser(recover=True, huge_tree=True))
    id_map = {sl_gml_id(el):el for el in root.iter() if isinstance(el.tag,str) and sl_gml_id(el)}
    records = []

    for building in root.xpath("//*[local-name()='Building']"):
        gml_id = sl_gml_id(building)
        value_el = building.xpath(".//*[local-name()='heightAboveGround']/*[local-name()='HeightAboveGround']//*[local-name()='value'][1]")
        height = float(value_el[0].text) if value_el and value_el[0].text else np.nan
        geometry = sl_footprint(building,id_map)

        records.append({
            "lod2_id":sl_text(building,"localId") or gml_id,
            "gml_id":gml_id,
            "creation_date":sl_text(building,"beginLifespanVersion"),
            "function":sl_href(building,"currentUse"),
            "condition":sl_href(building,"conditionOfConstruction"),
            "measured_height_m":height,
            "height_method":"height_above_ground" if pd.notna(height) else "missing",
            "footprint_method":"projected_3d_surfaces" if geometry is not None else "missing",
            "geometry":geometry
        })

    return gpd.GeoDataFrame(records,geometry="geometry",crs=SL_CRS)

sample = sl_get_wfs({
    "SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature",
    "TYPENAMES":SL_TYPENAME,"COUNT":"200","STARTINDEX":"0",
    "SRSNAME":"urn:ogc:def:crs:EPSG::25832"
})

sl_parser_test = extract_sl_buildings(sample.content)
sl_parser_test["footprint_area_m2"] = sl_parser_test.geometry.area

print("="*70)
print("SAARLAND DIRECT-POLYGON PARSER TEST")
print("="*70)
print(f"\nBuildings extracted : {len(sl_parser_test):,}")
print(f"Unique IDs          : {sl_parser_test.lod2_id.nunique():,}")
print(f"Measured heights    : {sl_parser_test.measured_height_m.notna().sum():,}")
print(f"Missing heights     : {sl_parser_test.measured_height_m.isna().sum():,}")
print(f"Usable geometries   : {sl_parser_test.geometry.notna().sum():,}")
print(f"Missing geometries  : {sl_parser_test.geometry.isna().sum():,}")
print(f"Minimum area        : {sl_parser_test.footprint_area_m2.min():,.2f} m²")
print(f"Median area         : {sl_parser_test.footprint_area_m2.median():,.2f} m²")
print(f"Maximum area        : {sl_parser_test.footprint_area_m2.max():,.2f} m²")

display(sl_parser_test[["lod2_id","measured_height_m","height_method","footprint_method","footprint_area_m2","geometry"]].head())

SAARLAND DIRECT-POLYGON PARSER TEST

Buildings extracted : 200
Unique IDs          : 200
Measured heights    : 186
Missing heights     : 14
Usable geometries   : 0
Missing geometries  : 200
Minimum area        : nan m²
Median area         : nan m²
Maximum area        : nan m²


/fast/home/o-olajuyigbe/miniforge3/envs/osm_env/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,lod2_id,measured_height_m,height_method,footprint_method,footprint_area_m2,geometry
0,Building_DESL64685862E1EB,1.908,height_above_ground,missing,NaN,None
1,Building_DESL6471706EC4A5,2.450,height_above_ground,missing,NaN,None
2,Building_DESL647171EEED51,17.573,height_above_ground,missing,NaN,None
3,Building_DESL6471722BDD9E,1.694,height_above_ground,missing,NaN,None
4,Building_DESL647173EAAB87,2.470,height_above_ground,missing,NaN,None


In [22]:
# ============================================================
# 92I — DIAGNOSE SAARLAND GEOMETRY STORAGE AND REFERENCES
# ============================================================

from lxml import etree
from collections import Counter

sample_path = SL_TEST_DIR/"bu-core3d_Building_sample.gml"
tree = etree.parse(sample_path, etree.XMLParser(recover=True, huge_tree=True))
root = tree.getroot()

def lname(el): return etree.QName(el.tag).localname if isinstance(el.tag,str) else None
def attrs(el): return {etree.QName(k).localname if k.startswith("{") else k:v for k,v in el.attrib.items()}
def short_xml(el,n=1200): return etree.tostring(el, encoding="unicode", pretty_print=True)[:n]

buildings = root.xpath("//*[local-name()='Building']")
first = buildings[0]

print("="*80)
print("FIRST BUILDING — GEOMETRY-RELATED ELEMENTS")
print("="*80)

geometry_names = {"geometry3DLoD2","BuildingGeometry3DLoD2","geometryMultiSurface","MultiSurface","surfaceMember","Polygon","posList"}

rows = []
for el in first.iter():
    if isinstance(el.tag,str) and lname(el) in geometry_names:
        rows.append({
            "tag":lname(el),
            "path":tree.getpath(el),
            "attributes":attrs(el),
            "text":(el.text or "").strip()[:100]
        })

display(pd.DataFrame(rows))

print("\nFirst Building XML:")
print(short_xml(first,5000))

print("\n"+"="*80)
print("DOCUMENT-LEVEL GEOMETRY OBJECTS")
print("="*80)

doc_rows = []
for el in root.iter():
    if not isinstance(el.tag,str): continue
    name = lname(el)
    if name in {"BuildingGeometry3DLoD2","MultiSurface","Polygon"}:
        doc_rows.append({
            "tag":name,
            "gml_id":sl_gml_id(el),
            "path":tree.getpath(el),
            "parent_tag":lname(el.getparent()) if el.getparent() is not None else None,
            "attributes":attrs(el)
        })

display(pd.DataFrame(doc_rows).head(40))
print(f"\nDocument BuildingGeometry3DLoD2 objects: {sum(x['tag']=='BuildingGeometry3DLoD2' for x in doc_rows):,}")
print(f"Document MultiSurface objects          : {sum(x['tag']=='MultiSurface' for x in doc_rows):,}")
print(f"Document Polygon objects               : {sum(x['tag']=='Polygon' for x in doc_rows):,}")

print("\n"+"="*80)
print("ALL REFERENCE-LIKE ATTRIBUTES")
print("="*80)

reference_rows = []
for el in root.iter():
    if not isinstance(el.tag,str): continue
    for key,value in attrs(el).items():
        if key.lower() in {"href","ref","idref","owns","nilreason"} or "#" in str(value) or "uuid" in str(value).lower():
            reference_rows.append({
                "tag":lname(el),
                "path":tree.getpath(el),
                "attribute":key,
                "value":value
            })

display(pd.DataFrame(reference_rows).head(100))

print("\nTop-level WFS member feature types:")
member_types = []
for member in root.xpath("//*[local-name()='member' or local-name()='featureMember']"):
    children = [x for x in member if isinstance(x.tag,str)]
    member_types.extend(lname(x) for x in children)

display(pd.Series(Counter(member_types),name="count").rename_axis("feature_type").reset_index())

FIRST BUILDING — GEOMETRY-RELATED ELEMENTS


,tag,path,attributes,text
0,geometry3DLoD2,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{},
1,BuildingGeometry3DLoD2,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{},
2,geometryMultiSurface,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{},
3,MultiSurface,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{'id': 'Building_DESL64685862E1EB_BU-CORE3D_GE...,
4,surfaceMember,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{},
5,Polygon,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{'id': 'GEOMETRY_334b995b-ee34-4042-b167-26fd4...,
6,posList,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{},0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.00...
7,surfaceMember,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{},
8,Polygon,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{'id': 'GEOMETRY_05e6405a-3af9-434c-90b7-14612...,
9,posList,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,{},0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.00...



First Building XML:
<bu-core3d:Building xmlns:bu-core3d="http://inspire.ec.europa.eu/schemas/bu-core3d/4.0" xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:wfs="http://www.opengis.net/wfs/2.0" gml:id="Building_DESL64685862E1EB">
      <gml:identifier codeSpace="http://inspire.ec.europa.eu/ids">https://registry.gdi-de.org/id/de.sl.inspire.citygml.bu-3D/Building_DESL64685862E1EB</gml:identifier>
      <bu-base:beginLifespanVersion xmlns:bu-base="http://inspire.ec.europa.eu/schemas/bu-base/4.0">2024-02-21T00:00:00</bu-base:beginLifespanVersion>
      <bu-base:conditionOfConstruction xmlns:bu-base="http://inspire.ec.europa.eu/schemas/bu-base/4.0" xsi:nil="true" nilReason="other:unpopulated"/>
      <bu-base:externalReference xmlns:bu-base="http://inspire.ec.europa.eu/schemas/bu-base/4.0">
        <bu-base:ExternalReference>
          <bu-base:informationSystem>http://repository.gdi-de.org/schemas/adv/citygml/fdv/art.htm#_9100</bu-base

,tag,gml_id,path,parent_tag,attributes
0,BuildingGeometry3DLoD2,None,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,geometry3DLoD2,{}
1,MultiSurface,Building_DESL64685862E1EB_BU-CORE3D_GEOMETRY3D...,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,geometryMultiSurface,{'id': 'Building_DESL64685862E1EB_BU-CORE3D_GE...
2,Polygon,GEOMETRY_334b995b-ee34-4042-b167-26fd46b0ea22,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,surfaceMember,{'id': 'GEOMETRY_334b995b-ee34-4042-b167-26fd4...
3,Polygon,GEOMETRY_05e6405a-3af9-434c-90b7-14612d0ee68c,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,surfaceMember,{'id': 'GEOMETRY_05e6405a-3af9-434c-90b7-14612...
4,Polygon,GEOMETRY_a505a2b4-7934-4aec-a265-378f14113be1,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,surfaceMember,{'id': 'GEOMETRY_a505a2b4-7934-4aec-a265-378f1...
5,Polygon,GEOMETRY_f71dcf2d-a87f-4e94-bd1e-200d9a3b2487,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,surfaceMember,{'id': 'GEOMETRY_f71dcf2d-a87f-4e94-bd1e-200d9...
6,Polygon,GEOMETRY_e3b91b18-97f3-4d5b-a3cd-bb40d466d9ef,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,surfaceMember,{'id': 'GEOMETRY_e3b91b18-97f3-4d5b-a3cd-bb40d...
7,Polygon,GEOMETRY_23c3cf55-5fe7-4eb3-a271-5759f810ed30,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,surfaceMember,{'id': 'GEOMETRY_23c3cf55-5fe7-4eb3-a271-5759f...
8,BuildingGeometry3DLoD2,None,/wfs:FeatureCollection/wfs:member[2]/bu-core3d...,geometry3DLoD2,{}
9,MultiSurface,Building_DESL6471706EC4A5_BU-CORE3D_GEOMETRY3D...,/wfs:FeatureCollection/wfs:member[2]/bu-core3d...,geometryMultiSurface,{'id': 'Building_DESL6471706EC4A5_BU-CORE3D_GE...



Document BuildingGeometry3DLoD2 objects: 19
Document MultiSurface objects          : 19
Document Polygon objects               : 181

ALL REFERENCE-LIKE ATTRIBUTES


,tag,path,attribute,value
0,conditionOfConstruction,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,nilReason,other:unpopulated
1,lowReference,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,nilReason,unknown
2,status,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,href,http://inspire.ec.europa.eu/codelist/HeightSta...
3,currentUse,/wfs:FeatureCollection/wfs:member[1]/bu-core3d...,href,https://registry.gdi-de.org/codelist/de.adv-on...
4,conditionOfConstruction,/wfs:FeatureCollection/wfs:member[2]/bu-core3d...,nilReason,other:unpopulated
...,...,...,...,...
76,currentUse,/wfs:FeatureCollection/wfs:member[19]/bu-core3...,href,http://inspire.ec.europa.eu/codelist/CurrentUs...
77,conditionOfConstruction,/wfs:FeatureCollection/wfs:member[20]/bu-core3...,nilReason,other:unpopulated
78,lowReference,/wfs:FeatureCollection/wfs:member[20]/bu-core3...,nilReason,unknown
79,status,/wfs:FeatureCollection/wfs:member[20]/bu-core3...,href,http://inspire.ec.europa.eu/codelist/HeightSta...



Top-level WFS member feature types:


,feature_type,count
0,Building,20


In [23]:
# ============================================================
# 92J — TEST SAARLAND GEOMETRY COORDINATES BY CRS
# ============================================================

from lxml import etree

crs_tests = {
    "default":None,
    "EPSG_7423":"urn:ogc:def:crs:EPSG::7423",
    "EPSG_4258":"urn:ogc:def:crs:EPSG::4258",
    "EPSG_25832":"urn:ogc:def:crs:EPSG::25832"
}

rows, responses = [], {}

for label,srs in crs_tests.items():
    params = {"SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature","TYPENAMES":SL_TYPENAME,"COUNT":"20","STARTINDEX":"0"}
    if srs: params["SRSNAME"] = srs

    try:
        r = sl_get_wfs(params)
        responses[label] = r.content
        root = etree.fromstring(r.content,etree.XMLParser(recover=True,huge_tree=True))
        pos_lists = root.xpath("//*[local-name()='posList']/text()")
        values = np.concatenate([np.fromstring(x,sep=" ") for x in pos_lists if x.strip()]) if pos_lists else np.array([])
        nonzero = values[np.abs(values)>1e-9]
        srs_names = sorted(set(root.xpath("//@srsName")))

        rows.append({
            "request":label,
            "status":"success",
            "pos_lists":len(pos_lists),
            "coordinate_values":len(values),
            "nonzero_values":len(nonzero),
            "minimum":values.min() if len(values) else np.nan,
            "maximum":values.max() if len(values) else np.nan,
            "first_values":" ".join(map(str,values[:12])),
            "returned_srs":"; ".join(srs_names)
        })
    except Exception as e:
        rows.append({"request":label,"status":str(e),"pos_lists":0,"coordinate_values":0,"nonzero_values":0,"minimum":np.nan,"maximum":np.nan,"first_values":None,"returned_srs":None})

sl_crs_test = pd.DataFrame(rows)
sl_crs_test.to_csv(SL_RAW_DIR/"sl_wfs_crs_coordinate_test.csv",index=False)

print("="*80)
print("SAARLAND WFS COORDINATE TEST")
print("="*80)
display(sl_crs_test)

for label,content in responses.items():
    (SL_TEST_DIR/f"sl_sample_{label}.gml").write_bytes(content)

SAARLAND WFS COORDINATE TEST


,request,status,pos_lists,coordinate_values,nonzero_values,minimum,maximum,first_values,returned_srs
0,default,success,200,5106,5106,6.760145,329.262000,49.351754 6.776324 214.726 49.351769 6.776355 ...,urn:ogc:def:crs:EPSG::7423
1,EPSG_7423,success,200,5106,5106,6.760145,329.262000,49.351754 6.776324 214.726 49.351769 6.776355 ...,urn:ogc:def:crs:EPSG::7423
2,EPSG_4258,success,200,3404,3404,6.760145,49.508206,49.351754 6.776324 49.351769 6.776355 49.35176...,urn:ogc:def:crs:EPSG::4258
3,EPSG_25832,success,200,3404,0,0.000000,0.000000,0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0,urn:ogc:def:crs:EPSG::25832


In [24]:
# ============================================================
# 92K — SAARLAND VALID-GEOMETRY PARSER TEST
# ============================================================

from shapely.geometry import Polygon
from shapely.ops import unary_union

def sl_first_href(el,name):
    for x in el.xpath(f".//*[local-name()='{name}']"):
        href = sl_xlink(x)
        if href: return href
    return None

def sl_ring_4258(ring):
    pos = ring.xpath(".//*[local-name()='posList'][1]")
    if not pos or not pos[0].text: return []
    values = np.fromstring(pos[0].text,sep=" ")
    if len(values)<6 or len(values)%2: return []
    latlon = values.reshape(-1,2)
    coords = np.column_stack([latlon[:,1],latlon[:,0]])  # x=longitude, y=latitude
    if len(coords)>=3 and not np.allclose(coords[0],coords[-1]): coords=np.vstack([coords,coords[0]])
    return [tuple(x) for x in coords]

def sl_polygon_4258(el):
    exterior = el.xpath("./*[local-name()='exterior']//*[local-name()='LinearRing'][1]")
    if not exterior: return None
    shell = sl_ring_4258(exterior[0])
    if len(shell)<4: return None
    holes = [r for r in (sl_ring_4258(x) for x in el.xpath("./*[local-name()='interior']//*[local-name()='LinearRing']")) if len(r)>=4]
    try:
        poly = Polygon(shell,holes)
        if not poly.is_valid: poly=poly.buffer(0)
        return poly if not poly.is_empty and poly.area>1e-12 else None
    except Exception:
        return None

def sl_footprint_4258(building):
    polygons = [p for p in (sl_polygon_4258(x) for x in building.xpath(".//*[local-name()='geometry3DLoD2']//*[local-name()='Polygon']")) if p is not None]
    if not polygons: return None
    geom = unary_union(polygons)
    if not geom.is_valid: geom=geom.buffer(0)
    if geom.geom_type=="GeometryCollection": geom=unary_union([g for g in geom.geoms if g.geom_type in {"Polygon","MultiPolygon"}])
    return geom if not geom.is_empty and geom.geom_type in {"Polygon","MultiPolygon"} else None

def extract_sl_buildings(content):
    root = etree.fromstring(content,etree.XMLParser(recover=True,huge_tree=True))
    records = []
    for building in root.xpath("//*[local-name()='Building']"):
        gml_id = sl_gml_id(building)
        value = building.xpath(".//*[local-name()='heightAboveGround']/*[local-name()='HeightAboveGround']//*[local-name()='value'][1]/text()")
        height = float(value[0]) if value else np.nan
        geom = sl_footprint_4258(building)
        records.append({
            "lod2_id":sl_text(building,"localId") or gml_id,
            "gml_id":gml_id,
            "creation_date":sl_text(building,"beginLifespanVersion"),
            "function":sl_first_href(building,"currentUse"),
            "condition":sl_first_href(building,"conditionOfConstruction"),
            "measured_height_m":height,
            "height_method":"height_above_ground" if pd.notna(height) else "missing",
            "footprint_method":"projected_lod2_surfaces" if geom is not None else "missing",
            "geometry":geom
        })
    return gpd.GeoDataFrame(records,geometry="geometry",crs="EPSG:4258").to_crs("EPSG:25832")

sample = sl_get_wfs({
    "SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature",
    "TYPENAMES":SL_TYPENAME,"COUNT":"200","STARTINDEX":"0",
    "SRSNAME":"urn:ogc:def:crs:EPSG::4258"
})

sl_parser_test = extract_sl_buildings(sample.content)
sl_parser_test["footprint_area_m2"] = sl_parser_test.geometry.area
sl_parser_test.to_parquet(SL_TEST_DIR/"sl_valid_geometry_parser_test.parquet",index=False)

print("="*70)
print("SAARLAND VALID-GEOMETRY PARSER TEST")
print("="*70)
print(f"\nBuildings extracted : {len(sl_parser_test):,}")
print(f"Unique IDs          : {sl_parser_test.lod2_id.nunique():,}")
print(f"Measured heights    : {sl_parser_test.measured_height_m.notna().sum():,}")
print(f"Missing heights     : {sl_parser_test.measured_height_m.isna().sum():,}")
print(f"Usable geometries   : {sl_parser_test.geometry.notna().sum():,}")
print(f"Missing geometries  : {sl_parser_test.geometry.isna().sum():,}")
print(f"Minimum area        : {sl_parser_test.footprint_area_m2.min():,.2f} m²")
print(f"Median area         : {sl_parser_test.footprint_area_m2.median():,.2f} m²")
print(f"Maximum area        : {sl_parser_test.footprint_area_m2.max():,.2f} m²")
print(f"Dataset bounds      : {np.round(sl_parser_test.total_bounds,2)}")
display(sl_parser_test.head())

SAARLAND VALID-GEOMETRY PARSER TEST

Buildings extracted : 200
Unique IDs          : 200
Measured heights    : 186
Missing heights     : 14
Usable geometries   : 186
Missing geometries  : 14
Minimum area        : 4.29 m²
Median area         : 93.53 m²
Maximum area        : 426.33 m²
Dataset bounds      : [ 337154.6  5456488.44  373080.63 5486549.83]


,lod2_id,gml_id,creation_date,function,condition,measured_height_m,height_method,footprint_method,geometry,footprint_area_m2
0,Building_DESL64685862E1EB,Building_DESL64685862E1EB,2024-02-21T00:00:00,https://registry.gdi-de.org/codelist/de.adv-on...,None,1.908,height_above_ground,projected_lod2_surfaces,"POLYGON ((338511.55 5468939.617, 338512.735 54...",5.816999
1,Building_DESL6471706EC4A5,Building_DESL6471706EC4A5,2024-02-20T00:00:00,https://registry.gdi-de.org/codelist/de.adv-on...,None,2.450,height_above_ground,projected_lod2_surfaces,"POLYGON ((337795.898 5456491.353, 337797.688 5...",4.599319
2,Building_DESL647171EEED51,Building_DESL647171EEED51,2024-02-23T00:00:00,https://registry.gdi-de.org/codelist/de.adv-on...,None,17.573,height_above_ground,projected_lod2_surfaces,"POLYGON ((340280.089 5460128.612, 340280.116 5...",5.457937
3,Building_DESL6471722BDD9E,Building_DESL6471722BDD9E,2024-02-21T00:00:00,https://registry.gdi-de.org/codelist/de.adv-on...,None,1.694,height_above_ground,projected_lod2_surfaces,"POLYGON ((339769.454 5459225.304, 339771.864 5...",4.467445
4,Building_DESL647173EAAB87,Building_DESL647173EAAB87,2024-02-22T00:00:00,https://registry.gdi-de.org/codelist/de.adv-on...,None,2.470,height_above_ground,projected_lod2_surfaces,"POLYGON ((342361.045 5459772.725, 342363.174 5...",5.462185


In [25]:
# ============================================================
# 92L — INSPECT SAARLAND MULTIPART PARENT BUILDINGS
# ============================================================

from collections import Counter
from lxml import etree

sample = sl_get_wfs({
    "SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature",
    "TYPENAMES":"bu-core3d:Building","COUNT":"200","STARTINDEX":"0",
    "SRSNAME":"urn:ogc:def:crs:EPSG::4258"
})

root = etree.fromstring(sample.content,etree.XMLParser(recover=True,huge_tree=True))
buildings = root.xpath("//*[local-name()='Building']")

def sl_has_height(b): return bool(b.xpath(".//*[local-name()='HeightAboveGround']//*[local-name()='value']/text()"))
def sl_has_geometry(b): return bool(b.xpath(".//*[local-name()='geometry3DLoD2']//*[local-name()='Polygon']"))
def sl_local(el): return etree.QName(el.tag).localname if isinstance(el.tag,str) else None
def sl_attrs(el): return {etree.QName(k).localname if k.startswith("{") else k:v for k,v in el.attrib.items()}

missing = [b for b in buildings if not sl_has_height(b) or not sl_has_geometry(b)]
rows, href_rows, tag_rows = [], [], []

for i,b in enumerate(missing,1):
    gml_id = sl_gml_id(b)
    direct_children = [sl_local(x) for x in b if isinstance(x.tag,str)]
    relevant_tags = sorted({sl_local(x) for x in b.iter() if isinstance(x.tag,str) and any(k in sl_local(x).lower() for k in ["part","parent","building","geometry","height","reference"])})

    rows.append({
        "missing_number":i,
        "gml_id":gml_id,
        "local_id":sl_text(b,"localId"),
        "has_height":sl_has_height(b),
        "has_geometry":sl_has_geometry(b),
        "direct_children":"; ".join(direct_children),
        "relevant_tags":"; ".join(relevant_tags)
    })

    for el in b.iter():
        if not isinstance(el.tag,str): continue
        tag_rows.append({"gml_id":gml_id,"tag":sl_local(el)})
        for key,value in sl_attrs(el).items():
            if key.lower() in {"href","ref","idref","nilreason"} or "part" in str(value).lower() or "uuid" in str(value).lower():
                href_rows.append({"gml_id":gml_id,"tag":sl_local(el),"attribute":key,"value":value})

missing_summary = pd.DataFrame(rows)
missing_refs = pd.DataFrame(href_rows)
missing_tags = pd.DataFrame(tag_rows).value_counts("tag").rename("count").reset_index()

print("="*80)
print("SAARLAND GEOMETRY-LESS BUILDING DIAGNOSIS")
print("="*80)
print(f"\nBuildings inspected       : {len(buildings):,}")
print(f"Missing height/geometry   : {len(missing):,}")

print("\nMissing-building structure:")
display(missing_summary)

print("\nReference-like attributes:")
display(missing_refs)

print("\nRelevant tags across missing buildings:")
display(missing_tags[missing_tags["tag"].str.contains("part|parent|building|geometry|height|reference",case=False,regex=True)])

if missing:
    print("\nFirst missing Building XML:")
    print(etree.tostring(missing[0],encoding="unicode",pretty_print=True)[:8000])

SAARLAND GEOMETRY-LESS BUILDING DIAGNOSIS

Buildings inspected       : 200
Missing height/geometry   : 14

Missing-building structure:


,missing_number,gml_id,local_id,has_height,has_geometry,direct_children,relevant_tags
0,1,Building_DESLA0000MS14188,Building_DESLA0000MS14188,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
1,2,Building_DESLA0002RO279B1,Building_DESLA0002RO279B1,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
2,3,Building_DESLA00044F906F2,Building_DESLA00044F906F2,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
3,4,Building_DESLA0004F06F40F,Building_DESLA0004F06F40F,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
4,5,Building_DESLA0005WP290AE,Building_DESLA0005WP290AE,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
5,6,Building_DESLA0005X994796,Building_DESLA0005X994796,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
6,7,Building_DESLA0005XK8C45D,Building_DESLA0005XK8C45D,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
7,8,Building_DESLA0006FMB966A,Building_DESLA0006FMB966A,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
8,9,Building_DESLA0006OR803C8,Building_DESLA0006OR803C8,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...
9,10,Building_DESLA0008Y9C40E6,Building_DESLA0008Y9C40E6,False,False,identifier; beginLifespanVersion; conditionOfC...,Building; ExternalReference; HeightAboveGround...



Reference-like attributes:


,gml_id,tag,attribute,value
0,Building_DESLA0000MS14188,conditionOfConstruction,nilReason,other:unpopulated
1,Building_DESLA0000MS14188,currentUse,href,http://inspire.ec.europa.eu/codelist/CurrentUs...
2,Building_DESLA0000MS14188,parts,href,https://geoportal.saarland.de/gdi-sl/inspirewf...
3,Building_DESLA0000MS14188,parts,href,https://geoportal.saarland.de/gdi-sl/inspirewf...
4,Building_DESLA0000MS14188,parts,href,https://geoportal.saarland.de/gdi-sl/inspirewf...
5,Building_DESLA0002RO279B1,conditionOfConstruction,nilReason,other:unpopulated
6,Building_DESLA0002RO279B1,currentUse,href,http://inspire.ec.europa.eu/codelist/CurrentUs...
7,Building_DESLA0002RO279B1,parts,href,https://geoportal.saarland.de/gdi-sl/inspirewf...
8,Building_DESLA0002RO279B1,parts,href,https://geoportal.saarland.de/gdi-sl/inspirewf...
9,Building_DESLA00044F906F2,conditionOfConstruction,nilReason,other:unpopulated



Relevant tags across missing buildings:


,tag,count
1,parts,30
4,Building,14
5,HeightAboveGround,14
7,ExternalReference,14
11,externalReference,14
12,heightAboveGround,14
13,heightReference,14
24,lowReference,14
28,reference,14



First missing Building XML:
<bu-core3d:Building xmlns:bu-core3d="http://inspire.ec.europa.eu/schemas/bu-core3d/4.0" xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:wfs="http://www.opengis.net/wfs/2.0" gml:id="Building_DESLA0000MS14188">
      <gml:identifier codeSpace="http://inspire.ec.europa.eu/ids">https://registry.gdi-de.org/id/de.sl.inspire.citygml.bu-3D/Building_DESLA0000MS14188</gml:identifier>
      <bu-base:beginLifespanVersion xmlns:bu-base="http://inspire.ec.europa.eu/schemas/bu-base/4.0">2024-03-06T00:00:00</bu-base:beginLifespanVersion>
      <bu-base:conditionOfConstruction xmlns:bu-base="http://inspire.ec.europa.eu/schemas/bu-base/4.0" xsi:nil="true" nilReason="other:unpopulated"/>
      <bu-base:externalReference xmlns:bu-base="http://inspire.ec.europa.eu/schemas/bu-base/4.0">
        <bu-base:ExternalReference>
          <bu-base:informationSystem>http://repository.gdi-de.org/schemas/adv/citygml/fdv/art.htm#_9100<

In [26]:
# ============================================================
# 92M — SAARLAND MULTIPART BUILDING RECONSTRUCTION TEST
# ============================================================

from urllib.parse import unquote
from shapely.ops import unary_union

sample = sl_get_wfs({"SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature","TYPENAMES":"bu-core3d:Building","COUNT":"200","STARTINDEX":"0","SRSNAME":"urn:ogc:def:crs:EPSG::4258"})
root = etree.fromstring(sample.content,etree.XMLParser(recover=True,huge_tree=True))

parents, parent_parts = [], {}
for building in root.xpath("//*[local-name()='Building']"):
    part_ids = []
    for el in building.xpath("./*[local-name()='parts']"):
        href = sl_xlink(el)
        if href: part_ids.append(unquote(href.split("#")[-1]))
    if part_ids:
        parent_id = sl_text(building,"localId") or sl_gml_id(building)
        parents.append(building); parent_parts[parent_id]=part_ids

requested_ids = sorted({pid for ids in parent_parts.values() for pid in ids})
part_records = {}

for start in tqdm(range(0,len(requested_ids),50),desc="Fetching referenced parts"):
    ids = requested_ids[start:start+50]
    response = sl_get_wfs({
        "SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature",
        "TYPENAMES":"bu-core3d:BuildingPart","RESOURCEID":",".join(ids),
        "SRSNAME":"urn:ogc:def:crs:EPSG::4258"
    })
    part_root = etree.fromstring(response.content,etree.XMLParser(recover=True,huge_tree=True))

    for part in part_root.xpath("//*[local-name()='BuildingPart']"):
        part_id = sl_text(part,"localId") or sl_gml_id(part)
        value = part.xpath(".//*[local-name()='HeightAboveGround']//*[local-name()='value'][1]/text()")
        part_records[part_id]={
            "height":float(value[0]) if value else np.nan,
            "geometry":sl_footprint_4258(part)
        }

records = []
for building in parents:
    parent_id = sl_text(building,"localId") or sl_gml_id(building)
    parts = [part_records[x] for x in parent_parts[parent_id] if x in part_records]
    heights = [x["height"] for x in parts if pd.notna(x["height"])]
    geometries = [x["geometry"] for x in parts if x["geometry"] is not None]
    geometry = unary_union(geometries) if geometries else None
    if geometry is not None and not geometry.is_valid: geometry=geometry.buffer(0)

    records.append({
        "lod2_id":parent_id,
        "building_part_count":len(parent_parts[parent_id]),
        "parts_returned":len(parts),
        "measured_height_m":max(heights) if heights else np.nan,
        "height_method":"maximum_building_part_height" if heights else "missing",
        "footprint_method":"union_building_part_surfaces" if geometry is not None else "missing",
        "geometry":geometry
    })

sl_multipart_test = gpd.GeoDataFrame(records,geometry="geometry",crs="EPSG:4258").to_crs("EPSG:25832")
sl_multipart_test["footprint_area_m2"] = sl_multipart_test.geometry.area

print("="*70)
print("SAARLAND MULTIPART RECONSTRUCTION TEST")
print("="*70)
print(f"\nParent buildings        : {len(parent_parts):,}")
print(f"Referenced part IDs     : {len(requested_ids):,}")
print(f"Parts returned          : {len(part_records):,}")
print(f"Missing referenced parts: {len(set(requested_ids)-set(part_records)):,}")
print(f"Parents with heights    : {sl_multipart_test.measured_height_m.notna().sum():,}")
print(f"Parents with geometries : {sl_multipart_test.geometry.notna().sum():,}")

display(sl_multipart_test)

Fetching referenced parts: 100%|███████████████████████████████| 1/1 [00:00<00:00,  2.76it/s]

SAARLAND MULTIPART RECONSTRUCTION TEST

Parent buildings        : 14
Referenced part IDs     : 30
Parts returned          : 30
Missing referenced parts: 0
Parents with heights    : 14
Parents with geometries : 14


,lod2_id,building_part_count,parts_returned,measured_height_m,height_method,footprint_method,geometry,footprint_area_m2
0,Building_DESLA0000MS14188,3,3,7.191,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((365262.574 5485560.493, 365257.885 5...",582.315297
1,Building_DESLA0002RO279B1,2,2,10.651,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((372037.727 5477451.755, 372034.607 5...",123.528955
2,Building_DESLA00044F906F2,2,2,7.475,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((372439.639 5477557.036, 372437.257 5...",708.291496
3,Building_DESLA0004F06F40F,2,2,8.577,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((352073.317 5467588.839, 352069.546 5...",155.341157
4,Building_DESLA0005WP290AE,2,2,11.611,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((372215.431 5477258.958, 372214.236 5...",176.064614
5,Building_DESLA0005X994796,2,2,13.184,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((372096.67 5477137.384, 372099.309 54...",140.296797
6,Building_DESLA0005XK8C45D,2,2,6.975,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((372028.475 5476935.658, 372033.178 5...",161.660239
7,Building_DESLA0006FMB966A,2,2,11.840,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((372095.039 5477312.604, 372097.641 5...",108.739228
8,Building_DESLA0006OR803C8,3,3,10.802,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((372033.543 5477604.567, 372031.48 54...",234.193892
9,Building_DESLA0008Y9C40E6,2,2,8.451,maximum_building_part_height,union_building_part_surfaces,"POLYGON ((360975.471 5486349.12, 360972.292 54...",167.973987


In [27]:
# ============================================================
# 93 — SAARLAND FULL RESUMABLE WFS EXTRACTION
# ============================================================

import math, time
from urllib.parse import unquote
from shapely.ops import unary_union

required = ["sl_get_wfs","sl_text","sl_gml_id","sl_xlink","sl_first_href","sl_footprint_4258"]
missing = [x for x in required if x not in globals()]
if missing: raise RuntimeError(f"Missing required functions: {missing}. Rerun Cells 92C–92M.")

SL_OUT, SL_PART, SL_MANIFEST = EXTRACTED_DIR/"SL", EXTRACTED_DIR/"SL"/"parts", EXTRACTED_DIR/"SL"/"manifests"
SL_PART.mkdir(parents=True,exist_ok=True); SL_MANIFEST.mkdir(parents=True,exist_ok=True)

SL_PAGE_SIZE, SL_PART_BATCH, SL_PAGE_RETRIES = 2000, 50, 3
SL_TYPENAME, SL_PART_TYPENAME = "bu-core3d:Building", "bu-core3d:BuildingPart"
SL_REQUEST_CRS, SL_OUTPUT_CRS = "urn:ogc:def:crs:EPSG::4258", "EPSG:25832"

hits = sl_get_wfs({"SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature","TYPENAMES":SL_TYPENAME,"RESULTTYPE":"hits"})
hits_root = etree.fromstring(hits.content,etree.XMLParser(recover=True,huge_tree=True))
SL_TOTAL = int(hits_root.get("numberMatched") or hits_root.get("numberOfFeatures"))
SL_PAGES = math.ceil(SL_TOTAL/SL_PAGE_SIZE)

def sl_float(values):
    try: return float(values[0]) if values else np.nan
    except: return np.nan

def sl_fetch_parts(part_ids):
    records = {}
    for start in range(0,len(part_ids),SL_PART_BATCH):
        ids = part_ids[start:start+SL_PART_BATCH]
        response = sl_get_wfs({"SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature","TYPENAMES":SL_PART_TYPENAME,"RESOURCEID":",".join(ids),"SRSNAME":SL_REQUEST_CRS})
        root = etree.fromstring(response.content,etree.XMLParser(recover=True,huge_tree=True))
        for part in root.xpath("//*[local-name()='BuildingPart']"):
            part_id = sl_text(part,"localId") or sl_gml_id(part)
            height = sl_float(part.xpath(".//*[local-name()='HeightAboveGround']//*[local-name()='value'][1]/text()"))
            records[part_id] = {"height":height,"geometry":sl_footprint_4258(part)}
    return records

def sl_parse_page(content,page_number,start_index):
    root = etree.fromstring(content,etree.XMLParser(recover=True,huge_tree=True))
    buildings, part_ids = [], set()

    for building in root.xpath("//*[local-name()='Building']"):
        ids = []
        for el in building.xpath("./*[local-name()='parts']"):
            href = sl_xlink(el)
            if href: ids.append(unquote(href.split("#")[-1]))
        part_ids.update(ids)
        direct_height = sl_float(building.xpath(".//*[local-name()='HeightAboveGround']//*[local-name()='value'][1]/text()"))
        buildings.append({
            "element":building,
            "gml_id":sl_gml_id(building),
            "lod2_id":sl_text(building,"localId") or sl_gml_id(building),
            "creation_date":sl_text(building,"beginLifespanVersion"),
            "function":sl_first_href(building,"currentUse"),
            "condition":sl_first_href(building,"conditionOfConstruction"),
            "parent_height_m":direct_height,
            "direct_geometry":sl_footprint_4258(building),
            "part_ids":ids
        })

    parts = sl_fetch_parts(sorted(part_ids)) if part_ids else {}
    records = []

    for building in buildings:
        available = [parts[x] for x in building["part_ids"] if x in parts]
        part_heights = [x["height"] for x in available if pd.notna(x["height"])]
        part_geometries = [x["geometry"] for x in available if x["geometry"] is not None]
        maximum_part_height = max(part_heights) if part_heights else np.nan
        part_geometry = unary_union(part_geometries) if part_geometries else None
        if part_geometry is not None and not part_geometry.is_valid: part_geometry = part_geometry.buffer(0)

        if pd.notna(building["parent_height_m"]):
            measured_height, height_method = building["parent_height_m"], "height_above_ground"
        elif pd.notna(maximum_part_height):
            measured_height, height_method = maximum_part_height, "maximum_building_part_height"
        else:
            measured_height, height_method = np.nan, "missing"

        if building["direct_geometry"] is not None:
            geometry, footprint_method = building["direct_geometry"], "projected_lod2_surfaces"
        elif part_geometry is not None:
            geometry, footprint_method = part_geometry, "union_building_part_surfaces"
        else:
            geometry, footprint_method = None, "missing"

        records.append({
            "lod2_id":building["lod2_id"],
            "gml_id":building["gml_id"],
            "creation_date":building["creation_date"],
            "function":building["function"],
            "condition":building["condition"],
            "roof_type":None,
            "measured_height_m":measured_height,
            "parent_height_m":building["parent_height_m"],
            "maximum_part_height_m":maximum_part_height,
            "building_part_count":len(building["part_ids"]),
            "parts_returned":len(available),
            "height_method":height_method,
            "height_source":height_method,
            "storeys_above_ground":np.nan,
            "footprint_method":footprint_method,
            "geometry":geometry,
            "state_code":"SL",
            "source_state":"Saarland",
            "source_crs":"EPSG:4258",
            "citygml_version":"INSPIRE bu-core3d 4.0",
            "source_file":f"WFS_page_{page_number:04d}",
            "source_path":f"STARTINDEX={start_index}&COUNT={SL_PAGE_SIZE}",
            "source_url":service_url
        })

    gdf = gpd.GeoDataFrame(records,geometry="geometry",crs="EPSG:4258").to_crs(SL_OUTPUT_CRS)
    gdf["footprint_area_m2"] = gdf.geometry.area
    return gdf, len(part_ids), len(parts)

print("="*72)
print("SAARLAND FULL INSPIRE LoD2 EXTRACTION")
print("="*72)
print(f"\nBuildings expected : {SL_TOTAL:,}")
print(f"Page size          : {SL_PAGE_SIZE:,}")
print(f"Expected pages     : {SL_PAGES:,}")
print(f"Raw GML saved      : False\n")

for page in tqdm(range(1,SL_PAGES+1),desc="Processing Saarland pages"):
    start_index = (page-1)*SL_PAGE_SIZE
    part_path = SL_PART/f"lod2_buildings_SL_part_{page:04d}.parquet"
    manifest_path = SL_MANIFEST/f"lod2_manifest_SL_part_{page:04d}.csv"

    if part_path.exists() and manifest_path.exists():
        old = pd.read_csv(manifest_path)
        if not old.empty and old.iloc[0].get("status")=="complete": continue

    error = None
    for attempt in range(1,SL_PAGE_RETRIES+1):
        try:
            response = sl_get_wfs({"SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature","TYPENAMES":SL_TYPENAME,"COUNT":SL_PAGE_SIZE,"STARTINDEX":start_index,"SRSNAME":SL_REQUEST_CRS})
            gdf, referenced_parts, returned_parts = sl_parse_page(response.content,page,start_index)
            part_path.unlink(missing_ok=True)
            gdf.to_parquet(part_path,index=False)
            manifest = {
                "page_number":page,
                "start_index":start_index,
                "requested_count":min(SL_PAGE_SIZE,SL_TOTAL-start_index),
                "building_count":len(gdf),
                "multipart_buildings":int(gdf.building_part_count.gt(0).sum()),
                "referenced_parts":referenced_parts,
                "returned_parts":returned_parts,
                "missing_part_references":referenced_parts-returned_parts,
                "status":"complete",
                "error":None,
                "part_file":str(part_path)
            }
            pd.DataFrame([manifest]).to_csv(manifest_path,index=False)
            break
        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            time.sleep(min(attempt*20,60))
    else:
        pd.DataFrame([{
            "page_number":page,"start_index":start_index,
            "requested_count":min(SL_PAGE_SIZE,SL_TOTAL-start_index),
            "building_count":0,"multipart_buildings":0,
            "referenced_parts":0,"returned_parts":0,
            "missing_part_references":0,"status":"failed",
            "error":error,"part_file":str(part_path)
        }]).to_csv(manifest_path,index=False)

manifest_files = sorted(SL_MANIFEST.glob("lod2_manifest_SL_part_*.csv"))
manifests = pd.concat([pd.read_csv(x) for x in manifest_files],ignore_index=True) if manifest_files else pd.DataFrame()
expected_pages = set(range(1,SL_PAGES+1))
recorded_pages = set(manifests.page_number.astype(int)) if not manifests.empty else set()
complete = manifests[manifests.status.eq("complete")].copy() if not manifests.empty else pd.DataFrame()
failed = manifests[~manifests.status.eq("complete")].copy() if not manifests.empty else pd.DataFrame()

ids, method_counts, footprint_counts = [], [], []
buildings = missing_ids = measured_heights = missing_heights = usable_geometries = missing_geometries = 0
minimum_height, maximum_height = np.inf, -np.inf

for path in sorted(SL_PART.glob("lod2_buildings_SL_part_*.parquet")):
    gdf = gpd.read_parquet(path,columns=["lod2_id","measured_height_m","height_method","footprint_method","geometry"])
    ids.append(gdf.lod2_id.astype("string"))
    buildings += len(gdf)
    missing_ids += int(gdf.lod2_id.isna().sum() + gdf.lod2_id.astype("string").str.strip().eq("").sum())
    measured_heights += int(gdf.measured_height_m.notna().sum())
    missing_heights += int(gdf.measured_height_m.isna().sum())
    usable = gdf.geometry.notna() & ~gdf.geometry.is_empty
    usable_geometries += int(usable.sum())
    missing_geometries += int((~usable).sum())
    heights = gdf.measured_height_m.dropna()
    if not heights.empty:
        minimum_height = min(minimum_height,float(heights.min()))
        maximum_height = max(maximum_height,float(heights.max()))
    method_counts.append(gdf.height_method.value_counts())
    footprint_counts.append(gdf.footprint_method.value_counts())

all_ids = pd.concat(ids,ignore_index=True) if ids else pd.Series(dtype="string")
duplicate_ids = int(all_ids.duplicated().sum())
height_summary = pd.concat(method_counts,axis=1).fillna(0).sum(axis=1).astype(int).rename("building_count").reset_index(names="height_method")
footprint_summary = pd.concat(footprint_counts,axis=1).fillna(0).sum(axis=1).astype(int).rename("building_count").reset_index(names="footprint_method")

summary = pd.DataFrame([{
    "state_code":"SL",
    "state_name":"Saarland",
    "source_files_expected":SL_PAGES,
    "source_files_recorded":len(manifests),
    "files_with_buildings":int(complete.building_count.gt(0).sum()) if not complete.empty else 0,
    "empty_files":int(complete.building_count.eq(0).sum()) if not complete.empty else 0,
    "failed_files":len(failed),
    "missing_source_files":len(expected_pages-recorded_pages),
    "unexpected_source_files":len(recorded_pages-expected_pages),
    "duplicate_manifest_sources":int(manifests.page_number.duplicated().sum()) if not manifests.empty else 0,
    "duplicate_building_ids":duplicate_ids,
    "buildings_expected":SL_TOTAL,
    "buildings":buildings,
    "missing_ids":missing_ids,
    "measured_heights":measured_heights,
    "missing_heights":missing_heights,
    "usable_geometries":usable_geometries,
    "missing_geometries":missing_geometries,
    "minimum_height_m":minimum_height if np.isfinite(minimum_height) else np.nan,
    "maximum_height_m":maximum_height if np.isfinite(maximum_height) else np.nan,
    "raw_gml_saved":False,
    "output_directory":str(SL_OUT)
}])

summary.to_csv(REPORT_DIR/"lod2_extraction_summary_SL.csv",index=False)
height_summary.to_csv(REPORT_DIR/"lod2_height_methods_SL.csv",index=False)
footprint_summary.to_csv(REPORT_DIR/"lod2_footprint_methods_SL.csv",index=False)
manifests.to_csv(REPORT_DIR/"lod2_processing_manifest_SL.csv",index=False)

registry_row = {
    "state_code":"SL","state_name":"Saarland","source_type":"INSPIRE WFS",
    "source_url":service_url,"source_crs":"EPSG:4258","output_crs":"EPSG:25832",
    "source_files":SL_PAGES,"buildings":buildings,"status":"complete" if len(failed)==0 and buildings==SL_TOTAL and duplicate_ids==0 else "incomplete"
}
if "source_registry" not in globals(): source_registry = pd.DataFrame()
if not source_registry.empty and "state_code" in source_registry: source_registry = source_registry[source_registry.state_code.ne("SL")]
source_registry = pd.concat([source_registry,pd.DataFrame([registry_row])],ignore_index=True)
if "source_registry_path" not in globals(): source_registry_path = REPORT_DIR/"lod2_source_registry.csv"
source_registry.to_csv(source_registry_path,index=False)

print("\n"+"="*72)
print("SAARLAND EXTRACTION VALIDATION")
print("="*72)
display(summary)

print("\nHeight methods:")
display(height_summary)

print("\nFootprint methods:")
display(footprint_summary)

if len(failed) or expected_pages-recorded_pages or buildings!=SL_TOTAL or duplicate_ids:
    print("\nVALIDATION FAILED — rerun Cell 93 to retry incomplete pages.")
    if len(failed): display(failed[["page_number","start_index","error"]])
else:
    print("\nVALIDATION PASSED — Saarland and all 16 German states are complete.")

SAARLAND FULL INSPIRE LoD2 EXTRACTION

Buildings expected : 738,006
Page size          : 2,000
Expected pages     : 370
Raw GML saved      : False



Processing Saarland pages: 100%|███████████████████████| 370/370 [32:47:20<00:00, 319.03s/it]


TypeError: Series.reset_index() got an unexpected keyword argument 'names'

In [29]:
# ============================================================
# 93B — REPAIR FAILED SAARLAND WFS PAGES
# ============================================================

import time
from urllib.parse import urlencode

SL_PART, SL_MANIFEST = EXTRACTED_DIR/"SL"/"parts", EXTRACTED_DIR/"SL"/"manifests"
SL_REPAIR_RAW = RAW_DIR/"SL"/"repair_chunks"
SL_REPAIR_RAW.mkdir(parents=True,exist_ok=True)

SL_TOTAL, SL_PAGE_SIZE, SL_REPAIR_CHUNK, SL_MIN_CHUNK = 738006, 2000, 500, 25
SL_TYPENAME, SL_REQUEST_CRS = "bu-core3d:Building", "urn:ogc:def:crs:EPSG::4258"

manifest_files = sorted(SL_MANIFEST.glob("lod2_manifest_SL_part_*.csv"))
manifests = pd.concat([pd.read_csv(x) for x in manifest_files],ignore_index=True)
failed_pages = manifests[manifests["status"].ne("complete")].sort_values("page_number").copy()

def fetch_sl_range(start,count,page):
    raw_path = SL_REPAIR_RAW/f"page_{page:04d}_start_{start:06d}_count_{count:04d}.gml"
    try:
        if raw_path.exists():
            content = raw_path.read_bytes()
        else:
            response = sl_get_wfs({
                "SERVICE":"WFS","VERSION":"2.0.0","REQUEST":"GetFeature",
                "TYPENAMES":SL_TYPENAME,"COUNT":count,"STARTINDEX":start,
                "SRSNAME":SL_REQUEST_CRS
            })
            content = response.content
            raw_path.write_bytes(content)

        gdf,referenced_parts,returned_parts = sl_parse_page(content,page,start)
        if len(gdf)!=count: raise RuntimeError(f"Expected {count:,} buildings but received {len(gdf):,}")
        return gdf,referenced_parts,returned_parts,1

    except Exception:
        raw_path.unlink(missing_ok=True)
        if count<=SL_MIN_CHUNK: raise
        left = count//2
        right = count-left
        gdf1,ref1,ret1,n1 = fetch_sl_range(start,left,page)
        gdf2,ref2,ret2,n2 = fetch_sl_range(start+left,right,page)
        gdf = gpd.GeoDataFrame(pd.concat([gdf1,gdf2],ignore_index=True),geometry="geometry",crs=gdf1.crs)
        return gdf,ref1+ref2,ret1+ret2,n1+n2

print("="*72)
print("SAARLAND FAILED-PAGE REPAIR")
print("="*72)
print(f"\nFailed pages       : {len(failed_pages):,}")
print(f"Buildings missing  : {failed_pages['requested_count'].sum():,.0f}")
print(f"Initial chunk size : {SL_REPAIR_CHUNK:,}")
print(f"Raw cache          : {SL_REPAIR_RAW}\n")

repaired,still_failed = 0,[]

for row in tqdm(failed_pages.itertuples(index=False),total=len(failed_pages),desc="Repairing Saarland pages"):
    page,start = int(row.page_number),int(row.start_index)
    requested = min(SL_PAGE_SIZE,SL_TOTAL-start)
    part_path = SL_PART/f"lod2_buildings_SL_part_{page:04d}.parquet"
    manifest_path = SL_MANIFEST/f"lod2_manifest_SL_part_{page:04d}.csv"

    try:
        frames,referenced_parts,returned_parts,requests_used = [],0,0,0

        for offset in range(0,requested,SL_REPAIR_CHUNK):
            count = min(SL_REPAIR_CHUNK,requested-offset)
            gdf,refs,returned,n_requests = fetch_sl_range(start+offset,count,page)
            frames.append(gdf); referenced_parts += refs; returned_parts += returned; requests_used += n_requests

        repaired_gdf = gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry="geometry",crs=frames[0].crs)

        if len(repaired_gdf)!=requested: raise RuntimeError(f"Expected {requested:,} rows but reconstructed {len(repaired_gdf):,}")
        if repaired_gdf["lod2_id"].duplicated().any(): raise RuntimeError(f"{repaired_gdf['lod2_id'].duplicated().sum():,} duplicate IDs inside repaired page")

        temp_path = part_path.with_suffix(".tmp.parquet")
        repaired_gdf.to_parquet(temp_path,index=False)
        temp_path.replace(part_path)

        pd.DataFrame([{
            "page_number":page,
            "start_index":start,
            "requested_count":requested,
            "building_count":len(repaired_gdf),
            "multipart_buildings":int(repaired_gdf["building_part_count"].gt(0).sum()),
            "referenced_parts":referenced_parts,
            "returned_parts":returned_parts,
            "missing_part_references":referenced_parts-returned_parts,
            "repair_requests":requests_used,
            "status":"complete",
            "error":None,
            "part_file":str(part_path)
        }]).to_csv(manifest_path,index=False)

        repaired += 1

    except Exception as e:
        error = f"{type(e).__name__}: {e}"
        still_failed.append({"page_number":page,"start_index":start,"error":error})
        pd.DataFrame([{
            "page_number":page,
            "start_index":start,
            "requested_count":requested,
            "building_count":0,
            "multipart_buildings":0,
            "referenced_parts":0,
            "returned_parts":0,
            "missing_part_references":0,
            "repair_requests":0,
            "status":"failed",
            "error":error,
            "part_file":str(part_path)
        }]).to_csv(manifest_path,index=False)

updated = pd.concat([pd.read_csv(x) for x in sorted(SL_MANIFEST.glob("lod2_manifest_SL_part_*.csv"))],ignore_index=True)
remaining = updated[updated["status"].ne("complete")]
recorded_buildings = int(updated.loc[updated["status"].eq("complete"),"building_count"].sum())

print("\n"+"="*72)
print("SAARLAND REPAIR RESULT")
print("="*72)
print(f"\nPages repaired       : {repaired:,}")
print(f"Pages still failed   : {len(remaining):,}")
print(f"Buildings recorded   : {recorded_buildings:,}")
print(f"Buildings expected   : {SL_TOTAL:,}")
print(f"Buildings remaining  : {SL_TOTAL-recorded_buildings:,}")

if not remaining.empty: display(remaining[["page_number","start_index","error"]])
else: print("\nAll failed pages were repaired. Run Cell 93A again for final validation.")

SAARLAND FAILED-PAGE REPAIR

Failed pages       : 52
Buildings missing  : 104,000
Initial chunk size : 500
Raw cache          : /fast/home/o-olajuyigbe/data/germany_lod2/raw/SL/repair_chunks



Repairing Saarland pages: 100%|███████████████████████████| 52/52 [3:11:09<00:00, 220.57s/it]



SAARLAND REPAIR RESULT

Pages repaired       : 52
Pages still failed   : 0
Buildings recorded   : 738,006
Buildings expected   : 738,006
Buildings remaining  : 0

All failed pages were repaired. Run Cell 93A again for final validation.


In [30]:
# ============================================================
# 93A — SAARLAND VALIDATION ONLY
# ============================================================

SL_OUT, SL_PART, SL_MANIFEST = EXTRACTED_DIR/"SL", EXTRACTED_DIR/"SL"/"parts", EXTRACTED_DIR/"SL"/"manifests"
SL_TOTAL, SL_PAGES = 738006, 370

manifest_files = sorted(SL_MANIFEST.glob("lod2_manifest_SL_part_*.csv"))
manifests = pd.concat([pd.read_csv(x) for x in manifest_files],ignore_index=True) if manifest_files else pd.DataFrame()
expected_pages = set(range(1,SL_PAGES+1))
recorded_pages = set(manifests["page_number"].astype(int)) if not manifests.empty else set()
complete = manifests[manifests["status"].eq("complete")].copy() if not manifests.empty else pd.DataFrame()
failed = manifests[~manifests["status"].eq("complete")].copy() if not manifests.empty else pd.DataFrame()

ids, method_counts, footprint_counts = [], [], []
buildings = missing_ids = measured_heights = missing_heights = usable_geometries = missing_geometries = 0
minimum_height, maximum_height = np.inf, -np.inf

for path in tqdm(sorted(SL_PART.glob("lod2_buildings_SL_part_*.parquet")),desc="Validating Saarland parts"):
    gdf = gpd.read_parquet(path,columns=["lod2_id","measured_height_m","height_method","footprint_method","geometry"])
    ids.append(gdf["lod2_id"].astype("string"))
    buildings += len(gdf)
    missing_ids += int(gdf["lod2_id"].isna().sum()+gdf["lod2_id"].astype("string").str.strip().eq("").sum())
    measured_heights += int(gdf["measured_height_m"].notna().sum())
    missing_heights += int(gdf["measured_height_m"].isna().sum())
    usable = gdf.geometry.notna() & ~gdf.geometry.is_empty
    usable_geometries += int(usable.sum()); missing_geometries += int((~usable).sum())
    heights = gdf["measured_height_m"].dropna()
    if not heights.empty:
        minimum_height = min(minimum_height,float(heights.min()))
        maximum_height = max(maximum_height,float(heights.max()))
    method_counts.append(gdf["height_method"].value_counts())
    footprint_counts.append(gdf["footprint_method"].value_counts())

all_ids = pd.concat(ids,ignore_index=True) if ids else pd.Series(dtype="string")
duplicate_ids = int(all_ids.duplicated().sum())

height_summary = pd.concat(method_counts,axis=1).fillna(0).sum(axis=1).astype(int).rename("building_count").reset_index().rename(columns={"index":"height_method"})
footprint_summary = pd.concat(footprint_counts,axis=1).fillna(0).sum(axis=1).astype(int).rename("building_count").reset_index().rename(columns={"index":"footprint_method"})

summary = pd.DataFrame([{
    "state_code":"SL",
    "state_name":"Saarland",
    "source_files_expected":SL_PAGES,
    "source_files_recorded":len(manifests),
    "files_with_buildings":int(complete["building_count"].gt(0).sum()) if not complete.empty else 0,
    "empty_files":int(complete["building_count"].eq(0).sum()) if not complete.empty else 0,
    "failed_files":len(failed),
    "missing_source_files":len(expected_pages-recorded_pages),
    "unexpected_source_files":len(recorded_pages-expected_pages),
    "duplicate_manifest_sources":int(manifests["page_number"].duplicated().sum()) if not manifests.empty else 0,
    "duplicate_building_ids":duplicate_ids,
    "buildings_expected":SL_TOTAL,
    "buildings":buildings,
    "missing_ids":missing_ids,
    "measured_heights":measured_heights,
    "missing_heights":missing_heights,
    "usable_geometries":usable_geometries,
    "missing_geometries":missing_geometries,
    "minimum_height_m":minimum_height if np.isfinite(minimum_height) else np.nan,
    "maximum_height_m":maximum_height if np.isfinite(maximum_height) else np.nan,
    "raw_gml_saved":False,
    "output_directory":str(SL_OUT)
}])

summary.to_csv(REPORT_DIR/"lod2_extraction_summary_SL.csv",index=False)
height_summary.to_csv(REPORT_DIR/"lod2_height_methods_SL.csv",index=False)
footprint_summary.to_csv(REPORT_DIR/"lod2_footprint_methods_SL.csv",index=False)
manifests.to_csv(REPORT_DIR/"lod2_processing_manifest_SL.csv",index=False)

print("="*72)
print("SAARLAND FINAL LoD2 VALIDATION")
print("="*72)
display(summary)

print("\nHeight methods:")
display(height_summary)

print("\nFootprint methods:")
display(footprint_summary)

valid = len(failed)==0 and len(expected_pages-recorded_pages)==0 and buildings==SL_TOTAL and duplicate_ids==0
print("\nVALIDATION PASSED — all 16 German states are complete." if valid else "\nVALIDATION FAILED — inspect the reported differences.")
if len(failed): display(failed[["page_number","start_index","error"]])

Validating Saarland parts: 100%|███████████████████████████| 370/370 [00:09<00:00, 37.15it/s]


SAARLAND FINAL LoD2 VALIDATION


,state_code,state_name,source_files_expected,source_files_recorded,files_with_buildings,empty_files,failed_files,missing_source_files,unexpected_source_files,duplicate_manifest_sources,...,buildings,missing_ids,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,raw_gml_saved,output_directory
0,SL,Saarland,370,370,370,0,0,0,0,0,...,738006,0,738006,0,738006,0,0.017,287.0,False,/fast/home/o-olajuyigbe/data/germany_lod2/extr...



Height methods:


,height_method,building_count
0,height_above_ground,698283
1,maximum_building_part_height,39723



Footprint methods:


,footprint_method,building_count
0,projected_lod2_surfaces,698283
1,union_building_part_surfaces,39723



VALIDATION PASSED — all 16 German states are complete.


In [31]:
# ============================================================
# 94 — NATIONAL LoD2 STATE INVENTORY
# ============================================================

state_summaries = []

for path in sorted(REPORT_DIR.glob("lod2_extraction_summary_*.csv")):
    df = pd.read_csv(path)
    if not df.empty: state_summaries.append(df.iloc[0])

lod2_inventory = pd.DataFrame(state_summaries).sort_values("state_code").reset_index(drop=True)

required_states = {"BB","BE","BW","BY","HB","HE","HH","MV","NI","NW","RP","SH","SL","SN","ST","TH"}
recorded_states = set(lod2_inventory["state_code"])
missing_states = sorted(required_states-recorded_states)
unexpected_states = sorted(recorded_states-required_states)

numeric_cols = ["buildings","measured_heights","missing_heights","usable_geometries","missing_geometries","failed_files","missing_source_files"]
for col in numeric_cols:
    if col in lod2_inventory: lod2_inventory[col] = pd.to_numeric(lod2_inventory[col],errors="coerce").fillna(0).astype("int64")

lod2_inventory["validation_passed"] = (
    lod2_inventory["failed_files"].eq(0)
    & lod2_inventory["missing_source_files"].eq(0)
    & lod2_inventory["missing_ids"].eq(0)
    & lod2_inventory["missing_heights"].eq(0)
)

lod2_inventory.to_csv(REPORT_DIR/"lod2_national_state_inventory.csv",index=False)

print("="*80)
print("GERMANY NATIONAL LoD2 INVENTORY")
print("="*80)
display(lod2_inventory[[
    "state_code","state_name","buildings","measured_heights",
    "missing_heights","usable_geometries","missing_geometries",
    "minimum_height_m","maximum_height_m","validation_passed"
]])

print(f"\nStates expected       : {len(required_states):,}")
print(f"States recorded       : {len(recorded_states):,}")
print(f"Total LoD2 buildings : {lod2_inventory['buildings'].sum():,}")
print(f"Usable geometries    : {lod2_inventory['usable_geometries'].sum():,}")
print(f"Missing geometries   : {lod2_inventory['missing_geometries'].sum():,}")
print(f"Missing states       : {missing_states}")
print(f"Unexpected states    : {unexpected_states}")
print(f"All states validated : {len(missing_states)==0 and lod2_inventory['validation_passed'].all()}")

GERMANY NATIONAL LoD2 INVENTORY


,state_code,state_name,buildings,measured_heights,missing_heights,usable_geometries,missing_geometries,minimum_height_m,maximum_height_m,validation_passed
0,BB,Brandenburg,2376794,2376794,0,2331500,45294,0.500,170.683,False
1,BE,Berlin,639658,639658,0,0,0,0.000,230.000,False
2,BW,Baden-Württemberg,6465296,6465296,0,6465292,4,0.143,269.000,True
3,BY,Bavaria,10111920,10111920,0,10111920,0,-0.030,292.980,True
4,HB,Bremen,331804,331804,0,0,0,0.010,249.020,False
5,HE,Hesse,4981859,4981859,0,4981787,72,-0.380,261.813,True
6,HH,Hamburg,388729,388729,0,0,0,0.737,158.470,False
7,MV,Mecklenburg-Western Pomerania,1332053,1332053,0,1332053,0,0.110,268.000,False
8,NI,Lower Saxony,6828051,6828051,0,6828014,37,0.000,344.375,False
9,NW,North Rhine-Westphalia,11870213,11870213,0,11870213,0,0.000,320.657,True



States expected       : 16
States recorded       : 16
Total LoD2 buildings : 58,049,619
Usable geometries    : 56,644,020
Missing geometries   : 45,408
Missing states       : []
Unexpected states    : []
All states validated : False


In [32]:
# ============================================================
# 95A — MATCHING DIRECTORY SETUP
# ============================================================

from pathlib import Path

BASE_DIR = Path("/fast/home/o-olajuyigbe/data/germany_lod2")
EXTRACTED_DIR = BASE_DIR/"extracted"
MATCHED_DIR = BASE_DIR/"matched_to_osm"
MATCHED_STATE_DIR = MATCHED_DIR/"states"
MATCH_AUDIT_DIR = MATCHED_DIR/"audit"
FINAL_DIR = BASE_DIR/"final"

for directory in [MATCHED_DIR,MATCHED_STATE_DIR,MATCH_AUDIT_DIR,FINAL_DIR]:
    directory.mkdir(parents=True,exist_ok=True)

print(f"Base directory     : {BASE_DIR}")
print(f"LoD2 inputs        : {EXTRACTED_DIR}")
print(f"Matched states     : {MATCHED_STATE_DIR}")
print(f"Matching audits    : {MATCH_AUDIT_DIR}")
print(f"Final outputs      : {FINAL_DIR}")

Base directory     : /fast/home/o-olajuyigbe/data/germany_lod2
LoD2 inputs        : /fast/home/o-olajuyigbe/data/germany_lod2/extracted
Matched states     : /fast/home/o-olajuyigbe/data/germany_lod2/matched_to_osm/states
Matching audits    : /fast/home/o-olajuyigbe/data/germany_lod2/matched_to_osm/audit
Final outputs      : /fast/home/o-olajuyigbe/data/germany_lod2/final


In [33]:
# ============================================================
# 95 — AUTOMATED NATIONAL STATE-BY-STATE MATCHING RUNNER
# ============================================================

import gc
from pathlib import Path
from tqdm.auto import tqdm

MATCHED_DIR = BASE_DIR/"matched_to_osm"
MATCHED_STATE_DIR = MATCHED_DIR/"states"
MATCH_AUDIT_DIR = MATCHED_DIR/"audit"
MATCHED_STATE_DIR.mkdir(parents=True,exist_ok=True)
MATCH_AUDIT_DIR.mkdir(parents=True,exist_ok=True)

GERMAN_STATES = ["HB","HH","BE","SL","SH","MV","ST","TH","SN","BB","HE","RP","BW","NI","BY","NW"]

def load_state_lod2(state_code):
    paths = sorted((EXTRACTED_DIR/state_code/"parts").glob(f"lod2_buildings_{state_code}_part_*.parquet"))
    if not paths: raise FileNotFoundError(f"No LoD2 parts found for {state_code}")
    frames = [gpd.read_parquet(path) for path in paths]
    gdf = gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry="geometry",crs=frames[0].crs)
    del frames
    return gdf

def load_state_osm(osm_source,state_code,state_column="state_code"):
    if isinstance(osm_source,(str,Path)):
        path = Path(osm_source)
        try:
            return gpd.read_parquet(path,filters=[(state_column,"==",state_code)])
        except Exception:
            gdf = gpd.read_parquet(path)
            result = gdf[gdf[state_column].eq(state_code)].copy()
            del gdf
            return result
    return osm_source[osm_source[state_column].eq(state_code)].copy()

def run_all_states(osm_source,match_function,state_column="state_code",states=GERMAN_STATES,overwrite=False,**match_kwargs):
    summaries = []

    for state_code in tqdm(states,desc="Matching German states"):
        output_path = MATCHED_STATE_DIR/f"osm_buildings_{state_code}_with_lod2.parquet"
        audit_path = MATCH_AUDIT_DIR/f"lod2_match_audit_{state_code}.parquet"

        if output_path.exists() and audit_path.exists() and not overwrite:
            summaries.append({
                "state_code":state_code,
                "status":"skipped_complete",
                "output_path":str(output_path),
                "audit_path":str(audit_path)
            })
            continue

        try:
            osm_state = load_state_osm(osm_source,state_code,state_column)
            lod2_state = load_state_lod2(state_code)

            if osm_state.crs is None: raise ValueError(f"OSM buildings for {state_code} have no CRS")
            if lod2_state.crs is None: raise ValueError(f"LoD2 buildings for {state_code} have no CRS")

            matched,audit,summary = match_function(
                osm_state=osm_state,
                lod2_state=lod2_state,
                state_code=state_code,
                **match_kwargs
            )

            matched.to_parquet(output_path,index=False)
            audit.to_parquet(audit_path,index=False)

            summary = dict(summary)
            summary.update({
                "state_code":state_code,
                "status":"complete",
                "osm_buildings":len(osm_state),
                "lod2_buildings":len(lod2_state),
                "matched_output_rows":len(matched),
                "audit_rows":len(audit),
                "output_path":str(output_path),
                "audit_path":str(audit_path)
            })
            summaries.append(summary)

        except Exception as e:
            summaries.append({
                "state_code":state_code,
                "status":"failed",
                "error":f"{type(e).__name__}: {e}",
                "output_path":str(output_path),
                "audit_path":str(audit_path)
            })

        finally:
            for name in ["osm_state","lod2_state","matched","audit"]:
                if name in locals(): del locals()[name]
            gc.collect()

        pd.DataFrame(summaries).to_csv(MATCH_AUDIT_DIR/"national_matching_progress.csv",index=False)

    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(MATCH_AUDIT_DIR/"national_matching_summary.csv",index=False)
    return summary_df

print("="*72)
print("NATIONAL MATCHING RUNNER READY")
print("="*72)
print("\nThe runner will process states sequentially and resume automatically.")
print("Next: define and validate match_state_lod2_to_osm() using Bremen.")

NATIONAL MATCHING RUNNER READY

The runner will process states sequentially and resume automatically.
Next: define and validate match_state_lod2_to_osm() using Bremen.


In [ ]:
# ============================================================
# 96A — FLEXIBLE LoD2 STATE LOADER
# ============================================================

import gc
from pathlib import Path

def find_state_lod2_paths(state_code):
    state_code = state_code.upper()
    roots = [EXTRACTED_DIR/state_code]

    summary_path = REPORT_DIR/f"lod2_extraction_summary_{state_code}.csv"
    if summary_path.exists():
        summary = pd.read_csv(summary_path)
        if not summary.empty and "output_directory" in summary:
            output_dir = Path(str(summary.iloc[0]["output_directory"]))
            roots.extend([output_dir,output_dir/"parts"])

        if not summary.empty and "state_name" in summary:
            state_name = str(summary.iloc[0]["state_name"])
            roots.extend([EXTRACTED_DIR/state_name,EXTRACTED_DIR/state_name.replace(" ","_")])

    paths = []
    for root in roots:
        if root.is_file() and root.suffix.lower()==".parquet":
            paths.append(root)
        elif root.exists():
            paths.extend(root.glob("*.parquet"))
            paths.extend(root.glob("parts/*.parquet"))
            paths.extend(root.rglob("*.parquet"))

    if not paths:
        paths.extend(EXTRACTED_DIR.rglob(f"*{state_code}*.parquet"))

    paths = sorted({
        p.resolve() for p in paths
        if p.is_file()
        and p.suffix.lower()==".parquet"
        and not any(x in p.name.lower() for x in ["audit","matched","test","temp","tmp"])
    })

    if not paths:
        available = sorted({p.parent.name for p in EXTRACTED_DIR.rglob("*.parquet")})
        raise FileNotFoundError(
            f"No LoD2 Parquet files found for {state_code}.\n"
            f"EXTRACTED_DIR: {EXTRACTED_DIR}\n"
            f"Available Parquet folders: {available[:30]}"
        )

    return paths

def load_state_lod2(state_code,columns=None):
    paths = find_state_lod2_paths(state_code)
    frames = [gpd.read_parquet(path,columns=columns) for path in tqdm(paths,desc=f"Loading {state_code} LoD2")]

    crs_values = {str(frame.crs) for frame in frames}
    if len(crs_values)>1: raise ValueError(f"{state_code} parts have inconsistent CRS values: {crs_values}")

    gdf = gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry="geometry",crs=frames[0].crs)
    del frames
    gc.collect()

    print("="*72)
    print(f"{state_code} LoD2 DATASET LOADED")
    print("="*72)
    print(f"\nFiles loaded      : {len(paths):,}")
    print(f"Buildings         : {len(gdf):,}")
    print(f"CRS               : {gdf.crs}")
    print(f"Missing IDs       : {gdf['lod2_id'].isna().sum():,}")
    print(f"Missing heights   : {gdf['measured_height_m'].isna().sum():,}")
    print(f"Missing geometry  : {(gdf.geometry.isna()|gdf.geometry.is_empty).sum():,}")
    print("\nFiles:")
    for path in paths[:10]: print(f"  {path}")
    if len(paths)>10: print(f"  ... and {len(paths)-10:,} more")

    return gdf

hb_lod2 = load_state_lod2("HB")

Loading HB LoD2:   0%|                                                 | 0/2 [00:00<?, ?it/s]

Loading HB LoD2: 100%|█████████████████████████████████████████| 2/2 [00:01<00:00,  1.19it/s]


HB LoD2 DATASET LOADED

Files loaded      : 2
Buildings         : 663,608
CRS               : {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89 / UTM zone 32N", "base_crs": {"type": "GeographicCRS", "name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}, {"name": "European Terrestri

In [36]:
# ============================================================
# 96B — LOAD ONLY FINAL LoD2 OUTPUTS
# ============================================================

import gc
from pathlib import Path

EXCLUDE_NAMES = ("raw","test","sample","backup","temp","tmp","audit","matched")

def find_state_lod2_paths(state_code):
    state_code = state_code.upper()
    state_dir = EXTRACTED_DIR/state_code
    if not state_dir.exists(): raise FileNotFoundError(f"Missing state directory: {state_dir}")

    part_paths = sorted((state_dir/"parts").glob(f"lod2_buildings_{state_code}_part_*.parquet"))
    if part_paths: return part_paths

    canonical = state_dir/f"lod2_buildings_{state_code}.parquet"
    if canonical.exists(): return [canonical]

    paths = sorted({
        p.resolve() for p in state_dir.rglob("*.parquet")
        if not any(x in p.stem.lower() for x in EXCLUDE_NAMES)
    })
    if not paths: raise FileNotFoundError(f"No final LoD2 Parquet files found for {state_code}")
    return paths

def load_state_lod2(state_code,columns=None,validate=True):
    paths = find_state_lod2_paths(state_code)
    frames = [gpd.read_parquet(path,columns=columns) for path in tqdm(paths,desc=f"Loading {state_code} LoD2")]
    crs_values = {str(frame.crs) for frame in frames}
    if len(crs_values)!=1: raise ValueError(f"Inconsistent CRS values for {state_code}: {crs_values}")

    gdf = gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry="geometry",crs=frames[0].crs)
    del frames; gc.collect()

    summary_path = REPORT_DIR/f"lod2_extraction_summary_{state_code}.csv"
    expected = int(pd.read_csv(summary_path).iloc[0]["buildings"]) if summary_path.exists() else None

    if validate:
        if expected is not None and len(gdf)!=expected: raise ValueError(f"{state_code}: loaded {len(gdf):,}, expected {expected:,}")
        if gdf["lod2_id"].duplicated().any(): raise ValueError(f"{state_code}: {gdf['lod2_id'].duplicated().sum():,} duplicate LoD2 IDs")

    print("="*72)
    print(f"{state_code} FINAL LoD2 DATASET")
    print("="*72)
    print(f"\nFiles loaded     : {len(paths):,}")
    print(f"Buildings loaded : {len(gdf):,}")
    print(f"Buildings expected: {expected:,}" if expected is not None else "Buildings expected: unavailable")
    print(f"Duplicate IDs    : {gdf['lod2_id'].duplicated().sum():,}")
    print(f"Missing heights  : {gdf['measured_height_m'].isna().sum():,}")
    print(f"Missing geometry : {(gdf.geometry.isna()|gdf.geometry.is_empty).sum():,}")
    print(f"CRS              : {gdf.crs.to_epsg() or gdf.crs.name}")
    print("\nFiles:")
    for path in paths: print(f"  {path}")

    return gdf

del hb_lod2
gc.collect()
hb_lod2 = load_state_lod2("HB")

Loading HB LoD2: 100%|█████████████████████████████████████████| 1/1 [00:00<00:00,  1.37it/s]

HB FINAL LoD2 DATASET

Files loaded     : 1
Buildings loaded : 331,804
Buildings expected: 331,804
Duplicate IDs    : 0
Missing heights  : 0
Missing geometry : 0
CRS              : 25832

Files:
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/HB/lod2_buildings_HB.parquet


In [38]:
# ============================================================
# 96D — READ SPATIAL OSM SUBSET FROM GEOPARQUET
# ============================================================

import json, gc
import pyarrow.parquet as pq
import shapely
from pathlib import Path
from pyproj import CRS
from shapely.geometry import box

OSM_BUILDING_FILE=Path("/fast/home/o-olajuyigbe/osm_project/data/processed/germany_buildings_classified_stage2.parquet")
if not OSM_BUILDING_FILE.exists(): raise FileNotFoundError(OSM_BUILDING_FILE)
if "hb_lod2" not in globals(): hb_lod2=load_state_lod2("HB")

def geoparquet_metadata(path):
    pf=pq.ParquetFile(path)
    metadata=pf.metadata.metadata or {}
    if b"geo" not in metadata: raise ValueError("The Parquet file has no GeoParquet metadata.")
    geo=json.loads(metadata[b"geo"])
    geometry_column=geo["primary_column"]
    crs=CRS.from_user_input(geo["columns"][geometry_column]["crs"])
    return pf,geometry_column,crs

def read_geoparquet_bbox(path,bbox,columns=None):
    pf,geometry_column,crs=geoparquet_metadata(path)
    columns=list(dict.fromkeys((columns or [])+[geometry_column]))
    query_box=box(*bbox)
    frames=[]

    for row_group in tqdm(range(pf.num_row_groups),desc="Scanning OSM row groups"):
        df=pf.read_row_group(row_group,columns=columns).to_pandas()
        values=df[geometry_column].to_numpy()

        if len(values)==0: continue
        if isinstance(values[0],(bytes,bytearray,memoryview)): geometries=shapely.from_wkb(values)
        else: geometries=np.asarray(values,dtype=object)

        bounds=shapely.bounds(geometries)
        candidate=(bounds[:,2]>=bbox[0])&(bounds[:,0]<=bbox[2])&(bounds[:,3]>=bbox[1])&(bounds[:,1]<=bbox[3])
        if not candidate.any(): continue

        indices=np.flatnonzero(candidate)
        exact=shapely.intersects(geometries[indices],query_box)
        indices=indices[exact]
        if len(indices)==0: continue

        selected=df.iloc[indices].copy()
        selected[geometry_column]=geometries[indices]
        frames.append(selected)

        del df,values,geometries,bounds
        gc.collect()

    if not frames: return gpd.GeoDataFrame(columns=columns,geometry=geometry_column,crs=crs)
    return gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry=geometry_column,crs=crs)

# Densest Bremen 2 × 2 km window
centroids=hb_lod2.geometry.centroid
grid=pd.DataFrame({"gx":np.floor(centroids.x/2000).astype("int64"),"gy":np.floor(centroids.y/2000).astype("int64")})
gx,gy=grid.value_counts().index[0]
hb_core=box(gx*2000,gy*2000,(gx+1)*2000,(gy+1)*2000)
hb_buffer=hb_core.buffer(50)
hb_lod2_sample=hb_lod2[hb_lod2.geometry.intersects(hb_buffer)].copy()

_,_,osm_crs=geoparquet_metadata(OSM_BUILDING_FILE)
osm_bbox=tuple(gpd.GeoSeries([hb_buffer],crs=hb_lod2.crs).to_crs(osm_crs).total_bounds)

try:
    hb_osm_buffer=gpd.read_parquet(OSM_BUILDING_FILE,columns=["id","geometry"],bbox=osm_bbox)
    read_method="GeoPandas GeoParquet bbox filter"
except Exception as e:
    print(f"Native bbox filtering unavailable: {type(e).__name__}: {e}")
    hb_osm_buffer=read_geoparquet_bbox(OSM_BUILDING_FILE,osm_bbox,columns=["id"])

hb_osm_metric=hb_osm_buffer.to_crs(hb_lod2.crs)
inside_core=hb_osm_metric.geometry.centroid.within(hb_core)
hb_osm_sample=hb_osm_buffer.loc[inside_core].copy()

hb_matched,hb_audit,hb_match_summary=match_state_lod2_to_osm(
    osm_state=hb_osm_sample,
    lod2_state=hb_lod2_sample,
    state_code="HB",
    min_intersection_m2=1.0,
    min_cover=0.50,
    min_iou=0.25,
    audit_top_n=3
)

selected=hb_matched[hb_matched["lod2_id"].notna()]

print("="*72)
print("BREMEN 2 × 2 KM MATCHING CALIBRATION")
print("="*72)
print(f"\nOSM read method       : {read_method if 'read_method' in globals() else 'PyArrow row-group scan'}")
print(f"OSM buildings tested : {len(hb_osm_sample):,}")
print(f"LoD2 candidates      : {len(hb_lod2_sample):,}")
display(pd.DataFrame([hb_match_summary]))

print("\nMatch-quality distribution:")
display(hb_matched["lod2_match_quality"].fillna("unmatched").value_counts().rename_axis("quality").reset_index(name="buildings"))

print("\nSelected-match overlap statistics:")
display(selected[["lod2_iou","lod2_osm_coverage","lod2_footprint_coverage","lod2_score","lod2_score_margin"]].describe().round(3))

Native bbox filtering unavailable: ValueError: Specifying 'bbox' not supported for this Parquet file (it should either have a bbox covering column or use 'point' encoding).


Scanning OSM row groups: 100%|███████████████████████████████| 38/38 [01:32<00:00,  2.43s/it]


KeyError: 'height_method'

In [39]:
# ============================================================
# 96E — STANDARDIZE LEGACY LoD2 SCHEMAS + RERUN MATCH
# ============================================================

def standardize_lod2_schema(gdf):
    gdf=gdf.copy()

    def copy_first(target,candidates,default=None):
        if target in gdf.columns: return
        source=next((c for c in candidates if c in gdf.columns),None)
        gdf[target]=gdf[source] if source else default

    copy_first("lod2_id",["gml_id","id","building_id"])
    copy_first("measured_height_m",["measured_height","height_m","height"])
    copy_first("height_method",["height_source","height_type","measured_height_source"])
    copy_first("footprint_method",["footprint_source","geometry_method","geometry_source"])

    if gdf["height_method"].isna().all():
        multipart=pd.Series(False,index=gdf.index)
        if "maximum_part_height_m" in gdf:
            multipart=gdf["maximum_part_height_m"].notna()
            if "parent_height_m" in gdf: multipart&=gdf["parent_height_m"].isna()
        elif "building_part_count" in gdf:
            multipart=pd.to_numeric(gdf["building_part_count"],errors="coerce").fillna(0).gt(0)

        gdf["height_method"]=np.select(
            [gdf["measured_height_m"].isna(),multipart],
            ["missing","maximum_building_part_height"],
            default="building_measured_height"
        )

    if gdf["footprint_method"].isna().all():
        usable=gdf.geometry.notna() & ~gdf.geometry.is_empty
        gdf["footprint_method"]=np.where(usable,"legacy_lod2_footprint","missing")

    return gdf

# Apply to objects already loaded; no OSM rescan required
hb_lod2=standardize_lod2_schema(hb_lod2)
hb_lod2_sample=standardize_lod2_schema(hb_lod2_sample)

# Ensure future state loads are standardized automatically
if "_load_state_lod2_unstandardized" not in globals():
    _load_state_lod2_unstandardized=load_state_lod2

def load_state_lod2(state_code,columns=None,validate=True):
    return standardize_lod2_schema(_load_state_lod2_unstandardized(state_code,columns=columns,validate=validate))

hb_matched,hb_audit,hb_match_summary=match_state_lod2_to_osm(
    osm_state=hb_osm_sample,
    lod2_state=hb_lod2_sample,
    state_code="HB",
    min_intersection_m2=1.0,
    min_cover=0.50,
    min_iou=0.25,
    audit_top_n=3
)

selected=hb_matched[hb_matched["lod2_id"].notna()]

print("="*72)
print("BREMEN 2 × 2 KM MATCHING CALIBRATION")
print("="*72)
print(f"\nOSM buildings tested : {len(hb_osm_sample):,}")
print(f"LoD2 candidates      : {len(hb_lod2_sample):,}")
print(f"LoD2 input columns   : {list(hb_lod2_sample.columns)}")
display(pd.DataFrame([hb_match_summary]))

print("\nMatch-quality distribution:")
display(
    hb_matched["lod2_match_quality"]
    .fillna("unmatched")
    .value_counts()
    .rename_axis("quality")
    .reset_index(name="buildings")
)

print("\nSelected-match overlap statistics:")
display(
    selected[
        ["lod2_iou","lod2_osm_coverage","lod2_footprint_coverage",
         "lod2_score","lod2_score_margin"]
    ].describe().round(3)
)

BREMEN 2 × 2 KM MATCHING CALIBRATION

OSM buildings tested : 8,598
LoD2 candidates      : 12,635
LoD2 input columns   : ['lod2_id', 'creation_date', 'function', 'roof_type', 'measured_height_m', 'storeys_above_ground', 'source_file', 'geometry', 'state_code', 'source_city', 'height_method', 'footprint_method']


,state_code,osm_buildings,lod2_buildings,candidate_pairs,accepted_pairs,matched_buildings,unmatched_buildings,match_rate_pct,high_quality_matches,medium_quality_matches,low_quality_matches,lod2_used,lod2_reused
0,HB,8598,12635,18740,9503,8487,111,98.709002,4638,3341,508,8169,548



Match-quality distribution:


,quality,buildings
0,high,4638
1,medium,3341
2,low,508
3,unmatched,111



Selected-match overlap statistics:


,lod2_iou,lod2_osm_coverage,lod2_footprint_coverage,lod2_score,lod2_score_margin
count,8487.000,8487.000,8487.000,8487.000,8487.000
mean,0.716,0.853,0.824,0.777,0.750
std,0.160,0.117,0.164,0.129,0.179
min,0.005,0.023,0.005,0.154,0.000
25%,0.653,0.816,0.772,0.727,0.707
50%,0.755,0.882,0.866,0.810,0.804
75%,0.826,0.927,0.933,0.867,0.865
max,0.990,1.000,1.000,0.992,0.992


In [40]:
# ============================================================
# 97 — BREMEN MATCH-THRESHOLD DIAGNOSTICS
# ============================================================

if "hb_audit" not in globals() or "hb_matched" not in globals():
    raise RuntimeError("Run Cell 96E first.")

candidates=hb_audit.copy()
candidates["current_rule"]=(candidates["intersection_area_m2"]>=1.0)&((candidates["iou"]>=0.25)|(candidates["osm_coverage"]>=0.50)|(candidates["lod2_coverage"]>=0.50))

# Allows genuine subdivisions, but requires meaningful coverage of both footprints
candidates["balanced_rule"]=(candidates["intersection_area_m2"]>=1.0)&(
    (candidates["iou"]>=0.25)
    |((candidates["osm_coverage"]>=0.70)&(candidates["lod2_coverage"]>=0.20))
    |((candidates["lod2_coverage"]>=0.70)&(candidates["osm_coverage"]>=0.20))
)

# More conservative alternative
candidates["strict_rule"]=(candidates["intersection_area_m2"]>=2.0)&(
    (candidates["iou"]>=0.30)
    |((candidates["osm_coverage"]>=0.80)&(candidates["lod2_coverage"]>=0.30))
    |((candidates["lod2_coverage"]>=0.80)&(candidates["osm_coverage"]>=0.30))
)

def evaluate_rule(df,rule):
    accepted=df[df[rule]].sort_values(["osm_id","score","intersection_area_m2"],ascending=[True,False,False])
    best=accepted.drop_duplicates("osm_id")
    reuse=best["lod2_id"].value_counts()
    return {
        "rule":rule,
        "matched_buildings":len(best),
        "match_rate_pct":100*len(best)/len(hb_osm_sample),
        "unmatched_buildings":len(hb_osm_sample)-len(best),
        "median_iou":best["iou"].median(),
        "minimum_iou":best["iou"].min(),
        "median_osm_coverage":best["osm_coverage"].median(),
        "median_lod2_coverage":best["lod2_coverage"].median(),
        "lod2_ids_reused":int(reuse.gt(1).sum()),
        "osm_matches_in_reuse":int(reuse[reuse.gt(1)].sum()),
        "maximum_reuse_count":int(reuse.max()) if len(reuse) else 0
    }

comparison=pd.DataFrame([evaluate_rule(candidates,x) for x in ["current_rule","balanced_rule","strict_rule"]])
comparison.to_csv(MATCH_AUDIT_DIR/"HB_threshold_comparison.csv",index=False)

selected=hb_matched[hb_matched["lod2_id"].notna()].copy()
selected["match_pattern"]=np.select(
    [
        (selected["lod2_iou"]>=0.70)&(selected["lod2_osm_coverage"]>=0.80)&(selected["lod2_footprint_coverage"]>=0.80),
        (selected["lod2_osm_coverage"]>=0.70)&(selected["lod2_footprint_coverage"]<0.50),
        (selected["lod2_footprint_coverage"]>=0.70)&(selected["lod2_osm_coverage"]<0.50),
        (selected["lod2_iou"]>=0.25),
        (selected["lod2_osm_coverage"]>=0.50)|(selected["lod2_footprint_coverage"]>=0.50)
    ],
    ["strong_balanced","osm_inside_lod2","lod2_inside_osm","moderate_balanced","weak_containment"],
    default="weak_sliver"
)

pattern_summary=selected.groupby("match_pattern").agg(
    buildings=("lod2_id","size"),
    median_iou=("lod2_iou","median"),
    minimum_iou=("lod2_iou","min"),
    median_osm_coverage=("lod2_osm_coverage","median"),
    median_lod2_coverage=("lod2_footprint_coverage","median")
).reset_index().sort_values("buildings",ascending=False)

reuse_counts=selected["lod2_id"].value_counts()
reused_ids=set(reuse_counts[reuse_counts.gt(1)].index)
reused=selected[selected["lod2_id"].isin(reused_ids)].copy()
reused["reuse_count"]=reused["lod2_id"].map(reuse_counts)

weak=selected[
    (selected["lod2_iou"]<0.25)
    &((selected["lod2_osm_coverage"]<0.70)|(selected["lod2_footprint_coverage"]<0.20))
    &((selected["lod2_footprint_coverage"]<0.70)|(selected["lod2_osm_coverage"]<0.20))
].sort_values("lod2_score").copy()

weak.to_parquet(MATCH_AUDIT_DIR/"HB_weak_matches.parquet",index=False)
reused.to_parquet(MATCH_AUDIT_DIR/"HB_reused_lod2_matches.parquet",index=False)

print("="*72)
print("BREMEN MATCH-THRESHOLD COMPARISON")
print("="*72)
display(comparison.round(3))

print("\nCurrent selected-match patterns:")
display(pattern_summary.round(3))

print("\nWeak matches rejected by the balanced rule:")
print(f"{len(weak):,} of {len(selected):,} current matches")
display(weak[[
    "id","lod2_id","lod2_iou","lod2_osm_coverage",
    "lod2_footprint_coverage","lod2_intersection_m2","lod2_score"
]].head(20).round(3))

print("\nLoD2 reuse distribution:")
display(
    reuse_counts.value_counts()
    .sort_index()
    .rename_axis("osm_buildings_per_lod2")
    .reset_index(name="lod2_ids")
)

BREMEN MATCH-THRESHOLD COMPARISON


,rule,matched_buildings,match_rate_pct,unmatched_buildings,median_iou,minimum_iou,median_osm_coverage,median_lod2_coverage,lod2_ids_reused,osm_matches_in_reuse,maximum_reuse_count
0,current_rule,8487,98.709,111,0.755,0.005,0.882,0.866,230,548,11
1,balanced_rule,8383,97.499,215,0.757,0.193,0.882,0.867,193,428,4
2,strict_rule,8234,95.766,364,0.759,0.285,0.883,0.869,118,250,3



Current selected-match patterns:


,match_pattern,buildings,median_iou,minimum_iou,median_osm_coverage,median_lod2_coverage
3,strong_balanced,4638,0.819,0.700,0.899,0.909
1,moderate_balanced,3193,0.662,0.257,0.810,0.770
2,osm_inside_lod2,460,0.317,0.005,0.902,0.331
0,lod2_inside_osm,181,0.405,0.023,0.419,0.949
4,weak_containment,15,0.167,0.034,0.620,0.206



Weak matches rejected by the balanced rule:
104 of 8,487 current matches


,id,lod2_id,lod2_iou,lod2_osm_coverage,lod2_footprint_coverage,lod2_intersection_m2,lod2_score
3477,188143684,DEHB01ALn00004Gl,0.034,0.514,0.035,70.946,0.154
3452,188115651,DEHB01ALn0001N3v,0.042,0.043,0.515,3.996,0.161
6558,231916908,DEHB01ALn0001NNM,0.014,0.835,0.014,31.564,0.219
3478,188143685,DEHB01ALn00004Gl,0.027,0.813,0.027,54.059,0.223
8574,1306688374,DEHB01ALn0001Oxk,0.094,0.606,0.100,22.731,0.224
5366,221486875,DEHB01ALk00001qU,0.023,0.827,0.023,23.155,0.224
3470,188143674,DEHB01ALn00000Og,0.005,0.899,0.005,15.908,0.228
2700,180714547,DEHB01ALn0001Oft,0.120,0.577,0.131,22.720,0.237
1,25491909,DEHB01ALn00000Og,0.013,0.936,0.013,44.184,0.244
8475,436023962,DEHB01ALn0001Ocs,0.099,0.679,0.104,33.372,0.245



LoD2 reuse distribution:


,osm_buildings_per_lod2,lod2_ids
0,1,7939
1,2,167
2,3,47
3,4,13
4,5,2
5,11,1


In [41]:
# ============================================================
# 98 — FINAL BALANCED LoD2–OSM MATCHER
# ============================================================

from shapely import area, intersection

def match_state_lod2_to_osm(osm_state,lod2_state,state_code,min_intersection_m2=1.0,min_iou=0.25,min_primary_cover=0.70,min_secondary_cover=0.20,audit_top_n=3):
    output_crs=osm_state.crs
    osm=osm_state.reset_index(drop=True).copy()
    lod2=standardize_lod2_schema(lod2_state.reset_index(drop=True))
    osm["_osm_row"]=np.arange(len(osm),dtype=np.int64)
    lod2["_lod2_row"]=np.arange(len(lod2),dtype=np.int64)

    osm_metric=osm[["_osm_row","geometry"]].to_crs(lod2.crs)
    lod2_metric=lod2[["_lod2_row","geometry"]].copy()
    osm_metric=osm_metric[osm_metric.geometry.notna()&~osm_metric.geometry.is_empty].copy()
    lod2_metric=lod2_metric[lod2_metric.geometry.notna()&~lod2_metric.geometry.is_empty].copy()

    pairs=gpd.sjoin(osm_metric,lod2_metric,how="inner",predicate="intersects").drop(columns="index_right").reset_index(drop=True)

    output_columns={
        "lod2_id":None,"lod2_height_m":np.nan,"lod2_height_method":None,
        "lod2_footprint_method":None,"lod2_match_method":None,
        "lod2_match_quality":None,"lod2_match_pattern":None,
        "lod2_iou":np.nan,"lod2_osm_coverage":np.nan,
        "lod2_footprint_coverage":np.nan,"lod2_intersection_m2":np.nan,
        "lod2_score":np.nan,"lod2_score_margin":np.nan,
        "lod2_reuse_count":np.nan,"lod2_reused":False
    }

    if pairs.empty:
        matched=osm.drop(columns="_osm_row")
        for col,value in output_columns.items(): matched[col]=value
        matched=gpd.GeoDataFrame(matched,geometry="geometry",crs=output_crs)
        return matched,pd.DataFrame(),{
            "state_code":state_code,"osm_buildings":len(osm),"lod2_buildings":len(lod2),
            "candidate_pairs":0,"accepted_pairs":0,"matched_buildings":0,
            "unmatched_buildings":len(osm),"match_rate_pct":0
        }

    osm_geom=osm_metric.set_index("_osm_row").geometry
    lod2_geom=lod2_metric.set_index("_lod2_row").geometry
    left=pairs["_osm_row"].map(osm_geom).array
    right=pairs["_lod2_row"].map(lod2_geom).array
    intersections=intersection(left,right)

    osm_areas=osm_geom.area
    lod2_areas=lod2_geom.area
    pairs["intersection_area_m2"]=area(intersections)
    pairs["osm_area_m2"]=pairs["_osm_row"].map(osm_areas)
    pairs["lod2_area_m2"]=pairs["_lod2_row"].map(lod2_areas)
    pairs["osm_coverage"]=pairs["intersection_area_m2"]/pairs["osm_area_m2"]
    pairs["lod2_coverage"]=pairs["intersection_area_m2"]/pairs["lod2_area_m2"]
    pairs["iou"]=pairs["intersection_area_m2"]/(pairs["osm_area_m2"]+pairs["lod2_area_m2"]-pairs["intersection_area_m2"])
    pairs["score"]=0.50*pairs["iou"]+0.25*pairs["osm_coverage"]+0.25*pairs["lod2_coverage"]

    pairs["accepted"]=(pairs["intersection_area_m2"]>=min_intersection_m2)&(
        (pairs["iou"]>=min_iou)
        |((pairs["osm_coverage"]>=min_primary_cover)&(pairs["lod2_coverage"]>=min_secondary_cover))
        |((pairs["lod2_coverage"]>=min_primary_cover)&(pairs["osm_coverage"]>=min_secondary_cover))
    )

    pairs=pairs.sort_values(
        ["_osm_row","accepted","score","intersection_area_m2"],
        ascending=[True,False,False,False]
    )
    pairs["candidate_rank"]=pairs.groupby("_osm_row").cumcount()+1

    accepted=pairs[pairs["accepted"]].copy()
    accepted["accepted_rank"]=accepted.groupby("_osm_row").cumcount()+1
    best=accepted[accepted["accepted_rank"].eq(1)].copy()

    second_score=accepted[accepted["accepted_rank"].eq(2)].set_index("_osm_row")["score"]
    best["score_margin"]=best["score"]-best["_osm_row"].map(second_score).fillna(0)
    reuse_count=best["_lod2_row"].value_counts()
    best["reuse_count"]=best["_lod2_row"].map(reuse_count)

    lod2_columns=["lod2_id","measured_height_m","height_method","footprint_method"]
    attributes=lod2[["_lod2_row"]+[c for c in lod2_columns if c in lod2.columns]]
    best=best.merge(attributes,on="_lod2_row",how="left")

    best["match_pattern"]=np.select(
        [
            (best["iou"]>=0.70)&(best["osm_coverage"]>=0.80)&(best["lod2_coverage"]>=0.80),
            (best["iou"]>=0.25),
            (best["osm_coverage"]>=0.70)&(best["lod2_coverage"]>=0.20),
            (best["lod2_coverage"]>=0.70)&(best["osm_coverage"]>=0.20)
        ],
        ["strong_balanced","moderate_balanced","osm_inside_lod2","lod2_inside_osm"],
        default="borderline"
    )

    best["match_quality"]=np.select(
        [
            best["match_pattern"].eq("strong_balanced"),
            best["match_pattern"].eq("moderate_balanced")&(best["score"]>=0.50),
            best["match_pattern"].isin(["osm_inside_lod2","lod2_inside_osm"])&(best["score"]>=0.35)
        ],
        ["high","medium","medium"],
        default="low"
    )

    matched=osm.copy()
    for col,value in output_columns.items(): matched[col]=value

    target=best["_osm_row"].to_numpy()
    assignments={
        "lod2_id":"lod2_id",
        "lod2_height_m":"measured_height_m",
        "lod2_height_method":"height_method",
        "lod2_footprint_method":"footprint_method",
        "lod2_match_quality":"match_quality",
        "lod2_match_pattern":"match_pattern",
        "lod2_iou":"iou",
        "lod2_osm_coverage":"osm_coverage",
        "lod2_footprint_coverage":"lod2_coverage",
        "lod2_intersection_m2":"intersection_area_m2",
        "lod2_score":"score",
        "lod2_score_margin":"score_margin",
        "lod2_reuse_count":"reuse_count"
    }

    for output_col,source_col in assignments.items():
        matched.loc[target,output_col]=best[source_col].to_numpy()

    matched.loc[target,"lod2_match_method"]="balanced_footprint_overlap"
    matched.loc[target,"lod2_reused"]=best["reuse_count"].gt(1).to_numpy()
    matched=gpd.GeoDataFrame(matched.drop(columns="_osm_row"),geometry="geometry",crs=output_crs)

    audit=pairs[pairs["candidate_rank"].le(audit_top_n)].copy()
    audit["osm_id"]=audit["_osm_row"].map(osm.set_index("_osm_row")["id"]) if "id" in osm.columns else audit["_osm_row"]
    audit["lod2_id"]=audit["_lod2_row"].map(lod2.set_index("_lod2_row")["lod2_id"])
    chosen=set(zip(best["_osm_row"],best["_lod2_row"]))
    audit["selected"]=[(a,b) in chosen for a,b in zip(audit["_osm_row"],audit["_lod2_row"])]
    audit=audit[[
        "osm_id","lod2_id","candidate_rank","accepted","selected",
        "intersection_area_m2","osm_area_m2","lod2_area_m2",
        "osm_coverage","lod2_coverage","iou","score"
    ]]

    summary={
        "state_code":state_code,
        "osm_buildings":len(osm),
        "lod2_buildings":len(lod2),
        "candidate_pairs":len(pairs),
        "accepted_pairs":len(accepted),
        "matched_buildings":len(best),
        "unmatched_buildings":len(osm)-len(best),
        "match_rate_pct":100*len(best)/len(osm) if len(osm) else 0,
        "high_quality_matches":int(best["match_quality"].eq("high").sum()),
        "medium_quality_matches":int(best["match_quality"].eq("medium").sum()),
        "low_quality_matches":int(best["match_quality"].eq("low").sum()),
        "lod2_used":best["_lod2_row"].nunique(),
        "lod2_ids_reused":int(reuse_count.gt(1).sum()),
        "matches_using_reused_lod2":int(reuse_count[reuse_count.gt(1)].sum()),
        "maximum_lod2_reuse":int(reuse_count.max()) if len(reuse_count) else 0
    }

    return matched,audit,summary

# Confirm the final function reproduces the balanced Bremen result
hb_matched,hb_audit,hb_match_summary=match_state_lod2_to_osm(
    hb_osm_sample,hb_lod2_sample,"HB"
)

print("="*72)
print("FINAL BALANCED MATCHER — BREMEN CHECK")
print("="*72)
display(pd.DataFrame([hb_match_summary]))

print("\nMatch patterns:")
display(
    hb_matched["lod2_match_pattern"]
    .fillna("unmatched")
    .value_counts()
    .rename_axis("match_pattern")
    .reset_index(name="buildings")
)

print("\nLoD2 reuse:")
display(
    hb_matched.loc[hb_matched["lod2_id"].notna(),"lod2_reuse_count"]
    .value_counts()
    .sort_index()
    .rename_axis("osm_buildings_per_lod2")
    .reset_index(name="osm_matches")
)

FINAL BALANCED MATCHER — BREMEN CHECK


,state_code,osm_buildings,lod2_buildings,candidate_pairs,accepted_pairs,matched_buildings,unmatched_buildings,match_rate_pct,high_quality_matches,medium_quality_matches,low_quality_matches,lod2_used,lod2_ids_reused,matches_using_reused_lod2,maximum_lod2_reuse
0,HB,8598,12635,18740,8630,8383,215,97.499418,4638,3465,280,8148,193,428,4



Match patterns:


,match_pattern,buildings
0,strong_balanced,4638
1,moderate_balanced,3686
2,unmatched,215
3,osm_inside_lod2,52
4,lod2_inside_osm,7



LoD2 reuse:


,osm_buildings_per_lod2,osm_matches
0,1.0,7955
1,2.0,312
2,3.0,96
3,4.0,20


In [43]:
# ============================================================
# 99A — NATIONAL MATCHING USING LoD2 STATE EXTENTS
# ============================================================

import gc
from pathlib import Path
from shapely.geometry import box

MATCHED_DIR=BASE_DIR/"matched_to_osm"
MATCH_TABLE_DIR=MATCHED_DIR/"state_match_tables"
MATCH_AUDIT_DIR=MATCHED_DIR/"audit"
MATCH_TABLE_DIR.mkdir(parents=True,exist_ok=True)
MATCH_AUDIT_DIR.mkdir(parents=True,exist_ok=True)

GERMAN_STATES=["HB","HH","BE","SL","SH","MV","ST","TH","SN","BB","HE","RP","BW","NI","BY","NW"]
OSM_MATCH_COLUMNS=["id"]

def load_osm_for_lod2_extent(lod2_state,buffer_m=100):
    metric_crs=lod2_state.crs
    bounds=lod2_state.total_bounds
    search_geom=box(*bounds).buffer(buffer_m)

    _,_,osm_crs=geoparquet_metadata(OSM_BUILDING_FILE)
    osm_bbox=tuple(gpd.GeoSeries([search_geom],crs=metric_crs).to_crs(osm_crs).total_bounds)

    osm=read_geoparquet_bbox(
        OSM_BUILDING_FILE,
        osm_bbox,
        columns=OSM_MATCH_COLUMNS
    )

    # Remove bbox false positives after reprojection
    osm_metric=osm.to_crs(metric_crs)
    keep=osm_metric.geometry.intersects(search_geom)
    return osm.loc[keep].copy()

def run_national_lod2_matching(states=GERMAN_STATES,overwrite=False):
    summaries=[]

    for state_code in tqdm(states,desc="Matching LoD2 states"):
        match_path=MATCH_TABLE_DIR/f"lod2_matches_{state_code}.parquet"
        audit_path=MATCH_AUDIT_DIR/f"lod2_match_audit_{state_code}.parquet"

        if match_path.exists() and audit_path.exists() and not overwrite:
            existing=pd.read_parquet(match_path)
            summaries.append({
                "state_code":state_code,
                "status":"skipped_complete",
                "matched_buildings":len(existing),
                "match_path":str(match_path)
            })
            continue

        try:
            lod2_state=load_state_lod2(state_code)
            osm_state=load_osm_for_lod2_extent(lod2_state,buffer_m=100)

            matched,audit,summary=match_state_lod2_to_osm(
                osm_state=osm_state,
                lod2_state=lod2_state,
                state_code=state_code
            )

            keep_cols=[
                "id","lod2_id","lod2_height_m","lod2_height_method",
                "lod2_footprint_method","lod2_match_method",
                "lod2_match_quality","lod2_match_pattern",
                "lod2_iou","lod2_osm_coverage",
                "lod2_footprint_coverage","lod2_intersection_m2",
                "lod2_score","lod2_score_margin",
                "lod2_reuse_count","lod2_reused"
            ]

            match_table=matched.loc[
                matched["lod2_id"].notna(),
                [c for c in keep_cols if c in matched.columns]
            ].copy()

            match_table["lod2_state_code"]=state_code
            match_table.to_parquet(match_path,index=False)
            audit.to_parquet(audit_path,index=False)

            summary.update({
                "status":"complete",
                "osm_bbox_candidates":len(osm_state),
                "saved_matches":len(match_table),
                "match_path":str(match_path),
                "audit_path":str(audit_path)
            })
            summaries.append(summary)

        except Exception as e:
            summaries.append({
                "state_code":state_code,
                "status":"failed",
                "error":f"{type(e).__name__}: {e}"
            })

        finally:
            for name in ["lod2_state","osm_state","matched","audit","match_table"]:
                if name in locals(): del locals()[name]
            gc.collect()

        pd.DataFrame(summaries).to_csv(
            MATCH_AUDIT_DIR/"national_matching_progress.csv",
            index=False
        )

    summary_df=pd.DataFrame(summaries)
    summary_df.to_csv(
        MATCH_AUDIT_DIR/"national_matching_summary.csv",
        index=False
    )
    return summary_df

print("="*76)
print("NATIONAL LoD2 MATCHING RUNNER READY")
print("="*76)
print("\nThe OSM dataset does not need a state column.")
print("Each state is selected spatially from its LoD2 extent.")
print("Only matched OSM IDs and LoD2 attributes will be saved.")
print("\nRun initially with Bremen only:")
print('hb_run = run_national_lod2_matching(states=["HB"])')

NATIONAL LoD2 MATCHING RUNNER READY

The OSM dataset does not need a state column.
Each state is selected spatially from its LoD2 extent.
Only matched OSM IDs and LoD2 attributes will be saved.

Run initially with Bremen only:
hb_run = run_national_lod2_matching(states=["HB"])


In [44]:
hb_run = run_national_lod2_matching(states=["HB"])

Loading HB LoD2: 100%|█████████████████████████████████████████| 1/1 [00:00<00:00,  1.25it/s]


HB FINAL LoD2 DATASET

Files loaded     : 1
Buildings loaded : 331,804
Buildings expected: 331,804
Duplicate IDs    : 0
Missing heights  : 0
Missing geometry : 0
CRS              : 25832

Files:
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/HB/lod2_buildings_HB.parquet


Matching LoD2 states: 100%|████████████████████████████████████| 1/1 [01:30<00:00, 90.11s/it]


In [45]:
# ============================================================
# 99B — VALIDATE COMPLETE BREMEN MATCHING RESULT
# ============================================================

hb_match_path=MATCH_TABLE_DIR/"lod2_matches_HB.parquet"
hb_audit_path=MATCH_AUDIT_DIR/"lod2_match_audit_HB.parquet"
hb_progress_path=MATCH_AUDIT_DIR/"national_matching_progress.csv"

if not hb_match_path.exists(): raise FileNotFoundError(hb_match_path)
if not hb_audit_path.exists(): raise FileNotFoundError(hb_audit_path)

hb_matches=pd.read_parquet(hb_match_path)
hb_audit_full=pd.read_parquet(hb_audit_path)
hb_progress=pd.read_csv(hb_progress_path)

duplicate_osm=int(hb_matches["id"].duplicated().sum())
duplicate_pairs=int(hb_matches.duplicated(["id","lod2_id"]).sum())
missing_heights=int(hb_matches["lod2_height_m"].isna().sum())
invalid_heights=int((pd.to_numeric(hb_matches["lod2_height_m"],errors="coerce")<=0).sum())

reuse=hb_matches["lod2_id"].value_counts()
quality=hb_matches["lod2_match_quality"].value_counts(dropna=False).rename_axis("quality").reset_index(name="matches")
patterns=hb_matches["lod2_match_pattern"].value_counts(dropna=False).rename_axis("pattern").reset_index(name="matches")

validation=pd.DataFrame([{
    "state_code":"HB",
    "saved_matches":len(hb_matches),
    "unique_osm_ids":hb_matches["id"].nunique(),
    "unique_lod2_ids":hb_matches["lod2_id"].nunique(),
    "duplicate_osm_ids":duplicate_osm,
    "duplicate_id_lod2_pairs":duplicate_pairs,
    "missing_heights":missing_heights,
    "nonpositive_heights":invalid_heights,
    "reused_lod2_ids":int(reuse.gt(1).sum()),
    "matches_using_reused_lod2":int(reuse[reuse.gt(1)].sum()),
    "maximum_lod2_reuse":int(reuse.max()),
    "minimum_iou":hb_matches["lod2_iou"].min(),
    "median_iou":hb_matches["lod2_iou"].median(),
    "minimum_score":hb_matches["lod2_score"].min(),
    "median_score":hb_matches["lod2_score"].median(),
    "validation_passed":duplicate_osm==0 and duplicate_pairs==0 and missing_heights==0
}])

print("="*76)
print("BREMEN COMPLETE-STATE MATCH VALIDATION")
print("="*76)

print("\nRunner summary:")
display(hb_progress[hb_progress["state_code"].eq("HB")])

print("\nSaved-match validation:")
display(validation.round(3))

print("\nMatch quality:")
display(quality)

print("\nMatch patterns:")
display(patterns)

print("\nOverlap statistics:")
display(
    hb_matches[
        ["lod2_iou","lod2_osm_coverage","lod2_footprint_coverage",
         "lod2_intersection_m2","lod2_score","lod2_score_margin"]
    ].describe().round(3)
)

print("\nVALIDATION PASSED — ready for all states." if validation.iloc[0]["validation_passed"] else "\nVALIDATION FAILED — inspect the reported differences.")

BREMEN COMPLETE-STATE MATCH VALIDATION

Runner summary:


,state_code,osm_buildings,lod2_buildings,candidate_pairs,accepted_pairs,matched_buildings,unmatched_buildings,match_rate_pct,high_quality_matches,medium_quality_matches,low_quality_matches,lod2_used,lod2_ids_reused,matches_using_reused_lod2,maximum_lod2_reuse,status,osm_bbox_candidates,saved_matches,match_path,audit_path
0,HB,341527,331804,366883,191171,180588,160939,52.876639,90096,82310,8182,173465,5336,12459,4,complete,341527,180588,/fast/home/o-olajuyigbe/data/germany_lod2/matc...,/fast/home/o-olajuyigbe/data/germany_lod2/matc...



Saved-match validation:


,state_code,saved_matches,unique_osm_ids,unique_lod2_ids,duplicate_osm_ids,duplicate_id_lod2_pairs,missing_heights,nonpositive_heights,reused_lod2_ids,matches_using_reused_lod2,maximum_lod2_reuse,minimum_iou,median_iou,minimum_score,median_score,validation_passed
0,HB,180588,180588,173465,0,0,0,0,5336,12459,4,0.185,0.746,0.318,0.803,True



Match quality:


...


Match patterns:


,pattern,matches
0,strong_balanced,90096
1,moderate_balanced,87762
2,osm_inside_lod2,2153
3,lod2_inside_osm,577



Overlap statistics:


,lod2_iou,lod2_osm_coverage,lod2_footprint_coverage,lod2_intersection_m2,lod2_score,lod2_score_margin
count,180588.000,180588.000,180588.000,180588.000,180588.000,180588.000
mean,0.704,0.817,0.849,161.306,0.769,0.744
std,0.166,0.135,0.168,992.633,0.133,0.194
min,0.185,0.200,0.200,1.109,0.318,0.000
25%,0.623,0.770,0.804,47.758,0.702,0.698
50%,0.746,0.849,0.905,70.726,0.803,0.803
75%,0.825,0.907,0.962,120.001,0.866,0.866
max,0.996,1.000,1.000,124491.213,0.997,0.997



VALIDATION PASSED — ready for all states.


In [47]:
# ============================================================
# 100 — NATIONAL LoD2–OSM SPATIAL MATCHING
# ============================================================

import gc, shapely
from pathlib import Path

COMMON_CRS="EPSG:3035"
FINAL_DIR=BASE_DIR/"final"
FINAL_DIR.mkdir(parents=True,exist_ok=True)

NATIONAL_LOD2_PATH=FINAL_DIR/"germany_lod2_buildings.parquet"
NATIONAL_MATCH_PATH=FINAL_DIR/"germany_osm_lod2_matches.parquet"
NATIONAL_MATCH_SUMMARY=REPORT_DIR/"national_osm_lod2_match_summary.csv"

STATES=["BB","BE","BW","BY","HB","HE","HH","MV","NI","NW","RP","SH","SL","SN","ST","TH"]
LOD2_COLUMNS=["lod2_id","measured_height_m","height_method","footprint_method","geometry"]

# ── 1. Combine all LoD2 states ───────────────────────────────────────────────
lod2_frames=[]

for state in tqdm(STATES,desc="Loading national LoD2"):
    gdf=load_state_lod2(state)
    gdf=standardize_lod2_schema(gdf)
    missing=[c for c in LOD2_COLUMNS if c not in gdf.columns]
    if missing: raise KeyError(f"{state} missing columns: {missing}")

    gdf=gdf[LOD2_COLUMNS].copy()
    gdf["lod2_state_code"]=state
    gdf=gdf[gdf.geometry.notna()&~gdf.geometry.is_empty].to_crs(COMMON_CRS)
    lod2_frames.append(gdf)
    del gdf
    gc.collect()

lod2=gpd.GeoDataFrame(pd.concat(lod2_frames,ignore_index=True),geometry="geometry",crs=COMMON_CRS)
del lod2_frames
gc.collect()

lod2["_lod2_row"]=np.arange(len(lod2),dtype=np.int64)
lod2["lod2_area_m2"]=shapely.area(lod2.geometry.array)

valid_lod2=np.isfinite(lod2["lod2_area_m2"])&lod2["lod2_area_m2"].gt(0)
lod2=lod2.loc[valid_lod2].reset_index(drop=True)
lod2["_lod2_row"]=np.arange(len(lod2),dtype=np.int64)

lod2.to_parquet(NATIONAL_LOD2_PATH,index=False)

print("="*76)
print("NATIONAL LoD2 DATASET")
print("="*76)
print(f"\nBuildings : {len(lod2):,}")
print(f"States    : {lod2['lod2_state_code'].nunique():,}")
print(f"CRS       : {lod2.crs.to_epsg()}")
print(f"Saved to  : {NATIONAL_LOD2_PATH}")

# ── 2. Load all OSM building footprints ──────────────────────────────────────
osm=gpd.read_parquet(OSM_BUILDING_FILE,columns=["id","geometry"])
osm=osm[osm.geometry.notna()&~osm.geometry.is_empty].to_crs(COMMON_CRS).reset_index(drop=True)
osm["_osm_row"]=np.arange(len(osm),dtype=np.int64)
osm["osm_area_m2"]=shapely.area(osm.geometry.array)

valid_osm=np.isfinite(osm["osm_area_m2"])&osm["osm_area_m2"].gt(0)
osm=osm.loc[valid_osm].reset_index(drop=True)
osm["_osm_row"]=np.arange(len(osm),dtype=np.int64)

print("\n"+"="*76)
print("NATIONAL OSM DATASET")
print("="*76)
print(f"\nBuildings : {len(osm):,}")
print(f"CRS       : {osm.crs.to_epsg()}")
print(f"ID duplicates: {osm['id'].duplicated().sum():,}")

# ── 3. Find every intersecting OSM–LoD2 candidate pair ──────────────────────
pairs=gpd.sjoin(
    osm[["_osm_row","geometry"]],
    lod2[["_lod2_row","geometry"]],
    how="inner",
    predicate="intersects"
).drop(columns="index_right").reset_index(drop=True)

print("\n"+"="*76)
print("NATIONAL SPATIAL JOIN")
print("="*76)
print(f"\nCandidate pairs: {len(pairs):,}")

# ── 4. Calculate exact overlap metrics ───────────────────────────────────────
osm_index=pairs["_osm_row"].to_numpy(dtype=np.int64)
lod2_index=pairs["_lod2_row"].to_numpy(dtype=np.int64)

osm_geometries=osm.geometry.array.take(osm_index)
lod2_geometries=lod2.geometry.array.take(lod2_index)
intersection_geometries=shapely.intersection(osm_geometries,lod2_geometries)

pairs["intersection_area_m2"]=shapely.area(intersection_geometries)
pairs["osm_area_m2"]=osm["osm_area_m2"].to_numpy()[osm_index]
pairs["lod2_area_m2"]=lod2["lod2_area_m2"].to_numpy()[lod2_index]

pairs["osm_coverage"]=pairs["intersection_area_m2"]/pairs["osm_area_m2"]
pairs["lod2_coverage"]=pairs["intersection_area_m2"]/pairs["lod2_area_m2"]
pairs["iou"]=pairs["intersection_area_m2"]/(
    pairs["osm_area_m2"]+
    pairs["lod2_area_m2"]-
    pairs["intersection_area_m2"]
)

pairs["score"]=(
    0.50*pairs["iou"]+
    0.25*pairs["osm_coverage"]+
    0.25*pairs["lod2_coverage"]
)

# Balanced rule validated using Bremen
pairs["accepted"]=(
    pairs["intersection_area_m2"].ge(1.0)
    &(
        pairs["iou"].ge(0.25)
        |(pairs["osm_coverage"].ge(0.70)&pairs["lod2_coverage"].ge(0.20))
        |(pairs["lod2_coverage"].ge(0.70)&pairs["osm_coverage"].ge(0.20))
    )
)

# ── 5. Select best LoD2 footprint for each OSM building ─────────────────────
accepted=pairs[pairs["accepted"]].sort_values(
    ["_osm_row","score","intersection_area_m2"],
    ascending=[True,False,False]
).copy()

accepted["candidate_rank"]=accepted.groupby("_osm_row").cumcount()+1
best=accepted[accepted["candidate_rank"].eq(1)].copy()

second_scores=accepted[accepted["candidate_rank"].eq(2)].set_index("_osm_row")["score"]
best["score_margin"]=best["score"]-best["_osm_row"].map(second_scores).fillna(0)

reuse_counts=best["_lod2_row"].value_counts()
best["lod2_reuse_count"]=best["_lod2_row"].map(reuse_counts)
best["lod2_reused"]=best["lod2_reuse_count"].gt(1)

best["match_pattern"]=np.select(
    [
        best["iou"].ge(0.70)&best["osm_coverage"].ge(0.80)&best["lod2_coverage"].ge(0.80),
        best["iou"].ge(0.25),
        best["osm_coverage"].ge(0.70)&best["lod2_coverage"].ge(0.20),
        best["lod2_coverage"].ge(0.70)&best["osm_coverage"].ge(0.20)
    ],
    ["strong_balanced","moderate_balanced","osm_inside_lod2","lod2_inside_osm"],
    default="borderline"
)

best["match_quality"]=np.select(
    [
        best["match_pattern"].eq("strong_balanced"),
        best["match_pattern"].eq("moderate_balanced")&best["score"].ge(0.50),
        best["match_pattern"].isin(["osm_inside_lod2","lod2_inside_osm"])&best["score"].ge(0.35)
    ],
    ["high","medium","medium"],
    default="low"
)

# ── 6. Attach IDs, heights and source attributes ─────────────────────────────
match_table=best.merge(
    osm[["_osm_row","id"]],
    on="_osm_row",
    how="left"
).merge(
    lod2[
        [
            "_lod2_row","lod2_id","lod2_state_code",
            "measured_height_m","height_method","footprint_method"
        ]
    ],
    on="_lod2_row",
    how="left"
)

match_table=match_table.rename(columns={
    "id":"osm_id",
    "measured_height_m":"lod2_height_m",
    "height_method":"lod2_height_method",
    "footprint_method":"lod2_footprint_method",
    "intersection_area_m2":"lod2_intersection_m2",
    "osm_coverage":"lod2_osm_coverage",
    "lod2_coverage":"lod2_footprint_coverage",
    "iou":"lod2_iou",
    "score":"lod2_score",
    "score_margin":"lod2_score_margin",
    "match_pattern":"lod2_match_pattern",
    "match_quality":"lod2_match_quality"
})

match_table["lod2_match_method"]="balanced_footprint_overlap"

keep_columns=[
    "_osm_row","osm_id","lod2_id","lod2_state_code",
    "lod2_height_m","lod2_height_method","lod2_footprint_method",
    "lod2_match_method","lod2_match_quality","lod2_match_pattern",
    "lod2_iou","lod2_osm_coverage","lod2_footprint_coverage",
    "lod2_intersection_m2","lod2_score","lod2_score_margin",
    "lod2_reuse_count","lod2_reused"
]

match_table=match_table[keep_columns].sort_values("_osm_row").reset_index(drop=True)
match_table.to_parquet(NATIONAL_MATCH_PATH,index=False)

# ── 7. National validation summary ───────────────────────────────────────────
summary=pd.DataFrame([{
    "osm_buildings":len(osm),
    "lod2_buildings":len(lod2),
    "candidate_pairs":len(pairs),
    "accepted_pairs":len(accepted),
    "matched_osm_buildings":len(match_table),
    "unmatched_osm_buildings":len(osm)-len(match_table),
    "osm_match_rate_pct":100*len(match_table)/len(osm),
    "unique_lod2_used":match_table["lod2_id"].nunique(),
    "high_quality_matches":int(match_table["lod2_match_quality"].eq("high").sum()),
    "medium_quality_matches":int(match_table["lod2_match_quality"].eq("medium").sum()),
    "low_quality_matches":int(match_table["lod2_match_quality"].eq("low").sum()),
    "duplicate_osm_rows":int(match_table["_osm_row"].duplicated().sum()),
    "missing_heights":int(match_table["lod2_height_m"].isna().sum()),
    "reused_lod2_ids":int(reuse_counts.gt(1).sum()),
    "maximum_lod2_reuse":int(reuse_counts.max()) if len(reuse_counts) else 0,
    "minimum_iou":match_table["lod2_iou"].min(),
    "median_iou":match_table["lod2_iou"].median(),
    "match_output":str(NATIONAL_MATCH_PATH)
}])

summary.to_csv(NATIONAL_MATCH_SUMMARY,index=False)

print("\n"+"="*76)
print("NATIONAL LoD2–OSM MATCHING RESULT")
print("="*76)
display(summary)

print("\nMatch quality:")
display(
    match_table["lod2_match_quality"]
    .value_counts()
    .rename_axis("quality")
    .reset_index(name="buildings")
)

print("\nMatch patterns:")
display(
    match_table["lod2_match_pattern"]
    .value_counts()
    .rename_axis("pattern")
    .reset_index(name="buildings")
)

print(f"\nNational LoD2 dataset : {NATIONAL_LOD2_PATH}")
print(f"National match table  : {NATIONAL_MATCH_PATH}")

Loading BB LoD2: 100%|███████████████████████████████████████| 78/78 [00:09<00:00,  8.27it/s]


BB FINAL LoD2 DATASET

Files loaded     : 78
Buildings loaded : 2,376,794
Buildings expected: 2,376,794
Duplicate IDs    : 0
Missing heights  : 0
Missing geometry : 45,294
CRS              : 25833

Files:
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0001.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0002.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0003.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0004.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0005.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0006.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0007.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0008.parquet
  /

Loading national LoD2:   6%|██▏                               | 1/16 [00:22<05:39, 22.64s/it]


FileNotFoundError: No final LoD2 Parquet files found for BE

In [48]:
# ============================================================
# 100A — ROBUST LoD2 FILE DISCOVERY BY VALIDATED ROW COUNT
# ============================================================

import gc
import pyarrow.parquet as pq
from pathlib import Path

BAD_TOKENS=("test","sample","temp","tmp","audit","matched","backup")

def state_expected_buildings(state_code):
    path=REPORT_DIR/f"lod2_extraction_summary_{state_code}.csv"
    if not path.exists(): return None
    df=pd.read_csv(path)
    return int(df.iloc[0]["buildings"]) if not df.empty else None

def parquet_metadata(path):
    try:
        pf=pq.ParquetFile(path)
        return {
            "path":Path(path),
            "rows":pf.metadata.num_rows,
            "columns":set(pf.schema_arrow.names),
            "size_gb":Path(path).stat().st_size/1024**3
        }
    except Exception:
        return None

def find_state_lod2_paths(state_code,verbose=False):
    state_code=state_code.upper()
    expected=state_expected_buildings(state_code)
    summary_path=REPORT_DIR/f"lod2_extraction_summary_{state_code}.csv"
    roots=[EXTRACTED_DIR/state_code]

    if summary_path.exists():
        summary=pd.read_csv(summary_path)
        if not summary.empty and "output_directory" in summary.columns:
            output=Path(str(summary.iloc[0]["output_directory"]))
            roots.extend([output,output/"parts"])

    files=sorted({
        p.resolve()
        for root in roots if root.exists()
        for p in (root.rglob("*.parquet") if root.is_dir() else [root])
        if p.is_file() and not any(x in p.stem.lower() for x in BAD_TOKENS)
    })

    info=[x for x in (parquet_metadata(p) for p in files) if x is not None]
    usable=[x for x in info if "geometry" in x["columns"] and any(c in x["columns"] for c in ["lod2_id","gml_id","id","building_id"])]

    if not usable:
        raise FileNotFoundError(f"No usable LoD2 Parquet files found for {state_code} under {roots}")

    canonical=(EXTRACTED_DIR/state_code/f"lod2_buildings_{state_code}.parquet").resolve()
    if canonical.exists() and any(x["path"]==canonical for x in usable): return [canonical]

    # Prefer a complete parts directory whose combined metadata equals the validated total
    groups={}
    for item in usable: groups.setdefault(item["path"].parent,[]).append(item)

    matching_groups=[
        sorted(items,key=lambda x:x["path"].name)
        for items in groups.values()
        if len(items)>1 and expected is not None and sum(x["rows"] for x in items)==expected
    ]
    if matching_groups:
        matching_groups.sort(key=lambda g:(g[0]["path"].parent.name!="parts",len(g)))
        return [x["path"] for x in matching_groups[0]]

    # Otherwise choose one complete final file
    singles=[x for x in usable if expected is not None and x["rows"]==expected]
    if singles:
        def score(item):
            name=item["path"].stem.lower()
            return (
                100 if name==f"lod2_buildings_{state_code.lower()}" else 0,
                30 if "final" in name or "clean" in name else 0,
                20 if state_code.lower() in name else 0,
                -100 if "raw" in name else 0,
                -len(item["path"].parts)
            )
        return [max(singles,key=score)["path"]]

    inventory=pd.DataFrame([{
        "path":str(x["path"]),
        "rows":x["rows"],
        "size_gb":x["size_gb"],
        "has_lod2_id":"lod2_id" in x["columns"],
        "has_geometry":"geometry" in x["columns"]
    } for x in usable]).sort_values("rows",ascending=False)

    if verbose: display(inventory)
    raise FileNotFoundError(
        f"{state_code}: no file or file group matches the validated total "
        f"of {expected:,} buildings. Run find_state_lod2_paths('{state_code}',verbose=True)."
    )

def load_state_lod2(state_code,columns=None,validate=True):
    paths=find_state_lod2_paths(state_code)
    frames=[gpd.read_parquet(path,columns=columns) for path in tqdm(paths,desc=f"Loading {state_code} LoD2")]
    crs_values={str(x.crs) for x in frames}
    if len(crs_values)!=1: raise ValueError(f"{state_code} has inconsistent CRS values: {crs_values}")

    gdf=gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry="geometry",crs=frames[0].crs)
    gdf=standardize_lod2_schema(gdf)
    expected=state_expected_buildings(state_code)

    if validate:
        if expected is not None and len(gdf)!=expected: raise ValueError(f"{state_code}: loaded {len(gdf):,}, expected {expected:,}")
        if gdf["lod2_id"].duplicated().any(): raise ValueError(f"{state_code}: {gdf['lod2_id'].duplicated().sum():,} duplicate IDs")

    print("="*72)
    print(f"{state_code} FINAL LoD2 DATASET")
    print("="*72)
    print(f"\nFiles loaded      : {len(paths):,}")
    print(f"Buildings loaded  : {len(gdf):,}")
    print(f"Buildings expected: {expected:,}" if expected is not None else "Buildings expected: unavailable")
    print(f"Duplicate IDs     : {gdf['lod2_id'].duplicated().sum():,}")
    print(f"Missing heights   : {gdf['measured_height_m'].isna().sum():,}")
    print(f"Missing geometry  : {(gdf.geometry.isna()|gdf.geometry.is_empty).sum():,}")
    print(f"CRS               : {gdf.crs.to_epsg() or gdf.crs.name}")
    print("\nFiles:")
    for path in paths[:10]: print(f"  {path}")
    if len(paths)>10: print(f"  ... and {len(paths)-10:,} more")

    del frames
    gc.collect()
    return gdf

# Confirm Berlin resolves correctly before rerunning Cell 100
be_lod2=load_state_lod2("BE")

FileNotFoundError: BE: no file or file group matches the validated total of 639,658 buildings. Run find_state_lod2_paths('BE',verbose=True).

In [49]:
# ============================================================
# 100B — DIAGNOSE BERLIN LoD2 FILES AND VALIDATION REPORT
# ============================================================

import pyarrow.parquet as pq
from pathlib import Path

state="BE"
state_dir=EXTRACTED_DIR/state
summary_path=REPORT_DIR/f"lod2_extraction_summary_{state}.csv"

print("="*80)
print("BERLIN LoD2 FILE DIAGNOSIS")
print("="*80)

if summary_path.exists():
    be_summary=pd.read_csv(summary_path)
    print("\nCurrent validation report:")
    display(be_summary.T)
else:
    be_summary=pd.DataFrame()
    print(f"\nMissing validation report: {summary_path}")

rows=[]
for path in sorted(state_dir.rglob("*.parquet")):
    try:
        pf=pq.ParquetFile(path)
        cols=pf.schema_arrow.names
        rows.append({
            "path":str(path),
            "parent":path.parent.name,
            "filename":path.name,
            "rows":pf.metadata.num_rows,
            "size_mb":path.stat().st_size/1024**2,
            "has_geometry":"geometry" in cols,
            "has_lod2_id":"lod2_id" in cols,
            "has_gml_id":"gml_id" in cols,
            "columns":", ".join(cols)
        })
    except Exception as e:
        rows.append({
            "path":str(path),"parent":path.parent.name,"filename":path.name,
            "rows":np.nan,"size_mb":np.nan,"has_geometry":False,
            "has_lod2_id":False,"has_gml_id":False,"columns":f"ERROR: {e}"
        })

be_inventory=pd.DataFrame(rows).sort_values(["parent","filename"]).reset_index(drop=True)

print(f"\nParquet files found: {len(be_inventory):,}")
display(be_inventory[[
    "parent","filename","rows","size_mb",
    "has_geometry","has_lod2_id","has_gml_id","path"
]])

print("\nRows grouped by directory:")
display(
    be_inventory.groupby("parent",dropna=False)
    .agg(files=("filename","size"),rows=("rows","sum"),size_mb=("size_mb","sum"))
    .reset_index()
    .sort_values("rows",ascending=False)
)

print("\nPossible complete files:")
display(
    be_inventory[
        be_inventory["has_geometry"]
        &(be_inventory["has_lod2_id"]|be_inventory["has_gml_id"])
    ][["filename","rows","size_mb","path"]].sort_values("rows",ascending=False)
)

BERLIN LoD2 FILE DIAGNOSIS

Current validation report:


,0
state_code,BE
state_name,Berlin
gml_files,925
successful_files,922
failed_files,3
buildings,639658
unique_lod2_ids,639658
valid_geometries,636938
measured_heights,639658
missing_heights,0



Parquet files found: 1


,parent,filename,rows,size_mb,has_geometry,has_lod2_id,has_gml_id,path
0,BE,lod2_buildings_BE_raw.parquet,641813,79.327885,True,True,False,/fast/home/o-olajuyigbe/data/germany_lod2/extr...



Rows grouped by directory:


,parent,files,rows,size_mb
0,BE,1,641813,79.327885



Possible complete files:


,filename,rows,size_mb,path
0,lod2_buildings_BE_raw.parquet,641813,79.327885,/fast/home/o-olajuyigbe/data/germany_lod2/extr...


In [50]:
# ============================================================
# 100C — CLEAN BERLIN RAW LoD2 AND CREATE CANONICAL FILE
# ============================================================

BE_RAW=EXTRACTED_DIR/"BE"/"lod2_buildings_BE_raw.parquet"
BE_FINAL=EXTRACTED_DIR/"BE"/"lod2_buildings_BE.parquet"
BE_DUPLICATES=REPORT_DIR/"lod2_duplicate_rows_BE.parquet"
BE_SUMMARY=REPORT_DIR/"lod2_extraction_summary_BE.csv"

be=gpd.read_parquet(BE_RAW)
summary=pd.read_csv(BE_SUMMARY)
expected=int(summary.iloc[0]["buildings"])

be["_usable_geometry"]=be.geometry.notna()&~be.geometry.is_empty
be["_height_available"]=be["measured_height_m"].notna()
duplicate_mask=be["lod2_id"].duplicated(False)
duplicate_rows=be.loc[duplicate_mask].copy()
duplicate_rows.to_parquet(BE_DUPLICATES,index=False)

height_conflicts=int(
    duplicate_rows.groupby("lod2_id")["measured_height_m"]
    .nunique(dropna=True).gt(1).sum()
) if not duplicate_rows.empty else 0

be_clean=(
    be.sort_values(
        ["lod2_id","_usable_geometry","_height_available"],
        ascending=[True,False,False],
        kind="mergesort"
    )
    .drop_duplicates("lod2_id",keep="first")
    .drop(columns=["_usable_geometry","_height_available"])
    .reset_index(drop=True)
)

raw_rows=len(be)
duplicate_rows_removed=raw_rows-len(be_clean)
usable_geometry=be_clean.geometry.notna()&~be_clean.geometry.is_empty

if len(be_clean)!=expected: raise ValueError(f"Cleaned rows {len(be_clean):,} != validated total {expected:,}")
if be_clean["lod2_id"].duplicated().any(): raise ValueError("Duplicate Berlin LoD2 IDs remain.")
if be_clean["measured_height_m"].isna().any(): raise ValueError("Missing Berlin heights remain.")

be_clean.to_parquet(BE_FINAL,index=False)

summary.loc[0,"raw_rows"]=raw_rows
summary.loc[0,"duplicate_rows_removed"]=duplicate_rows_removed
summary.loc[0,"buildings"]=len(be_clean)
summary.loc[0,"unique_lod2_ids"]=be_clean["lod2_id"].nunique()
summary.loc[0,"valid_geometries"]=int(usable_geometry.sum())
summary.loc[0,"missing_geometries"]=int((~usable_geometry).sum())
summary.loc[0,"output_file"]=str(BE_FINAL)
summary.to_csv(BE_SUMMARY,index=False)

print("="*72)
print("BERLIN CLEANED LoD2 VALIDATION")
print("="*72)
print(f"\nRaw rows                 : {raw_rows:,}")
print(f"Unique LoD2 IDs          : {be['lod2_id'].nunique():,}")
print(f"Duplicate rows removed   : {duplicate_rows_removed:,}")
print(f"Duplicate height conflicts: {height_conflicts:,}")
print(f"Final buildings          : {len(be_clean):,}")
print(f"Usable geometries        : {usable_geometry.sum():,}")
print(f"Missing geometries       : {(~usable_geometry).sum():,}")
print(f"Missing heights          : {be_clean['measured_height_m'].isna().sum():,}")
print(f"Output                    : {BE_FINAL}")
print(f"Duplicate audit           : {BE_DUPLICATES}")

ValueError: Cleaned rows 641,813 != validated total 639,658

In [51]:
# ============================================================
# 100D — DIAGNOSE BERLIN RAW/REPORT COUNT DIFFERENCE
# ============================================================

from pathlib import Path

BE_RAW=EXTRACTED_DIR/"BE"/"lod2_buildings_BE_raw.parquet"
be=gpd.read_parquet(BE_RAW)
expected=int(pd.read_csv(REPORT_DIR/"lod2_extraction_summary_BE.csv").iloc[0]["buildings"])

def norm_source(x):
    if pd.isna(x): return None
    return Path(str(x)).name.lower()

print("="*76)
print("BERLIN RAW/REPORT COUNT DIAGNOSIS")
print("="*76)
print(f"\nRaw rows                 : {len(be):,}")
print(f"Validated report rows    : {expected:,}")
print(f"Difference               : {len(be)-expected:,}")
print(f"Unique LoD2 IDs          : {be['lod2_id'].nunique(dropna=True):,}")
print(f"Duplicate LoD2 IDs       : {be['lod2_id'].duplicated().sum():,}")
print(f"Missing/blank IDs        : {(be['lod2_id'].isna()|be['lod2_id'].astype('string').str.strip().eq('')).sum():,}")
print(f"Missing geometries       : {(be.geometry.isna()|be.geometry.is_empty).sum():,}")
print(f"Invalid geometries       : {(be.geometry.notna()&~be.geometry.is_empty&~be.geometry.is_valid).sum():,}")
print(f"Missing heights          : {be['measured_height_m'].isna().sum():,}")
print(f"Unique source files      : {be['source_file'].nunique() if 'source_file' in be else 'unavailable'}")

# Search all Berlin-related CSV reports/manifests
csv_paths=sorted({
    *REPORT_DIR.glob("*BE*.csv"),
    *(EXTRACTED_DIR/"BE").rglob("*.csv")
})

report_inventory=[]
failed_sources=set()
successful_sources=set()

for path in csv_paths:
    try:
        df=pd.read_csv(path)
    except Exception:
        continue

    report_inventory.append({
        "file":path.name,
        "rows":len(df),
        "columns":", ".join(df.columns)
    })

    source_col=next((c for c in ["source_file","filename","file","source_path","gml_file"] if c in df.columns),None)
    status_col=next((c for c in ["status","processing_status","result"] if c in df.columns),None)

    if source_col and status_col:
        status=df[status_col].astype("string").str.lower()
        sources=df[source_col].map(norm_source)
        failed_sources.update(sources[status.str.contains("fail|error",na=False)].dropna())
        successful_sources.update(sources[status.str.contains("success|complete|processed",na=False)].dropna())

print("\nBerlin report/manifest files:")
display(pd.DataFrame(report_inventory))

if "source_file" in be.columns:
    be["_source_norm"]=be["source_file"].map(norm_source)
    failed_rows=be[be["_source_norm"].isin(failed_sources)].copy()
    non_success_rows=be[~be["_source_norm"].isin(successful_sources)].copy() if successful_sources else pd.DataFrame()

    print("\nManifest comparison:")
    print(f"Failed sources identified       : {len(failed_sources):,}")
    print(f"Successful sources identified   : {len(successful_sources):,}")
    print(f"Rows from failed sources        : {len(failed_rows):,}")
    print(f"Rows outside successful sources : {len(non_success_rows):,}" if successful_sources else "No successful-source manifest detected.")

    if failed_sources:
        display(
            be[be["_source_norm"].isin(failed_sources)]
            .groupby(["_source_norm"],dropna=False)
            .agg(rows=("lod2_id","size"),unique_ids=("lod2_id","nunique"),missing_geometry=("geometry",lambda s:s.isna().sum()))
            .reset_index()
        )

    if len(failed_rows)==len(be)-expected:
        print("\nCAUSE CONFIRMED — the 2,155 extra records came from the three failed/partial GML files.")
    else:
        print("\nCause not yet confirmed. Inspect the manifest inventory and source-file counts above.")

print("\nLargest source files in the raw dataset:")
if "source_file" in be.columns:
    display(
        be.groupby("_source_norm",dropna=False)
        .size()
        .sort_values(ascending=False)
        .head(20)
        .rename("buildings")
        .reset_index()
    )

BERLIN RAW/REPORT COUNT DIAGNOSIS

Raw rows                 : 641,813
Validated report rows    : 639,658
Difference               : 2,155
Unique LoD2 IDs          : 641,813
Duplicate LoD2 IDs       : 0
Missing/blank IDs        : 0
Missing geometries       : 247
Invalid geometries       : 3
Missing heights          : 0
Unique source files      : 924

Berlin report/manifest files:


,file,rows,columns
0,lod2_download_errors_BE.csv,1,"filename, download_url, error"
1,lod2_extraction_summary_BE.csv,1,"state_code, state_name, gml_files, successful_..."
2,lod2_failed_files_BE.csv,3,"state_code, source_file, error"



Manifest comparison:
Failed sources identified       : 0
Successful sources identified   : 0
Rows from failed sources        : 0
No successful-source manifest detected.

Cause not yet confirmed. Inspect the manifest inventory and source-file counts above.

Largest source files in the raw dataset:


,_source_norm,buildings
0,lod2_33_380_5831_1_be.xml,2514
1,lod2_33_396_5810_1_be.xml,2416
2,lod2_33_391_5807_1_be.xml,2389
3,lod2_33_380_5830_1_be.xml,2341
4,lod2_33_376_5819_1_be.xml,2328
5,lod2_33_402_5806_1_be.xml,2306
6,lod2_33_373_5823_1_be.xml,2302
7,lod2_33_375_5821_1_be.xml,2300
8,lod2_33_405_5816_1_be.xml,2283
9,lod2_33_374_5822_1_be.xml,2265


In [52]:
# ============================================================
# 100E — RESOLVE BERLIN FAILED/PARTIAL SOURCE ROWS
# ============================================================

from pathlib import Path

BE_RAW=EXTRACTED_DIR/"BE"/"lod2_buildings_BE_raw.parquet"
BE_FINAL=EXTRACTED_DIR/"BE"/"lod2_buildings_BE.parquet"
BE_FAILED=REPORT_DIR/"lod2_failed_files_BE.csv"
BE_SUMMARY=REPORT_DIR/"lod2_extraction_summary_BE.csv"
BE_REMOVED=REPORT_DIR/"lod2_removed_failed_source_rows_BE.parquet"

be=gpd.read_parquet(BE_RAW)
failed=pd.read_csv(BE_FAILED)
summary=pd.read_csv(BE_SUMMARY)
expected=int(summary.iloc[0]["buildings"])
difference=len(be)-expected

def source_key(x):
    if pd.isna(x): return None
    name=Path(str(x)).name.lower()
    return Path(name).stem.replace(".gml","").replace(".xml","")

be["_source_key"]=be["source_file"].map(source_key)
failed["_source_key"]=failed["source_file"].map(source_key)
failed_keys=set(failed["_source_key"].dropna())

failed_source_rows=be[be["_source_key"].isin(failed_keys)].copy()
failed_counts=(
    failed_source_rows.groupby(["_source_key","source_file"],dropna=False)
    .agg(rows=("lod2_id","size"),unique_ids=("lod2_id","nunique"),missing_geometry=("geometry",lambda x:(x.isna()|x.is_empty).sum()))
    .reset_index()
)

print("="*76)
print("BERLIN FAILED-SOURCE RECONCILIATION")
print("="*76)
print(f"\nRaw rows                      : {len(be):,}")
print(f"Validated rows                : {expected:,}")
print(f"Required removal              : {difference:,}")
print(f"Failed files listed           : {len(failed):,}")
print(f"Failed files found in raw data: {failed_source_rows['source_file'].nunique():,}")
print(f"Rows from failed sources      : {len(failed_source_rows):,}")

print("\nFailed-file report:")
display(failed[["source_file","error","_source_key"]])

print("\nRows associated with failed sources:")
display(failed_counts)

if len(failed_source_rows)!=difference:
    raise ValueError(
        f"Failed-source rows ({len(failed_source_rows):,}) do not equal the report difference ({difference:,}). "
        "No rows were removed."
    )

be_clean=be[~be["_source_key"].isin(failed_keys)].drop(columns="_source_key").reset_index(drop=True)
failed_source_rows.drop(columns="_source_key").to_parquet(BE_REMOVED,index=False)

if len(be_clean)!=expected: raise ValueError(f"Cleaned rows {len(be_clean):,} != expected {expected:,}")
if be_clean["lod2_id"].duplicated().any(): raise ValueError("Duplicate LoD2 IDs remain.")
if be_clean["measured_height_m"].isna().any(): raise ValueError("Missing heights remain.")

usable=be_clean.geometry.notna()&~be_clean.geometry.is_empty
be_clean.to_parquet(BE_FINAL,index=False)

summary.loc[0,"raw_rows"]=len(be)
summary.loc[0,"failed_source_rows_removed"]=len(failed_source_rows)
summary.loc[0,"buildings"]=len(be_clean)
summary.loc[0,"unique_lod2_ids"]=be_clean["lod2_id"].nunique()
summary.loc[0,"valid_geometries"]=int(usable.sum())
summary.loc[0,"missing_geometries"]=int((~usable).sum())
summary.loc[0,"output_file"]=str(BE_FINAL)
summary.to_csv(BE_SUMMARY,index=False)

print("\n"+"="*76)
print("BERLIN CLEANED LoD2 VALIDATION")
print("="*76)
print(f"\nFinal buildings     : {len(be_clean):,}")
print(f"Unique LoD2 IDs     : {be_clean['lod2_id'].nunique():,}")
print(f"Missing heights     : {be_clean['measured_height_m'].isna().sum():,}")
print(f"Usable geometries   : {usable.sum():,}")
print(f"Missing geometries  : {(~usable).sum():,}")
print(f"Canonical output    : {BE_FINAL}")
print(f"Removed-row audit   : {BE_REMOVED}")

BERLIN FAILED-SOURCE RECONCILIATION

Raw rows                      : 641,813
Validated rows                : 639,658
Required removal              : 2,155
Failed files listed           : 3
Failed files found in raw data: 2
Rows from failed sources      : 2,155

Failed-file report:


,source_file,error,_source_key
0,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,Unknown column geometry,lod2_33_376_5827_1_be
1,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,TopologyException: side location conflict at 3...,lod2_33_382_5819_1_be
2,/fast/home/o-olajuyigbe/data/germany_lod2/raw/...,TopologyException: side location conflict at 3...,lod2_33_391_5826_1_be



Rows associated with failed sources:


,_source_key,source_file,rows,unique_ids,missing_geometry
0,lod2_33_382_5819_1_be,LoD2_33_382_5819_1_BE.xml,1013,1013,0
1,lod2_33_391_5826_1_be,LoD2_33_391_5826_1_BE.xml,1142,1142,0



BERLIN CLEANED LoD2 VALIDATION

Final buildings     : 639,658
Unique LoD2 IDs     : 639,658
Missing heights     : 0
Usable geometries   : 639,411
Missing geometries  : 247
Canonical output    : /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BE/lod2_buildings_BE.parquet
Removed-row audit   : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/lod2_removed_failed_source_rows_BE.parquet


In [53]:
# ============================================================
# 100F — VALIDATE ALL STATE FILE SETS BEFORE NATIONAL LOADING
# ============================================================

STATES=["BB","BE","BW","BY","HB","HE","HH","MV","NI","NW","RP","SH","SL","SN","ST","TH"]
rows=[]

for state in tqdm(STATES,desc="Checking state LoD2 files"):
    try:
        paths=find_state_lod2_paths(state)
        metadata=[parquet_metadata(path) for path in paths]
        expected=state_expected_buildings(state)
        recorded=sum(x["rows"] for x in metadata if x)
        rows.append({
            "state_code":state,
            "files":len(paths),
            "expected_buildings":expected,
            "recorded_buildings":recorded,
            "difference":recorded-expected if expected is not None else np.nan,
            "size_gb":sum(x["size_gb"] for x in metadata if x),
            "status":"ready" if expected==recorded else "count_mismatch",
            "first_file":str(paths[0])
        })
    except Exception as e:
        rows.append({
            "state_code":state,"files":0,"expected_buildings":state_expected_buildings(state),
            "recorded_buildings":0,"difference":np.nan,"size_gb":0,
            "status":"failed","first_file":None,"error":f"{type(e).__name__}: {e}"
        })

state_file_validation=pd.DataFrame(rows)
state_file_validation.to_csv(REPORT_DIR/"lod2_state_file_validation.csv",index=False)

print("="*80)
print("GERMANY LoD2 FILE-SET VALIDATION")
print("="*80)
display(state_file_validation[[
    "state_code","files","expected_buildings","recorded_buildings",
    "difference","size_gb","status"
]].round({"size_gb":3}))

print(f"\nStates ready       : {state_file_validation['status'].eq('ready').sum():,}/16")
print(f"Expected buildings : {state_file_validation['expected_buildings'].sum():,}")
print(f"Recorded buildings : {state_file_validation['recorded_buildings'].sum():,}")
print(f"Total size         : {state_file_validation['size_gb'].sum():,.2f} GB")

if state_file_validation["status"].eq("ready").all():
    print("\nVALIDATION PASSED — all states can be loaded for the national spatial join.")
else:
    print("\nVALIDATION FAILED — resolve the states shown below before restarting Cell 100.")
    display(state_file_validation[state_file_validation["status"].ne("ready")])

Checking state LoD2 files: 100%|█████████████████████████████| 16/16 [00:02<00:00,  6.55it/s]


GERMANY LoD2 FILE-SET VALIDATION


,state_code,files,expected_buildings,recorded_buildings,difference,size_gb,status
0,BB,78,2376794,2376794,0.0,0.216,ready
1,BE,1,639658,639658,0.0,0.077,ready
2,BW,122,6465296,6465296,0.0,0.628,ready
3,BY,61,10111920,10111920,0.0,1.027,ready
4,HB,1,331804,331804,0.0,0.035,ready
5,HE,33,4981859,4981859,0.0,0.512,ready
6,HH,1,388729,388729,0.0,0.042,ready
7,MV,26,1332053,1332053,0.0,0.119,ready
8,NI,0,6828051,0,NaN,0.000,failed
9,NW,117,11870213,11870213,0.0,1.230,ready



States ready       : 15/16
Expected buildings : 58,049,619
Recorded buildings : 51,221,568
Total size         : 5.19 GB

VALIDATION FAILED — resolve the states shown below before restarting Cell 100.


,state_code,files,expected_buildings,recorded_buildings,difference,size_gb,status,first_file,error
8,NI,0,6828051,0,NaN,0.0,failed,None,FileNotFoundError: NI: no file or file group m...


In [54]:
# ============================================================
# 100G — LOCATE LOWER SAXONY LoD2 FILES
# ============================================================

import pyarrow.parquet as pq
from pathlib import Path

state="NI"
expected=state_expected_buildings(state)
summary_path=REPORT_DIR/f"lod2_extraction_summary_{state}.csv"
ni_summary=pd.read_csv(summary_path)

print("="*80)
print("LOWER SAXONY LoD2 FILE DISCOVERY")
print("="*80)
print("\nValidation report:")
display(ni_summary.T)

tokens=("ni","niedersachsen","lower_saxony","lower saxony")
candidate_roots=[EXTRACTED_DIR]

if "output_directory" in ni_summary.columns:
    candidate_roots.append(Path(str(ni_summary.iloc[0]["output_directory"])))
if "output_file" in ni_summary.columns:
    candidate_roots.append(Path(str(ni_summary.iloc[0]["output_file"])).parent)

files=set()
for root in candidate_roots:
    if not root.exists(): continue
    if root.is_file() and root.suffix.lower()==".parquet": files.add(root.resolve())
    else:
        for path in root.rglob("*.parquet"):
            text=str(path).lower()
            if any(token in text for token in tokens): files.add(path.resolve())

rows=[]
for path in sorted(files):
    try:
        pf=pq.ParquetFile(path)
        columns=pf.schema_arrow.names
        rows.append({
            "path":str(path),
            "parent":str(path.parent),
            "filename":path.name,
            "rows":pf.metadata.num_rows,
            "size_mb":path.stat().st_size/1024**2,
            "has_geometry":"geometry" in columns,
            "has_lod2_id":"lod2_id" in columns,
            "has_gml_id":"gml_id" in columns
        })
    except Exception as e:
        rows.append({"path":str(path),"parent":str(path.parent),"filename":path.name,"rows":np.nan,"size_mb":np.nan,"error":str(e)})

ni_inventory=pd.DataFrame(rows)

print(f"\nExpected buildings : {expected:,}")
print(f"Candidate files    : {len(ni_inventory):,}")

if ni_inventory.empty:
    print("\nNo filename/path matches were found. Searching all LoD2 Parquet directories by combined row count...")
    all_rows=[]
    for path in tqdm(list(EXTRACTED_DIR.rglob("*.parquet")),desc="Reading Parquet metadata"):
        try:
            pf=pq.ParquetFile(path)
            columns=pf.schema_arrow.names
            if "geometry" in columns and any(c in columns for c in ["lod2_id","gml_id","id","building_id"]):
                all_rows.append({"path":str(path.resolve()),"parent":str(path.resolve().parent),"filename":path.name,"rows":pf.metadata.num_rows,"size_mb":path.stat().st_size/1024**2})
        except Exception:
            pass
    ni_inventory=pd.DataFrame(all_rows)

display(ni_inventory.sort_values(["parent","filename"]) if not ni_inventory.empty else ni_inventory)

if not ni_inventory.empty:
    grouped=(
        ni_inventory.groupby("parent",dropna=False)
        .agg(files=("filename","size"),rows=("rows","sum"),size_mb=("size_mb","sum"))
        .reset_index()
    )
    grouped["difference"]=grouped["rows"]-expected

    print("\nDirectory totals closest to the validated count:")
    display(grouped.iloc[grouped["difference"].abs().argsort()[:20]])

    exact=grouped[grouped["rows"].eq(expected)]
    if not exact.empty:
        print("\nEXACT MATCH FOUND:")
        display(exact)
    else:
        print("\nNo exact directory total found yet.")

LOWER SAXONY LoD2 FILE DISCOVERY

Validation report:


,0
state_code,NI
state_name,Lower Saxony
gml_files_expected,37928
files_with_buildings,37927
empty_files,1
files_accounted_for,37928
missing_source_files,0
unexpected_source_files,0
source_files_in_multiple_parts,0
failed_files,0



Expected buildings : 6,828,051
Candidate files    : 36


,path,parent,filename,rows,size_mb,has_geometry,has_lod2_id,has_gml_id
0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_part_0001.parquet,9001,1.106100,True,True,False
1,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_part_0002.parquet,4961,0.653766,True,True,False
2,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_part_0003.parquet,1134862,115.404383,True,True,False
3,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_part_0004.parquet,1572537,160.728450,True,True,False
4,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_part_0005.parquet,1496001,147.068261,True,True,False
5,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_part_0006.parquet,50871,4.914240,True,True,False
6,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_repair_0001.parquet,93272,10.004617,True,True,False
7,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_repair_0002.parquet,63937,7.228084,True,True,False
8,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_repair_0003.parquet,88894,9.928548,True,True,False
9,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,lod2_buildings_NI_repair_0004.parquet,95436,10.673816,True,True,False



Directory totals closest to the validated count:


,parent,files,rows,size_mb,difference
0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,6,4268233,429.875199,-2559818
1,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,30,2559818,277.083851,-4268233



No exact directory total found yet.


In [55]:
# ============================================================
# 100H — SUPPORT MULTIPLE VALID LoD2 PART DIRECTORIES
# ============================================================

from itertools import combinations

def find_state_lod2_paths(state_code,verbose=False):
    state_code=state_code.upper()
    expected=state_expected_buildings(state_code)
    summary_path=REPORT_DIR/f"lod2_extraction_summary_{state_code}.csv"
    roots=[EXTRACTED_DIR/state_code]

    if summary_path.exists():
        summary=pd.read_csv(summary_path)
        if not summary.empty:
            for col in ["output_directory","output_file"]:
                if col in summary.columns and pd.notna(summary.iloc[0][col]):
                    output=Path(str(summary.iloc[0][col]))
                    roots.extend([output,output.parent,output/"parts"] if output.suffix=="" else [output.parent])

    files=sorted({
        p.resolve()
        for root in roots if root.exists()
        for p in ([root] if root.is_file() else root.rglob("*.parquet"))
        if p.is_file() and not any(x in p.stem.lower() for x in BAD_TOKENS)
    })

    info=[x for x in (parquet_metadata(p) for p in files) if x is not None]
    usable=[x for x in info if "geometry" in x["columns"] and any(c in x["columns"] for c in ["lod2_id","gml_id","id","building_id"])]

    if not usable: raise FileNotFoundError(f"No usable LoD2 Parquet files found for {state_code}")

    canonical=(EXTRACTED_DIR/state_code/f"lod2_buildings_{state_code}.parquet").resolve()
    if canonical.exists() and any(x["path"]==canonical for x in usable): return [canonical]

    # One complete file
    singles=[x for x in usable if expected is not None and x["rows"]==expected]
    if singles:
        singles.sort(key=lambda x:("raw" in x["path"].stem.lower(),len(x["path"].parts)))
        return [singles[0]["path"]]

    # Group files by directory
    groups={}
    for item in usable: groups.setdefault(item["path"].parent,[]).append(item)

    group_info=[]
    for parent,items in groups.items():
        items=sorted(items,key=lambda x:x["path"].name)
        group_info.append({
            "parent":parent,
            "items":items,
            "rows":sum(x["rows"] for x in items),
            "files":len(items)
        })

    # One complete directory
    exact_groups=[g for g in group_info if g["rows"]==expected]
    if exact_groups:
        exact_groups.sort(key=lambda g:(g["parent"].name!="parts",g["files"]))
        return [x["path"] for x in exact_groups[0]["items"]]

    # Combination of directories, e.g. NI original + repair parts
    exact_combinations=[]
    for n in range(2,len(group_info)+1):
        for combo in combinations(group_info,n):
            if sum(g["rows"] for g in combo)==expected:
                paths=sorted([x["path"] for g in combo for x in g["items"]])
                exact_combinations.append((n,len(paths),combo,paths))
        if exact_combinations: break

    if exact_combinations:
        exact_combinations.sort(key=lambda x:(x[0],x[1]))
        chosen=exact_combinations[0]

        if verbose:
            display(pd.DataFrame([{
                "parent":str(g["parent"]),
                "files":g["files"],
                "rows":g["rows"]
            } for g in chosen[2]]))

        return chosen[3]

    inventory=pd.DataFrame([{
        "path":str(x["path"]),
        "parent":str(x["path"].parent),
        "rows":x["rows"],
        "size_gb":x["size_gb"]
    } for x in usable])

    if verbose:
        display(inventory.groupby("parent").agg(files=("path","size"),rows=("rows","sum"),size_gb=("size_gb","sum")).reset_index())

    raise FileNotFoundError(f"{state_code}: no file combination matches the validated total of {expected:,}")

# Confirm the combined Lower Saxony dataset
ni_paths=find_state_lod2_paths("NI",verbose=True)
ni_metadata=[parquet_metadata(path) for path in ni_paths]

print("="*76)
print("LOWER SAXONY COMBINED FILE SET")
print("="*76)
print(f"\nFiles selected     : {len(ni_paths):,}")
print(f"Original parts     : {sum('part_' in p.name for p in ni_paths):,}")
print(f"Repair parts       : {sum('repair_' in p.name for p in ni_paths):,}")
print(f"Buildings recorded : {sum(x['rows'] for x in ni_metadata):,}")
print(f"Buildings expected : {state_expected_buildings('NI'):,}")
print(f"Total size         : {sum(x['size_gb'] for x in ni_metadata):,.3f} GB")

,parent,files,rows
0,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,6,4268233
1,/fast/home/o-olajuyigbe/data/germany_lod2/extr...,30,2559818


LOWER SAXONY COMBINED FILE SET

Files selected     : 36
Original parts     : 6
Repair parts       : 30
Buildings recorded : 6,828,051
Buildings expected : 6,828,051
Total size         : 0.690 GB


In [56]:
# ============================================================
# 100I — FINAL 16-STATE FILE-SET VALIDATION
# ============================================================

STATES=["BB","BE","BW","BY","HB","HE","HH","MV","NI","NW","RP","SH","SL","SN","ST","TH"]
rows=[]

for state in tqdm(STATES,desc="Validating state file sets"):
    try:
        paths=find_state_lod2_paths(state)
        metadata=[parquet_metadata(path) for path in paths]
        expected=state_expected_buildings(state)
        recorded=sum(x["rows"] for x in metadata if x)

        rows.append({
            "state_code":state,
            "files":len(paths),
            "expected_buildings":expected,
            "recorded_buildings":recorded,
            "difference":recorded-expected,
            "size_gb":sum(x["size_gb"] for x in metadata if x),
            "status":"ready" if recorded==expected else "count_mismatch"
        })
    except Exception as e:
        rows.append({
            "state_code":state,
            "files":0,
            "expected_buildings":state_expected_buildings(state),
            "recorded_buildings":0,
            "difference":np.nan,
            "size_gb":0,
            "status":"failed",
            "error":f"{type(e).__name__}: {e}"
        })

state_file_validation=pd.DataFrame(rows)
state_file_validation.to_csv(REPORT_DIR/"lod2_state_file_validation.csv",index=False)

print("="*80)
print("GERMANY LoD2 FINAL FILE-SET VALIDATION")
print("="*80)
display(state_file_validation.round({"size_gb":3}))

print(f"\nStates ready       : {state_file_validation['status'].eq('ready').sum():,}/16")
print(f"Expected buildings : {state_file_validation['expected_buildings'].sum():,}")
print(f"Recorded buildings : {state_file_validation['recorded_buildings'].sum():,}")
print(f"Total size         : {state_file_validation['size_gb'].sum():,.2f} GB")

if state_file_validation["status"].eq("ready").all():
    print("\nVALIDATION PASSED — all 16 states are ready for the national spatial join.")
else:
    print("\nVALIDATION FAILED:")
    display(state_file_validation[state_file_validation["status"].ne("ready")])

Validating state file sets: 100%|████████████████████████████| 16/16 [00:01<00:00,  8.70it/s]

GERMANY LoD2 FINAL FILE-SET VALIDATION


,state_code,files,expected_buildings,recorded_buildings,difference,size_gb,status
0,BB,78,2376794,2376794,0,0.216,ready
1,BE,1,639658,639658,0,0.077,ready
2,BW,122,6465296,6465296,0,0.628,ready
3,BY,61,10111920,10111920,0,1.027,ready
4,HB,1,331804,331804,0,0.035,ready
5,HE,33,4981859,4981859,0,0.512,ready
6,HH,1,388729,388729,0,0.042,ready
7,MV,26,1332053,1332053,0,0.119,ready
8,NI,36,6828051,6828051,0,0.690,ready
9,NW,117,11870213,11870213,0,1.230,ready



States ready       : 16/16
Expected buildings : 58,049,619
Recorded buildings : 58,049,619
Total size         : 5.88 GB

VALIDATION PASSED — all 16 states are ready for the national spatial join.


In [57]:
# ============================================================
# 101 — NATIONAL LoD2–OSM BEST-OVERLAP MATCHING
# ============================================================

import gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from tqdm.auto import tqdm

COMMON_CRS="EPSG:3035"
STATES=["BB","BE","BW","BY","HB","HE","HH","MV","NI","NW","RP","SH","SL","SN","ST","TH"]
OSM_BUILDING_FILE=Path("/fast/home/o-olajuyigbe/osm_project/data/processed/germany_buildings_classified_stage2.parquet")
FINAL_DIR=BASE_DIR/"final"
FINAL_DIR.mkdir(parents=True,exist_ok=True)

NATIONAL_LOD2_PATH=FINAL_DIR/"germany_lod2_buildings.parquet"
NATIONAL_MATCH_PATH=FINAL_DIR/"germany_osm_lod2_matches.parquet"
NATIONAL_MATCH_SUMMARY=REPORT_DIR/"national_osm_lod2_match_summary.csv"
STATE_LOAD_SUMMARY=REPORT_DIR/"national_lod2_loading_summary.csv"

for name in ["lod2_frames","lod2","osm","pairs","accepted","best","match_table","be","be_clean","be_lod2"]:
    if name in globals(): del globals()[name]
gc.collect()

# ── 1. Load and standardize all LoD2 footprints ──────────────────────────────
lod2_frames=[]
state_stats=[]
lod2_columns=["lod2_id","measured_height_m","height_method","footprint_method","geometry"]

for state in tqdm(STATES,desc="Loading national LoD2"):
    gdf=load_state_lod2(state)
    gdf=standardize_lod2_schema(gdf)

    missing=[c for c in lod2_columns if c not in gdf.columns]
    if missing: raise KeyError(f"{state} missing columns: {missing}")

    loaded=len(gdf)
    usable=gdf.geometry.notna()&~gdf.geometry.is_empty
    invalid=usable&~np.asarray(shapely.is_valid(gdf.geometry.array))

    if invalid.any():
        gdf.loc[invalid,"geometry"]=list(shapely.make_valid(gdf.loc[invalid].geometry.array))

    usable=gdf.geometry.notna()&~gdf.geometry.is_empty
    gdf=gdf.loc[usable,lod2_columns].copy()
    gdf["lod2_state_code"]=state
    gdf=gdf.to_crs(COMMON_CRS)

    state_stats.append({
        "state_code":state,
        "records_loaded":loaded,
        "missing_geometry":loaded-int(usable.sum()),
        "invalid_geometry_repaired":int(invalid.sum()),
        "records_retained":len(gdf)
    })

    lod2_frames.append(gdf)
    del gdf
    gc.collect()

lod2=gpd.GeoDataFrame(pd.concat(lod2_frames,ignore_index=True,copy=False),geometry="geometry",crs=COMMON_CRS)
del lod2_frames
gc.collect()

lod2["lod2_area_m2"]=shapely.area(lod2.geometry.array)
valid_area=np.isfinite(lod2["lod2_area_m2"])&lod2["lod2_area_m2"].gt(0)
zero_area_count=int((~valid_area).sum())
lod2=lod2.loc[valid_area].reset_index(drop=True)
lod2["_lod2_row"]=np.arange(len(lod2),dtype=np.int64)

state_load_summary=pd.DataFrame(state_stats)
state_load_summary.to_csv(STATE_LOAD_SUMMARY,index=False)

print("="*80)
print("NATIONAL LoD2 DATASET")
print("="*80)
print(f"\nValidated records      : {state_file_validation['expected_buildings'].sum():,}")
print(f"Matchable footprints   : {len(lod2):,}")
print(f"Missing geometries     : {state_load_summary['missing_geometry'].sum():,}")
print(f"Invalid geometries fixed: {state_load_summary['invalid_geometry_repaired'].sum():,}")
print(f"Zero-area geometries   : {zero_area_count:,}")
print(f"CRS                    : {lod2.crs.to_epsg()}")

lod2.to_parquet(NATIONAL_LOD2_PATH,index=False)
print(f"Saved national LoD2    : {NATIONAL_LOD2_PATH}")

# ── 2. Load all OSM buildings ────────────────────────────────────────────────
osm=gpd.read_parquet(OSM_BUILDING_FILE,columns=["id","geometry"])
osm=osm[osm.geometry.notna()&~osm.geometry.is_empty].copy()

invalid_osm=~np.asarray(shapely.is_valid(osm.geometry.array))
if invalid_osm.any():
    osm.loc[invalid_osm,"geometry"]=list(shapely.make_valid(osm.loc[invalid_osm].geometry.array))

osm=osm[osm.geometry.notna()&~osm.geometry.is_empty].to_crs(COMMON_CRS).reset_index(drop=True)
osm["osm_area_m2"]=shapely.area(osm.geometry.array)
valid_osm_area=np.isfinite(osm["osm_area_m2"])&osm["osm_area_m2"].gt(0)
osm=osm.loc[valid_osm_area].reset_index(drop=True)
osm["_osm_row"]=np.arange(len(osm),dtype=np.int64)

if osm["id"].duplicated().any():
    raise ValueError(f"OSM contains {osm['id'].duplicated().sum():,} duplicate IDs.")

print("\n"+"="*80)
print("NATIONAL OSM DATASET")
print("="*80)
print(f"\nMatchable buildings    : {len(osm):,}")
print(f"Invalid geometries fixed: {invalid_osm.sum():,}")
print(f"CRS                    : {osm.crs.to_epsg()}")

# ── 3. Generate all intersecting candidate pairs ─────────────────────────────
join_start=time.time()

pairs=gpd.sjoin(
    osm[["_osm_row","geometry"]],
    lod2[["_lod2_row","geometry"]],
    how="inner",
    predicate="intersects"
)

pairs=pd.DataFrame(pairs[["_osm_row","_lod2_row"]]).reset_index(drop=True)
candidate_pairs=len(pairs)

print("\n"+"="*80)
print("NATIONAL SPATIAL JOIN")
print("="*80)
print(f"\nCandidate pairs        : {candidate_pairs:,}")

# ── 4. Calculate exact overlap area ──────────────────────────────────────────
osm_idx=pairs["_osm_row"].to_numpy(dtype=np.int64)
lod2_idx=pairs["_lod2_row"].to_numpy(dtype=np.int64)

intersection_area=shapely.area(
    shapely.intersection(
        osm.geometry.array.take(osm_idx),
        lod2.geometry.array.take(lod2_idx)
    )
)

pairs["intersection_area_m2"]=intersection_area
pairs=pairs[np.isfinite(pairs["intersection_area_m2"])&pairs["intersection_area_m2"].ge(1.0)].copy()

del intersection_area, osm_idx, lod2_idx
gc.collect()

osm_idx=pairs["_osm_row"].to_numpy(dtype=np.int64)
lod2_idx=pairs["_lod2_row"].to_numpy(dtype=np.int64)
pairs["osm_area_m2"]=osm["osm_area_m2"].to_numpy()[osm_idx]
pairs["lod2_area_m2"]=lod2["lod2_area_m2"].to_numpy()[lod2_idx]

pairs["osm_coverage"]=(pairs["intersection_area_m2"]/pairs["osm_area_m2"]).clip(0,1)
pairs["lod2_coverage"]=(pairs["intersection_area_m2"]/pairs["lod2_area_m2"]).clip(0,1)

union_area=pairs["osm_area_m2"]+pairs["lod2_area_m2"]-pairs["intersection_area_m2"]
pairs["iou"]=(pairs["intersection_area_m2"]/union_area).clip(0,1)

pairs["score"]=0.50*pairs["iou"]+0.25*pairs["osm_coverage"]+0.25*pairs["lod2_coverage"]

# Balanced overlap rule previously calibrated using Bremen
accepted=pairs[
    pairs["iou"].ge(0.25)
    |(pairs["osm_coverage"].ge(0.70)&pairs["lod2_coverage"].ge(0.20))
    |(pairs["lod2_coverage"].ge(0.70)&pairs["osm_coverage"].ge(0.20))
].copy()

if accepted.empty:
    raise RuntimeError("No acceptable LoD2–OSM matches were found.")

# ── 5. Select the best LoD2 footprint for every OSM building ─────────────────
accepted=accepted.sort_values(
    ["_osm_row","score","intersection_area_m2","iou","_lod2_row"],
    ascending=[True,False,False,False,True],
    kind="mergesort"
)

accepted["candidate_rank"]=accepted.groupby("_osm_row",sort=False).cumcount()+1
best=accepted[accepted["candidate_rank"].eq(1)].copy()

second_scores=accepted[accepted["candidate_rank"].eq(2)].set_index("_osm_row")["score"]
best["score_margin"]=best["score"]-best["_osm_row"].map(second_scores).fillna(0)

reuse_counts=best["_lod2_row"].value_counts()
best["lod2_reuse_count"]=best["_lod2_row"].map(reuse_counts).astype(np.int32)
best["lod2_reused"]=best["lod2_reuse_count"].gt(1)

best["match_pattern"]=np.select(
    [
        best["iou"].ge(0.70)&best["osm_coverage"].ge(0.80)&best["lod2_coverage"].ge(0.80),
        best["iou"].ge(0.25),
        best["osm_coverage"].ge(0.70)&best["lod2_coverage"].ge(0.20),
        best["lod2_coverage"].ge(0.70)&best["osm_coverage"].ge(0.20)
    ],
    ["strong_balanced","moderate_balanced","osm_inside_lod2","lod2_inside_osm"],
    default="borderline"
)

best["match_quality"]=np.select(
    [
        best["match_pattern"].eq("strong_balanced"),
        best["match_pattern"].eq("moderate_balanced")&best["score"].ge(0.50),
        best["match_pattern"].isin(["osm_inside_lod2","lod2_inside_osm"])&best["score"].ge(0.35)
    ],
    ["high","medium","medium"],
    default="low"
)

# ── 6. Build the national ID-to-height match table ───────────────────────────
osm_idx=best["_osm_row"].to_numpy(dtype=np.int64)
lod2_idx=best["_lod2_row"].to_numpy(dtype=np.int64)

match_table=pd.DataFrame({
    "_osm_row":osm_idx,
    "osm_id":osm["id"].to_numpy()[osm_idx],
    "_lod2_row":lod2_idx,
    "lod2_id":lod2["lod2_id"].to_numpy()[lod2_idx],
    "lod2_state_code":lod2["lod2_state_code"].to_numpy()[lod2_idx],
    "lod2_height_m":lod2["measured_height_m"].to_numpy()[lod2_idx],
    "lod2_height_method":lod2["height_method"].to_numpy()[lod2_idx],
    "lod2_footprint_method":lod2["footprint_method"].to_numpy()[lod2_idx],
    "lod2_match_quality":best["match_quality"].to_numpy(),
    "lod2_match_pattern":best["match_pattern"].to_numpy(),
    "lod2_iou":best["iou"].to_numpy(),
    "lod2_osm_coverage":best["osm_coverage"].to_numpy(),
    "lod2_footprint_coverage":best["lod2_coverage"].to_numpy(),
    "lod2_intersection_m2":best["intersection_area_m2"].to_numpy(),
    "lod2_score":best["score"].to_numpy(),
    "lod2_score_margin":best["score_margin"].to_numpy(),
    "lod2_reuse_count":best["lod2_reuse_count"].to_numpy(),
    "lod2_reused":best["lod2_reused"].to_numpy()
})

match_table["lod2_uid"]=match_table["lod2_state_code"].astype("string")+":"+match_table["lod2_id"].astype("string")
match_table["lod2_match_method"]="balanced_footprint_overlap"
match_table=match_table.sort_values("_osm_row").reset_index(drop=True)

if match_table["_osm_row"].duplicated().any():
    raise ValueError("More than one final LoD2 match remains for an OSM building.")
if match_table["osm_id"].duplicated().any():
    raise ValueError("More than one final match remains for an OSM ID.")
if match_table["lod2_height_m"].isna().any():
    raise ValueError("Matched rows contain missing LoD2 heights.")

match_table.to_parquet(NATIONAL_MATCH_PATH,index=False)

# ── 7. Save and print national validation summary ────────────────────────────
summary=pd.DataFrame([{
    "osm_buildings":len(osm),
    "lod2_records":int(state_file_validation["expected_buildings"].sum()),
    "lod2_matchable_footprints":len(lod2),
    "candidate_pairs":candidate_pairs,
    "pairs_with_intersection_ge_1m2":len(pairs),
    "accepted_pairs":len(accepted),
    "matched_osm_buildings":len(match_table),
    "unmatched_osm_buildings":len(osm)-len(match_table),
    "osm_match_rate_pct":100*len(match_table)/len(osm),
    "unique_lod2_footprints_used":match_table["_lod2_row"].nunique(),
    "high_quality_matches":int(match_table["lod2_match_quality"].eq("high").sum()),
    "medium_quality_matches":int(match_table["lod2_match_quality"].eq("medium").sum()),
    "low_quality_matches":int(match_table["lod2_match_quality"].eq("low").sum()),
    "reused_lod2_footprints":int(reuse_counts.gt(1).sum()),
    "maximum_lod2_reuse":int(reuse_counts.max()),
    "minimum_iou":match_table["lod2_iou"].min(),
    "median_iou":match_table["lod2_iou"].median(),
    "median_osm_coverage":match_table["lod2_osm_coverage"].median(),
    "median_lod2_coverage":match_table["lod2_footprint_coverage"].median(),
    "runtime_minutes":(time.time()-join_start)/60,
    "match_output":str(NATIONAL_MATCH_PATH)
}])

summary.to_csv(NATIONAL_MATCH_SUMMARY,index=False)

print("\n"+"="*80)
print("NATIONAL LoD2–OSM MATCHING RESULT")
print("="*80)
display(summary.T)

print("\nMatch quality:")
display(match_table["lod2_match_quality"].value_counts().rename_axis("quality").reset_index(name="buildings"))

print("\nMatch pattern:")
display(match_table["lod2_match_pattern"].value_counts().rename_axis("pattern").reset_index(name="buildings"))

print(f"\nNational LoD2 file : {NATIONAL_LOD2_PATH}")
print(f"National matches   : {NATIONAL_MATCH_PATH}")
print(f"Summary            : {NATIONAL_MATCH_SUMMARY}")

Loading BB LoD2: 100%|███████████████████████████████████████| 78/78 [00:08<00:00,  9.21it/s]


BB FINAL LoD2 DATASET

Files loaded      : 78
Buildings loaded  : 2,376,794
Buildings expected: 2,376,794
Duplicate IDs     : 0
Missing heights   : 0
Missing geometry  : 45,294
CRS               : 25833

Files:
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0001.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0002.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0003.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0004.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0005.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0006.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0007.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BB/parts/lod2_buildings_BB_part_0008.parqu

Loading BE LoD2: 100%|█████████████████████████████████████████| 1/1 [00:01<00:00,  1.11s/it]


BE FINAL LoD2 DATASET

Files loaded      : 1
Buildings loaded  : 639,658
Buildings expected: 639,658
Duplicate IDs     : 0
Missing heights   : 0
Missing geometry  : 247
CRS               : 25833

Files:
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BE/lod2_buildings_BE.parquet


Loading national LoD2:  12%|████▎                             | 2/16 [01:02<07:20, 31.43s/it]


ValueError: BW: 10 duplicate IDs

In [58]:
# ============================================================
# 101A — VALIDATE AND RESOLVE BW DUPLICATE LoD2 IDs
# ============================================================

import gc
from pathlib import Path

BW_FINAL=EXTRACTED_DIR/"BW"/"lod2_buildings_BW.parquet"
BW_AUDIT=REPORT_DIR/"lod2_duplicate_rows_BW.parquet"
BW_GROUP_AUDIT=REPORT_DIR/"lod2_duplicate_groups_BW.csv"
BW_SUMMARY=REPORT_DIR/"lod2_extraction_summary_BW.csv"

bw=load_state_lod2("BW",validate=False)
bw=standardize_lod2_schema(bw)

duplicate_mask=bw["lod2_id"].duplicated(False)
duplicate_rows=bw.loc[duplicate_mask].copy()
duplicate_rows.to_parquet(BW_AUDIT,index=False)

def compare_duplicate_group(group):
    heights=pd.to_numeric(group["measured_height_m"],errors="coerce")
    usable=group.geometry.notna()&~group.geometry.is_empty
    geometries=list(group.loc[usable].geometry)

    same_height=heights.notna().all() and np.allclose(heights,heights.iloc[0],rtol=0,atol=1e-6)

    same_geometry=True
    if len(geometries)>1:
        reference=shapely.make_valid(geometries[0])
        try:
            same_geometry=all(bool(shapely.equals(reference,shapely.make_valid(geometry))) for geometry in geometries[1:])
        except Exception:
            same_geometry=all(bool(shapely.equals_exact(reference,shapely.make_valid(geometry),tolerance=0.001)) for geometry in geometries[1:])

    return pd.Series({
        "rows":len(group),
        "extra_rows":len(group)-1,
        "unique_heights":heights.nunique(dropna=True),
        "usable_geometries":int(usable.sum()),
        "same_height":same_height,
        "same_geometry":same_geometry,
        "safe_to_deduplicate":same_height and same_geometry,
        "source_files":" | ".join(sorted(group["source_file"].astype(str).unique())) if "source_file" in group else ""
    })

duplicate_groups=(
    duplicate_rows.groupby("lod2_id",dropna=False)
    .apply(compare_duplicate_group,include_groups=False)
    .reset_index()
)

duplicate_groups.to_csv(BW_GROUP_AUDIT,index=False)

print("="*80)
print("BADEN-WÜRTTEMBERG DUPLICATE-ID VALIDATION")
print("="*80)
print(f"\nRows loaded              : {len(bw):,}")
print(f"Duplicated IDs           : {duplicate_groups['lod2_id'].nunique():,}")
print(f"Rows with duplicated IDs : {len(duplicate_rows):,}")
print(f"Extra duplicate rows     : {duplicate_groups['extra_rows'].sum():,}")
print(f"Safe duplicate groups    : {duplicate_groups['safe_to_deduplicate'].sum():,}")
print(f"Conflicting groups       : {(~duplicate_groups['safe_to_deduplicate']).sum():,}")

display(duplicate_groups)

conflicts=duplicate_groups[~duplicate_groups["safe_to_deduplicate"]]
if not conflicts.empty:
    display(duplicate_rows[duplicate_rows["lod2_id"].isin(conflicts["lod2_id"])])
    raise ValueError(
        f"{len(conflicts):,} BW duplicate-ID groups have conflicting heights or geometries. "
        "No records were removed."
    )

# Keep the best copy: usable geometry, valid geometry, largest area
bw["_usable_geometry"]=bw.geometry.notna()&~bw.geometry.is_empty
bw["_valid_geometry"]=bw["_usable_geometry"]&np.asarray(shapely.is_valid(bw.geometry.array))
bw["_geometry_area"]=np.where(bw["_usable_geometry"],shapely.area(bw.geometry.array),-1)

bw_clean=(
    bw.sort_values(
        ["lod2_id","_usable_geometry","_valid_geometry","_geometry_area"],
        ascending=[True,False,False,False],
        kind="mergesort"
    )
    .drop_duplicates("lod2_id",keep="first")
    .drop(columns=["_usable_geometry","_valid_geometry","_geometry_area"])
    .reset_index(drop=True)
)

removed=len(bw)-len(bw_clean)

if removed!=duplicate_groups["extra_rows"].sum():
    raise ValueError(f"Removed {removed:,} rows, expected {duplicate_groups['extra_rows'].sum():,}.")
if bw_clean["lod2_id"].duplicated().any():
    raise ValueError("Duplicate BW LoD2 IDs remain.")
if bw_clean["measured_height_m"].isna().any():
    raise ValueError("Missing BW heights remain.")

bw_clean.to_parquet(BW_FINAL,index=False)

summary=pd.read_csv(BW_SUMMARY)
summary.loc[0,"raw_rows"]=len(bw)
summary.loc[0,"duplicate_rows_removed"]=removed
summary.loc[0,"buildings"]=len(bw_clean)
summary.loc[0,"unique_lod2_ids"]=bw_clean["lod2_id"].nunique()

usable=bw_clean.geometry.notna()&~bw_clean.geometry.is_empty
if "valid_geometries" in summary.columns: summary.loc[0,"valid_geometries"]=int(usable.sum())
if "usable_geometries" in summary.columns: summary.loc[0,"usable_geometries"]=int(usable.sum())
if "missing_geometries" in summary.columns: summary.loc[0,"missing_geometries"]=int((~usable).sum())
if "output_file" in summary.columns: summary.loc[0,"output_file"]=str(BW_FINAL)
if "output_directory" in summary.columns: summary.loc[0,"output_directory"]=str(BW_FINAL.parent)

summary.to_csv(BW_SUMMARY,index=False)

print("\n"+"="*80)
print("BADEN-WÜRTTEMBERG CLEANED LoD2 DATASET")
print("="*80)
print(f"\nOriginal records       : {len(bw):,}")
print(f"Duplicate rows removed : {removed:,}")
print(f"Final records          : {len(bw_clean):,}")
print(f"Unique LoD2 IDs        : {bw_clean['lod2_id'].nunique():,}")
print(f"Missing heights        : {bw_clean['measured_height_m'].isna().sum():,}")
print(f"Missing geometries     : {(~usable).sum():,}")
print(f"Canonical output       : {BW_FINAL}")
print(f"Duplicate-row audit    : {BW_AUDIT}")
print(f"Duplicate-group audit  : {BW_GROUP_AUDIT}")

del bw, bw_clean, duplicate_rows
gc.collect()

Loading BW LoD2: 100%|█████████████████████████████████████| 122/122 [00:21<00:00,  5.71it/s]


BW FINAL LoD2 DATASET

Files loaded      : 122
Buildings loaded  : 6,465,296
Buildings expected: 6,465,296
Duplicate IDs     : 10
Missing heights   : 0
Missing geometry  : 4
CRS               : 25832

Files:
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BW/parts/lod2_buildings_BW_part_0001.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BW/parts/lod2_buildings_BW_part_0002.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BW/parts/lod2_buildings_BW_part_0003.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BW/parts/lod2_buildings_BW_part_0004.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BW/parts/lod2_buildings_BW_part_0005.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BW/parts/lod2_buildings_BW_part_0006.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BW/parts/lod2_buildings_BW_part_0007.parquet
  /fast/home/o-olajuyigbe/data/germany_lod2/extracted/BW/parts/lod2_buildings_BW_part_0008.parquet


,lod2_id,rows,extra_rows,unique_heights,usable_geometries,same_height,same_geometry,safe_to_deduplicate,source_files
0,DEBW_51000004HUv,2,1,2,2,False,False,False,LoD2_32_513_5443_1_BW.gml
1,DEBW_B010000BPNh,4,3,1,4,True,False,False,LoD2_32_443_5332_1_BW.gml | LoD2_32_444_5332_1...
2,DEBW_B010000BSXH,4,3,1,4,True,False,False,LoD2_32_513_5391_1_BW.gml | LoD2_32_514_5391_1...
3,DEBW_B010001jE2m,4,3,1,4,True,False,False,LoD2_32_550_5299_1_BW.gml | LoD2_32_551_5299_1...


,lod2_id,creation_date,function,roof_type,measured_height_m,parent_height_m,maximum_part_height_m,building_part_count,height_method,storeys_above_ground,...,footprint_method,geometry,state_code,source_state,source_crs,citygml_version,height_source,source_archive,source_member,source_path
931558,DEBW_B010000BPNh,2024-01-01,53001_1800,1000,1.000,1.000,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((444003.795 5332161.962, 443999.845 5...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_443_5332_2_bw.zip,LoD2_32_443_5332_2_bw/LoD2_32_443_5332_1_BW.gml,LoD2_32_443_5332_2_bw.zip::LoD2_32_443_5332_2_...
931559,DEBW_B010000BPNh,2025-12-25,53001_1800,1000,1.000,1.000,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((444012.396 5332164.631, 444013.924 5...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_443_5332_2_bw.zip,LoD2_32_443_5332_2_bw/LoD2_32_443_5332_1_BW.gml,LoD2_32_443_5332_2_bw.zip::LoD2_32_443_5332_2_...
931720,DEBW_B010000BPNh,2024-01-01,53001_1800,1000,1.000,1.000,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((444003.795 5332161.962, 443999.845 5...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_443_5332_2_bw.zip,LoD2_32_443_5332_2_bw/LoD2_32_444_5332_1_BW.gml,LoD2_32_443_5332_2_bw.zip::LoD2_32_443_5332_2_...
931721,DEBW_B010000BPNh,2025-12-25,53001_1800,1000,1.000,1.000,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((444012.396 5332164.631, 444013.924 5...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_443_5332_2_bw.zip,LoD2_32_443_5332_2_bw/LoD2_32_444_5332_1_BW.gml,LoD2_32_443_5332_2_bw.zip::LoD2_32_443_5332_2_...
3828675,DEBW_B010000BSXH,2026-02-08,53001_1800,1000,1.000,1.000,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((513999.941 5391429.764, 514000.211 5...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_513_5390_2_bw.zip,LoD2_32_513_5390_2_bw/LoD2_32_513_5391_1_BW.gml,LoD2_32_513_5390_2_bw.zip::LoD2_32_513_5390_2_...
3828676,DEBW_B010000BSXH,2021-01-13,53001_1800,1000,1.000,1.000,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((514003.94 5391429.681, 514000.941 53...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_513_5390_2_bw.zip,LoD2_32_513_5390_2_bw/LoD2_32_513_5391_1_BW.gml,LoD2_32_513_5390_2_bw.zip::LoD2_32_513_5390_2_...
3828815,DEBW_B010000BSXH,2026-02-08,53001_1800,1000,1.000,1.000,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((513999.941 5391429.764, 514000.211 5...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_513_5390_2_bw.zip,LoD2_32_513_5390_2_bw/LoD2_32_514_5391_1_BW.gml,LoD2_32_513_5390_2_bw.zip::LoD2_32_513_5390_2_...
3828817,DEBW_B010000BSXH,2021-01-13,53001_1800,1000,1.000,1.000,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((514003.94 5391429.681, 514000.941 53...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_513_5390_2_bw.zip,LoD2_32_513_5390_2_bw/LoD2_32_514_5391_1_BW.gml,LoD2_32_513_5390_2_bw.zip::LoD2_32_513_5390_2_...
3896867,DEBW_51000004HUv,2026-02-10,31001_3021,9999,9.555,9.555,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((513997.66 5443404.01, 514020.45 5443...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_513_5442_2_bw.zip,LoD2_32_513_5442_2_bw/LoD2_32_513_5443_1_BW.gml,LoD2_32_513_5442_2_bw.zip::LoD2_32_513_5442_2_...
3896868,DEBW_51000004HUv,2019-01-21,31001_3021,5000,22.537,22.537,NaN,0,building_measured_height,NaN,...,ground_surface,"POLYGON ((513996.52 5443351.83, 513989.89 5443...",BW,Baden-Württemberg,EPSG:25832,1.0,building_measured_height,LoD2_32_513_5442_2_bw.zip,LoD2_32_513_5442_2_bw/LoD2_32_513_5443_1_BW.gml,LoD2_32_513_5442_2_bw.zip::LoD2_32_513_5442_2_...


ValueError: 4 BW duplicate-ID groups have conflicting heights or geometries. No records were removed.

In [59]:
# ============================================================
# 101B — LOAD LoD2 SAFELY WITH NON-UNIQUE SOURCE IDs
# ============================================================

def remove_exact_lod2_duplicates(gdf):
    repeated=gdf["lod2_id"].duplicated(False)
    drop_mask=pd.Series(False,index=gdf.index)

    if repeated.any():
        check=gdf.loc[repeated,["lod2_id","measured_height_m","geometry"]].copy()
        check["_geometry_key"]=shapely.to_wkb(shapely.normalize(check.geometry.array),hex=True)
        drop_mask.loc[check.index]=check.duplicated(["lod2_id","measured_height_m","_geometry_key"],keep="first")

    return gdf.loc[~drop_mask].reset_index(drop=True),int(drop_mask.sum())

def load_state_lod2(state_code,columns=None,validate=True):
    paths=find_state_lod2_paths(state_code)
    frames=[gpd.read_parquet(path,columns=columns) for path in tqdm(paths,desc=f"Loading {state_code} LoD2")]

    crs_values={str(frame.crs) for frame in frames}
    if len(crs_values)!=1: raise ValueError(f"{state_code} has inconsistent CRS values: {crs_values}")

    gdf=gpd.GeoDataFrame(pd.concat(frames,ignore_index=True,copy=False),geometry="geometry",crs=frames[0].crs)
    gdf=standardize_lod2_schema(gdf)

    expected=state_expected_buildings(state_code)
    source_records=len(gdf)
    repeated_id_rows=int(gdf["lod2_id"].duplicated().sum())

    if validate and expected is not None and source_records!=expected:
        raise ValueError(f"{state_code}: loaded {source_records:,}, expected {expected:,}")

    gdf,exact_duplicates_removed=remove_exact_lod2_duplicates(gdf)
    remaining_id_collisions=int(gdf["lod2_id"].duplicated().sum())

    print("="*72)
    print(f"{state_code} LoD2 MATCHING DATASET")
    print("="*72)
    print(f"\nFiles loaded             : {len(paths):,}")
    print(f"Source records           : {source_records:,}")
    print(f"Expected source records  : {expected:,}" if expected is not None else "Expected source records  : unavailable")
    print(f"Repeated-ID extra rows   : {repeated_id_rows:,}")
    print(f"Exact duplicates removed : {exact_duplicates_removed:,}")
    print(f"Distinct ID collisions   : {remaining_id_collisions:,}")
    print(f"Matching records         : {len(gdf):,}")
    print(f"Missing heights          : {gdf['measured_height_m'].isna().sum():,}")
    print(f"Missing geometries       : {(gdf.geometry.isna()|gdf.geometry.is_empty).sum():,}")
    print(f"CRS                      : {gdf.crs.to_epsg() or gdf.crs.name}")

    del frames
    gc.collect()
    return gdf

# Confirm the correct BW interpretation
bw_test=load_state_lod2("BW")

print("\nExpected BW result:")
print("  Source records           : 6,465,296")
print("  Exact duplicates removed : 6")
print("  Distinct ID collisions   : 4")
print("  Matching records         : 6,465,290")

del bw_test
gc.collect()

Loading BW LoD2: 100%|█████████████████████████████████████| 122/122 [00:22<00:00,  5.33it/s]


BW LoD2 MATCHING DATASET

Files loaded             : 122
Source records           : 6,465,296
Expected source records  : 6,465,296
Repeated-ID extra rows   : 10
Exact duplicates removed : 6
Distinct ID collisions   : 4
Matching records         : 6,465,290
Missing heights          : 0
Missing geometries       : 4
CRS                      : 25832

Expected BW result:
  Source records           : 6,465,296
  Exact duplicates removed : 6
  Distinct ID collisions   : 4
  Matching records         : 6,465,290


0

In [60]:
# ============================================================
# 101 — NATIONAL LoD2–OSM BEST-OVERLAP MATCHING
# ============================================================

import gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from tqdm.auto import tqdm

COMMON_CRS="EPSG:3035"
STATES=["BB","BE","BW","BY","HB","HE","HH","MV","NI","NW","RP","SH","SL","SN","ST","TH"]
OSM_BUILDING_FILE=Path("/fast/home/o-olajuyigbe/osm_project/data/processed/germany_buildings_classified_stage2.parquet")

FINAL_DIR=BASE_DIR/"final"
FINAL_DIR.mkdir(parents=True,exist_ok=True)

NATIONAL_LOD2_PATH=FINAL_DIR/"germany_lod2_buildings.parquet"
NATIONAL_MATCH_PATH=FINAL_DIR/"germany_osm_lod2_matches.parquet"
NATIONAL_MATCH_SUMMARY=REPORT_DIR/"national_osm_lod2_match_summary.csv"
STATE_LOAD_SUMMARY=REPORT_DIR/"national_lod2_loading_summary.csv"

for name in ["lod2_frames","lod2","osm","pairs","accepted","best","match_table"]:
    if name in globals(): del globals()[name]
gc.collect()

def remove_exact_lod2_duplicates(gdf):
    repeated=gdf["lod2_id"].duplicated(False)
    if not repeated.any(): return gdf.reset_index(drop=True),0

    check=gdf.loc[repeated,["lod2_id","measured_height_m","geometry"]].copy()
    check["_geometry_key"]=None
    usable=check.geometry.notna()&~check.geometry.is_empty

    check.loc[usable,"_geometry_key"]=shapely.to_wkb(
        shapely.normalize(check.loc[usable].geometry.array),
        hex=True
    )

    duplicate_indices=check.index[
        check.duplicated(["lod2_id","measured_height_m","_geometry_key"],keep="first")
    ]

    return gdf.drop(index=duplicate_indices).reset_index(drop=True),len(duplicate_indices)

# ── 1. Load all LoD2 state datasets ──────────────────────────────────────────
lod2_frames=[]
state_stats=[]
lod2_columns=["lod2_id","measured_height_m","height_method","footprint_method","geometry"]

for state in tqdm(STATES,desc="Loading national LoD2"):
    paths=find_state_lod2_paths(state)
    frames=[gpd.read_parquet(path) for path in tqdm(paths,desc=f"Loading {state}",leave=False)]

    crs_values={str(frame.crs) for frame in frames}
    if len(crs_values)!=1: raise ValueError(f"{state}: inconsistent CRS values: {crs_values}")

    gdf=gpd.GeoDataFrame(pd.concat(frames,ignore_index=True,copy=False),geometry="geometry",crs=frames[0].crs)
    gdf=standardize_lod2_schema(gdf)

    expected=state_expected_buildings(state)
    source_records=len(gdf)

    if source_records!=expected:
        raise ValueError(f"{state}: loaded {source_records:,}, expected {expected:,}")

    missing=[column for column in lod2_columns if column not in gdf.columns]
    if missing: raise KeyError(f"{state} missing columns: {missing}")

    gdf,exact_duplicates_removed=remove_exact_lod2_duplicates(gdf)

    usable=gdf.geometry.notna()&~gdf.geometry.is_empty
    invalid=usable&~np.asarray(shapely.is_valid(gdf.geometry.array))

    if invalid.any():
        gdf.loc[invalid,"geometry"]=list(shapely.make_valid(gdf.loc[invalid].geometry.array))

    usable=gdf.geometry.notna()&~gdf.geometry.is_empty
    gdf=gdf.loc[usable,lod2_columns].copy()
    gdf["lod2_state_code"]=state
    gdf=gdf.to_crs(COMMON_CRS)

    state_stats.append({
        "state_code":state,
        "source_records":source_records,
        "expected_records":expected,
        "exact_duplicates_removed":exact_duplicates_removed,
        "distinct_id_collisions":int(gdf["lod2_id"].duplicated().sum()),
        "missing_geometry":source_records-exact_duplicates_removed-len(gdf),
        "invalid_geometry_repaired":int(invalid.sum()),
        "matching_records":len(gdf)
    })

    lod2_frames.append(gdf)
    del gdf,frames
    gc.collect()

lod2=gpd.GeoDataFrame(
    pd.concat(lod2_frames,ignore_index=True,copy=False),
    geometry="geometry",
    crs=COMMON_CRS
)

del lod2_frames
gc.collect()

lod2["lod2_area_m2"]=shapely.area(lod2.geometry.array)
valid_area=np.isfinite(lod2["lod2_area_m2"])&lod2["lod2_area_m2"].gt(0)
zero_area_count=int((~valid_area).sum())

lod2=lod2.loc[valid_area].reset_index(drop=True)
lod2["_lod2_row"]=np.arange(len(lod2),dtype=np.int64)

state_load_summary=pd.DataFrame(state_stats)
state_load_summary.to_csv(STATE_LOAD_SUMMARY,index=False)

print("="*80)
print("NATIONAL LoD2 DATASET")
print("="*80)
print(f"\nValidated source records : {state_load_summary['source_records'].sum():,}")
print(f"Exact duplicates removed : {state_load_summary['exact_duplicates_removed'].sum():,}")
print(f"Matchable footprints     : {len(lod2):,}")
print(f"Missing geometries       : {state_load_summary['missing_geometry'].sum():,}")
print(f"Invalid geometries fixed : {state_load_summary['invalid_geometry_repaired'].sum():,}")
print(f"Zero-area geometries     : {zero_area_count:,}")
print(f"Distinct ID collisions   : {lod2.duplicated(['lod2_state_code','lod2_id']).sum():,}")
print(f"CRS                      : {lod2.crs.to_epsg()}")

lod2.to_parquet(NATIONAL_LOD2_PATH,index=False)
print(f"Saved national LoD2      : {NATIONAL_LOD2_PATH}")

# ── 2. Load all OSM buildings ────────────────────────────────────────────────
osm=gpd.read_parquet(OSM_BUILDING_FILE,columns=["id","geometry"])
osm=osm[osm.geometry.notna()&~osm.geometry.is_empty].copy()

invalid_osm=~np.asarray(shapely.is_valid(osm.geometry.array))

if invalid_osm.any():
    osm.loc[invalid_osm,"geometry"]=list(shapely.make_valid(osm.loc[invalid_osm].geometry.array))

osm=osm[osm.geometry.notna()&~osm.geometry.is_empty].to_crs(COMMON_CRS).reset_index(drop=True)
osm["osm_area_m2"]=shapely.area(osm.geometry.array)

valid_osm_area=np.isfinite(osm["osm_area_m2"])&osm["osm_area_m2"].gt(0)
osm=osm.loc[valid_osm_area].reset_index(drop=True)
osm["_osm_row"]=np.arange(len(osm),dtype=np.int64)

if osm["id"].duplicated().any():
    raise ValueError(f"OSM contains {osm['id'].duplicated().sum():,} duplicate IDs.")

print("\n"+"="*80)
print("NATIONAL OSM DATASET")
print("="*80)
print(f"\nMatchable buildings     : {len(osm):,}")
print(f"Invalid geometries fixed: {invalid_osm.sum():,}")
print(f"CRS                     : {osm.crs.to_epsg()}")

# ── 3. Find all intersecting footprint pairs ─────────────────────────────────
start=time.time()

pairs=gpd.sjoin(
    osm[["_osm_row","geometry"]],
    lod2[["_lod2_row","geometry"]],
    how="inner",
    predicate="intersects"
)

pairs=pd.DataFrame(pairs[["_osm_row","_lod2_row"]]).reset_index(drop=True)
candidate_pairs=len(pairs)

print("\n"+"="*80)
print("NATIONAL SPATIAL JOIN")
print("="*80)
print(f"\nCandidate pairs: {candidate_pairs:,}")

# ── 4. Calculate exact overlap metrics ───────────────────────────────────────
osm_idx=pairs["_osm_row"].to_numpy(dtype=np.int64)
lod2_idx=pairs["_lod2_row"].to_numpy(dtype=np.int64)

pairs["intersection_area_m2"]=shapely.area(
    shapely.intersection(
        osm.geometry.array.take(osm_idx),
        lod2.geometry.array.take(lod2_idx)
    )
)

pairs=pairs[
    np.isfinite(pairs["intersection_area_m2"])
    &pairs["intersection_area_m2"].ge(1.0)
].copy()

osm_idx=pairs["_osm_row"].to_numpy(dtype=np.int64)
lod2_idx=pairs["_lod2_row"].to_numpy(dtype=np.int64)

pairs["osm_area_m2"]=osm["osm_area_m2"].to_numpy()[osm_idx]
pairs["lod2_area_m2"]=lod2["lod2_area_m2"].to_numpy()[lod2_idx]
pairs["osm_coverage"]=(pairs["intersection_area_m2"]/pairs["osm_area_m2"]).clip(0,1)
pairs["lod2_coverage"]=(pairs["intersection_area_m2"]/pairs["lod2_area_m2"]).clip(0,1)

union_area=pairs["osm_area_m2"]+pairs["lod2_area_m2"]-pairs["intersection_area_m2"]
pairs["iou"]=(pairs["intersection_area_m2"]/union_area).clip(0,1)
pairs["score"]=0.50*pairs["iou"]+0.25*pairs["osm_coverage"]+0.25*pairs["lod2_coverage"]

# Balanced overlap rule calibrated using Bremen
accepted=pairs[
    pairs["iou"].ge(0.25)
    |(pairs["osm_coverage"].ge(0.70)&pairs["lod2_coverage"].ge(0.20))
    |(pairs["lod2_coverage"].ge(0.70)&pairs["osm_coverage"].ge(0.20))
].copy()

if accepted.empty:
    raise RuntimeError("No acceptable LoD2–OSM matches were found.")

# ── 5. Retain the best LoD2 footprint per OSM building ──────────────────────
accepted=accepted.sort_values(
    ["_osm_row","score","intersection_area_m2","iou","_lod2_row"],
    ascending=[True,False,False,False,True],
    kind="mergesort"
)

accepted["candidate_rank"]=accepted.groupby("_osm_row",sort=False).cumcount()+1
best=accepted[accepted["candidate_rank"].eq(1)].copy()

second_scores=accepted[accepted["candidate_rank"].eq(2)].set_index("_osm_row")["score"]
best["score_margin"]=best["score"]-best["_osm_row"].map(second_scores).fillna(0)

reuse_counts=best["_lod2_row"].value_counts()
best["lod2_reuse_count"]=best["_lod2_row"].map(reuse_counts).astype(np.int32)
best["lod2_reused"]=best["lod2_reuse_count"].gt(1)

best["match_pattern"]=np.select(
    [
        best["iou"].ge(0.70)&best["osm_coverage"].ge(0.80)&best["lod2_coverage"].ge(0.80),
        best["iou"].ge(0.25),
        best["osm_coverage"].ge(0.70)&best["lod2_coverage"].ge(0.20),
        best["lod2_coverage"].ge(0.70)&best["osm_coverage"].ge(0.20)
    ],
    ["strong_balanced","moderate_balanced","osm_inside_lod2","lod2_inside_osm"],
    default="borderline"
)

best["match_quality"]=np.select(
    [
        best["match_pattern"].eq("strong_balanced"),
        best["match_pattern"].eq("moderate_balanced")&best["score"].ge(0.50),
        best["match_pattern"].isin(["osm_inside_lod2","lod2_inside_osm"])&best["score"].ge(0.35)
    ],
    ["high","medium","medium"],
    default="low"
)

# ── 6. Create final OSM-to-LoD2 height table ─────────────────────────────────
osm_idx=best["_osm_row"].to_numpy(dtype=np.int64)
lod2_idx=best["_lod2_row"].to_numpy(dtype=np.int64)

match_table=pd.DataFrame({
    "_osm_row":osm_idx,
    "osm_id":osm["id"].to_numpy()[osm_idx],
    "_lod2_row":lod2_idx,
    "lod2_id":lod2["lod2_id"].to_numpy()[lod2_idx],
    "lod2_state_code":lod2["lod2_state_code"].to_numpy()[lod2_idx],
    "lod2_height_m":lod2["measured_height_m"].to_numpy()[lod2_idx],
    "lod2_height_method":lod2["height_method"].to_numpy()[lod2_idx],
    "lod2_footprint_method":lod2["footprint_method"].to_numpy()[lod2_idx],
    "lod2_match_quality":best["match_quality"].to_numpy(),
    "lod2_match_pattern":best["match_pattern"].to_numpy(),
    "lod2_iou":best["iou"].to_numpy(),
    "lod2_osm_coverage":best["osm_coverage"].to_numpy(),
    "lod2_footprint_coverage":best["lod2_coverage"].to_numpy(),
    "lod2_intersection_m2":best["intersection_area_m2"].to_numpy(),
    "lod2_score":best["score"].to_numpy(),
    "lod2_score_margin":best["score_margin"].to_numpy(),
    "lod2_reuse_count":best["lod2_reuse_count"].to_numpy(),
    "lod2_reused":best["lod2_reused"].to_numpy()
})

match_table["lod2_uid"]=(
    match_table["lod2_state_code"].astype("string")+":"+
    match_table["lod2_id"].astype("string")+":"+
    match_table["_lod2_row"].astype("string")
)

match_table["lod2_match_method"]="balanced_footprint_overlap"
match_table=match_table.sort_values("_osm_row").reset_index(drop=True)

if match_table["_osm_row"].duplicated().any():
    raise ValueError("More than one final match remains for an OSM row.")
if match_table["osm_id"].duplicated().any():
    raise ValueError("More than one final match remains for an OSM ID.")
if match_table["lod2_height_m"].isna().any():
    raise ValueError("Matched rows contain missing LoD2 heights.")

match_table.to_parquet(NATIONAL_MATCH_PATH,index=False)

# ── 7. National validation summary ───────────────────────────────────────────
summary=pd.DataFrame([{
    "osm_buildings":len(osm),
    "lod2_source_records":int(state_load_summary["source_records"].sum()),
    "lod2_exact_duplicates_removed":int(state_load_summary["exact_duplicates_removed"].sum()),
    "lod2_matchable_footprints":len(lod2),
    "candidate_pairs":candidate_pairs,
    "pairs_with_intersection_ge_1m2":len(pairs),
    "accepted_pairs":len(accepted),
    "matched_osm_buildings":len(match_table),
    "unmatched_osm_buildings":len(osm)-len(match_table),
    "osm_match_rate_pct":100*len(match_table)/len(osm),
    "unique_lod2_footprints_used":match_table["_lod2_row"].nunique(),
    "high_quality_matches":int(match_table["lod2_match_quality"].eq("high").sum()),
    "medium_quality_matches":int(match_table["lod2_match_quality"].eq("medium").sum()),
    "low_quality_matches":int(match_table["lod2_match_quality"].eq("low").sum()),
    "reused_lod2_footprints":int(reuse_counts.gt(1).sum()),
    "maximum_lod2_reuse":int(reuse_counts.max()),
    "minimum_iou":match_table["lod2_iou"].min(),
    "median_iou":match_table["lod2_iou"].median(),
    "median_osm_coverage":match_table["lod2_osm_coverage"].median(),
    "median_lod2_coverage":match_table["lod2_footprint_coverage"].median(),
    "runtime_minutes":(time.time()-start)/60,
    "match_output":str(NATIONAL_MATCH_PATH)
}])

summary.to_csv(NATIONAL_MATCH_SUMMARY,index=False)

print("\n"+"="*80)
print("NATIONAL LoD2–OSM MATCHING RESULT")
print("="*80)
display(summary.T)

print("\nMatch quality:")
display(match_table["lod2_match_quality"].value_counts().rename_axis("quality").reset_index(name="buildings"))

print("\nMatch pattern:")
display(match_table["lod2_match_pattern"].value_counts().rename_axis("pattern").reset_index(name="buildings"))

print(f"\nNational LoD2 file : {NATIONAL_LOD2_PATH}")
print(f"National matches   : {NATIONAL_MATCH_PATH}")
print(f"Summary            : {NATIONAL_MATCH_SUMMARY}")

Loading national LoD2: 100%|█████████████████████████████████| 16/16 [10:18<00:00, 38.68s/it]


NATIONAL LoD2 DATASET

Validated source records : 58,049,619
Exact duplicates removed : 2,418
Matchable footprints     : 58,001,546
Missing geometries       : 45,655
Invalid geometries fixed : 914
Zero-area geometries     : 0
Distinct ID collisions   : 406
CRS                      : 3035
Saved national LoD2      : /fast/home/o-olajuyigbe/data/germany_lod2/final/germany_lod2_buildings.parquet

NATIONAL OSM DATASET

Matchable buildings     : 38,802,372
Invalid geometries fixed: 0
CRS                     : 3035

NATIONAL SPATIAL JOIN

Candidate pairs: 66,137,713

NATIONAL LoD2–OSM MATCHING RESULT


,0
osm_buildings,38802372
lod2_source_records,58049619
lod2_exact_duplicates_removed,2418
lod2_matchable_footprints,58001546
candidate_pairs,66137713
pairs_with_intersection_ge_1m2,53295472
accepted_pairs,37514730
matched_osm_buildings,34882239
unmatched_osm_buildings,3920133
osm_match_rate_pct,89.897182



Match quality:


,quality,buildings
0,high,18753509
1,medium,14350237
2,low,1778493



Match pattern:


,pattern,buildings
0,strong_balanced,18753509
1,moderate_balanced,15788623
2,osm_inside_lod2,216595
3,lod2_inside_osm,123512



National LoD2 file : /fast/home/o-olajuyigbe/data/germany_lod2/final/germany_lod2_buildings.parquet
National matches   : /fast/home/o-olajuyigbe/data/germany_lod2/final/germany_osm_lod2_matches.parquet
Summary            : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/national_osm_lod2_match_summary.csv


In [61]:
# ============================================================
# 102 — NATIONAL MATCH-QUALITY AND HEIGHT AUDIT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

if "match_table" not in globals():
    match_table=pd.read_parquet(NATIONAL_MATCH_PATH)

if "state_load_summary" not in globals():
    state_load_summary=pd.read_csv(STATE_LOAD_SUMMARY)

AUDIT_DIR=FINAL_DIR/"audit"
AUDIT_DIR.mkdir(parents=True,exist_ok=True)
MATCH_AUDIT_PATH=AUDIT_DIR/"germany_osm_lod2_questionable_matches.parquet"
MATCH_STATE_SUMMARY=REPORT_DIR/"national_osm_lod2_match_summary_by_state.csv"
HEIGHT_SUMMARY_PATH=REPORT_DIR/"national_osm_lod2_height_summary.csv"

if "accepted" in globals():
    candidate_counts=accepted.groupby("_osm_row",sort=False).size()
    match_table["accepted_candidate_count"]=match_table["_osm_row"].map(candidate_counts).fillna(1).astype(np.int16)
else:
    match_table["accepted_candidate_count"]=np.nan

match_table["height_nonpositive"]=match_table["lod2_height_m"].le(0)
match_table["height_below_1m"]=match_table["lod2_height_m"].lt(1)
match_table["height_above_100m"]=match_table["lod2_height_m"].gt(100)
match_table["height_above_200m"]=match_table["lod2_height_m"].gt(200)
match_table["ambiguous_match"]=match_table["accepted_candidate_count"].gt(1)&match_table["lod2_score_margin"].lt(0.05)

total=len(match_table)
quality_summary=(
    match_table["lod2_match_quality"].value_counts()
    .reindex(["high","medium","low"],fill_value=0)
    .rename_axis("quality").reset_index(name="buildings")
)
quality_summary["percentage"]=100*quality_summary["buildings"]/total

state_summary=(
    match_table.groupby("lod2_state_code",dropna=False)
    .agg(
        matched_osm=("osm_id","size"),
        unique_lod2_used=("_lod2_row","nunique"),
        median_height_m=("lod2_height_m","median"),
        median_iou=("lod2_iou","median"),
        median_osm_coverage=("lod2_osm_coverage","median"),
        median_lod2_coverage=("lod2_footprint_coverage","median"),
        high_quality=("lod2_match_quality",lambda x:x.eq("high").sum()),
        medium_quality=("lod2_match_quality",lambda x:x.eq("medium").sum()),
        low_quality=("lod2_match_quality",lambda x:x.eq("low").sum()),
        nonpositive_heights=("height_nonpositive","sum"),
        reused_matches=("lod2_reused","sum"),
        ambiguous_matches=("ambiguous_match","sum")
    )
    .reset_index()
)

state_summary["low_quality_pct"]=100*state_summary["low_quality"]/state_summary["matched_osm"]
state_summary["nonpositive_height_pct"]=100*state_summary["nonpositive_heights"]/state_summary["matched_osm"]
state_summary["ambiguous_match_pct"]=100*state_summary["ambiguous_matches"]/state_summary["matched_osm"]
state_summary.to_csv(MATCH_STATE_SUMMARY,index=False)

height_quantiles=match_table["lod2_height_m"].quantile([0,0.001,0.01,0.05,0.25,0.5,0.75,0.95,0.99,0.999,1]).rename_axis("quantile").reset_index(name="height_m")
height_quantiles.to_csv(HEIGHT_SUMMARY_PATH,index=False)

questionable=match_table[
    match_table["lod2_match_quality"].eq("low")
    |match_table["ambiguous_match"]
    |match_table["height_nonpositive"]
    |match_table["height_above_100m"]
    |match_table["lod2_reused"]
].copy()

questionable.to_parquet(MATCH_AUDIT_PATH,index=False)

print("="*80)
print("NATIONAL LoD2–OSM MATCH AUDIT")
print("="*80)
print(f"\nMatched OSM buildings       : {total:,}")
print(f"Match coverage              : {100*total/38_802_372:.2f}%")
print(f"High + medium quality       : {match_table['lod2_match_quality'].isin(['high','medium']).sum():,} ({100*match_table['lod2_match_quality'].isin(['high','medium']).mean():.2f}%)")
print(f"Low quality                 : {match_table['lod2_match_quality'].eq('low').sum():,} ({100*match_table['lod2_match_quality'].eq('low').mean():.2f}%)")
print(f"Multiple accepted candidates: {match_table['accepted_candidate_count'].gt(1).sum():,}")
print(f"Ambiguous margin < 0.05     : {match_table['ambiguous_match'].sum():,}")
print(f"Reused LoD2 footprint       : {match_table['lod2_reused'].sum():,}")
print(f"Nonpositive heights         : {match_table['height_nonpositive'].sum():,}")
print(f"Heights below 1 m           : {match_table['height_below_1m'].sum():,}")
print(f"Heights above 100 m         : {match_table['height_above_100m'].sum():,}")
print(f"Heights above 200 m         : {match_table['height_above_200m'].sum():,}")

print("\nMatch quality:")
display(quality_summary)

print("\nHeight distribution:")
display(height_quantiles)

print("\nState-level matching summary:")
display(state_summary.sort_values("matched_osm",ascending=False).round({
    "median_height_m":2,
    "median_iou":3,
    "median_osm_coverage":3,
    "median_lod2_coverage":3,
    "low_quality_pct":2,
    "nonpositive_height_pct":3,
    "ambiguous_match_pct":2
}))

print("\nExact duplicate removal by state:")
display(
    state_load_summary[
        ["state_code","source_records","exact_duplicates_removed","distinct_id_collisions","missing_geometry","matching_records"]
    ].sort_values("exact_duplicates_removed",ascending=False)
)

print(f"\nQuestionable-match audit: {MATCH_AUDIT_PATH}")
print(f"State summary            : {MATCH_STATE_SUMMARY}")
print(f"Height summary           : {HEIGHT_SUMMARY_PATH}")

NATIONAL LoD2–OSM MATCH AUDIT

Matched OSM buildings       : 34,882,239
Match coverage              : 89.90%
High + medium quality       : 33,103,746 (94.90%)
Low quality                 : 1,778,493 (5.10%)
Multiple accepted candidates: 2,465,186
Ambiguous margin < 0.05     : 701,350
Reused LoD2 footprint       : 1,508,457
Nonpositive heights         : 4
Heights below 1 m           : 18,407
Heights above 100 m         : 430
Heights above 200 m         : 68

Match quality:


,quality,buildings,percentage
0,high,18753509,53.762343
1,medium,14350237,41.139094
2,low,1778493,5.098563



Height distribution:


,quantile,height_m
0,0.000,-15.447
1,0.001,1.327
2,0.010,2.211
3,0.050,2.492
4,0.250,4.443
5,0.500,7.993
6,0.750,9.929
7,0.950,13.578
8,0.990,19.130
9,0.999,28.712



State-level matching summary:


,lod2_state_code,matched_osm,unique_lod2_used,median_height_m,median_iou,median_osm_coverage,median_lod2_coverage,high_quality,medium_quality,low_quality,nonpositive_heights,reused_matches,ambiguous_matches,low_quality_pct,nonpositive_height_pct,ambiguous_match_pct
9,NW,8405597,8242582,7.24,0.950,0.978,0.978,7187479,1082349,135769,0,304778,59844,1.62,0.0,0.71
3,BY,5716339,5595962,8.36,0.669,0.765,0.909,1792924,3503083,420332,0,233237,179174,7.35,0.0,3.13
2,BW,4453428,4396746,8.70,0.806,0.887,0.942,2708872,1582133,162423,0,108672,61308,3.65,0.0,1.38
8,NI,3465774,3379401,7.72,0.682,0.784,0.906,1209223,1995769,260782,0,166184,88174,7.52,0.0,2.54
5,HE,2337861,2300107,8.91,0.725,0.803,0.945,1028400,1173251,136210,0,71800,73005,5.83,0.0,3.12
10,RP,1881461,1836253,8.80,0.712,0.802,0.928,763274,1007280,110907,0,88037,42381,5.89,0.0,2.25
13,SN,1528112,1451549,8.24,0.726,0.870,0.880,707106,723620,97386,2,144623,24685,6.37,0.0,1.62
0,BB,1447526,1423945,6.32,0.771,0.869,0.942,772166,606772,68588,0,44828,29805,4.74,0.0,2.06
11,SH,1187852,1150999,7.43,0.655,0.773,0.878,354580,725683,107589,0,69218,33552,9.06,0.0,2.82
15,TH,1141495,1114190,7.91,0.690,0.788,0.920,419212,641408,80875,0,52457,41081,7.09,0.0,3.60



Exact duplicate removal by state:


,state_code,source_records,exact_duplicates_removed,distinct_id_collisions,missing_geometry,matching_records
5,HE,4981859,1402,0,72,4980385
15,TH,2333261,932,3,0,2332329
13,SN,2290890,63,377,0,2290827
8,NI,6828051,10,22,37,6828004
2,BW,6465296,6,4,4,6465286
3,BY,10111920,5,0,0,10111915
0,BB,2376794,0,0,45294,2331500
1,BE,639658,0,0,247,639411
7,MV,1332053,0,0,0,1332053
6,HH,388729,0,0,0,388729



Questionable-match audit: /fast/home/o-olajuyigbe/data/germany_lod2/final/audit/germany_osm_lod2_questionable_matches.parquet
State summary            : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/national_osm_lod2_match_summary_by_state.csv
Height summary           : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/national_osm_lod2_height_summary.csv


In [62]:
# ============================================================
# 103 — MERGE LoD2 HEIGHTS INTO NATIONAL OSM BUILDINGS
# ============================================================

import gc
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

OSM_INPUT=Path("/fast/home/o-olajuyigbe/osm_project/data/processed/germany_buildings_classified_stage2.parquet")
MATCH_INPUT=FINAL_DIR/"germany_osm_lod2_matches.parquet"
OSM_LOD2_OUTPUT=FINAL_DIR/"germany_buildings_classified_stage2_lod2.parquet"
MERGE_SUMMARY=REPORT_DIR/"national_osm_lod2_merge_summary.csv"

HEIGHT_MAX_M=400.0

matches=pd.read_parquet(MATCH_INPUT)

# Preserve the original source height
matches=matches.rename(columns={"lod2_height_m":"lod2_height_raw_m"})

matches["lod2_height_nonpositive"]=matches["lod2_height_raw_m"].le(0)
matches["lod2_height_below_1m"]=matches["lod2_height_raw_m"].between(0,1, inclusive="neither")
matches["lod2_height_extreme"]=matches["lod2_height_raw_m"].gt(HEIGHT_MAX_M)
matches["lod2_height_plausible"]=matches["lod2_height_raw_m"].gt(0)&matches["lod2_height_raw_m"].le(HEIGHT_MAX_M)
matches["lod2_match_trusted"]=matches["lod2_match_quality"].isin(["high","medium"])
matches["lod2_height_available"]=matches["lod2_height_plausible"]
matches["lod2_height_trusted"]=matches["lod2_height_plausible"]&matches["lod2_match_trusted"]

# Height for descriptive use: all plausible accepted matches
matches["lod2_height_m"]=matches["lod2_height_raw_m"].where(matches["lod2_height_plausible"])

# Conservative height for ML/modeling: plausible high- or medium-quality matches only
matches["lod2_height_model_m"]=matches["lod2_height_raw_m"].where(matches["lod2_height_trusted"])

matches["lod2_height_status"]=np.select(
    [
        matches["lod2_height_nonpositive"],
        matches["lod2_height_extreme"],
        matches["lod2_match_quality"].eq("low"),
        matches["lod2_height_below_1m"]
    ],
    [
        "invalid_nonpositive",
        "invalid_extreme",
        "plausible_low_match_quality",
        "plausible_below_1m"
    ],
    default="trusted"
)

merge_columns=[
    "osm_id","lod2_uid","lod2_id","lod2_state_code",
    "lod2_height_raw_m","lod2_height_m","lod2_height_model_m",
    "lod2_height_method","lod2_footprint_method",
    "lod2_height_status","lod2_height_available","lod2_height_trusted",
    "lod2_height_nonpositive","lod2_height_below_1m","lod2_height_extreme",
    "lod2_match_quality","lod2_match_pattern","lod2_match_method",
    "lod2_iou","lod2_osm_coverage","lod2_footprint_coverage",
    "lod2_intersection_m2","lod2_score","lod2_score_margin",
    "lod2_reuse_count","lod2_reused"
]

matches=matches[merge_columns].rename(columns={"osm_id":"id"})

if matches["id"].duplicated().any():
    raise ValueError(f"Match table contains {matches['id'].duplicated().sum():,} duplicate OSM IDs.")

osm=gpd.read_parquet(OSM_INPUT)

# Allow the cell to be rerun safely
existing_lod2_columns=[column for column in merge_columns if column!="osm_id" and column in osm.columns]
if existing_lod2_columns:
    osm=osm.drop(columns=existing_lod2_columns)

source_rows=len(osm)
source_ids=osm["id"].nunique()

osm_lod2=osm.merge(matches,on="id",how="left",validate="one_to_one")

if len(osm_lod2)!=source_rows:
    raise ValueError(f"Row count changed from {source_rows:,} to {len(osm_lod2):,}.")
if osm_lod2["id"].nunique()!=source_ids:
    raise ValueError("The number of unique OSM IDs changed during the merge.")
if osm_lod2["geometry"].isna().sum()!=osm["geometry"].isna().sum():
    raise ValueError("Geometry availability changed during the merge.")

osm_lod2=gpd.GeoDataFrame(osm_lod2,geometry="geometry",crs=osm.crs)
osm_lod2.to_parquet(OSM_LOD2_OUTPUT,index=False)

summary=pd.DataFrame([{
    "osm_buildings":len(osm_lod2),
    "matched_lod2_heights":int(osm_lod2["lod2_height_raw_m"].notna().sum()),
    "plausible_lod2_heights":int(osm_lod2["lod2_height_m"].notna().sum()),
    "trusted_model_heights":int(osm_lod2["lod2_height_model_m"].notna().sum()),
    "unmatched_buildings":int(osm_lod2["lod2_uid"].isna().sum()),
    "nonpositive_heights_removed":int(osm_lod2["lod2_height_nonpositive"].fillna(False).sum()),
    "extreme_heights_removed":int(osm_lod2["lod2_height_extreme"].fillna(False).sum()),
    "below_1m_retained":int(osm_lod2["lod2_height_below_1m"].fillna(False).sum()),
    "low_quality_heights_excluded_from_model":int(
        (osm_lod2["lod2_match_quality"].eq("low")&osm_lod2["lod2_height_m"].notna()).sum()
    ),
    "raw_height_coverage_pct":100*osm_lod2["lod2_height_raw_m"].notna().mean(),
    "plausible_height_coverage_pct":100*osm_lod2["lod2_height_m"].notna().mean(),
    "trusted_model_height_coverage_pct":100*osm_lod2["lod2_height_model_m"].notna().mean(),
    "output_file":str(OSM_LOD2_OUTPUT)
}])

summary.to_csv(MERGE_SUMMARY,index=False)

print("="*80)
print("NATIONAL OSM–LoD2 MERGE VALIDATION")
print("="*80)
display(summary.T)

print("\nHeight status:")
display(
    osm_lod2["lod2_height_status"]
    .fillna("unmatched")
    .value_counts()
    .rename_axis("status")
    .reset_index(name="buildings")
)

print("\nMatch quality:")
display(
    osm_lod2["lod2_match_quality"]
    .fillna("unmatched")
    .value_counts()
    .rename_axis("quality")
    .reset_index(name="buildings")
)

print(f"\nFinal national database: {OSM_LOD2_OUTPUT}")
print(f"Merge summary          : {MERGE_SUMMARY}")

del osm,matches
gc.collect()

/tmp/ipykernel_109491/4056828836.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "nonpositive_heights_removed":int(osm_lod2["lod2_height_nonpositive"].fillna(False).sum()),
/tmp/ipykernel_109491/4056828836.py:99: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "extreme_heights_removed":int(osm_lod2["lod2_height_extreme"].fillna(False).sum()),
/tmp/ipykernel_109491/4056828836.py:100: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead.

NATIONAL OSM–LoD2 MERGE VALIDATION


,0
osm_buildings,38802372
matched_lod2_heights,34882239
plausible_lod2_heights,34882188
trusted_model_heights,33103695
unmatched_buildings,3920133
nonpositive_heights_removed,4
extreme_heights_removed,47
below_1m_retained,18403
low_quality_heights_excluded_from_model,1778493
raw_height_coverage_pct,89.897182



Height status:


,status,buildings
0,trusted,33086479
1,unmatched,3920133
2,plausible_low_match_quality,1778493
3,plausible_below_1m,17216
4,invalid_extreme,47
5,invalid_nonpositive,4



Match quality:


,quality,buildings
0,high,18753509
1,medium,14350237
2,unmatched,3920133
3,low,1778493



Final national database: /fast/home/o-olajuyigbe/data/germany_lod2/final/germany_buildings_classified_stage2_lod2.parquet
Merge summary          : /fast/home/o-olajuyigbe/data/germany_lod2/quality_reports/national_osm_lod2_merge_summary.csv


0